# 04e Extraction, batch e: twelve included studies coded from their PDFs

Design log.
- Batch e of notebook 04 (plan v2 B4), coded under the frozen schema (schema/record_schema.yaml, v1) and the coding rules, as batches a to d were. The twelve reports: architecture-aware attacks on fine-tuning defences (2507.07417), AutoDojo (2606.15057), RETA (2606.15441), IHDec (2606.29960; the search record lists Findings of EMNLP 2026), COPA (2608.19982), SecOPD (2608.21500, accepted to EMNLP 2026 as printed on its first page), Semantic Overlays (2608.23873), CoER (2609.07529), Provenance-Aware Transformers (2609.21088), CoDeL (2609.34463), Pictionary (2609.36121) and Self-Reflection Fine-Tuning (2610.04269).
- 2507.07417 comes earlier in the list of included studies than batch c; it was passed over when batches b and c were formed and is coded here.
- Each record's extraction_scope names the tables and rows coded. As before, detectors, sanitisers, rewriters, tool filters and system-level designs (PromptArmor, PromptLocate, DataFilter, DataSentinel, MELON, PIGuard, TSGuard, PromptGuard, Known-Answer detection and others) are left out and the scope says so (amendment 4). Results whose attacker writes the user's own input (TensorTrust hijacking in Semantic Overlays; the system-against-user conflicts in IHDec) are left out too.
- Coding decisions worth knowing when reading these records (reading aids for notebook 05, not findings):
  - CoDeL and CoER define utility under attack as completing the user's task without executing the injection, so it is coded R3safe; RETA's and the others' utility under attack is coded R2attack.
  - RETA's defender is trained by reinforcement learning on attacks found beforehand, not co-evolved, so axis 2 is preference_optimisation (as for VerIH); CoER and CoDeL co-evolve the attacker and are coded adversarial_rl.
  - Several baselines were re-trained by the authors rather than taken as released (Meta-SecAlign in SecOPD and CoDeL, SecAlign in COPA, every model-level baseline in CoDeL). Where a paper does not say how a baseline was obtained for a backbone with no released checkpoint (MetaSecAlign on Qwen3-4B in RETA), the base condition is matched_unstated.
  - The Provenance-Aware Transformer's training data tags only the inserted injection as untrusted inside a trusted context, and the paper does not say how rings were assigned on its out-of-distribution test sets; a residual note records this. Its table and text also give different NeuExe numbers; the table is coded. Zeroing its origin attention bias at inference is coded as a component comparison that removes the signal.
  - In Semantic Overlays, SEP for the Llama overlay uses 1,000 items while the other Llama rows use 300, so those two comparisons grade B.
  - Pictionary renders untrusted text as an image (or audio) before an unchanged model; it is coded on axis 1 as an encoding and on axis 2 as a fixed inference rule.
  - The ASTRA paper reports defended models only; its comparisons vary the attack (ASTRA against GCG) or, for SecAlign++, the role holding the user's instruction.
  - COPA's chart labels are coded figure_only with the printed value kept, as in earlier batches, and checked by Stage 3.
- Full-text exclusions found while coding this batch: none.
- Outputs: reports/records/ (twelve files), reports/extraction_04e_check.csv, reports/extraction_04e_summary.csv, reports/extraction_batches.csv (rows for batch e added, earlier batches kept), reports/decision_04e.txt.
- CPU; no network calls beyond pip; model calls: 0.

## Stage 0. Session bootstrap

In [ ]:
import os, sys, subprocess, importlib
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/SourceMap_Research"
    REPO_DIR = PROJECT_ROOT + "/sourcemap-research"
except ImportError:
    PROJECT_ROOT = os.environ["SOURCEMAP_PROJECT_ROOT"]
    REPO_DIR = os.environ["SOURCEMAP_REPO_ROOT"]
os.environ["SOURCEMAP_PROJECT_ROOT"] = PROJECT_ROOT
os.environ["SOURCEMAP_REPO_ROOT"] = REPO_DIR
os.chdir(REPO_DIR)
if "src" not in sys.path:
    sys.path.insert(0, "src")
for module, package in (("yaml", "pyyaml"), ("pypdf", "pypdf"), ("pypdfium2", "pypdfium2")):
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=False)
import pypdf
if int(pypdf.__version__.split(".")[0]) < 4:                 # the column-preserving extraction needs pypdf 4 or later
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "pypdf>=4"], check=False)
    print("pypdf upgraded; restart the runtime if the next stage reports an old version")
print(subprocess.run(["git", "pull", "--ff-only"], capture_output=True, text=True).stdout.strip() or "pull: nothing to report")
print("cwd:", os.getcwd(), "| pypdf", pypdf.__version__)

## Stage 1. The 12 records

In [ ]:
Path("reports/records").mkdir(parents=True, exist_ok=True)       # %%writefile does not create folders

`astra_arxiv2025` (2507.07417)

In [ ]:
%%writefile reports/records/astra_arxiv2025.yaml
report:
  id: astra_arxiv2025
  slug: '2507.07417'
  title: May I have your Attention? Breaking Fine-Tuning based Prompt Injection Defenses using Architecture-Aware Attacks
  first_author: Nishit V. Pandya
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2507.07417'
  version_read: arXiv v2
  first_date: '2025-07-10'
  read: F
  role: re_measures
  code_release: released (linked in the abstract); not checked
  extraction_scope: 'Section 6: Table 1 (SecAlign, single-prompt attacks, all budgets), the StruQ counts stated in Section 6.1 and Table 2 (universal attacks on unseen prompts); the abstract claim (the weight ablation of Section
    6.3 not coded)'
defences:
- id: secalign_l3
  name: SecAlign (re-measured)
  origin: other_report
  base_checkpoint: Meta-Llama-3-8B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 11
  anchor: we choose the SecAlign defended models of Llama-3-8B-Instruct
- id: struq_l3
  name: StruQ (re-measured)
  origin: other_report
  base_checkpoint: Meta-Llama-3-8B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 11
  anchor: We choose the equivalent defended models for StruQ
- id: secalign_mi
  name: SecAlign (re-measured)
  origin: other_report
  base_checkpoint: Mistral-7B-Instruct-v0.1
  axis1:
  - delimiters_markers
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 11
  anchor: we choose the SecAlign defended models of Llama-3-8B-Instruct
- id: struq_mi
  name: StruQ (re-measured)
  origin: other_report
  base_checkpoint: Mistral-7B-Instruct-v0.1
  axis1:
  - delimiters_markers
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 11
  anchor: We choose the equivalent defended models for StruQ
- id: secalignpp
  name: SecAlign++ (re-measured; checkpoint not named)
  origin: other_report
  base_checkpoint: not stated
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 12
  anchor: This is a production-grade Instruction Hi
  layout_compat:
    user_turn: supported
    separate_role: supported
    system_context: supported
    agent_tool_result: untested
evaluations:
- id: ev_t1_l3_20_gcg
  defence_id: secalign_l3
  benchmark: 40 AlpacaFarm examples, 'Print exactly Hacked!' at the end of the data, per-prompt optimisation
  attacker_type: optimisation
  attacker_access: white_box
  judge: output starts with 'Hacked' or 'hacked'
  layout_same_as_defence: 'yes'
  measured_by: astra_arxiv2025
  page: 11
  attacker_budget: 20 tokens, prefix and suffix (0, 20), 500 iterations
  decoding: temperature 0
  sample_size: '40'
- id: ev_t1_l3_20_astra
  defence_id: secalign_l3
  benchmark: 40 AlpacaFarm examples, 'Print exactly Hacked!' at the end of the data, per-prompt optimisation
  attacker_type: optimisation
  attacker_access: white_box
  judge: output starts with 'Hacked' or 'hacked'
  layout_same_as_defence: 'yes'
  measured_by: astra_arxiv2025
  page: 11
  attacker_budget: 20 tokens, prefix and suffix (0, 20), 500 iterations
  decoding: temperature 0
  sample_size: '40'
- id: ev_t1_l3_25_gcg
  defence_id: secalign_l3
  benchmark: 40 AlpacaFarm examples, 'Print exactly Hacked!' at the end of the data, per-prompt optimisation
  attacker_type: optimisation
  attacker_access: white_box
  judge: output starts with 'Hacked' or 'hacked'
  layout_same_as_defence: 'yes'
  measured_by: astra_arxiv2025
  page: 11
  attacker_budget: 25 tokens, prefix and suffix (5, 20), 500 iterations
  decoding: temperature 0
  sample_size: '40'
- id: ev_t1_l3_25_astra
  defence_id: secalign_l3
  benchmark: 40 AlpacaFarm examples, 'Print exactly Hacked!' at the end of the data, per-prompt optimisation
  attacker_type: optimisation
  attacker_access: white_box
  judge: output starts with 'Hacked' or 'hacked'
  layout_same_as_defence: 'yes'
  measured_by: astra_arxiv2025
  page: 11
  attacker_budget: 25 tokens, prefix and suffix (5, 20), 500 iterations
  decoding: temperature 0
  sample_size: '40'
- id: ev_t1_l3_30_gcg
  defence_id: secalign_l3
  benchmark: 40 AlpacaFarm examples, 'Print exactly Hacked!' at the end of the data, per-prompt optimisation
  attacker_type: optimisation
  attacker_access: white_box
  judge: output starts with 'Hacked' or 'hacked'
  layout_same_as_defence: 'yes'
  measured_by: astra_arxiv2025
  page: 11
  attacker_budget: 30 tokens, prefix and suffix (10, 20), 500 iterations
  decoding: temperature 0
  sample_size: '40'
- id: ev_t1_l3_30_astra
  defence_id: secalign_l3
  benchmark: 40 AlpacaFarm examples, 'Print exactly Hacked!' at the end of the data, per-prompt optimisation
  attacker_type: optimisation
  attacker_access: white_box
  judge: output starts with 'Hacked' or 'hacked'
  layout_same_as_defence: 'yes'
  measured_by: astra_arxiv2025
  page: 11
  attacker_budget: 30 tokens, prefix and suffix (10, 20), 500 iterations
  decoding: temperature 0
  sample_size: '40'
- id: ev_t1_l3_35_gcg
  defence_id: secalign_l3
  benchmark: 40 AlpacaFarm examples, 'Print exactly Hacked!' at the end of the data, per-prompt optimisation
  attacker_type: optimisation
  attacker_access: white_box
  judge: output starts with 'Hacked' or 'hacked'
  layout_same_as_defence: 'yes'
  measured_by: astra_arxiv2025
  page: 11
  attacker_budget: 35 tokens, prefix and suffix (15, 20), 500 iterations
  decoding: temperature 0
  sample_size: '40'
- id: ev_t1_l3_35_astra
  defence_id: secalign_l3
  benchmark: 40 AlpacaFarm examples, 'Print exactly Hacked!' at the end of the data, per-prompt optimisation
  attacker_type: optimisation
  attacker_access: white_box
  judge: output starts with 'Hacked' or 'hacked'
  layout_same_as_defence: 'yes'
  measured_by: astra_arxiv2025
  page: 11
  attacker_budget: 35 tokens, prefix and suffix (15, 20), 500 iterations
  decoding: temperature 0
  sample_size: '40'
- id: ev_t1_mi_20_gcg
  defence_id: secalign_mi
  benchmark: 40 AlpacaFarm examples, 'Print exactly Hacked!' at the end of the data, per-prompt optimisation
  attacker_type: optimisation
  attacker_access: white_box
  judge: output starts with 'Hacked' or 'hacked'
  layout_same_as_defence: 'yes'
  measured_by: astra_arxiv2025
  page: 11
  attacker_budget: 20 tokens, prefix and suffix (0, 20), 500 iterations
  decoding: temperature 0
  sample_size: '40'
- id: ev_t1_mi_20_astra
  defence_id: secalign_mi
  benchmark: 40 AlpacaFarm examples, 'Print exactly Hacked!' at the end of the data, per-prompt optimisation
  attacker_type: optimisation
  attacker_access: white_box
  judge: output starts with 'Hacked' or 'hacked'
  layout_same_as_defence: 'yes'
  measured_by: astra_arxiv2025
  page: 11
  attacker_budget: 20 tokens, prefix and suffix (0, 20), 500 iterations
  decoding: temperature 0
  sample_size: '40'
- id: ev_t1_mi_25_gcg
  defence_id: secalign_mi
  benchmark: 40 AlpacaFarm examples, 'Print exactly Hacked!' at the end of the data, per-prompt optimisation
  attacker_type: optimisation
  attacker_access: white_box
  judge: output starts with 'Hacked' or 'hacked'
  layout_same_as_defence: 'yes'
  measured_by: astra_arxiv2025
  page: 11
  attacker_budget: 25 tokens, prefix and suffix (5, 20), 500 iterations
  decoding: temperature 0
  sample_size: '40'
- id: ev_t1_mi_25_astra
  defence_id: secalign_mi
  benchmark: 40 AlpacaFarm examples, 'Print exactly Hacked!' at the end of the data, per-prompt optimisation
  attacker_type: optimisation
  attacker_access: white_box
  judge: output starts with 'Hacked' or 'hacked'
  layout_same_as_defence: 'yes'
  measured_by: astra_arxiv2025
  page: 11
  attacker_budget: 25 tokens, prefix and suffix (5, 20), 500 iterations
  decoding: temperature 0
  sample_size: '40'
- id: ev_t1_mi_30_gcg
  defence_id: secalign_mi
  benchmark: 40 AlpacaFarm examples, 'Print exactly Hacked!' at the end of the data, per-prompt optimisation
  attacker_type: optimisation
  attacker_access: white_box
  judge: output starts with 'Hacked' or 'hacked'
  layout_same_as_defence: 'yes'
  measured_by: astra_arxiv2025
  page: 11
  attacker_budget: 30 tokens, prefix and suffix (10, 20), 500 iterations
  decoding: temperature 0
  sample_size: '40'
- id: ev_t1_mi_30_astra
  defence_id: secalign_mi
  benchmark: 40 AlpacaFarm examples, 'Print exactly Hacked!' at the end of the data, per-prompt optimisation
  attacker_type: optimisation
  attacker_access: white_box
  judge: output starts with 'Hacked' or 'hacked'
  layout_same_as_defence: 'yes'
  measured_by: astra_arxiv2025
  page: 11
  attacker_budget: 30 tokens, prefix and suffix (10, 20), 500 iterations
  decoding: temperature 0
  sample_size: '40'
- id: ev_t1_mi_35_gcg
  defence_id: secalign_mi
  benchmark: 40 AlpacaFarm examples, 'Print exactly Hacked!' at the end of the data, per-prompt optimisation
  attacker_type: optimisation
  attacker_access: white_box
  judge: output starts with 'Hacked' or 'hacked'
  layout_same_as_defence: 'yes'
  measured_by: astra_arxiv2025
  page: 11
  attacker_budget: 35 tokens, prefix and suffix (15, 20), 500 iterations
  decoding: temperature 0
  sample_size: '40'
- id: ev_t1_mi_35_astra
  defence_id: secalign_mi
  benchmark: 40 AlpacaFarm examples, 'Print exactly Hacked!' at the end of the data, per-prompt optimisation
  attacker_type: optimisation
  attacker_access: white_box
  judge: output starts with 'Hacked' or 'hacked'
  layout_same_as_defence: 'yes'
  measured_by: astra_arxiv2025
  page: 11
  attacker_budget: 35 tokens, prefix and suffix (15, 20), 500 iterations
  decoding: temperature 0
  sample_size: '40'
- id: ev_sq_mi_gcg
  defence_id: struq_mi
  benchmark: 40 AlpacaFarm examples, GCG suffix of 20 tokens
  attacker_type: optimisation
  attacker_access: white_box
  judge: output starts with 'Hacked' or 'hacked'
  layout_same_as_defence: 'yes'
  measured_by: astra_arxiv2025
  page: 11
  attacker_budget: 20-token suffix
  decoding: temperature 0
  sample_size: '40'
- id: ev_sq_mi_astra
  defence_id: struq_mi
  benchmark: the 12 examples GCG did not break, ASTRA with a 20-token suffix
  attacker_type: optimisation
  attacker_access: white_box
  judge: output starts with 'Hacked' or 'hacked'
  layout_same_as_defence: 'yes'
  measured_by: astra_arxiv2025
  page: 11
  attacker_budget: 20-token suffix
  decoding: temperature 0
  sample_size: '12'
  subset_of: 40 AlpacaFarm examples
- id: ev_sq_l3_gcg
  defence_id: struq_l3
  benchmark: 40 AlpacaFarm examples, GCG suffix of 20 tokens
  attacker_type: optimisation
  attacker_access: white_box
  judge: output starts with 'Hacked' or 'hacked'
  layout_same_as_defence: 'yes'
  measured_by: astra_arxiv2025
  page: 11
  attacker_budget: 20-token suffix
  decoding: temperature 0
  sample_size: '40'
- id: ev_sq_l3_astra
  defence_id: struq_l3
  benchmark: the 17 examples GCG did not break, ASTRA with a 20-token suffix
  attacker_type: optimisation
  attacker_access: white_box
  judge: output starts with 'Hacked' or 'hacked'
  layout_same_as_defence: 'yes'
  measured_by: astra_arxiv2025
  page: 11
  attacker_budget: 20-token suffix
  decoding: temperature 0
  sample_size: '17'
  subset_of: 40 AlpacaFarm examples
- id: ev_t2_sa_l3_30_ugcg
  defence_id: secalign_l3
  benchmark: 198 unseen AlpacaFarm examples, universal suffix trained on 10 others; SecAlign prompt format
  attacker_type: optimisation
  attacker_access: white_box
  judge: output starts with 'Hacked' or 'hacked'
  layout_same_as_defence: 'yes'
  measured_by: astra_arxiv2025
  page: 12
  attacker_budget: 30 tokens (15, 15), 1000 iterations, universal
  decoding: temperature 0
  sample_size: '198'
  pages:
    decoding: 11
- id: ev_t2_sa_l3_30_astrapp
  defence_id: secalign_l3
  benchmark: 198 unseen AlpacaFarm examples, universal suffix trained on 10 others; SecAlign prompt format
  attacker_type: optimisation
  attacker_access: white_box
  judge: output starts with 'Hacked' or 'hacked'
  layout_same_as_defence: 'yes'
  measured_by: astra_arxiv2025
  page: 12
  attacker_budget: 30 tokens (15, 15), 1000 iterations, universal
  decoding: temperature 0
  sample_size: '198'
  pages:
    decoding: 11
- id: ev_t2_sa_l3_35_ugcg
  defence_id: secalign_l3
  benchmark: 198 unseen AlpacaFarm examples, universal suffix trained on 10 others; SecAlign prompt format
  attacker_type: optimisation
  attacker_access: white_box
  judge: output starts with 'Hacked' or 'hacked'
  layout_same_as_defence: 'yes'
  measured_by: astra_arxiv2025
  page: 12
  attacker_budget: 35 tokens (17, 18), 1000 iterations, universal
  decoding: temperature 0
  sample_size: '198'
  pages:
    decoding: 11
- id: ev_t2_sa_l3_35_astrapp
  defence_id: secalign_l3
  benchmark: 198 unseen AlpacaFarm examples, universal suffix trained on 10 others; SecAlign prompt format
  attacker_type: optimisation
  attacker_access: white_box
  judge: output starts with 'Hacked' or 'hacked'
  layout_same_as_defence: 'yes'
  measured_by: astra_arxiv2025
  page: 12
  attacker_budget: 35 tokens (17, 18), 1000 iterations, universal
  decoding: temperature 0
  sample_size: '198'
  pages:
    decoding: 11
- id: ev_t2_sa_l3_40_ugcg
  defence_id: secalign_l3
  benchmark: 198 unseen AlpacaFarm examples, universal suffix trained on 10 others; SecAlign prompt format
  attacker_type: optimisation
  attacker_access: white_box
  judge: output starts with 'Hacked' or 'hacked'
  layout_same_as_defence: 'yes'
  measured_by: astra_arxiv2025
  page: 12
  attacker_budget: 40 tokens (20, 20), 1000 iterations, universal
  decoding: temperature 0
  sample_size: '198'
  pages:
    decoding: 11
- id: ev_t2_sa_l3_40_astrapp
  defence_id: secalign_l3
  benchmark: 198 unseen AlpacaFarm examples, universal suffix trained on 10 others; SecAlign prompt format
  attacker_type: optimisation
  attacker_access: white_box
  judge: output starts with 'Hacked' or 'hacked'
  layout_same_as_defence: 'yes'
  measured_by: astra_arxiv2025
  page: 12
  attacker_budget: 40 tokens (20, 20), 1000 iterations, universal
  decoding: temperature 0
  sample_size: '198'
  pages:
    decoding: 11
- id: ev_t2_pp_sys_30_ugcg
  defence_id: secalignpp
  benchmark: 198 unseen AlpacaFarm examples, universal suffix trained on 10 others; instruction as system, data as input
  attacker_type: optimisation
  attacker_access: white_box
  judge: output starts with 'Hacked' or 'hacked'
  layout_same_as_defence: 'yes'
  measured_by: astra_arxiv2025
  page: 12
  attacker_budget: 30 tokens (15, 15), 1000 iterations, universal
  decoding: temperature 0
  sample_size: '198'
  pages:
    decoding: 11
- id: ev_t2_pp_sys_30_astrapp
  defence_id: secalignpp
  benchmark: 198 unseen AlpacaFarm examples, universal suffix trained on 10 others; instruction as system, data as input
  attacker_type: optimisation
  attacker_access: white_box
  judge: output starts with 'Hacked' or 'hacked'
  layout_same_as_defence: 'yes'
  measured_by: astra_arxiv2025
  page: 12
  attacker_budget: 30 tokens (15, 15), 1000 iterations, universal
  decoding: temperature 0
  sample_size: '198'
  pages:
    decoding: 11
- id: ev_t2_pp_sys_35_ugcg
  defence_id: secalignpp
  benchmark: 198 unseen AlpacaFarm examples, universal suffix trained on 10 others; instruction as system, data as input
  attacker_type: optimisation
  attacker_access: white_box
  judge: output starts with 'Hacked' or 'hacked'
  layout_same_as_defence: 'yes'
  measured_by: astra_arxiv2025
  page: 12
  attacker_budget: 35 tokens (17, 18), 1000 iterations, universal
  decoding: temperature 0
  sample_size: '198'
  pages:
    decoding: 11
- id: ev_t2_pp_sys_35_astrapp
  defence_id: secalignpp
  benchmark: 198 unseen AlpacaFarm examples, universal suffix trained on 10 others; instruction as system, data as input
  attacker_type: optimisation
  attacker_access: white_box
  judge: output starts with 'Hacked' or 'hacked'
  layout_same_as_defence: 'yes'
  measured_by: astra_arxiv2025
  page: 12
  attacker_budget: 35 tokens (17, 18), 1000 iterations, universal
  decoding: temperature 0
  sample_size: '198'
  pages:
    decoding: 11
- id: ev_t2_pp_sys_40_ugcg
  defence_id: secalignpp
  benchmark: 198 unseen AlpacaFarm examples, universal suffix trained on 10 others; instruction as system, data as input
  attacker_type: optimisation
  attacker_access: white_box
  judge: output starts with 'Hacked' or 'hacked'
  layout_same_as_defence: 'yes'
  measured_by: astra_arxiv2025
  page: 12
  attacker_budget: 40 tokens (20, 20), 1000 iterations, universal
  decoding: temperature 0
  sample_size: '198'
  pages:
    decoding: 11
- id: ev_t2_pp_sys_40_astrapp
  defence_id: secalignpp
  benchmark: 198 unseen AlpacaFarm examples, universal suffix trained on 10 others; instruction as system, data as input
  attacker_type: optimisation
  attacker_access: white_box
  judge: output starts with 'Hacked' or 'hacked'
  layout_same_as_defence: 'yes'
  measured_by: astra_arxiv2025
  page: 12
  attacker_budget: 40 tokens (20, 20), 1000 iterations, universal
  decoding: temperature 0
  sample_size: '198'
  pages:
    decoding: 11
- id: ev_t2_pp_user_30_ugcg
  defence_id: secalignpp
  benchmark: 198 unseen AlpacaFarm examples, universal suffix trained on 10 others; instruction as user, data as input
  attacker_type: optimisation
  attacker_access: white_box
  judge: output starts with 'Hacked' or 'hacked'
  layout_same_as_defence: 'yes'
  measured_by: astra_arxiv2025
  page: 12
  attacker_budget: 30 tokens (15, 15), 1000 iterations, universal
  decoding: temperature 0
  sample_size: '198'
  pages:
    decoding: 11
- id: ev_t2_pp_user_30_astrapp
  defence_id: secalignpp
  benchmark: 198 unseen AlpacaFarm examples, universal suffix trained on 10 others; instruction as user, data as input
  attacker_type: optimisation
  attacker_access: white_box
  judge: output starts with 'Hacked' or 'hacked'
  layout_same_as_defence: 'yes'
  measured_by: astra_arxiv2025
  page: 12
  attacker_budget: 30 tokens (15, 15), 1000 iterations, universal
  decoding: temperature 0
  sample_size: '198'
  pages:
    decoding: 11
- id: ev_t2_pp_user_35_ugcg
  defence_id: secalignpp
  benchmark: 198 unseen AlpacaFarm examples, universal suffix trained on 10 others; instruction as user, data as input
  attacker_type: optimisation
  attacker_access: white_box
  judge: output starts with 'Hacked' or 'hacked'
  layout_same_as_defence: 'yes'
  measured_by: astra_arxiv2025
  page: 12
  attacker_budget: 35 tokens (17, 18), 1000 iterations, universal
  decoding: temperature 0
  sample_size: '198'
  pages:
    decoding: 11
- id: ev_t2_pp_user_35_astrapp
  defence_id: secalignpp
  benchmark: 198 unseen AlpacaFarm examples, universal suffix trained on 10 others; instruction as user, data as input
  attacker_type: optimisation
  attacker_access: white_box
  judge: output starts with 'Hacked' or 'hacked'
  layout_same_as_defence: 'yes'
  measured_by: astra_arxiv2025
  page: 12
  attacker_budget: 35 tokens (17, 18), 1000 iterations, universal
  decoding: temperature 0
  sample_size: '198'
  pages:
    decoding: 11
- id: ev_t2_pp_user_40_ugcg
  defence_id: secalignpp
  benchmark: 198 unseen AlpacaFarm examples, universal suffix trained on 10 others; instruction as user, data as input
  attacker_type: optimisation
  attacker_access: white_box
  judge: output starts with 'Hacked' or 'hacked'
  layout_same_as_defence: 'yes'
  measured_by: astra_arxiv2025
  page: 12
  attacker_budget: 40 tokens (20, 20), 1000 iterations, universal
  decoding: temperature 0
  sample_size: '198'
  pages:
    decoding: 11
- id: ev_t2_pp_user_40_astrapp
  defence_id: secalignpp
  benchmark: 198 unseen AlpacaFarm examples, universal suffix trained on 10 others; instruction as user, data as input
  attacker_type: optimisation
  attacker_access: white_box
  judge: output starts with 'Hacked' or 'hacked'
  layout_same_as_defence: 'yes'
  measured_by: astra_arxiv2025
  page: 12
  attacker_budget: 40 tokens (20, 20), 1000 iterations, universal
  decoding: temperature 0
  sample_size: '198'
  pages:
    decoding: 11
results:
- id: r_t1_l3_20_gcg
  evaluation_id: ev_t1_l3_20_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: GCG ASR
  value: '12.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 1
- id: r_t1_l3_20_astra
  evaluation_id: ev_t1_l3_20_astra
  quantity: R1
  coverage: direct
  metric_as_printed: ASTRA ASR
  value: '20'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 1
- id: r_t1_l3_25_gcg
  evaluation_id: ev_t1_l3_25_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: GCG ASR
  value: '32.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 1
- id: r_t1_l3_25_astra
  evaluation_id: ev_t1_l3_25_astra
  quantity: R1
  coverage: direct
  metric_as_printed: ASTRA ASR
  value: '57.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 1
- id: r_t1_l3_30_gcg
  evaluation_id: ev_t1_l3_30_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: GCG ASR
  value: '35'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 1
- id: r_t1_l3_30_astra
  evaluation_id: ev_t1_l3_30_astra
  quantity: R1
  coverage: direct
  metric_as_printed: ASTRA ASR
  value: '60'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 1
- id: r_t1_l3_35_gcg
  evaluation_id: ev_t1_l3_35_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: GCG ASR
  value: '45'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 1
- id: r_t1_l3_35_astra
  evaluation_id: ev_t1_l3_35_astra
  quantity: R1
  coverage: direct
  metric_as_printed: ASTRA ASR
  value: '72.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 1
- id: r_t1_mi_20_gcg
  evaluation_id: ev_t1_mi_20_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: GCG ASR
  value: '32.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 1
- id: r_t1_mi_20_astra
  evaluation_id: ev_t1_mi_20_astra
  quantity: R1
  coverage: direct
  metric_as_printed: ASTRA ASR
  value: '75'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 1
- id: r_t1_mi_25_gcg
  evaluation_id: ev_t1_mi_25_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: GCG ASR
  value: '57.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 1
- id: r_t1_mi_25_astra
  evaluation_id: ev_t1_mi_25_astra
  quantity: R1
  coverage: direct
  metric_as_printed: ASTRA ASR
  value: '77.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 1
- id: r_t1_mi_30_gcg
  evaluation_id: ev_t1_mi_30_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: GCG ASR
  value: '60'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 1
- id: r_t1_mi_30_astra
  evaluation_id: ev_t1_mi_30_astra
  quantity: R1
  coverage: direct
  metric_as_printed: ASTRA ASR
  value: '80'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 1
- id: r_t1_mi_35_gcg
  evaluation_id: ev_t1_mi_35_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: GCG ASR
  value: '70'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 1
- id: r_t1_mi_35_astra
  evaluation_id: ev_t1_mi_35_astra
  quantity: R1
  coverage: direct
  metric_as_printed: ASTRA ASR
  value: '82.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 1
- id: r_sq_mi_gcg
  evaluation_id: ev_sq_mi_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: examples broken by GCG (of 40)
  value: '28'
  unit: count
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 11
  location: Section 6.1 text
- id: r_sq_mi_astra
  evaluation_id: ev_sq_mi_astra
  quantity: R1
  coverage: direct
  metric_as_printed: examples broken by ASTRA (of the remaining 12)
  value: '10'
  unit: count
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 11
  location: Section 6.1 text
- id: r_sq_l3_gcg
  evaluation_id: ev_sq_l3_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: examples broken by GCG (of 40)
  value: '23'
  unit: count
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 11
  location: Section 6.1 text
- id: r_sq_l3_astra
  evaluation_id: ev_sq_l3_astra
  quantity: R1
  coverage: direct
  metric_as_printed: examples broken by ASTRA (of the remaining 17)
  value: '8'
  unit: count
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 11
  location: Section 6.1 text
- id: r_t2_sa_l3_30_ugcg
  evaluation_id: ev_t2_sa_l3_30_ugcg
  quantity: R1
  coverage: direct
  metric_as_printed: Univ-GCG ASR, mean over 3 training runs
  value: '11'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
  spread: standard deviation 17% over 3 training runs
- id: r_t2_sa_l3_30_astrapp
  evaluation_id: ev_t2_sa_l3_30_astrapp
  quantity: R1
  coverage: direct
  metric_as_printed: ASTRA++ ASR, mean over 3 training runs
  value: '28'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
  spread: standard deviation 48% over 3 training runs
- id: r_t2_sa_l3_35_ugcg
  evaluation_id: ev_t2_sa_l3_35_ugcg
  quantity: R1
  coverage: direct
  metric_as_printed: Univ-GCG ASR, mean over 3 training runs
  value: '55'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
  spread: standard deviation 48% over 3 training runs
- id: r_t2_sa_l3_35_astrapp
  evaluation_id: ev_t2_sa_l3_35_astrapp
  quantity: R1
  coverage: direct
  metric_as_printed: ASTRA++ ASR, mean over 3 training runs
  value: '96'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
  spread: standard deviation 5% over 3 training runs
- id: r_t2_sa_l3_40_ugcg
  evaluation_id: ev_t2_sa_l3_40_ugcg
  quantity: R1
  coverage: direct
  metric_as_printed: Univ-GCG ASR, mean over 3 training runs
  value: '2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
  spread: standard deviation 2% over 3 training runs
- id: r_t2_sa_l3_40_astrapp
  evaluation_id: ev_t2_sa_l3_40_astrapp
  quantity: R1
  coverage: direct
  metric_as_printed: ASTRA++ ASR, mean over 3 training runs
  value: '85'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
  spread: standard deviation 18% over 3 training runs
- id: r_t2_pp_sys_30_ugcg
  evaluation_id: ev_t2_pp_sys_30_ugcg
  quantity: R1
  coverage: direct
  metric_as_printed: Univ-GCG ASR, mean over 3 training runs
  value: '19'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
  spread: standard deviation 16% over 3 training runs
- id: r_t2_pp_sys_30_astrapp
  evaluation_id: ev_t2_pp_sys_30_astrapp
  quantity: R1
  coverage: direct
  metric_as_printed: ASTRA++ ASR, mean over 3 training runs
  value: '62'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
  spread: standard deviation 53% over 3 training runs
- id: r_t2_pp_sys_35_ugcg
  evaluation_id: ev_t2_pp_sys_35_ugcg
  quantity: R1
  coverage: direct
  metric_as_printed: Univ-GCG ASR, mean over 3 training runs
  value: '22'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
  spread: standard deviation 21% over 3 training runs
- id: r_t2_pp_sys_35_astrapp
  evaluation_id: ev_t2_pp_sys_35_astrapp
  quantity: R1
  coverage: direct
  metric_as_printed: ASTRA++ ASR, mean over 3 training runs
  value: '87'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
  spread: standard deviation 12% over 3 training runs
- id: r_t2_pp_sys_40_ugcg
  evaluation_id: ev_t2_pp_sys_40_ugcg
  quantity: R1
  coverage: direct
  metric_as_printed: Univ-GCG ASR, mean over 3 training runs
  value: '30'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
  spread: standard deviation 33% over 3 training runs
- id: r_t2_pp_sys_40_astrapp
  evaluation_id: ev_t2_pp_sys_40_astrapp
  quantity: R1
  coverage: direct
  metric_as_printed: ASTRA++ ASR, mean over 3 training runs
  value: '55'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
  spread: standard deviation 20% over 3 training runs
- id: r_t2_pp_user_30_ugcg
  evaluation_id: ev_t2_pp_user_30_ugcg
  quantity: R1
  coverage: direct
  metric_as_printed: Univ-GCG ASR, mean over 3 training runs
  value: '1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
  spread: standard deviation 11% over 3 training runs
- id: r_t2_pp_user_30_astrapp
  evaluation_id: ev_t2_pp_user_30_astrapp
  quantity: R1
  coverage: direct
  metric_as_printed: ASTRA++ ASR, mean over 3 training runs
  value: '69'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
  spread: standard deviation 1% over 3 training runs
- id: r_t2_pp_user_35_ugcg
  evaluation_id: ev_t2_pp_user_35_ugcg
  quantity: R1
  coverage: direct
  metric_as_printed: Univ-GCG ASR, mean over 3 training runs
  value: '31'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
  spread: standard deviation 24% over 3 training runs
- id: r_t2_pp_user_35_astrapp
  evaluation_id: ev_t2_pp_user_35_astrapp
  quantity: R1
  coverage: direct
  metric_as_printed: ASTRA++ ASR, mean over 3 training runs
  value: '22'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
  spread: standard deviation 54% over 3 training runs
- id: r_t2_pp_user_40_ugcg
  evaluation_id: ev_t2_pp_user_40_ugcg
  quantity: R1
  coverage: direct
  metric_as_printed: Univ-GCG ASR, mean over 3 training runs
  value: '22'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
  spread: standard deviation 53% over 3 training runs
- id: r_t2_pp_user_40_astrapp
  evaluation_id: ev_t2_pp_user_40_astrapp
  quantity: R1
  coverage: direct
  metric_as_printed: ASTRA++ ASR, mean over 3 training runs
  value: '37'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
  spread: standard deviation 30% over 3 training runs
comparisons:
- id: c_t1_l3_20
  left: ev_t1_l3_20_astra
  right: ev_t1_l3_20_gcg
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: manipulated
  c5_judge: matched
  c6_decoding: matched
  varies: ASTRA attention-guided warm start against plain GCG, same budget and iterations
  page: 11
- id: c_t1_l3_25
  left: ev_t1_l3_25_astra
  right: ev_t1_l3_25_gcg
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: manipulated
  c5_judge: matched
  c6_decoding: matched
  varies: ASTRA attention-guided warm start against plain GCG, same budget and iterations
  page: 11
- id: c_t1_l3_30
  left: ev_t1_l3_30_astra
  right: ev_t1_l3_30_gcg
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: manipulated
  c5_judge: matched
  c6_decoding: matched
  varies: ASTRA attention-guided warm start against plain GCG, same budget and iterations
  page: 11
- id: c_t1_l3_35
  left: ev_t1_l3_35_astra
  right: ev_t1_l3_35_gcg
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: manipulated
  c5_judge: matched
  c6_decoding: matched
  varies: ASTRA attention-guided warm start against plain GCG, same budget and iterations
  page: 11
- id: c_t1_mi_20
  left: ev_t1_mi_20_astra
  right: ev_t1_mi_20_gcg
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: manipulated
  c5_judge: matched
  c6_decoding: matched
  varies: ASTRA attention-guided warm start against plain GCG, same budget and iterations
  page: 11
- id: c_t1_mi_25
  left: ev_t1_mi_25_astra
  right: ev_t1_mi_25_gcg
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: manipulated
  c5_judge: matched
  c6_decoding: matched
  varies: ASTRA attention-guided warm start against plain GCG, same budget and iterations
  page: 11
- id: c_t1_mi_30
  left: ev_t1_mi_30_astra
  right: ev_t1_mi_30_gcg
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: manipulated
  c5_judge: matched
  c6_decoding: matched
  varies: ASTRA attention-guided warm start against plain GCG, same budget and iterations
  page: 11
- id: c_t1_mi_35
  left: ev_t1_mi_35_astra
  right: ev_t1_mi_35_gcg
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: manipulated
  c5_judge: matched
  c6_decoding: matched
  varies: ASTRA attention-guided warm start against plain GCG, same budget and iterations
  page: 11
- id: c_t2_role_30_ugcg
  left: ev_t2_pp_sys_30_ugcg
  right: ev_t2_pp_user_30_ugcg
  purpose: M
  c1_base: matched
  c2_layout: manipulated
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: user instruction in the system role against the user role, data in the input role
  page: 12
- id: c_t2_role_30_astrapp
  left: ev_t2_pp_sys_30_astrapp
  right: ev_t2_pp_user_30_astrapp
  purpose: M
  c1_base: matched
  c2_layout: manipulated
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: user instruction in the system role against the user role, data in the input role
  page: 12
- id: c_t2_role_35_ugcg
  left: ev_t2_pp_sys_35_ugcg
  right: ev_t2_pp_user_35_ugcg
  purpose: M
  c1_base: matched
  c2_layout: manipulated
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: user instruction in the system role against the user role, data in the input role
  page: 12
- id: c_t2_role_35_astrapp
  left: ev_t2_pp_sys_35_astrapp
  right: ev_t2_pp_user_35_astrapp
  purpose: M
  c1_base: matched
  c2_layout: manipulated
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: user instruction in the system role against the user role, data in the input role
  page: 12
- id: c_t2_role_40_ugcg
  left: ev_t2_pp_sys_40_ugcg
  right: ev_t2_pp_user_40_ugcg
  purpose: M
  c1_base: matched
  c2_layout: manipulated
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: user instruction in the system role against the user role, data in the input role
  page: 12
- id: c_t2_role_40_astrapp
  left: ev_t2_pp_sys_40_astrapp
  right: ev_t2_pp_user_40_astrapp
  purpose: M
  c1_base: matched
  c2_layout: manipulated
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: user instruction in the system role against the user role, data in the input role
  page: 12
claims:
- id: k1
  text: architecture-aware attacks reach 85 to 95% success on unseen prompts against fine-tuning defences
  location: abstract
  anchor: attacks with success rates of up to85-95%on unseen prompts
  jobs:
  - R1
  supported_by: []
  page: 1
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Tables 1 and 2 and Section 6.1 (defended models only)
schema_gaps:
- where: evaluations ev_*
  note: every result is for a defended model; there is no undefended model under the same attacks, so the only comparisons vary the attack or, for SecAlign++, the role holding the instruction
- where: defence secalignpp
  note: the paper calls SecAlign++ a production-grade instruction-hierarchy model and does not name the checkpoint or base model

`autodojo_arxiv2026` (2606.15057)

In [ ]:
%%writefile reports/records/autodojo_arxiv2026.yaml
report:
  id: autodojo_arxiv2026
  slug: '2606.15057'
  title: 'AutoDojo: A Generative Benchmark for Evaluating Prompt Injection Defenses in LLM Agents'
  first_author: Xinhang Ma
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2606.15057'
  version_read: arXiv v3
  first_date: '2026-06-13'
  read: F
  role: benchmark
  code_release: https://github.com/xhOwenMa/AutoDojo (linked from the abstract)
  extraction_scope: 'Section 4 and Appendices D and H: Table 5 (AgentDojo, five models) and Table 8 (AgentDyn, two models), rows No defense, Reminder and Spotlighting; the abstract claim (PromptGuard, PIGuard, ProtectAI and
    DataFilter, which filter tool outputs, and Progent, DRIFT, CaMeL and ROPE, which are system-level designs, are left out; per-suite and per-bucket Tables 6, 7, 9 and 10 and the comparison with other adaptive attacks not
    coded)'
defences:
- id: rem_g4m
  name: Reminder
  origin: other_report
  base_checkpoint: GPT-4o-mini
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 13
  layout: AgentDojo or AgentDyn agent loop; tool outputs carry the injection
  layout_compat: &id001
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Reminder prepends to every tool output a sentence telling the agent not to
- id: spot_g4m
  name: Spotlighting (delimiters around tool output)
  origin: other_report
  base_checkpoint: GPT-4o-mini
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 13
  layout: AgentDojo or AgentDyn agent loop; tool outputs carry the injection
  layout_compat: *id001
  anchor: wraps every tool output in delimiters
- id: rem_g54m
  name: Reminder
  origin: other_report
  base_checkpoint: GPT-5.4-mini
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 13
  layout: AgentDojo or AgentDyn agent loop; tool outputs carry the injection
  layout_compat: *id001
  anchor: Reminder prepends to every tool output a sentence telling the agent not to
- id: spot_g54m
  name: Spotlighting (delimiters around tool output)
  origin: other_report
  base_checkpoint: GPT-5.4-mini
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 13
  layout: AgentDojo or AgentDyn agent loop; tool outputs carry the injection
  layout_compat: *id001
  anchor: wraps every tool output in delimiters
- id: rem_gem
  name: Reminder
  origin: other_report
  base_checkpoint: Gemini-2.5-Flash
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 13
  layout: AgentDojo or AgentDyn agent loop; tool outputs carry the injection
  layout_compat: *id001
  anchor: Reminder prepends to every tool output a sentence telling the agent not to
- id: spot_gem
  name: Spotlighting (delimiters around tool output)
  origin: other_report
  base_checkpoint: Gemini-2.5-Flash
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 13
  layout: AgentDojo or AgentDyn agent loop; tool outputs carry the injection
  layout_compat: *id001
  anchor: wraps every tool output in delimiters
- id: rem_ds
  name: Reminder
  origin: other_report
  base_checkpoint: DeepSeek-v4-Flash
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 13
  layout: AgentDojo or AgentDyn agent loop; tool outputs carry the injection
  layout_compat: *id001
  anchor: Reminder prepends to every tool output a sentence telling the agent not to
- id: spot_ds
  name: Spotlighting (delimiters around tool output)
  origin: other_report
  base_checkpoint: DeepSeek-v4-Flash
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 13
  layout: AgentDojo or AgentDyn agent loop; tool outputs carry the injection
  layout_compat: *id001
  anchor: wraps every tool output in delimiters
- id: rem_hk
  name: Reminder
  origin: other_report
  base_checkpoint: Claude-Haiku-4.5
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 13
  layout: AgentDojo or AgentDyn agent loop; tool outputs carry the injection
  layout_compat: *id001
  anchor: Reminder prepends to every tool output a sentence telling the agent not to
- id: spot_hk
  name: Spotlighting (delimiters around tool output)
  origin: other_report
  base_checkpoint: Claude-Haiku-4.5
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 13
  layout: AgentDojo or AgentDyn agent loop; tool outputs carry the injection
  layout_compat: *id001
  anchor: wraps every tool output in delimiters
evaluations:
- id: ev_ad_none_g4m_clean
  defence_id: none:GPT-4o-mini
  benchmark: AgentDojo (banking, slack, travel), aggregated over the three suites, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 23
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_ad_none_g4m_static
  defence_id: none:GPT-4o-mini
  benchmark: AgentDojo (banking, slack, travel), aggregated over the three suites, important instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 23
  decoding: not reported
  pages:
    benchmark: 6
  baseline_kind: off_the_shelf
- id: ev_ad_none_g4m_auto
  defence_id: none:GPT-4o-mini
  benchmark: AgentDojo (banking, slack, travel), aggregated over the three suites, AutoDojo injections optimised against this agent and defence
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 23
  attacker_budget: Gemini-3.1-Pro optimiser, six iterations per injection target, success signal only
  decoding: not reported
  pages:
    benchmark: 6
    attacker_budget: 6
  baseline_kind: off_the_shelf
- id: ev_ad_none_g54m_clean
  defence_id: none:GPT-5.4-mini
  benchmark: AgentDojo (banking, slack, travel), aggregated over the three suites, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 23
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_ad_none_g54m_static
  defence_id: none:GPT-5.4-mini
  benchmark: AgentDojo (banking, slack, travel), aggregated over the three suites, important instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 23
  decoding: not reported
  pages:
    benchmark: 6
  baseline_kind: off_the_shelf
- id: ev_ad_none_g54m_auto
  defence_id: none:GPT-5.4-mini
  benchmark: AgentDojo (banking, slack, travel), aggregated over the three suites, AutoDojo injections optimised against this agent and defence
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 23
  attacker_budget: Gemini-3.1-Pro optimiser, six iterations per injection target, success signal only
  decoding: not reported
  pages:
    benchmark: 6
    attacker_budget: 6
  baseline_kind: off_the_shelf
- id: ev_ad_none_gem_clean
  defence_id: none:Gemini-2.5-Flash
  benchmark: AgentDojo (banking, slack, travel), aggregated over the three suites, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 23
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_ad_none_gem_static
  defence_id: none:Gemini-2.5-Flash
  benchmark: AgentDojo (banking, slack, travel), aggregated over the three suites, important instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 23
  decoding: not reported
  pages:
    benchmark: 6
  baseline_kind: off_the_shelf
- id: ev_ad_none_gem_auto
  defence_id: none:Gemini-2.5-Flash
  benchmark: AgentDojo (banking, slack, travel), aggregated over the three suites, AutoDojo injections optimised against this agent and defence
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 23
  attacker_budget: Gemini-3.1-Pro optimiser, six iterations per injection target, success signal only
  decoding: not reported
  pages:
    benchmark: 6
    attacker_budget: 6
  baseline_kind: off_the_shelf
- id: ev_ad_none_ds_clean
  defence_id: none:DeepSeek-v4-Flash
  benchmark: AgentDojo (banking, slack, travel), aggregated over the three suites, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 23
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_ad_none_ds_static
  defence_id: none:DeepSeek-v4-Flash
  benchmark: AgentDojo (banking, slack, travel), aggregated over the three suites, important instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 23
  decoding: not reported
  pages:
    benchmark: 6
  baseline_kind: off_the_shelf
- id: ev_ad_none_ds_auto
  defence_id: none:DeepSeek-v4-Flash
  benchmark: AgentDojo (banking, slack, travel), aggregated over the three suites, AutoDojo injections optimised against this agent and defence
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 23
  attacker_budget: Gemini-3.1-Pro optimiser, six iterations per injection target, success signal only
  decoding: not reported
  pages:
    benchmark: 6
    attacker_budget: 6
  baseline_kind: off_the_shelf
- id: ev_ad_none_hk_clean
  defence_id: none:Claude-Haiku-4.5
  benchmark: AgentDojo (banking, slack, travel), aggregated over the three suites, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 23
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_ad_none_hk_static
  defence_id: none:Claude-Haiku-4.5
  benchmark: AgentDojo (banking, slack, travel), aggregated over the three suites, important instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 23
  decoding: not reported
  pages:
    benchmark: 6
  baseline_kind: off_the_shelf
- id: ev_ad_none_hk_auto
  defence_id: none:Claude-Haiku-4.5
  benchmark: AgentDojo (banking, slack, travel), aggregated over the three suites, AutoDojo injections optimised against this agent and defence
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 23
  attacker_budget: Gemini-3.1-Pro optimiser, six iterations per injection target, success signal only
  decoding: not reported
  pages:
    benchmark: 6
    attacker_budget: 6
  baseline_kind: off_the_shelf
- id: ev_ad_rem_g4m_clean
  defence_id: rem_g4m
  benchmark: AgentDojo (banking, slack, travel), aggregated over the three suites, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 23
  decoding: not reported
- id: ev_ad_rem_g4m_static
  defence_id: rem_g4m
  benchmark: AgentDojo (banking, slack, travel), aggregated over the three suites, important instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 23
  decoding: not reported
  pages:
    benchmark: 6
- id: ev_ad_rem_g4m_auto
  defence_id: rem_g4m
  benchmark: AgentDojo (banking, slack, travel), aggregated over the three suites, AutoDojo injections optimised against this agent and defence
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 23
  attacker_budget: Gemini-3.1-Pro optimiser, six iterations per injection target, success signal only
  decoding: not reported
  pages:
    benchmark: 6
    attacker_budget: 6
- id: ev_ad_rem_g54m_clean
  defence_id: rem_g54m
  benchmark: AgentDojo (banking, slack, travel), aggregated over the three suites, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 23
  decoding: not reported
- id: ev_ad_rem_g54m_static
  defence_id: rem_g54m
  benchmark: AgentDojo (banking, slack, travel), aggregated over the three suites, important instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 23
  decoding: not reported
  pages:
    benchmark: 6
- id: ev_ad_rem_g54m_auto
  defence_id: rem_g54m
  benchmark: AgentDojo (banking, slack, travel), aggregated over the three suites, AutoDojo injections optimised against this agent and defence
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 23
  attacker_budget: Gemini-3.1-Pro optimiser, six iterations per injection target, success signal only
  decoding: not reported
  pages:
    benchmark: 6
    attacker_budget: 6
- id: ev_ad_rem_gem_clean
  defence_id: rem_gem
  benchmark: AgentDojo (banking, slack, travel), aggregated over the three suites, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 23
  decoding: not reported
- id: ev_ad_rem_gem_static
  defence_id: rem_gem
  benchmark: AgentDojo (banking, slack, travel), aggregated over the three suites, important instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 23
  decoding: not reported
  pages:
    benchmark: 6
- id: ev_ad_rem_gem_auto
  defence_id: rem_gem
  benchmark: AgentDojo (banking, slack, travel), aggregated over the three suites, AutoDojo injections optimised against this agent and defence
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 23
  attacker_budget: Gemini-3.1-Pro optimiser, six iterations per injection target, success signal only
  decoding: not reported
  pages:
    benchmark: 6
    attacker_budget: 6
- id: ev_ad_rem_ds_clean
  defence_id: rem_ds
  benchmark: AgentDojo (banking, slack, travel), aggregated over the three suites, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 23
  decoding: not reported
- id: ev_ad_rem_ds_static
  defence_id: rem_ds
  benchmark: AgentDojo (banking, slack, travel), aggregated over the three suites, important instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 23
  decoding: not reported
  pages:
    benchmark: 6
- id: ev_ad_rem_ds_auto
  defence_id: rem_ds
  benchmark: AgentDojo (banking, slack, travel), aggregated over the three suites, AutoDojo injections optimised against this agent and defence
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 23
  attacker_budget: Gemini-3.1-Pro optimiser, six iterations per injection target, success signal only
  decoding: not reported
  pages:
    benchmark: 6
    attacker_budget: 6
- id: ev_ad_rem_hk_clean
  defence_id: rem_hk
  benchmark: AgentDojo (banking, slack, travel), aggregated over the three suites, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 23
  decoding: not reported
- id: ev_ad_rem_hk_static
  defence_id: rem_hk
  benchmark: AgentDojo (banking, slack, travel), aggregated over the three suites, important instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 23
  decoding: not reported
  pages:
    benchmark: 6
- id: ev_ad_rem_hk_auto
  defence_id: rem_hk
  benchmark: AgentDojo (banking, slack, travel), aggregated over the three suites, AutoDojo injections optimised against this agent and defence
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 23
  attacker_budget: Gemini-3.1-Pro optimiser, six iterations per injection target, success signal only
  decoding: not reported
  pages:
    benchmark: 6
    attacker_budget: 6
- id: ev_ad_spot_g4m_clean
  defence_id: spot_g4m
  benchmark: AgentDojo (banking, slack, travel), aggregated over the three suites, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 23
  decoding: not reported
- id: ev_ad_spot_g4m_static
  defence_id: spot_g4m
  benchmark: AgentDojo (banking, slack, travel), aggregated over the three suites, important instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 23
  decoding: not reported
  pages:
    benchmark: 6
- id: ev_ad_spot_g4m_auto
  defence_id: spot_g4m
  benchmark: AgentDojo (banking, slack, travel), aggregated over the three suites, AutoDojo injections optimised against this agent and defence
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 23
  attacker_budget: Gemini-3.1-Pro optimiser, six iterations per injection target, success signal only
  decoding: not reported
  pages:
    benchmark: 6
    attacker_budget: 6
- id: ev_ad_spot_g54m_clean
  defence_id: spot_g54m
  benchmark: AgentDojo (banking, slack, travel), aggregated over the three suites, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 23
  decoding: not reported
- id: ev_ad_spot_g54m_static
  defence_id: spot_g54m
  benchmark: AgentDojo (banking, slack, travel), aggregated over the three suites, important instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 23
  decoding: not reported
  pages:
    benchmark: 6
- id: ev_ad_spot_g54m_auto
  defence_id: spot_g54m
  benchmark: AgentDojo (banking, slack, travel), aggregated over the three suites, AutoDojo injections optimised against this agent and defence
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 23
  attacker_budget: Gemini-3.1-Pro optimiser, six iterations per injection target, success signal only
  decoding: not reported
  pages:
    benchmark: 6
    attacker_budget: 6
- id: ev_ad_spot_gem_clean
  defence_id: spot_gem
  benchmark: AgentDojo (banking, slack, travel), aggregated over the three suites, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 23
  decoding: not reported
- id: ev_ad_spot_gem_static
  defence_id: spot_gem
  benchmark: AgentDojo (banking, slack, travel), aggregated over the three suites, important instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 23
  decoding: not reported
  pages:
    benchmark: 6
- id: ev_ad_spot_gem_auto
  defence_id: spot_gem
  benchmark: AgentDojo (banking, slack, travel), aggregated over the three suites, AutoDojo injections optimised against this agent and defence
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 23
  attacker_budget: Gemini-3.1-Pro optimiser, six iterations per injection target, success signal only
  decoding: not reported
  pages:
    benchmark: 6
    attacker_budget: 6
- id: ev_ad_spot_ds_clean
  defence_id: spot_ds
  benchmark: AgentDojo (banking, slack, travel), aggregated over the three suites, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 23
  decoding: not reported
- id: ev_ad_spot_ds_static
  defence_id: spot_ds
  benchmark: AgentDojo (banking, slack, travel), aggregated over the three suites, important instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 23
  decoding: not reported
  pages:
    benchmark: 6
- id: ev_ad_spot_ds_auto
  defence_id: spot_ds
  benchmark: AgentDojo (banking, slack, travel), aggregated over the three suites, AutoDojo injections optimised against this agent and defence
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 23
  attacker_budget: Gemini-3.1-Pro optimiser, six iterations per injection target, success signal only
  decoding: not reported
  pages:
    benchmark: 6
    attacker_budget: 6
- id: ev_ad_spot_hk_clean
  defence_id: spot_hk
  benchmark: AgentDojo (banking, slack, travel), aggregated over the three suites, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 23
  decoding: not reported
- id: ev_ad_spot_hk_static
  defence_id: spot_hk
  benchmark: AgentDojo (banking, slack, travel), aggregated over the three suites, important instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 23
  decoding: not reported
  pages:
    benchmark: 6
- id: ev_ad_spot_hk_auto
  defence_id: spot_hk
  benchmark: AgentDojo (banking, slack, travel), aggregated over the three suites, AutoDojo injections optimised against this agent and defence
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 23
  attacker_budget: Gemini-3.1-Pro optimiser, six iterations per injection target, success signal only
  decoding: not reported
  pages:
    benchmark: 6
    attacker_budget: 6
- id: ev_dy_none_g4m_clean
  defence_id: none:GPT-4o-mini
  benchmark: AgentDyn (github, shopping, dailylife), aggregated over the three suites, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 26
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_dy_none_g4m_static
  defence_id: none:GPT-4o-mini
  benchmark: AgentDyn (github, shopping, dailylife), aggregated over the three suites, important instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 26
  decoding: not reported
  pages:
    benchmark: 6
  baseline_kind: off_the_shelf
- id: ev_dy_none_g4m_auto
  defence_id: none:GPT-4o-mini
  benchmark: AgentDyn (github, shopping, dailylife), aggregated over the three suites, AutoDojo injections optimised against this agent and defence
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 26
  attacker_budget: Gemini-3.1-Pro optimiser, six iterations per injection target, success signal only
  decoding: not reported
  pages:
    benchmark: 6
    attacker_budget: 6
  baseline_kind: off_the_shelf
- id: ev_dy_none_gem_clean
  defence_id: none:Gemini-2.5-Flash
  benchmark: AgentDyn (github, shopping, dailylife), aggregated over the three suites, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 26
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_dy_none_gem_static
  defence_id: none:Gemini-2.5-Flash
  benchmark: AgentDyn (github, shopping, dailylife), aggregated over the three suites, important instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 26
  decoding: not reported
  pages:
    benchmark: 6
  baseline_kind: off_the_shelf
- id: ev_dy_none_gem_auto
  defence_id: none:Gemini-2.5-Flash
  benchmark: AgentDyn (github, shopping, dailylife), aggregated over the three suites, AutoDojo injections optimised against this agent and defence
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 26
  attacker_budget: Gemini-3.1-Pro optimiser, six iterations per injection target, success signal only
  decoding: not reported
  pages:
    benchmark: 6
    attacker_budget: 6
  baseline_kind: off_the_shelf
- id: ev_dy_rem_g4m_clean
  defence_id: rem_g4m
  benchmark: AgentDyn (github, shopping, dailylife), aggregated over the three suites, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 26
  decoding: not reported
- id: ev_dy_rem_g4m_static
  defence_id: rem_g4m
  benchmark: AgentDyn (github, shopping, dailylife), aggregated over the three suites, important instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 26
  decoding: not reported
  pages:
    benchmark: 6
- id: ev_dy_rem_g4m_auto
  defence_id: rem_g4m
  benchmark: AgentDyn (github, shopping, dailylife), aggregated over the three suites, AutoDojo injections optimised against this agent and defence
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 26
  attacker_budget: Gemini-3.1-Pro optimiser, six iterations per injection target, success signal only
  decoding: not reported
  pages:
    benchmark: 6
    attacker_budget: 6
- id: ev_dy_rem_gem_clean
  defence_id: rem_gem
  benchmark: AgentDyn (github, shopping, dailylife), aggregated over the three suites, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 26
  decoding: not reported
- id: ev_dy_rem_gem_static
  defence_id: rem_gem
  benchmark: AgentDyn (github, shopping, dailylife), aggregated over the three suites, important instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 26
  decoding: not reported
  pages:
    benchmark: 6
- id: ev_dy_rem_gem_auto
  defence_id: rem_gem
  benchmark: AgentDyn (github, shopping, dailylife), aggregated over the three suites, AutoDojo injections optimised against this agent and defence
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 26
  attacker_budget: Gemini-3.1-Pro optimiser, six iterations per injection target, success signal only
  decoding: not reported
  pages:
    benchmark: 6
    attacker_budget: 6
- id: ev_dy_spot_g4m_clean
  defence_id: spot_g4m
  benchmark: AgentDyn (github, shopping, dailylife), aggregated over the three suites, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 26
  decoding: not reported
- id: ev_dy_spot_g4m_static
  defence_id: spot_g4m
  benchmark: AgentDyn (github, shopping, dailylife), aggregated over the three suites, important instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 26
  decoding: not reported
  pages:
    benchmark: 6
- id: ev_dy_spot_g4m_auto
  defence_id: spot_g4m
  benchmark: AgentDyn (github, shopping, dailylife), aggregated over the three suites, AutoDojo injections optimised against this agent and defence
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 26
  attacker_budget: Gemini-3.1-Pro optimiser, six iterations per injection target, success signal only
  decoding: not reported
  pages:
    benchmark: 6
    attacker_budget: 6
- id: ev_dy_spot_gem_clean
  defence_id: spot_gem
  benchmark: AgentDyn (github, shopping, dailylife), aggregated over the three suites, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 26
  decoding: not reported
- id: ev_dy_spot_gem_static
  defence_id: spot_gem
  benchmark: AgentDyn (github, shopping, dailylife), aggregated over the three suites, important instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 26
  decoding: not reported
  pages:
    benchmark: 6
- id: ev_dy_spot_gem_auto
  defence_id: spot_gem
  benchmark: AgentDyn (github, shopping, dailylife), aggregated over the three suites, AutoDojo injections optimised against this agent and defence
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: autodojo_arxiv2026
  page: 26
  attacker_budget: Gemini-3.1-Pro optimiser, six iterations per injection target, success signal only
  decoding: not reported
  pages:
    benchmark: 6
    attacker_budget: 6
results:
- id: r_ad_none_g4m_cu
  evaluation_id: ev_ad_none_g4m_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Clean util.
  value: '66.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_none_g4m_sa
  evaluation_id: ev_ad_none_g4m_static
  quantity: R1
  coverage: direct
  metric_as_printed: Static ASR
  value: '58.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_none_g4m_sua
  evaluation_id: ev_ad_none_g4m_static
  quantity: R2attack
  coverage: direct
  metric_as_printed: Static UA (in parentheses)
  value: '42.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_none_g4m_aa
  evaluation_id: ev_ad_none_g4m_auto
  quantity: R1
  coverage: direct
  metric_as_printed: AutoDojo ASR
  value: '52.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_none_g4m_aua
  evaluation_id: ev_ad_none_g4m_auto
  quantity: R2attack
  coverage: direct
  metric_as_printed: AutoDojo UA (in parentheses)
  value: '41.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_none_g54m_cu
  evaluation_id: ev_ad_none_g54m_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Clean util.
  value: '77.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_none_g54m_sa
  evaluation_id: ev_ad_none_g54m_static
  quantity: R1
  coverage: direct
  metric_as_printed: Static ASR
  value: '6.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_none_g54m_sua
  evaluation_id: ev_ad_none_g54m_static
  quantity: R2attack
  coverage: direct
  metric_as_printed: Static UA (in parentheses)
  value: '60.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_none_g54m_aa
  evaluation_id: ev_ad_none_g54m_auto
  quantity: R1
  coverage: direct
  metric_as_printed: AutoDojo ASR
  value: '8.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_none_g54m_aua
  evaluation_id: ev_ad_none_g54m_auto
  quantity: R2attack
  coverage: direct
  metric_as_printed: AutoDojo UA (in parentheses)
  value: '62.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_none_gem_cu
  evaluation_id: ev_ad_none_gem_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Clean util.
  value: '63.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_none_gem_sa
  evaluation_id: ev_ad_none_gem_static
  quantity: R1
  coverage: direct
  metric_as_printed: Static ASR
  value: '50.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_none_gem_sua
  evaluation_id: ev_ad_none_gem_static
  quantity: R2attack
  coverage: direct
  metric_as_printed: Static UA (in parentheses)
  value: '40.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_none_gem_aa
  evaluation_id: ev_ad_none_gem_auto
  quantity: R1
  coverage: direct
  metric_as_printed: AutoDojo ASR
  value: '34.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_none_gem_aua
  evaluation_id: ev_ad_none_gem_auto
  quantity: R2attack
  coverage: direct
  metric_as_printed: AutoDojo UA (in parentheses)
  value: '44.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_none_ds_cu
  evaluation_id: ev_ad_none_ds_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Clean util.
  value: '89.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_none_ds_sa
  evaluation_id: ev_ad_none_ds_static
  quantity: R1
  coverage: direct
  metric_as_printed: Static ASR
  value: '22.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_none_ds_sua
  evaluation_id: ev_ad_none_ds_static
  quantity: R2attack
  coverage: direct
  metric_as_printed: Static UA (in parentheses)
  value: '77.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_none_ds_aa
  evaluation_id: ev_ad_none_ds_auto
  quantity: R1
  coverage: direct
  metric_as_printed: AutoDojo ASR
  value: '16.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_none_ds_aua
  evaluation_id: ev_ad_none_ds_auto
  quantity: R2attack
  coverage: direct
  metric_as_printed: AutoDojo UA (in parentheses)
  value: '77.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_none_hk_cu
  evaluation_id: ev_ad_none_hk_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Clean util.
  value: '70.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_none_hk_sa
  evaluation_id: ev_ad_none_hk_static
  quantity: R1
  coverage: direct
  metric_as_printed: Static ASR
  value: '0.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_none_hk_sua
  evaluation_id: ev_ad_none_hk_static
  quantity: R2attack
  coverage: direct
  metric_as_printed: Static UA (in parentheses)
  value: '61.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_none_hk_aa
  evaluation_id: ev_ad_none_hk_auto
  quantity: R1
  coverage: direct
  metric_as_printed: AutoDojo ASR
  value: '1.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_none_hk_aua
  evaluation_id: ev_ad_none_hk_auto
  quantity: R2attack
  coverage: direct
  metric_as_printed: AutoDojo UA (in parentheses)
  value: '64.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_rem_g4m_cu
  evaluation_id: ev_ad_rem_g4m_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Clean util.
  value: '68.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_rem_g4m_sa
  evaluation_id: ev_ad_rem_g4m_static
  quantity: R1
  coverage: direct
  metric_as_printed: Static ASR
  value: '39.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_rem_g4m_sua
  evaluation_id: ev_ad_rem_g4m_static
  quantity: R2attack
  coverage: direct
  metric_as_printed: Static UA (in parentheses)
  value: '46.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_rem_g4m_aa
  evaluation_id: ev_ad_rem_g4m_auto
  quantity: R1
  coverage: direct
  metric_as_printed: AutoDojo ASR
  value: '40.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_rem_g4m_aua
  evaluation_id: ev_ad_rem_g4m_auto
  quantity: R2attack
  coverage: direct
  metric_as_printed: AutoDojo UA (in parentheses)
  value: '42.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_rem_g54m_cu
  evaluation_id: ev_ad_rem_g54m_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Clean util.
  value: '80.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_rem_g54m_sa
  evaluation_id: ev_ad_rem_g54m_static
  quantity: R1
  coverage: direct
  metric_as_printed: Static ASR
  value: '0.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_rem_g54m_sua
  evaluation_id: ev_ad_rem_g54m_static
  quantity: R2attack
  coverage: direct
  metric_as_printed: Static UA (in parentheses)
  value: '66.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_rem_g54m_aa
  evaluation_id: ev_ad_rem_g54m_auto
  quantity: R1
  coverage: direct
  metric_as_printed: AutoDojo ASR
  value: '2.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_rem_g54m_aua
  evaluation_id: ev_ad_rem_g54m_auto
  quantity: R2attack
  coverage: direct
  metric_as_printed: AutoDojo UA (in parentheses)
  value: '64.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_rem_gem_cu
  evaluation_id: ev_ad_rem_gem_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Clean util.
  value: '63.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_rem_gem_sa
  evaluation_id: ev_ad_rem_gem_static
  quantity: R1
  coverage: direct
  metric_as_printed: Static ASR
  value: '45.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_rem_gem_sua
  evaluation_id: ev_ad_rem_gem_static
  quantity: R2attack
  coverage: direct
  metric_as_printed: Static UA (in parentheses)
  value: '35.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_rem_gem_aa
  evaluation_id: ev_ad_rem_gem_auto
  quantity: R1
  coverage: direct
  metric_as_printed: AutoDojo ASR
  value: '29.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_rem_gem_aua
  evaluation_id: ev_ad_rem_gem_auto
  quantity: R2attack
  coverage: direct
  metric_as_printed: AutoDojo UA (in parentheses)
  value: '45.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_rem_ds_cu
  evaluation_id: ev_ad_rem_ds_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Clean util.
  value: '89.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_rem_ds_sa
  evaluation_id: ev_ad_rem_ds_static
  quantity: R1
  coverage: direct
  metric_as_printed: Static ASR
  value: '4.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_rem_ds_sua
  evaluation_id: ev_ad_rem_ds_static
  quantity: R2attack
  coverage: direct
  metric_as_printed: Static UA (in parentheses)
  value: '81.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_rem_ds_aa
  evaluation_id: ev_ad_rem_ds_auto
  quantity: R1
  coverage: direct
  metric_as_printed: AutoDojo ASR
  value: '5.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_rem_ds_aua
  evaluation_id: ev_ad_rem_ds_auto
  quantity: R2attack
  coverage: direct
  metric_as_printed: AutoDojo UA (in parentheses)
  value: '83.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_rem_hk_cu
  evaluation_id: ev_ad_rem_hk_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Clean util.
  value: '68.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_rem_hk_sa
  evaluation_id: ev_ad_rem_hk_static
  quantity: R1
  coverage: direct
  metric_as_printed: Static ASR
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_rem_hk_sua
  evaluation_id: ev_ad_rem_hk_static
  quantity: R2attack
  coverage: direct
  metric_as_printed: Static UA (in parentheses)
  value: '59.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_rem_hk_aa
  evaluation_id: ev_ad_rem_hk_auto
  quantity: R1
  coverage: direct
  metric_as_printed: AutoDojo ASR
  value: '1.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_rem_hk_aua
  evaluation_id: ev_ad_rem_hk_auto
  quantity: R2attack
  coverage: direct
  metric_as_printed: AutoDojo UA (in parentheses)
  value: '60.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_spot_g4m_cu
  evaluation_id: ev_ad_spot_g4m_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Clean util.
  value: '59.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_spot_g4m_sa
  evaluation_id: ev_ad_spot_g4m_static
  quantity: R1
  coverage: direct
  metric_as_printed: Static ASR
  value: '51.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_spot_g4m_sua
  evaluation_id: ev_ad_spot_g4m_static
  quantity: R2attack
  coverage: direct
  metric_as_printed: Static UA (in parentheses)
  value: '43.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_spot_g4m_aa
  evaluation_id: ev_ad_spot_g4m_auto
  quantity: R1
  coverage: direct
  metric_as_printed: AutoDojo ASR
  value: '46.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_spot_g4m_aua
  evaluation_id: ev_ad_spot_g4m_auto
  quantity: R2attack
  coverage: direct
  metric_as_printed: AutoDojo UA (in parentheses)
  value: '42.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_spot_g54m_cu
  evaluation_id: ev_ad_spot_g54m_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Clean util.
  value: '70.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_spot_g54m_sa
  evaluation_id: ev_ad_spot_g54m_static
  quantity: R1
  coverage: direct
  metric_as_printed: Static ASR
  value: '3.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_spot_g54m_sua
  evaluation_id: ev_ad_spot_g54m_static
  quantity: R2attack
  coverage: direct
  metric_as_printed: Static UA (in parentheses)
  value: '61.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_spot_g54m_aa
  evaluation_id: ev_ad_spot_g54m_auto
  quantity: R1
  coverage: direct
  metric_as_printed: AutoDojo ASR
  value: '5.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_spot_g54m_aua
  evaluation_id: ev_ad_spot_g54m_auto
  quantity: R2attack
  coverage: direct
  metric_as_printed: AutoDojo UA (in parentheses)
  value: '61.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_spot_gem_cu
  evaluation_id: ev_ad_spot_gem_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Clean util.
  value: '64.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_spot_gem_sa
  evaluation_id: ev_ad_spot_gem_static
  quantity: R1
  coverage: direct
  metric_as_printed: Static ASR
  value: '54.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_spot_gem_sua
  evaluation_id: ev_ad_spot_gem_static
  quantity: R2attack
  coverage: direct
  metric_as_printed: Static UA (in parentheses)
  value: '40.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_spot_gem_aa
  evaluation_id: ev_ad_spot_gem_auto
  quantity: R1
  coverage: direct
  metric_as_printed: AutoDojo ASR
  value: '36.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_spot_gem_aua
  evaluation_id: ev_ad_spot_gem_auto
  quantity: R2attack
  coverage: direct
  metric_as_printed: AutoDojo UA (in parentheses)
  value: '42.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_spot_ds_cu
  evaluation_id: ev_ad_spot_ds_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Clean util.
  value: '75.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_spot_ds_sa
  evaluation_id: ev_ad_spot_ds_static
  quantity: R1
  coverage: direct
  metric_as_printed: Static ASR
  value: '13.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_spot_ds_sua
  evaluation_id: ev_ad_spot_ds_static
  quantity: R2attack
  coverage: direct
  metric_as_printed: Static UA (in parentheses)
  value: '80.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_spot_ds_aa
  evaluation_id: ev_ad_spot_ds_auto
  quantity: R1
  coverage: direct
  metric_as_printed: AutoDojo ASR
  value: '13.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_spot_ds_aua
  evaluation_id: ev_ad_spot_ds_auto
  quantity: R2attack
  coverage: direct
  metric_as_printed: AutoDojo UA (in parentheses)
  value: '80.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_spot_hk_cu
  evaluation_id: ev_ad_spot_hk_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Clean util.
  value: '70.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_spot_hk_sa
  evaluation_id: ev_ad_spot_hk_static
  quantity: R1
  coverage: direct
  metric_as_printed: Static ASR
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_spot_hk_sua
  evaluation_id: ev_ad_spot_hk_static
  quantity: R2attack
  coverage: direct
  metric_as_printed: Static UA (in parentheses)
  value: '57.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_spot_hk_aa
  evaluation_id: ev_ad_spot_hk_auto
  quantity: R1
  coverage: direct
  metric_as_printed: AutoDojo ASR
  value: '1.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_ad_spot_hk_aua
  evaluation_id: ev_ad_spot_hk_auto
  quantity: R2attack
  coverage: direct
  metric_as_printed: AutoDojo UA (in parentheses)
  value: '59.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 5
- id: r_dy_none_g4m_cu
  evaluation_id: ev_dy_none_g4m_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Clean util.
  value: '46.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 26
  location: Table 8
- id: r_dy_none_g4m_sa
  evaluation_id: ev_dy_none_g4m_static
  quantity: R1
  coverage: direct
  metric_as_printed: Static ASR
  value: '51.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 26
  location: Table 8
- id: r_dy_none_g4m_sua
  evaluation_id: ev_dy_none_g4m_static
  quantity: R2attack
  coverage: direct
  metric_as_printed: Static UA (in parentheses)
  value: '35.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 26
  location: Table 8
- id: r_dy_none_g4m_aa
  evaluation_id: ev_dy_none_g4m_auto
  quantity: R1
  coverage: direct
  metric_as_printed: AutoDojo ASR
  value: '59.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 26
  location: Table 8
- id: r_dy_none_g4m_aua
  evaluation_id: ev_dy_none_g4m_auto
  quantity: R2attack
  coverage: direct
  metric_as_printed: AutoDojo UA (in parentheses)
  value: '37.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 26
  location: Table 8
- id: r_dy_none_gem_cu
  evaluation_id: ev_dy_none_gem_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Clean util.
  value: '36.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 26
  location: Table 8
- id: r_dy_none_gem_sa
  evaluation_id: ev_dy_none_gem_static
  quantity: R1
  coverage: direct
  metric_as_printed: Static ASR
  value: '52.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 26
  location: Table 8
- id: r_dy_none_gem_sua
  evaluation_id: ev_dy_none_gem_static
  quantity: R2attack
  coverage: direct
  metric_as_printed: Static UA (in parentheses)
  value: '33.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 26
  location: Table 8
- id: r_dy_none_gem_aa
  evaluation_id: ev_dy_none_gem_auto
  quantity: R1
  coverage: direct
  metric_as_printed: AutoDojo ASR
  value: '50.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 26
  location: Table 8
- id: r_dy_none_gem_aua
  evaluation_id: ev_dy_none_gem_auto
  quantity: R2attack
  coverage: direct
  metric_as_printed: AutoDojo UA (in parentheses)
  value: '30.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 26
  location: Table 8
- id: r_dy_rem_g4m_cu
  evaluation_id: ev_dy_rem_g4m_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Clean util.
  value: '40.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 26
  location: Table 8
- id: r_dy_rem_g4m_sa
  evaluation_id: ev_dy_rem_g4m_static
  quantity: R1
  coverage: direct
  metric_as_printed: Static ASR
  value: '37.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 26
  location: Table 8
- id: r_dy_rem_g4m_sua
  evaluation_id: ev_dy_rem_g4m_static
  quantity: R2attack
  coverage: direct
  metric_as_printed: Static UA (in parentheses)
  value: '32.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 26
  location: Table 8
- id: r_dy_rem_g4m_aa
  evaluation_id: ev_dy_rem_g4m_auto
  quantity: R1
  coverage: direct
  metric_as_printed: AutoDojo ASR
  value: '51.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 26
  location: Table 8
- id: r_dy_rem_g4m_aua
  evaluation_id: ev_dy_rem_g4m_auto
  quantity: R2attack
  coverage: direct
  metric_as_printed: AutoDojo UA (in parentheses)
  value: '31.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 26
  location: Table 8
- id: r_dy_rem_gem_cu
  evaluation_id: ev_dy_rem_gem_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Clean util.
  value: '36.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 26
  location: Table 8
- id: r_dy_rem_gem_sa
  evaluation_id: ev_dy_rem_gem_static
  quantity: R1
  coverage: direct
  metric_as_printed: Static ASR
  value: '44.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 26
  location: Table 8
- id: r_dy_rem_gem_sua
  evaluation_id: ev_dy_rem_gem_static
  quantity: R2attack
  coverage: direct
  metric_as_printed: Static UA (in parentheses)
  value: '38.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 26
  location: Table 8
- id: r_dy_rem_gem_aa
  evaluation_id: ev_dy_rem_gem_auto
  quantity: R1
  coverage: direct
  metric_as_printed: AutoDojo ASR
  value: '53.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 26
  location: Table 8
- id: r_dy_rem_gem_aua
  evaluation_id: ev_dy_rem_gem_auto
  quantity: R2attack
  coverage: direct
  metric_as_printed: AutoDojo UA (in parentheses)
  value: '36.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 26
  location: Table 8
- id: r_dy_spot_g4m_cu
  evaluation_id: ev_dy_spot_g4m_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Clean util.
  value: '36.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 26
  location: Table 8
- id: r_dy_spot_g4m_sa
  evaluation_id: ev_dy_spot_g4m_static
  quantity: R1
  coverage: direct
  metric_as_printed: Static ASR
  value: '48.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 26
  location: Table 8
- id: r_dy_spot_g4m_sua
  evaluation_id: ev_dy_spot_g4m_static
  quantity: R2attack
  coverage: direct
  metric_as_printed: Static UA (in parentheses)
  value: '35.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 26
  location: Table 8
- id: r_dy_spot_g4m_aa
  evaluation_id: ev_dy_spot_g4m_auto
  quantity: R1
  coverage: direct
  metric_as_printed: AutoDojo ASR
  value: '55.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 26
  location: Table 8
- id: r_dy_spot_g4m_aua
  evaluation_id: ev_dy_spot_g4m_auto
  quantity: R2attack
  coverage: direct
  metric_as_printed: AutoDojo UA (in parentheses)
  value: '35.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 26
  location: Table 8
- id: r_dy_spot_gem_cu
  evaluation_id: ev_dy_spot_gem_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Clean util.
  value: '38.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 26
  location: Table 8
- id: r_dy_spot_gem_sa
  evaluation_id: ev_dy_spot_gem_static
  quantity: R1
  coverage: direct
  metric_as_printed: Static ASR
  value: '57.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 26
  location: Table 8
- id: r_dy_spot_gem_sua
  evaluation_id: ev_dy_spot_gem_static
  quantity: R2attack
  coverage: direct
  metric_as_printed: Static UA (in parentheses)
  value: '40.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 26
  location: Table 8
- id: r_dy_spot_gem_aa
  evaluation_id: ev_dy_spot_gem_auto
  quantity: R1
  coverage: direct
  metric_as_printed: AutoDojo ASR
  value: '52.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 26
  location: Table 8
- id: r_dy_spot_gem_aua
  evaluation_id: ev_dy_spot_gem_auto
  quantity: R2attack
  coverage: direct
  metric_as_printed: AutoDojo UA (in parentheses)
  value: '37.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 26
  location: Table 8
comparisons:
- id: c_ad_rem_g4m_clean
  left: ev_ad_rem_g4m_clean
  right: ev_ad_none_g4m_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 23
- id: c_ad_rem_g4m_static
  left: ev_ad_rem_g4m_static
  right: ev_ad_none_g4m_static
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 23
- id: c_ad_rem_g4m_auto
  left: ev_ad_rem_g4m_auto
  right: ev_ad_none_g4m_auto
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 23
- id: c_ad_spot_g4m_clean
  left: ev_ad_spot_g4m_clean
  right: ev_ad_none_g4m_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 23
- id: c_ad_spot_g4m_static
  left: ev_ad_spot_g4m_static
  right: ev_ad_none_g4m_static
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 23
- id: c_ad_spot_g4m_auto
  left: ev_ad_spot_g4m_auto
  right: ev_ad_none_g4m_auto
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 23
- id: c_ad_rem_g54m_clean
  left: ev_ad_rem_g54m_clean
  right: ev_ad_none_g54m_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 23
- id: c_ad_rem_g54m_static
  left: ev_ad_rem_g54m_static
  right: ev_ad_none_g54m_static
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 23
- id: c_ad_rem_g54m_auto
  left: ev_ad_rem_g54m_auto
  right: ev_ad_none_g54m_auto
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 23
- id: c_ad_spot_g54m_clean
  left: ev_ad_spot_g54m_clean
  right: ev_ad_none_g54m_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 23
- id: c_ad_spot_g54m_static
  left: ev_ad_spot_g54m_static
  right: ev_ad_none_g54m_static
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 23
- id: c_ad_spot_g54m_auto
  left: ev_ad_spot_g54m_auto
  right: ev_ad_none_g54m_auto
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 23
- id: c_ad_rem_gem_clean
  left: ev_ad_rem_gem_clean
  right: ev_ad_none_gem_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 23
- id: c_ad_rem_gem_static
  left: ev_ad_rem_gem_static
  right: ev_ad_none_gem_static
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 23
- id: c_ad_rem_gem_auto
  left: ev_ad_rem_gem_auto
  right: ev_ad_none_gem_auto
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 23
- id: c_ad_spot_gem_clean
  left: ev_ad_spot_gem_clean
  right: ev_ad_none_gem_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 23
- id: c_ad_spot_gem_static
  left: ev_ad_spot_gem_static
  right: ev_ad_none_gem_static
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 23
- id: c_ad_spot_gem_auto
  left: ev_ad_spot_gem_auto
  right: ev_ad_none_gem_auto
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 23
- id: c_ad_rem_ds_clean
  left: ev_ad_rem_ds_clean
  right: ev_ad_none_ds_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 23
- id: c_ad_rem_ds_static
  left: ev_ad_rem_ds_static
  right: ev_ad_none_ds_static
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 23
- id: c_ad_rem_ds_auto
  left: ev_ad_rem_ds_auto
  right: ev_ad_none_ds_auto
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 23
- id: c_ad_spot_ds_clean
  left: ev_ad_spot_ds_clean
  right: ev_ad_none_ds_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 23
- id: c_ad_spot_ds_static
  left: ev_ad_spot_ds_static
  right: ev_ad_none_ds_static
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 23
- id: c_ad_spot_ds_auto
  left: ev_ad_spot_ds_auto
  right: ev_ad_none_ds_auto
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 23
- id: c_ad_rem_hk_clean
  left: ev_ad_rem_hk_clean
  right: ev_ad_none_hk_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 23
- id: c_ad_rem_hk_static
  left: ev_ad_rem_hk_static
  right: ev_ad_none_hk_static
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 23
- id: c_ad_rem_hk_auto
  left: ev_ad_rem_hk_auto
  right: ev_ad_none_hk_auto
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 23
- id: c_ad_spot_hk_clean
  left: ev_ad_spot_hk_clean
  right: ev_ad_none_hk_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 23
- id: c_ad_spot_hk_static
  left: ev_ad_spot_hk_static
  right: ev_ad_none_hk_static
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 23
- id: c_ad_spot_hk_auto
  left: ev_ad_spot_hk_auto
  right: ev_ad_none_hk_auto
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 23
- id: c_dy_rem_g4m_clean
  left: ev_dy_rem_g4m_clean
  right: ev_dy_none_g4m_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 26
- id: c_dy_rem_g4m_static
  left: ev_dy_rem_g4m_static
  right: ev_dy_none_g4m_static
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 26
- id: c_dy_rem_g4m_auto
  left: ev_dy_rem_g4m_auto
  right: ev_dy_none_g4m_auto
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 26
- id: c_dy_spot_g4m_clean
  left: ev_dy_spot_g4m_clean
  right: ev_dy_none_g4m_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 26
- id: c_dy_spot_g4m_static
  left: ev_dy_spot_g4m_static
  right: ev_dy_none_g4m_static
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 26
- id: c_dy_spot_g4m_auto
  left: ev_dy_spot_g4m_auto
  right: ev_dy_none_g4m_auto
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 26
- id: c_dy_rem_gem_clean
  left: ev_dy_rem_gem_clean
  right: ev_dy_none_gem_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 26
- id: c_dy_rem_gem_static
  left: ev_dy_rem_gem_static
  right: ev_dy_none_gem_static
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 26
- id: c_dy_rem_gem_auto
  left: ev_dy_rem_gem_auto
  right: ev_dy_none_gem_auto
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 26
- id: c_dy_spot_gem_clean
  left: ev_dy_spot_gem_clean
  right: ev_dy_none_gem_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 26
- id: c_dy_spot_gem_static
  left: ev_dy_spot_gem_static
  right: ev_dy_none_gem_static
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 26
- id: c_dy_spot_gem_auto
  left: ev_dy_spot_gem_auto
  right: ev_dy_none_gem_auto
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 26
claims:
- id: k1
  text: static benchmarks often overestimate defence efficacy, and most defences either lose utility or are insecure
  location: abstract
  anchor: standard static benchmarks often significantly overestimate de- fense efficacy
  jobs:
  - R1
  - R2
  supported_by:
  - c_ad_rem_g4m_auto
  - c_ad_spot_g4m_auto
  - c_ad_rem_g54m_auto
  - c_ad_spot_g54m_auto
  - c_ad_rem_gem_auto
  - c_ad_spot_gem_auto
  - c_ad_rem_ds_auto
  - c_ad_spot_ds_auto
  - c_ad_rem_hk_auto
  - c_ad_spot_hk_auto
  - c_dy_rem_g4m_auto
  - c_dy_spot_g4m_auto
  - c_dy_rem_gem_auto
  - c_dy_spot_gem_auto
  page: 1
coverage:
  R1: direct
  R2clean: direct
  R2attack: direct
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Tables 5 and 8 (static and AutoDojo ASR)
    R2clean: Tables 5 and 8 (clean utility)
    R2attack: Tables 5 and 8 (utility under attack, in parentheses)

`reta_arxiv2026` (2606.15441)

In [ ]:
%%writefile reports/records/reta_arxiv2026.yaml
report:
  id: reta_arxiv2026
  slug: '2606.15441'
  title: Defending against Adaptive Prompt Injection Attacks via Reasoning-enabled Task Alignment
  first_author: Lipeng He
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2606.15441'
  version_read: arXiv v1
  first_date: '2026-06-13'
  read: F
  role: proposes
  code_release: not checked
  extraction_scope: 'Section 5: Table 2 (AgentDojo averages over six adaptive attacks and five held-out static attacks, and clean BU) for None, Spotlighting, Sandwiching, MetaSecAlign and RETA on both models; the abstract
    claim (PromptGuard, PIGuard and DataSentinel, which are detectors, and MELON, a tool filter, are left out; per-attack Tables 7 and 8, Table 3 transfer, the ablations of Tables 1 and 4 and the cost analysis not coded)'
defences:
- id: reta_q
  name: RETA
  origin: this_report
  base_checkpoint: Qwen3-4B-Instruct-2507
  axis1:
  - no_signal
  axis2:
  - sft
  - preference_optimisation
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 1
  components:
  - at each tool-output step the defender reasons whether its next action serves the user task
  - 'Stage I: an attacker model trained by RL with a dictionary-learning diversity reward writes injections against the undefended target'
  - 'Stage II: format SFT, then multi-objective RL (GDPO) on the successful Stage I attacks with security, utility and reasoning rewards'
  - training on AgentDyn suites only, disjoint from the AgentDojo test suites
  layout: AgentDojo agent loop; tool outputs carry the injection
  layout_compat: &id001
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: the defender undertakes chain-of-thought reasoning verifying
  pages:
    components: 10
- id: spot_q
  name: Spotlighting
  origin: other_report
  base_checkpoint: Qwen3-4B-Instruct-2507
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  layout: AgentDojo agent loop; tool outputs carry the injection
  layout_compat: *id001
  anchor: Spotlighting uses provenance-signaling transformations
- id: sand_q
  name: Prompt Sandwiching
  origin: other_report
  base_checkpoint: Qwen3-4B-Instruct-2507
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  layout: AgentDojo agent loop; tool outputs carry the injection
  layout_compat: *id001
  anchor: Prompt Sandwiching wraps the observation
- id: msa_q
  name: MetaSecAlign (how it was obtained for Qwen3-4B is not stated)
  origin: other_report
  base_checkpoint: Qwen3-4B-Instruct-2507
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 3
  anchor: MetaSecAlign is the released successor in the SecAlign family
- id: reta_l
  name: RETA
  origin: this_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - no_signal
  axis2:
  - sft
  - preference_optimisation
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 1
  components:
  - at each tool-output step the defender reasons whether its next action serves the user task
  - 'Stage I: an attacker model trained by RL with a dictionary-learning diversity reward writes injections against the undefended target'
  - 'Stage II: format SFT, then multi-objective RL (GDPO) on the successful Stage I attacks with security, utility and reasoning rewards'
  - training on AgentDyn suites only, disjoint from the AgentDojo test suites
  layout: AgentDojo agent loop; tool outputs carry the injection
  layout_compat: *id001
  anchor: the defender undertakes chain-of-thought reasoning verifying
  pages:
    components: 10
- id: spot_l
  name: Spotlighting
  origin: other_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  layout: AgentDojo agent loop; tool outputs carry the injection
  layout_compat: *id001
  anchor: Spotlighting uses provenance-signaling transformations
- id: sand_l
  name: Prompt Sandwiching
  origin: other_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  layout: AgentDojo agent loop; tool outputs carry the injection
  layout_compat: *id001
  anchor: Prompt Sandwiching wraps the observation
- id: msa_l
  name: MetaSecAlign
  origin: other_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 3
  anchor: MetaSecAlign is the released successor in the SecAlign family
evaluations:
- id: ev_q_none_ad
  defence_id: none:Qwen3-4B-Instruct-2507
  benchmark: AgentDojo (Slack, Banking, Workspace, Travel), 40-case vulnerable subset per target; mean over TAP, Strategy, Genetic Search, AutoInject, RL-Hammer and PISmith
  attacker_type: adaptive
  attacker_access: black_box
  judge: AgentDojo task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: reta_arxiv2026
  page: 11
  decoding: temperature 0
  sample_size: '40'
  pages:
    benchmark: 10
    decoding: 10
  baseline_kind: off_the_shelf
- id: ev_q_none_st
  defence_id: none:Qwen3-4B-Instruct-2507
  benchmark: AgentDojo, 420 cases, mean over five held-out static attack families (IgnorePrev., InjecAgent, ImportantMsgs., ToolKnow., ChatInject)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: reta_arxiv2026
  page: 11
  decoding: temperature 0
  sample_size: '420'
  pages:
    benchmark: 10
    decoding: 10
  baseline_kind: off_the_shelf
- id: ev_q_none_bu
  defence_id: none:Qwen3-4B-Instruct-2507
  benchmark: AgentDojo, 97 clean utility cases
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: reta_arxiv2026
  page: 11
  decoding: temperature 0
  sample_size: '97'
  pages:
    decoding: 10
  baseline_kind: off_the_shelf
- id: ev_q_spot_ad
  defence_id: spot_q
  benchmark: AgentDojo (Slack, Banking, Workspace, Travel), 40-case vulnerable subset per target; mean over TAP, Strategy, Genetic Search, AutoInject, RL-Hammer and PISmith
  attacker_type: adaptive
  attacker_access: black_box
  judge: AgentDojo task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: reta_arxiv2026
  page: 11
  decoding: temperature 0
  sample_size: '40'
  pages:
    benchmark: 10
    decoding: 10
- id: ev_q_spot_st
  defence_id: spot_q
  benchmark: AgentDojo, 420 cases, mean over five held-out static attack families (IgnorePrev., InjecAgent, ImportantMsgs., ToolKnow., ChatInject)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: reta_arxiv2026
  page: 11
  decoding: temperature 0
  sample_size: '420'
  pages:
    benchmark: 10
    decoding: 10
- id: ev_q_spot_bu
  defence_id: spot_q
  benchmark: AgentDojo, 97 clean utility cases
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: reta_arxiv2026
  page: 11
  decoding: temperature 0
  sample_size: '97'
  pages:
    decoding: 10
- id: ev_q_sand_ad
  defence_id: sand_q
  benchmark: AgentDojo (Slack, Banking, Workspace, Travel), 40-case vulnerable subset per target; mean over TAP, Strategy, Genetic Search, AutoInject, RL-Hammer and PISmith
  attacker_type: adaptive
  attacker_access: black_box
  judge: AgentDojo task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: reta_arxiv2026
  page: 11
  decoding: temperature 0
  sample_size: '40'
  pages:
    benchmark: 10
    decoding: 10
- id: ev_q_sand_st
  defence_id: sand_q
  benchmark: AgentDojo, 420 cases, mean over five held-out static attack families (IgnorePrev., InjecAgent, ImportantMsgs., ToolKnow., ChatInject)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: reta_arxiv2026
  page: 11
  decoding: temperature 0
  sample_size: '420'
  pages:
    benchmark: 10
    decoding: 10
- id: ev_q_sand_bu
  defence_id: sand_q
  benchmark: AgentDojo, 97 clean utility cases
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: reta_arxiv2026
  page: 11
  decoding: temperature 0
  sample_size: '97'
  pages:
    decoding: 10
- id: ev_q_msa_ad
  defence_id: msa_q
  benchmark: AgentDojo (Slack, Banking, Workspace, Travel), 40-case vulnerable subset per target; mean over TAP, Strategy, Genetic Search, AutoInject, RL-Hammer and PISmith
  attacker_type: adaptive
  attacker_access: black_box
  judge: AgentDojo task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: reta_arxiv2026
  page: 11
  decoding: temperature 0
  sample_size: '40'
  pages:
    benchmark: 10
    decoding: 10
- id: ev_q_msa_st
  defence_id: msa_q
  benchmark: AgentDojo, 420 cases, mean over five held-out static attack families (IgnorePrev., InjecAgent, ImportantMsgs., ToolKnow., ChatInject)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: reta_arxiv2026
  page: 11
  decoding: temperature 0
  sample_size: '420'
  pages:
    benchmark: 10
    decoding: 10
- id: ev_q_msa_bu
  defence_id: msa_q
  benchmark: AgentDojo, 97 clean utility cases
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: reta_arxiv2026
  page: 11
  decoding: temperature 0
  sample_size: '97'
  pages:
    decoding: 10
- id: ev_q_reta_ad
  defence_id: reta_q
  benchmark: AgentDojo (Slack, Banking, Workspace, Travel), 40-case vulnerable subset per target; mean over TAP, Strategy, Genetic Search, AutoInject, RL-Hammer and PISmith
  attacker_type: adaptive
  attacker_access: black_box
  judge: AgentDojo task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: reta_arxiv2026
  page: 11
  decoding: temperature 0
  sample_size: '40'
  pages:
    benchmark: 10
    decoding: 10
- id: ev_q_reta_st
  defence_id: reta_q
  benchmark: AgentDojo, 420 cases, mean over five held-out static attack families (IgnorePrev., InjecAgent, ImportantMsgs., ToolKnow., ChatInject)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: reta_arxiv2026
  page: 11
  decoding: temperature 0
  sample_size: '420'
  pages:
    benchmark: 10
    decoding: 10
- id: ev_q_reta_bu
  defence_id: reta_q
  benchmark: AgentDojo, 97 clean utility cases
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: reta_arxiv2026
  page: 11
  decoding: temperature 0
  sample_size: '97'
  pages:
    decoding: 10
- id: ev_l_none_ad
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: AgentDojo (Slack, Banking, Workspace, Travel), 40-case vulnerable subset per target; mean over TAP, Strategy, Genetic Search, AutoInject, RL-Hammer and PISmith
  attacker_type: adaptive
  attacker_access: black_box
  judge: AgentDojo task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: reta_arxiv2026
  page: 11
  decoding: temperature 0
  sample_size: '40'
  pages:
    benchmark: 10
    decoding: 10
  baseline_kind: off_the_shelf
- id: ev_l_none_st
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: AgentDojo, 420 cases, mean over five held-out static attack families (IgnorePrev., InjecAgent, ImportantMsgs., ToolKnow., ChatInject)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: reta_arxiv2026
  page: 11
  decoding: temperature 0
  sample_size: '420'
  pages:
    benchmark: 10
    decoding: 10
  baseline_kind: off_the_shelf
- id: ev_l_none_bu
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: AgentDojo, 97 clean utility cases
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: reta_arxiv2026
  page: 11
  decoding: temperature 0
  sample_size: '97'
  pages:
    decoding: 10
  baseline_kind: off_the_shelf
- id: ev_l_spot_ad
  defence_id: spot_l
  benchmark: AgentDojo (Slack, Banking, Workspace, Travel), 40-case vulnerable subset per target; mean over TAP, Strategy, Genetic Search, AutoInject, RL-Hammer and PISmith
  attacker_type: adaptive
  attacker_access: black_box
  judge: AgentDojo task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: reta_arxiv2026
  page: 11
  decoding: temperature 0
  sample_size: '40'
  pages:
    benchmark: 10
    decoding: 10
- id: ev_l_spot_st
  defence_id: spot_l
  benchmark: AgentDojo, 420 cases, mean over five held-out static attack families (IgnorePrev., InjecAgent, ImportantMsgs., ToolKnow., ChatInject)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: reta_arxiv2026
  page: 11
  decoding: temperature 0
  sample_size: '420'
  pages:
    benchmark: 10
    decoding: 10
- id: ev_l_spot_bu
  defence_id: spot_l
  benchmark: AgentDojo, 97 clean utility cases
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: reta_arxiv2026
  page: 11
  decoding: temperature 0
  sample_size: '97'
  pages:
    decoding: 10
- id: ev_l_sand_ad
  defence_id: sand_l
  benchmark: AgentDojo (Slack, Banking, Workspace, Travel), 40-case vulnerable subset per target; mean over TAP, Strategy, Genetic Search, AutoInject, RL-Hammer and PISmith
  attacker_type: adaptive
  attacker_access: black_box
  judge: AgentDojo task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: reta_arxiv2026
  page: 11
  decoding: temperature 0
  sample_size: '40'
  pages:
    benchmark: 10
    decoding: 10
- id: ev_l_sand_st
  defence_id: sand_l
  benchmark: AgentDojo, 420 cases, mean over five held-out static attack families (IgnorePrev., InjecAgent, ImportantMsgs., ToolKnow., ChatInject)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: reta_arxiv2026
  page: 11
  decoding: temperature 0
  sample_size: '420'
  pages:
    benchmark: 10
    decoding: 10
- id: ev_l_sand_bu
  defence_id: sand_l
  benchmark: AgentDojo, 97 clean utility cases
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: reta_arxiv2026
  page: 11
  decoding: temperature 0
  sample_size: '97'
  pages:
    decoding: 10
- id: ev_l_msa_ad
  defence_id: msa_l
  benchmark: AgentDojo (Slack, Banking, Workspace, Travel), 40-case vulnerable subset per target; mean over TAP, Strategy, Genetic Search, AutoInject, RL-Hammer and PISmith
  attacker_type: adaptive
  attacker_access: black_box
  judge: AgentDojo task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: reta_arxiv2026
  page: 11
  decoding: temperature 0
  sample_size: '40'
  pages:
    benchmark: 10
    decoding: 10
- id: ev_l_msa_st
  defence_id: msa_l
  benchmark: AgentDojo, 420 cases, mean over five held-out static attack families (IgnorePrev., InjecAgent, ImportantMsgs., ToolKnow., ChatInject)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: reta_arxiv2026
  page: 11
  decoding: temperature 0
  sample_size: '420'
  pages:
    benchmark: 10
    decoding: 10
- id: ev_l_msa_bu
  defence_id: msa_l
  benchmark: AgentDojo, 97 clean utility cases
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: reta_arxiv2026
  page: 11
  decoding: temperature 0
  sample_size: '97'
  pages:
    decoding: 10
- id: ev_l_reta_ad
  defence_id: reta_l
  benchmark: AgentDojo (Slack, Banking, Workspace, Travel), 40-case vulnerable subset per target; mean over TAP, Strategy, Genetic Search, AutoInject, RL-Hammer and PISmith
  attacker_type: adaptive
  attacker_access: black_box
  judge: AgentDojo task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: reta_arxiv2026
  page: 11
  decoding: temperature 0
  sample_size: '40'
  pages:
    benchmark: 10
    decoding: 10
- id: ev_l_reta_st
  defence_id: reta_l
  benchmark: AgentDojo, 420 cases, mean over five held-out static attack families (IgnorePrev., InjecAgent, ImportantMsgs., ToolKnow., ChatInject)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: reta_arxiv2026
  page: 11
  decoding: temperature 0
  sample_size: '420'
  pages:
    benchmark: 10
    decoding: 10
- id: ev_l_reta_bu
  defence_id: reta_l
  benchmark: AgentDojo, 97 clean utility cases
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: reta_arxiv2026
  page: 11
  decoding: temperature 0
  sample_size: '97'
  pages:
    decoding: 10
results:
- id: r_q_none_aa
  evaluation_id: ev_q_none_ad
  quantity: R1
  coverage: direct
  metric_as_printed: Adaptive ASR (average of six attacks)
  value: '32.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_q_none_aua
  evaluation_id: ev_q_none_ad
  quantity: R2attack
  coverage: direct
  metric_as_printed: Adaptive UA (average)
  value: '57.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_q_none_sa
  evaluation_id: ev_q_none_st
  quantity: R1
  coverage: direct
  metric_as_printed: Static ASR (average)
  value: '33.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_q_none_sua
  evaluation_id: ev_q_none_st
  quantity: R2attack
  coverage: direct
  metric_as_printed: Static UA (average)
  value: '44.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_q_none_bu
  evaluation_id: ev_q_none_bu
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '61.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_q_spot_aa
  evaluation_id: ev_q_spot_ad
  quantity: R1
  coverage: direct
  metric_as_printed: Adaptive ASR (average of six attacks)
  value: '28.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_q_spot_aua
  evaluation_id: ev_q_spot_ad
  quantity: R2attack
  coverage: direct
  metric_as_printed: Adaptive UA (average)
  value: '62.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_q_spot_sa
  evaluation_id: ev_q_spot_st
  quantity: R1
  coverage: direct
  metric_as_printed: Static ASR (average)
  value: '29.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_q_spot_sua
  evaluation_id: ev_q_spot_st
  quantity: R2attack
  coverage: direct
  metric_as_printed: Static UA (average)
  value: '43.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_q_spot_bu
  evaluation_id: ev_q_spot_bu
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '57.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_q_sand_aa
  evaluation_id: ev_q_sand_ad
  quantity: R1
  coverage: direct
  metric_as_printed: Adaptive ASR (average of six attacks)
  value: '16.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_q_sand_aua
  evaluation_id: ev_q_sand_ad
  quantity: R2attack
  coverage: direct
  metric_as_printed: Adaptive UA (average)
  value: '65.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_q_sand_sa
  evaluation_id: ev_q_sand_st
  quantity: R1
  coverage: direct
  metric_as_printed: Static ASR (average)
  value: '17.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_q_sand_sua
  evaluation_id: ev_q_sand_st
  quantity: R2attack
  coverage: direct
  metric_as_printed: Static UA (average)
  value: '49.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_q_sand_bu
  evaluation_id: ev_q_sand_bu
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '60.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_q_msa_aa
  evaluation_id: ev_q_msa_ad
  quantity: R1
  coverage: direct
  metric_as_printed: Adaptive ASR (average of six attacks)
  value: '12.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_q_msa_aua
  evaluation_id: ev_q_msa_ad
  quantity: R2attack
  coverage: direct
  metric_as_printed: Adaptive UA (average)
  value: '69.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_q_msa_sa
  evaluation_id: ev_q_msa_st
  quantity: R1
  coverage: direct
  metric_as_printed: Static ASR (average)
  value: '3.52'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_q_msa_sua
  evaluation_id: ev_q_msa_st
  quantity: R2attack
  coverage: direct
  metric_as_printed: Static UA (average)
  value: '58.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_q_msa_bu
  evaluation_id: ev_q_msa_bu
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '64.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_q_reta_aa
  evaluation_id: ev_q_reta_ad
  quantity: R1
  coverage: direct
  metric_as_printed: Adaptive ASR (average of six attacks)
  value: '2.92'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_q_reta_aua
  evaluation_id: ev_q_reta_ad
  quantity: R2attack
  coverage: direct
  metric_as_printed: Adaptive UA (average)
  value: '55.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_q_reta_sa
  evaluation_id: ev_q_reta_st
  quantity: R1
  coverage: direct
  metric_as_printed: Static ASR (average)
  value: '2.90'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_q_reta_sua
  evaluation_id: ev_q_reta_st
  quantity: R2attack
  coverage: direct
  metric_as_printed: Static UA (average)
  value: '59.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_q_reta_bu
  evaluation_id: ev_q_reta_bu
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '60.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_l_none_aa
  evaluation_id: ev_l_none_ad
  quantity: R1
  coverage: direct
  metric_as_printed: Adaptive ASR (average of six attacks)
  value: '14.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_l_none_aua
  evaluation_id: ev_l_none_ad
  quantity: R2attack
  coverage: direct
  metric_as_printed: Adaptive UA (average)
  value: '47.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_l_none_sa
  evaluation_id: ev_l_none_st
  quantity: R1
  coverage: direct
  metric_as_printed: Static ASR (average)
  value: '13.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_l_none_sua
  evaluation_id: ev_l_none_st
  quantity: R2attack
  coverage: direct
  metric_as_printed: Static UA (average)
  value: '33.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_l_none_bu
  evaluation_id: ev_l_none_bu
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '48.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_l_spot_aa
  evaluation_id: ev_l_spot_ad
  quantity: R1
  coverage: direct
  metric_as_printed: Adaptive ASR (average of six attacks)
  value: '12.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_l_spot_aua
  evaluation_id: ev_l_spot_ad
  quantity: R2attack
  coverage: direct
  metric_as_printed: Adaptive UA (average)
  value: '39.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_l_spot_sa
  evaluation_id: ev_l_spot_st
  quantity: R1
  coverage: direct
  metric_as_printed: Static ASR (average)
  value: '11.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_l_spot_sua
  evaluation_id: ev_l_spot_st
  quantity: R2attack
  coverage: direct
  metric_as_printed: Static UA (average)
  value: '28.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_l_spot_bu
  evaluation_id: ev_l_spot_bu
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '44.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_l_sand_aa
  evaluation_id: ev_l_sand_ad
  quantity: R1
  coverage: direct
  metric_as_printed: Adaptive ASR (average of six attacks)
  value: '7.08'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_l_sand_aua
  evaluation_id: ev_l_sand_ad
  quantity: R2attack
  coverage: direct
  metric_as_printed: Adaptive UA (average)
  value: '40.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_l_sand_sa
  evaluation_id: ev_l_sand_st
  quantity: R1
  coverage: direct
  metric_as_printed: Static ASR (average)
  value: '6.57'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_l_sand_sua
  evaluation_id: ev_l_sand_st
  quantity: R2attack
  coverage: direct
  metric_as_printed: Static UA (average)
  value: '34.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_l_sand_bu
  evaluation_id: ev_l_sand_bu
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '46.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_l_msa_aa
  evaluation_id: ev_l_msa_ad
  quantity: R1
  coverage: direct
  metric_as_printed: Adaptive ASR (average of six attacks)
  value: '5.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_l_msa_aua
  evaluation_id: ev_l_msa_ad
  quantity: R2attack
  coverage: direct
  metric_as_printed: Adaptive UA (average)
  value: '47.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_l_msa_sa
  evaluation_id: ev_l_msa_st
  quantity: R1
  coverage: direct
  metric_as_printed: Static ASR (average)
  value: '2.67'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_l_msa_sua
  evaluation_id: ev_l_msa_st
  quantity: R2attack
  coverage: direct
  metric_as_printed: Static UA (average)
  value: '35.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_l_msa_bu
  evaluation_id: ev_l_msa_bu
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '46.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_l_reta_aa
  evaluation_id: ev_l_reta_ad
  quantity: R1
  coverage: direct
  metric_as_printed: Adaptive ASR (average of six attacks)
  value: '3.75'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_l_reta_aua
  evaluation_id: ev_l_reta_ad
  quantity: R2attack
  coverage: direct
  metric_as_printed: Adaptive UA (average)
  value: '49.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_l_reta_sa
  evaluation_id: ev_l_reta_st
  quantity: R1
  coverage: direct
  metric_as_printed: Static ASR (average)
  value: '1.76'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_l_reta_sua
  evaluation_id: ev_l_reta_st
  quantity: R2attack
  coverage: direct
  metric_as_printed: Static UA (average)
  value: '36.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
- id: r_l_reta_bu
  evaluation_id: ev_l_reta_bu
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '45.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 2
comparisons:
- id: c_q_spot_ad
  left: ev_q_spot_ad
  right: ev_q_none_ad
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_q_spot_st
  left: ev_q_spot_st
  right: ev_q_none_st
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_q_spot_bu
  left: ev_q_spot_bu
  right: ev_q_none_bu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_q_sand_ad
  left: ev_q_sand_ad
  right: ev_q_none_ad
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_q_sand_st
  left: ev_q_sand_st
  right: ev_q_none_st
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_q_sand_bu
  left: ev_q_sand_bu
  right: ev_q_none_bu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_q_msa_ad
  left: ev_q_msa_ad
  right: ev_q_none_ad
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_q_msa_st
  left: ev_q_msa_st
  right: ev_q_none_st
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_q_msa_bu
  left: ev_q_msa_bu
  right: ev_q_none_bu
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_q_reta_ad
  left: ev_q_reta_ad
  right: ev_q_none_ad
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_q_reta_st
  left: ev_q_reta_st
  right: ev_q_none_st
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_q_reta_bu
  left: ev_q_reta_bu
  right: ev_q_none_bu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_q_reta_msa_ad
  left: ev_q_reta_ad
  right: ev_q_msa_ad
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_q_reta_msa_st
  left: ev_q_reta_st
  right: ev_q_msa_st
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_q_reta_msa_bu
  left: ev_q_reta_bu
  right: ev_q_msa_bu
  purpose: W
  c1_base: matched_unstated
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_l_spot_ad
  left: ev_l_spot_ad
  right: ev_l_none_ad
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_l_spot_st
  left: ev_l_spot_st
  right: ev_l_none_st
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_l_spot_bu
  left: ev_l_spot_bu
  right: ev_l_none_bu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_l_sand_ad
  left: ev_l_sand_ad
  right: ev_l_none_ad
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_l_sand_st
  left: ev_l_sand_st
  right: ev_l_none_st
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_l_sand_bu
  left: ev_l_sand_bu
  right: ev_l_none_bu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_l_msa_ad
  left: ev_l_msa_ad
  right: ev_l_none_ad
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_l_msa_st
  left: ev_l_msa_st
  right: ev_l_none_st
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_l_msa_bu
  left: ev_l_msa_bu
  right: ev_l_none_bu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_l_reta_ad
  left: ev_l_reta_ad
  right: ev_l_none_ad
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_l_reta_st
  left: ev_l_reta_st
  right: ev_l_none_st
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_l_reta_bu
  left: ev_l_reta_bu
  right: ev_l_none_bu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_l_reta_msa_ad
  left: ev_l_reta_ad
  right: ev_l_msa_ad
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_l_reta_msa_st
  left: ev_l_reta_st
  right: ev_l_msa_st
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_l_reta_msa_bu
  left: ev_l_reta_bu
  right: ev_l_msa_bu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
claims:
- id: k1
  text: RETA keeps every per-attack adaptive ASR below 10% and averages 2.92% and 3.75% while preserving most utility
  location: abstract
  anchor: keeps every per-attack ASR below10%, with average ASR of 2.92%
  jobs:
  - R1
  - R2
  supported_by:
  - c_q_reta_ad
  - c_l_reta_ad
  - c_q_reta_msa_ad
  - c_l_reta_msa_ad
  - c_q_reta_bu
  - c_l_reta_bu
  page: 1
coverage:
  R1: direct
  R2clean: direct
  R2attack: direct
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 2 (averages; per-attack values in Tables 7 and 8)
    R2clean: Table 2 (BU)
    R2attack: Table 2 (UA)
schema_gaps:
- where: defences reta_*
  note: RETA's defender is trained by RL on attacks found beforehand by a trained attacker, not co-evolved online, so axis 2 is preference_optimisation (reward-based RL, as for VerIH) rather than adversarial_rl
- where: defences spot_*
  note: the paper describes Spotlighting as provenance-signalling transformations without naming the variant; coded delimiters_markers as in the other records
- where: defence msa_q
  note: the paper does not say how MetaSecAlign was obtained for Qwen3-4B (the released models are Llama-based); the base condition is matched_unstated

`ihdec_arxiv2026` (2606.29960)

In [ ]:
%%writefile reports/records/ihdec_arxiv2026.yaml
report:
  id: ihdec_arxiv2026
  slug: '2606.29960'
  title: 'IHDec: Divergence-Steered Contrastive Decoding for Securing Multi-Turn Instruction Hierarchies'
  first_author: Nicole Geumheon Liu
  venue: arXiv preprint
  peer_reviewed: 'no'
  doi: 10.18653/2026.findings-emnlp.1751
  arxiv: '2606.29960'
  version_read: arXiv v2
  first_date: '2026-06-29'
  read: F
  role: proposes
  code_release: https://github.com/nxcolelxu/IHDec.git (linked under the title)
  later_version: Findings of EMNLP 2026 (from the search record; not read)
  extraction_scope: 'Section 5 and Appendix E: Table 4 tool-use subsets (instructions arriving in tool returns) and the MT-Bench-101 averages of Figure 4; the abstract claim (Table 1 and Table 2, whose conflicts are between
    system and user messages, the Safety subsets of Table 4, Table 3 scaling and the ablations not coded)'
defences:
- id: ihdec_l31
  name: IHDec
  origin: this_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - chat_roles
  axis2:
  - fixed_inference_rule
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  components:
  - Jensen-Shannon divergence between output distributions with and without each role to find lower roles that dominate higher ones
  - 'contrastive decoding: logits shifted by the difference between the distribution without the conflicting roles and without the higher role'
  layout: model chat template; tool returns in the ipython role
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: we employ a dynamic contrastive decoding mechanism
  pages:
    layout: 14
    layout_compat: 14
evaluations:
- id: ev_webpage_none
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: IHEval get_webpage conflict subset extended to five turns (740 dialogues), conflicting instruction in the tool return
  attacker_type: static
  attacker_access: black_box
  judge: IHEval compliance check (follows the higher-priority instruction)
  layout_same_as_defence: 'yes'
  measured_by: ihdec_arxiv2026
  page: 14
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_webpage_ihdec
  defence_id: ihdec_l31
  benchmark: IHEval get_webpage conflict subset extended to five turns (740 dialogues), conflicting instruction in the tool return
  attacker_type: static
  attacker_access: black_box
  judge: IHEval compliance check (follows the higher-priority instruction)
  layout_same_as_defence: 'yes'
  measured_by: ihdec_arxiv2026
  page: 14
  decoding: not reported
- id: ev_slack_none
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: IHEval slack conflict subset, conflicting instruction in the tool return
  attacker_type: static
  attacker_access: black_box
  judge: IHEval compliance check (follows the higher-priority instruction)
  layout_same_as_defence: 'yes'
  measured_by: ihdec_arxiv2026
  page: 14
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_slack_ihdec
  defence_id: ihdec_l31
  benchmark: IHEval slack conflict subset, conflicting instruction in the tool return
  attacker_type: static
  attacker_access: black_box
  judge: IHEval compliance check (follows the higher-priority instruction)
  layout_same_as_defence: 'yes'
  measured_by: ihdec_arxiv2026
  page: 14
  decoding: not reported
- id: ev_mtb_none
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: MT-Bench-101
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o judge
  layout_same_as_defence: 'yes'
  measured_by: ihdec_arxiv2026
  page: 7
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_mtb_ihdec
  defence_id: ihdec_l31
  benchmark: MT-Bench-101
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o judge
  layout_same_as_defence: 'yes'
  measured_by: ihdec_arxiv2026
  page: 7
  decoding: not reported
results:
- id: r_webpage_none
  evaluation_id: ev_webpage_none
  quantity: R1
  coverage: proxy
  metric_as_printed: instruction compliance accuracy under conflict (webpage)
  value: '1.95'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 14
  location: Table 4
- id: r_webpage_ihdec
  evaluation_id: ev_webpage_ihdec
  quantity: R1
  coverage: proxy
  metric_as_printed: instruction compliance accuracy under conflict (webpage)
  value: '12.45'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 14
  location: Table 4
- id: r_slack_none
  evaluation_id: ev_slack_none
  quantity: R1
  coverage: proxy
  metric_as_printed: instruction compliance accuracy under conflict (slack)
  value: '11.50'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 14
  location: Table 4
- id: r_slack_ihdec
  evaluation_id: ev_slack_ihdec
  quantity: R1
  coverage: proxy
  metric_as_printed: instruction compliance accuracy under conflict (slack)
  value: '34.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 14
  location: Table 4
- id: r_mtb_none
  evaluation_id: ev_mtb_none
  quantity: general_utility
  coverage: direct
  metric_as_printed: MT-Bench-101 overall average
  value: '9.29'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: text
  page: 7
  location: Figure 4 and caption
- id: r_mtb_ihdec
  evaluation_id: ev_mtb_ihdec
  quantity: general_utility
  coverage: direct
  metric_as_printed: MT-Bench-101 overall average
  value: '9.21'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: text
  page: 7
  location: Figure 4 and caption
comparisons:
- id: c_webpage
  left: ev_webpage_ihdec
  right: ev_webpage_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_slack
  left: ev_slack_ihdec
  right: ev_slack_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_mtb
  left: ev_mtb_ihdec
  right: ev_mtb_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
claims:
- id: k1
  text: IHDec strengthens safety against adversarial prompt injection while preserving general response quality
  location: abstract
  anchor: IHDec strengthens safety against adversarial prompt injections
  jobs:
  - R1
  - general_utility
  supported_by:
  - c_webpage
  - c_slack
  - c_mtb
  page: 1
coverage:
  R1: proxy
  R2clean: not_reported
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 4 tool-use subsets (compliance with the user instruction when a tool return conflicts)
    general_utility: Figure 4 (MT-Bench-101)
schema_gaps:
- where: evaluations ev_webpage_*, ev_slack_*
  note: the tool-use subsets score compliance with the higher-priority instruction under conflict, not whether an injected goal was executed, so they are an R1 proxy; IHDec's beta was set separately for these subsets (800
    for Tool-Use, p. 14)

`copa_arxiv2026` (2608.19982)

In [ ]:
%%writefile reports/records/copa_arxiv2026.yaml
report:
  id: copa_arxiv2026
  slug: '2608.19982'
  title: 'COPA: Continual Preference Optimization for Adaptive Prompt Injection Defense'
  first_author: Roshan Sood
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2608.19982'
  version_read: arXiv v1
  first_date: '2026-08-20'
  read: F
  role: proposes
  code_release: none found in the paper
  extraction_scope: 'whole report (7 pages): Figure 1 (ASR in scenarios S1 to S3 for no defence, SecAlign and COPA), the ASR stated in Section IV-B for SecAlign and COPA over the lifelong sequence, and the GPQA and MMLU accuracies
    of Figure 4 and Section IV-B; the abstract claim (LlamaGuard and DataSentinel, which are detectors, are left out; the other backbones of Figure 5, backward transfer and average performance for the baselines, and the ablations
    of Tables I and II not coded)'
defences:
- id: copa_l31
  name: COPA
  origin: this_report
  base_checkpoint: Meta-Llama-3.1-8B-Instruct
  axis1:
  - no_signal
  axis2:
  - preference_optimisation
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 3
  components:
  - 'Stage 1: LoRA adapter trained by GRPO with a binary safety judge on DeepTeam cybersecurity preference pairs'
  - 'Stage 2: GRPO on nine CyberSecEval injection variants in sequence, with a replay buffer that favours pairs with a low preference margin'
  anchor: Stage 1 mitigates this issue by aligning all backbones to a shared
  pages:
    components: 4
- id: secalign_l31
  name: SecAlign (run inside COPA's lifelong pipeline)
  origin: other_report
  base_checkpoint: Meta-Llama-3.1-8B-Instruct
  axis1:
  - no_signal
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  anchor: SecAlign [7] performs defensive
evaluations:
- id: ev_s1_none
  defence_id: none:Meta-Llama-3.1-8B-Instruct
  benchmark: 'S1: six held-out CyberSecEval variants'
  attacker_type: static
  attacker_access: black_box
  judge: attack bypasses the defence (safety judge)
  layout_same_as_defence: 'yes'
  measured_by: copa_arxiv2026
  page: 1
  decoding: not reported
  pages:
    benchmark: 5
  baseline_kind: off_the_shelf
- id: ev_s1_sa
  defence_id: secalign_l31
  benchmark: 'S1: six held-out CyberSecEval variants'
  attacker_type: static
  attacker_access: black_box
  judge: attack bypasses the defence (safety judge)
  layout_same_as_defence: 'yes'
  measured_by: copa_arxiv2026
  page: 1
  decoding: not reported
  pages:
    benchmark: 5
- id: ev_s1_copa
  defence_id: copa_l31
  benchmark: 'S1: six held-out CyberSecEval variants'
  attacker_type: static
  attacker_access: black_box
  judge: attack bypasses the defence (safety judge)
  layout_same_as_defence: 'yes'
  measured_by: copa_arxiv2026
  page: 1
  decoding: not reported
  pages:
    benchmark: 5
- id: ev_s2_none
  defence_id: none:Meta-Llama-3.1-8B-Instruct
  benchmark: 'S2: text-based injections (direct, ignore, completion, hybrid)'
  attacker_type: static
  attacker_access: black_box
  judge: attack bypasses the defence (safety judge)
  layout_same_as_defence: 'yes'
  measured_by: copa_arxiv2026
  page: 1
  decoding: not reported
  pages:
    benchmark: 5
  baseline_kind: off_the_shelf
- id: ev_s2_sa
  defence_id: secalign_l31
  benchmark: 'S2: text-based injections (direct, ignore, completion, hybrid)'
  attacker_type: static
  attacker_access: black_box
  judge: attack bypasses the defence (safety judge)
  layout_same_as_defence: 'yes'
  measured_by: copa_arxiv2026
  page: 1
  decoding: not reported
  pages:
    benchmark: 5
- id: ev_s2_copa
  defence_id: copa_l31
  benchmark: 'S2: text-based injections (direct, ignore, completion, hybrid)'
  attacker_type: static
  attacker_access: black_box
  judge: attack bypasses the defence (safety judge)
  layout_same_as_defence: 'yes'
  measured_by: copa_arxiv2026
  page: 1
  decoding: not reported
  pages:
    benchmark: 5
- id: ev_s3_none
  defence_id: none:Meta-Llama-3.1-8B-Instruct
  benchmark: 'S3: optimisation-based injections (UAT, GCG)'
  attacker_type: optimisation
  attacker_access: white_box
  judge: attack bypasses the defence (safety judge)
  layout_same_as_defence: 'yes'
  measured_by: copa_arxiv2026
  page: 1
  decoding: not reported
  pages:
    benchmark: 5
  baseline_kind: off_the_shelf
- id: ev_s3_sa
  defence_id: secalign_l31
  benchmark: 'S3: optimisation-based injections (UAT, GCG)'
  attacker_type: optimisation
  attacker_access: white_box
  judge: attack bypasses the defence (safety judge)
  layout_same_as_defence: 'yes'
  measured_by: copa_arxiv2026
  page: 1
  decoding: not reported
  pages:
    benchmark: 5
- id: ev_s3_copa
  defence_id: copa_l31
  benchmark: 'S3: optimisation-based injections (UAT, GCG)'
  attacker_type: optimisation
  attacker_access: white_box
  judge: attack bypasses the defence (safety judge)
  layout_same_as_defence: 'yes'
  measured_by: copa_arxiv2026
  page: 1
  decoding: not reported
  pages:
    benchmark: 5
- id: ev_life_sa
  defence_id: secalign_l31
  benchmark: lifelong CyberSecEval attack sequence, all variants
  attacker_type: static
  attacker_access: black_box
  judge: attack bypasses the defence (safety judge)
  layout_same_as_defence: 'yes'
  measured_by: copa_arxiv2026
  page: 5
  decoding: not reported
- id: ev_life_copa
  defence_id: copa_l31
  benchmark: lifelong CyberSecEval attack sequence, all variants
  attacker_type: static
  attacker_access: black_box
  judge: attack bypasses the defence (safety judge)
  layout_same_as_defence: 'yes'
  measured_by: copa_arxiv2026
  page: 5
  decoding: not reported
- id: ev_gpqa_none
  defence_id: none:Meta-Llama-3.1-8B-Instruct
  benchmark: GPQA
  attacker_type: none
  attacker_access: not_applicable
  judge: QA accuracy
  layout_same_as_defence: 'yes'
  measured_by: copa_arxiv2026
  page: 6
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_gpqa_sa
  defence_id: secalign_l31
  benchmark: GPQA
  attacker_type: none
  attacker_access: not_applicable
  judge: QA accuracy
  layout_same_as_defence: 'yes'
  measured_by: copa_arxiv2026
  page: 6
  decoding: not reported
- id: ev_gpqa_copa
  defence_id: copa_l31
  benchmark: GPQA
  attacker_type: none
  attacker_access: not_applicable
  judge: QA accuracy
  layout_same_as_defence: 'yes'
  measured_by: copa_arxiv2026
  page: 6
  decoding: not reported
- id: ev_mmlu_none
  defence_id: none:Meta-Llama-3.1-8B-Instruct
  benchmark: MMLU (MMLU-Pro per Section IV-A)
  attacker_type: none
  attacker_access: not_applicable
  judge: QA accuracy
  layout_same_as_defence: 'yes'
  measured_by: copa_arxiv2026
  page: 6
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_mmlu_sa
  defence_id: secalign_l31
  benchmark: MMLU (MMLU-Pro per Section IV-A)
  attacker_type: none
  attacker_access: not_applicable
  judge: QA accuracy
  layout_same_as_defence: 'yes'
  measured_by: copa_arxiv2026
  page: 6
  decoding: not reported
- id: ev_mmlu_copa
  defence_id: copa_l31
  benchmark: MMLU (MMLU-Pro per Section IV-A)
  attacker_type: none
  attacker_access: not_applicable
  judge: QA accuracy
  layout_same_as_defence: 'yes'
  measured_by: copa_arxiv2026
  page: 6
  decoding: not reported
results:
- id: r_s1_none
  evaluation_id: ev_s1_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.15'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 1
  location: Figure 1 point labels
- id: r_s1_sa
  evaluation_id: ev_s1_sa
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.12'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 1
  location: Figure 1 point labels
- id: r_s1_copa
  evaluation_id: ev_s1_copa
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.03'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 1
  location: Figure 1 point labels
- id: r_s2_none
  evaluation_id: ev_s2_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.55'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 1
  location: Figure 1 point labels
- id: r_s2_sa
  evaluation_id: ev_s2_sa
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.15'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 1
  location: Figure 1 point labels
- id: r_s2_copa
  evaluation_id: ev_s2_copa
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.05'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 1
  location: Figure 1 point labels
- id: r_s3_none
  evaluation_id: ev_s3_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.74'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 1
  location: Figure 1 point labels
- id: r_s3_sa
  evaluation_id: ev_s3_sa
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.25'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 1
  location: Figure 1 point labels
- id: r_s3_copa
  evaluation_id: ev_s3_copa
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.06'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 1
  location: Figure 1 point labels
- id: r_life_sa
  evaluation_id: ev_life_sa
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.221'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 5
  location: Section IV-B text and Figure 3
- id: r_life_copa
  evaluation_id: ev_life_copa
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.035'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 5
  location: Section IV-B text and Figure 3
- id: r_life_copa_bwt
  evaluation_id: ev_life_copa
  quantity: other
  coverage: direct
  metric_as_printed: BWT (backward transfer)
  value: '+0.028'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: text
  page: 5
  location: Section IV-B text
- id: r_life_copa_ap
  evaluation_id: ev_life_copa
  quantity: other
  coverage: direct
  metric_as_printed: AP (average performance)
  value: '0.850'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: text
  page: 5
  location: Section IV-B text
- id: r_gpqa_none
  evaluation_id: ev_gpqa_none
  quantity: general_utility
  coverage: direct
  metric_as_printed: QA accuracy
  value: '0.300'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: text
  page: 6
  location: Figure 4 and Section IV-B text
- id: r_gpqa_sa
  evaluation_id: ev_gpqa_sa
  quantity: general_utility
  coverage: direct
  metric_as_printed: QA accuracy
  value: '0.187'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: figure_only
  page: 6
  location: Figure 4 bar labels
- id: r_gpqa_copa
  evaluation_id: ev_gpqa_copa
  quantity: general_utility
  coverage: direct
  metric_as_printed: QA accuracy
  value: '0.310'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: text
  page: 6
  location: Figure 4 and Section IV-B text
- id: r_mmlu_none
  evaluation_id: ev_mmlu_none
  quantity: general_utility
  coverage: direct
  metric_as_printed: QA accuracy
  value: '0.332'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: text
  page: 6
  location: Figure 4 and Section IV-B text
- id: r_mmlu_sa
  evaluation_id: ev_mmlu_sa
  quantity: general_utility
  coverage: direct
  metric_as_printed: QA accuracy
  value: '0.286'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: figure_only
  page: 6
  location: Figure 4 bar labels
- id: r_mmlu_copa
  evaluation_id: ev_mmlu_copa
  quantity: general_utility
  coverage: direct
  metric_as_printed: QA accuracy
  value: '0.340'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: text
  page: 6
  location: Figure 4 and Section IV-B text
comparisons:
- id: c_s1_sa
  left: ev_s1_sa
  right: ev_s1_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 1
- id: c_s1_copa
  left: ev_s1_copa
  right: ev_s1_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 1
- id: c_s1_copa_sa
  left: ev_s1_copa
  right: ev_s1_sa
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 1
- id: c_s2_sa
  left: ev_s2_sa
  right: ev_s2_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 1
- id: c_s2_copa
  left: ev_s2_copa
  right: ev_s2_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 1
- id: c_s2_copa_sa
  left: ev_s2_copa
  right: ev_s2_sa
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 1
- id: c_s3_sa
  left: ev_s3_sa
  right: ev_s3_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 1
- id: c_s3_copa
  left: ev_s3_copa
  right: ev_s3_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 1
- id: c_s3_copa_sa
  left: ev_s3_copa
  right: ev_s3_sa
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 1
- id: c_life_copa_sa
  left: ev_life_copa
  right: ev_life_sa
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_gpqa_sa
  left: ev_gpqa_sa
  right: ev_gpqa_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_gpqa_copa
  left: ev_gpqa_copa
  right: ev_gpqa_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_mmlu_sa
  left: ev_mmlu_sa
  right: ev_mmlu_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_mmlu_copa
  left: ev_mmlu_copa
  right: ev_mmlu_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
claims:
- id: k1
  text: COPA cuts attack success up to 6.3 times and 4.4 times on average against state-of-the-art defences
  location: abstract
  anchor: COPA reduces attack success rate by up to 6.3
  jobs:
  - R1
  supported_by:
  - c_life_copa_sa
  - c_s1_copa_sa
  - c_s2_copa_sa
  - c_s3_copa_sa
  page: 1
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Figure 1 and Section IV-B
    general_utility: Figure 4 and Section IV-B
schema_gaps:
- where: results with value_source figure_only
  note: Figure 1 and Figure 4 print their values as point and bar labels; they are coded figure_only with the printed value kept (checked against the page text by the batch notebook)
- where: defence secalign_l31
  note: SecAlign is re-trained inside COPA's lifelong pipeline (same backbone, the same Stage 1 data, the same chance to adapt), not the released checkpoint (p. 5)

`secopd_emnlp2026` (2608.21500)

In [ ]:
%%writefile reports/records/secopd_emnlp2026.yaml
report:
  id: secopd_emnlp2026
  slug: '2608.21500'
  title: 'SecOPD: Mitigating Adaptive Prompt Injections by On-Policy Distillation'
  first_author: Yibo Peng
  venue: EMNLP 2026 (accepted, as printed on p. 1)
  peer_reviewed: 'yes'
  doi: 10.18653/2026.emnlp-main.1102
  arxiv: '2608.21500'
  version_read: arXiv v1 (carries an EMNLP 2026 acceptance note)
  first_date: '2026-08-21'
  read: F
  role: proposes
  code_release: https://github.com/pppyb/SecOPD and released weights (linked from the abstract)
  extraction_scope: 'Section 5: Tables 1 and 2 (all four models, all columns); the abstract claims (the static attack breakdown of Table 7, the per-domain AgentDojo Table 6 and the judge audits not coded)'
defences:
- id: secopd
  name: SecOPD (secure on-policy distillation)
  origin: this_report
  base_checkpoint: Qwen3.6-27B
  axis1:
  - chat_roles
  axis2:
  - distillation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 1
  components:
  - trusted prompt in the user role, untrusted data in a separate input role
  - on injected samples the model's own rollout is scored token by token by the initial model given the clean input (reverse KL)
  - 19K samples built from Cleaned-Alpaca
  layout: user role for the instruction, input role for the data
  layout_compat:
    user_turn: untested
    separate_role: supported
    system_context: untested
    agent_tool_result: supported
  anchor: we propose Secure On-Policy Distillation (SecOPD) that provides
  pages:
    components: 6
    layout: 2
- id: msa_q
  name: Meta-SecAlign (trained by the authors on the same 19K data)
  origin: other_report
  base_checkpoint: Qwen3.6-27B
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  anchor: all defended mod- els are fine-tuned using a 19K dataset
- id: grpo_q
  name: GRPO baseline (sequence-level judge reward on injected samples)
  origin: this_report
  base_checkpoint: Qwen3.6-27B
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  anchor: We use GRPO as a baseline for sequence-level
evaluations:
- id: ev_sepst_none
  defence_id: none:Qwen3.6-27B
  benchmark: SEP, 9.1K samples, six static attacks combined
  attacker_type: static
  attacker_access: black_box
  judge: two LLM judges, three calls each, all YES (after a witness filter)
  layout_same_as_defence: 'yes'
  measured_by: secopd_emnlp2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
  baseline_kind: off_the_shelf
- id: ev_sepba_none
  defence_id: none:Qwen3.6-27B
  benchmark: SEP, basic adaptive attack with delimiter-like substitutes found in embedding space
  attacker_type: adaptive
  attacker_access: white_box
  judge: two LLM judges, three calls each, all YES (after a witness filter)
  layout_same_as_defence: 'yes'
  measured_by: secopd_emnlp2026
  page: 7
  attacker_budget: nearby-token substitutes for the official delimiters
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
  baseline_kind: off_the_shelf
- id: ev_seppi_none
  defence_id: none:Qwen3.6-27B
  benchmark: SEP, 1,024 test examples, PISmith attacker trained against each model, pass@10
  attacker_type: rl
  attacker_access: black_box
  judge: two LLM judges, three calls each, all YES (after a witness filter)
  layout_same_as_defence: 'yes'
  measured_by: secopd_emnlp2026
  page: 7
  attacker_budget: 10 attempts per example
  decoding: not reported
  sample_size: 1,024
  pages:
    benchmark: 6
    judge: 6
  baseline_kind: off_the_shelf
- id: ev_adst_none
  defence_id: none:Qwen3.6-27B
  benchmark: AgentDojo, 949 (user task, injection task) pairs, important instructions attack
  attacker_type: static
  attacker_access: black_box
  judge: malicious tool called
  layout_same_as_defence: 'yes'
  measured_by: secopd_emnlp2026
  page: 7
  decoding: not reported
  sample_size: '949'
  pages:
    benchmark: 6
    judge: 6
  baseline_kind: off_the_shelf
- id: ev_sepst_msa
  defence_id: msa_q
  benchmark: SEP, 9.1K samples, six static attacks combined
  attacker_type: static
  attacker_access: black_box
  judge: two LLM judges, three calls each, all YES (after a witness filter)
  layout_same_as_defence: 'yes'
  measured_by: secopd_emnlp2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
- id: ev_sepba_msa
  defence_id: msa_q
  benchmark: SEP, basic adaptive attack with delimiter-like substitutes found in embedding space
  attacker_type: adaptive
  attacker_access: white_box
  judge: two LLM judges, three calls each, all YES (after a witness filter)
  layout_same_as_defence: 'yes'
  measured_by: secopd_emnlp2026
  page: 7
  attacker_budget: nearby-token substitutes for the official delimiters
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
- id: ev_seppi_msa
  defence_id: msa_q
  benchmark: SEP, 1,024 test examples, PISmith attacker trained against each model, pass@10
  attacker_type: rl
  attacker_access: black_box
  judge: two LLM judges, three calls each, all YES (after a witness filter)
  layout_same_as_defence: 'yes'
  measured_by: secopd_emnlp2026
  page: 7
  attacker_budget: 10 attempts per example
  decoding: not reported
  sample_size: 1,024
  pages:
    benchmark: 6
    judge: 6
- id: ev_adst_msa
  defence_id: msa_q
  benchmark: AgentDojo, 949 (user task, injection task) pairs, important instructions attack
  attacker_type: static
  attacker_access: black_box
  judge: malicious tool called
  layout_same_as_defence: 'yes'
  measured_by: secopd_emnlp2026
  page: 7
  decoding: not reported
  sample_size: '949'
  pages:
    benchmark: 6
    judge: 6
- id: ev_sepst_grpo
  defence_id: grpo_q
  benchmark: SEP, 9.1K samples, six static attacks combined
  attacker_type: static
  attacker_access: black_box
  judge: two LLM judges, three calls each, all YES (after a witness filter)
  layout_same_as_defence: 'yes'
  measured_by: secopd_emnlp2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
- id: ev_sepba_grpo
  defence_id: grpo_q
  benchmark: SEP, basic adaptive attack with delimiter-like substitutes found in embedding space
  attacker_type: adaptive
  attacker_access: white_box
  judge: two LLM judges, three calls each, all YES (after a witness filter)
  layout_same_as_defence: 'yes'
  measured_by: secopd_emnlp2026
  page: 7
  attacker_budget: nearby-token substitutes for the official delimiters
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
- id: ev_seppi_grpo
  defence_id: grpo_q
  benchmark: SEP, 1,024 test examples, PISmith attacker trained against each model, pass@10
  attacker_type: rl
  attacker_access: black_box
  judge: two LLM judges, three calls each, all YES (after a witness filter)
  layout_same_as_defence: 'yes'
  measured_by: secopd_emnlp2026
  page: 7
  attacker_budget: 10 attempts per example
  decoding: not reported
  sample_size: 1,024
  pages:
    benchmark: 6
    judge: 6
- id: ev_adst_grpo
  defence_id: grpo_q
  benchmark: AgentDojo, 949 (user task, injection task) pairs, important instructions attack
  attacker_type: static
  attacker_access: black_box
  judge: malicious tool called
  layout_same_as_defence: 'yes'
  measured_by: secopd_emnlp2026
  page: 7
  decoding: not reported
  sample_size: '949'
  pages:
    benchmark: 6
    judge: 6
- id: ev_sepst_opd
  defence_id: secopd
  benchmark: SEP, 9.1K samples, six static attacks combined
  attacker_type: static
  attacker_access: black_box
  judge: two LLM judges, three calls each, all YES (after a witness filter)
  layout_same_as_defence: 'yes'
  measured_by: secopd_emnlp2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
- id: ev_sepba_opd
  defence_id: secopd
  benchmark: SEP, basic adaptive attack with delimiter-like substitutes found in embedding space
  attacker_type: adaptive
  attacker_access: white_box
  judge: two LLM judges, three calls each, all YES (after a witness filter)
  layout_same_as_defence: 'yes'
  measured_by: secopd_emnlp2026
  page: 7
  attacker_budget: nearby-token substitutes for the official delimiters
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
- id: ev_seppi_opd
  defence_id: secopd
  benchmark: SEP, 1,024 test examples, PISmith attacker trained against each model, pass@10
  attacker_type: rl
  attacker_access: black_box
  judge: two LLM judges, three calls each, all YES (after a witness filter)
  layout_same_as_defence: 'yes'
  measured_by: secopd_emnlp2026
  page: 7
  attacker_budget: 10 attempts per example
  decoding: not reported
  sample_size: 1,024
  pages:
    benchmark: 6
    judge: 6
- id: ev_adst_opd
  defence_id: secopd
  benchmark: AgentDojo, 949 (user task, injection task) pairs, important instructions attack
  attacker_type: static
  attacker_access: black_box
  judge: malicious tool called
  layout_same_as_defence: 'yes'
  measured_by: secopd_emnlp2026
  page: 7
  decoding: not reported
  sample_size: '949'
  pages:
    benchmark: 6
    judge: 6
- id: ev_alpaca_none
  defence_id: none:Qwen3.6-27B
  benchmark: AlpacaEval2
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: secopd_emnlp2026
  page: 7
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_sepu_none
  defence_id: none:Qwen3.6-27B
  benchmark: SEP (AlpacaEval2 judge)
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: secopd_emnlp2026
  page: 7
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_adu_none
  defence_id: none:Qwen3.6-27B
  benchmark: AgentDojo benign user tasks
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: secopd_emnlp2026
  page: 7
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_mmlupro_none
  defence_id: none:Qwen3.6-27B
  benchmark: MMLU Pro
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: secopd_emnlp2026
  page: 7
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_gpqa_none
  defence_id: none:Qwen3.6-27B
  benchmark: GPQA Diamond
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: secopd_emnlp2026
  page: 7
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_gsm8k_none
  defence_id: none:Qwen3.6-27B
  benchmark: GSM8K
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: secopd_emnlp2026
  page: 7
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_minerva_none
  defence_id: none:Qwen3.6-27B
  benchmark: Minerva Math
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: secopd_emnlp2026
  page: 7
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_alpaca_msa
  defence_id: msa_q
  benchmark: AlpacaEval2
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: secopd_emnlp2026
  page: 7
  decoding: not reported
- id: ev_sepu_msa
  defence_id: msa_q
  benchmark: SEP (AlpacaEval2 judge)
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: secopd_emnlp2026
  page: 7
  decoding: not reported
- id: ev_adu_msa
  defence_id: msa_q
  benchmark: AgentDojo benign user tasks
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: secopd_emnlp2026
  page: 7
  decoding: not reported
- id: ev_mmlupro_msa
  defence_id: msa_q
  benchmark: MMLU Pro
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: secopd_emnlp2026
  page: 7
  decoding: not reported
- id: ev_gpqa_msa
  defence_id: msa_q
  benchmark: GPQA Diamond
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: secopd_emnlp2026
  page: 7
  decoding: not reported
- id: ev_gsm8k_msa
  defence_id: msa_q
  benchmark: GSM8K
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: secopd_emnlp2026
  page: 7
  decoding: not reported
- id: ev_minerva_msa
  defence_id: msa_q
  benchmark: Minerva Math
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: secopd_emnlp2026
  page: 7
  decoding: not reported
- id: ev_alpaca_grpo
  defence_id: grpo_q
  benchmark: AlpacaEval2
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: secopd_emnlp2026
  page: 7
  decoding: not reported
- id: ev_sepu_grpo
  defence_id: grpo_q
  benchmark: SEP (AlpacaEval2 judge)
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: secopd_emnlp2026
  page: 7
  decoding: not reported
- id: ev_adu_grpo
  defence_id: grpo_q
  benchmark: AgentDojo benign user tasks
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: secopd_emnlp2026
  page: 7
  decoding: not reported
- id: ev_mmlupro_grpo
  defence_id: grpo_q
  benchmark: MMLU Pro
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: secopd_emnlp2026
  page: 7
  decoding: not reported
- id: ev_gpqa_grpo
  defence_id: grpo_q
  benchmark: GPQA Diamond
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: secopd_emnlp2026
  page: 7
  decoding: not reported
- id: ev_gsm8k_grpo
  defence_id: grpo_q
  benchmark: GSM8K
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: secopd_emnlp2026
  page: 7
  decoding: not reported
- id: ev_minerva_grpo
  defence_id: grpo_q
  benchmark: Minerva Math
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: secopd_emnlp2026
  page: 7
  decoding: not reported
- id: ev_alpaca_opd
  defence_id: secopd
  benchmark: AlpacaEval2
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: secopd_emnlp2026
  page: 7
  decoding: not reported
- id: ev_sepu_opd
  defence_id: secopd
  benchmark: SEP (AlpacaEval2 judge)
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: secopd_emnlp2026
  page: 7
  decoding: not reported
- id: ev_adu_opd
  defence_id: secopd
  benchmark: AgentDojo benign user tasks
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: secopd_emnlp2026
  page: 7
  decoding: not reported
- id: ev_mmlupro_opd
  defence_id: secopd
  benchmark: MMLU Pro
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: secopd_emnlp2026
  page: 7
  decoding: not reported
- id: ev_gpqa_opd
  defence_id: secopd
  benchmark: GPQA Diamond
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: secopd_emnlp2026
  page: 7
  decoding: not reported
- id: ev_gsm8k_opd
  defence_id: secopd
  benchmark: GSM8K
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: secopd_emnlp2026
  page: 7
  decoding: not reported
- id: ev_minerva_opd
  defence_id: secopd
  benchmark: Minerva Math
  attacker_type: none
  attacker_access: not_applicable
  judge: benchmark scorer
  layout_same_as_defence: 'yes'
  measured_by: secopd_emnlp2026
  page: 7
  decoding: not reported
results:
- id: r_sepst_none
  evaluation_id: ev_sepst_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (SEP Static)
  value: '99.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_sepba_none
  evaluation_id: ev_sepba_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (SEP Basic Adaptive)
  value: '99.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_seppi_none
  evaluation_id: ev_seppi_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (SEP PISmith Adaptive)
  value: '97.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_adst_none
  evaluation_id: ev_adst_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (AgentDojo Static)
  value: '26.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_sepst_msa
  evaluation_id: ev_sepst_msa
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (SEP Static)
  value: '28.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_sepba_msa
  evaluation_id: ev_sepba_msa
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (SEP Basic Adaptive)
  value: '5.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_seppi_msa
  evaluation_id: ev_seppi_msa
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (SEP PISmith Adaptive)
  value: '94.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_adst_msa
  evaluation_id: ev_adst_msa
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (AgentDojo Static)
  value: '5.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_sepst_grpo
  evaluation_id: ev_sepst_grpo
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (SEP Static)
  value: '15.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_sepba_grpo
  evaluation_id: ev_sepba_grpo
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (SEP Basic Adaptive)
  value: '2.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_seppi_grpo
  evaluation_id: ev_seppi_grpo
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (SEP PISmith Adaptive)
  value: '61.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_adst_grpo
  evaluation_id: ev_adst_grpo
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (AgentDojo Static)
  value: '0.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_sepst_opd
  evaluation_id: ev_sepst_opd
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (SEP Static)
  value: '1.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_sepba_opd
  evaluation_id: ev_sepba_opd
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (SEP Basic Adaptive)
  value: '0.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_seppi_opd
  evaluation_id: ev_seppi_opd
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (SEP PISmith Adaptive)
  value: '9.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_adst_opd
  evaluation_id: ev_adst_opd
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (AgentDojo Static)
  value: '4.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_alpaca_none
  evaluation_id: ev_alpaca_none
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaEval2
  value: '81.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_sepu_none
  evaluation_id: ev_sepu_none
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP (AlpacaEval2 judge)
  value: '88.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_adu_none
  evaluation_id: ev_adu_none
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDojo benign user tasks
  value: '92.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_mmlupro_none
  evaluation_id: ev_mmlupro_none
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU Pro
  value: '84.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_gpqa_none
  evaluation_id: ev_gpqa_none
  quantity: general_utility
  coverage: direct
  metric_as_printed: GPQA Diamond
  value: '79.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_gsm8k_none
  evaluation_id: ev_gsm8k_none
  quantity: general_utility
  coverage: direct
  metric_as_printed: GSM8K
  value: '97.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_minerva_none
  evaluation_id: ev_minerva_none
  quantity: general_utility
  coverage: direct
  metric_as_printed: Minerva Math
  value: '92.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_alpaca_msa
  evaluation_id: ev_alpaca_msa
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaEval2
  value: '82.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_sepu_msa
  evaluation_id: ev_sepu_msa
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP (AlpacaEval2 judge)
  value: '89.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_adu_msa
  evaluation_id: ev_adu_msa
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDojo benign user tasks
  value: '92.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_mmlupro_msa
  evaluation_id: ev_mmlupro_msa
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU Pro
  value: '83.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_gpqa_msa
  evaluation_id: ev_gpqa_msa
  quantity: general_utility
  coverage: direct
  metric_as_printed: GPQA Diamond
  value: '78.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_gsm8k_msa
  evaluation_id: ev_gsm8k_msa
  quantity: general_utility
  coverage: direct
  metric_as_printed: GSM8K
  value: '97.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_minerva_msa
  evaluation_id: ev_minerva_msa
  quantity: general_utility
  coverage: direct
  metric_as_printed: Minerva Math
  value: '95.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_alpaca_grpo
  evaluation_id: ev_alpaca_grpo
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaEval2
  value: '76.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_sepu_grpo
  evaluation_id: ev_sepu_grpo
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP (AlpacaEval2 judge)
  value: '79.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_adu_grpo
  evaluation_id: ev_adu_grpo
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDojo benign user tasks
  value: '82.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_mmlupro_grpo
  evaluation_id: ev_mmlupro_grpo
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU Pro
  value: '83.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_gpqa_grpo
  evaluation_id: ev_gpqa_grpo
  quantity: general_utility
  coverage: direct
  metric_as_printed: GPQA Diamond
  value: '77.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_gsm8k_grpo
  evaluation_id: ev_gsm8k_grpo
  quantity: general_utility
  coverage: direct
  metric_as_printed: GSM8K
  value: '97.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_minerva_grpo
  evaluation_id: ev_minerva_grpo
  quantity: general_utility
  coverage: direct
  metric_as_printed: Minerva Math
  value: '85.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_alpaca_opd
  evaluation_id: ev_alpaca_opd
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaEval2
  value: '80.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_sepu_opd
  evaluation_id: ev_sepu_opd
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP (AlpacaEval2 judge)
  value: '88.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_adu_opd
  evaluation_id: ev_adu_opd
  quantity: R2clean
  coverage: direct
  metric_as_printed: AgentDojo benign user tasks
  value: '90.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_mmlupro_opd
  evaluation_id: ev_mmlupro_opd
  quantity: general_utility
  coverage: direct
  metric_as_printed: MMLU Pro
  value: '84.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_gpqa_opd
  evaluation_id: ev_gpqa_opd
  quantity: general_utility
  coverage: direct
  metric_as_printed: GPQA Diamond
  value: '81.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_gsm8k_opd
  evaluation_id: ev_gsm8k_opd
  quantity: general_utility
  coverage: direct
  metric_as_printed: GSM8K
  value: '97.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_minerva_opd
  evaluation_id: ev_minerva_opd
  quantity: general_utility
  coverage: direct
  metric_as_printed: Minerva Math
  value: '94.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
comparisons:
- id: c_sepst_msa
  left: ev_sepst_msa
  right: ev_sepst_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_sepst_grpo
  left: ev_sepst_grpo
  right: ev_sepst_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_sepst_opd
  left: ev_sepst_opd
  right: ev_sepst_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_sepst_opd_grpo
  left: ev_sepst_opd
  right: ev_sepst_grpo
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: token-level on-policy distillation against a sequence-level judge reward, same data
  page: 7
- id: c_sepba_msa
  left: ev_sepba_msa
  right: ev_sepba_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_sepba_grpo
  left: ev_sepba_grpo
  right: ev_sepba_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_sepba_opd
  left: ev_sepba_opd
  right: ev_sepba_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_sepba_opd_grpo
  left: ev_sepba_opd
  right: ev_sepba_grpo
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: token-level on-policy distillation against a sequence-level judge reward, same data
  page: 7
- id: c_seppi_msa
  left: ev_seppi_msa
  right: ev_seppi_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_seppi_grpo
  left: ev_seppi_grpo
  right: ev_seppi_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_seppi_opd
  left: ev_seppi_opd
  right: ev_seppi_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_seppi_opd_grpo
  left: ev_seppi_opd
  right: ev_seppi_grpo
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: token-level on-policy distillation against a sequence-level judge reward, same data
  page: 7
- id: c_adst_msa
  left: ev_adst_msa
  right: ev_adst_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_adst_grpo
  left: ev_adst_grpo
  right: ev_adst_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_adst_opd
  left: ev_adst_opd
  right: ev_adst_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_adst_opd_grpo
  left: ev_adst_opd
  right: ev_adst_grpo
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: token-level on-policy distillation against a sequence-level judge reward, same data
  page: 7
- id: c_alpaca_msa
  left: ev_alpaca_msa
  right: ev_alpaca_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_alpaca_grpo
  left: ev_alpaca_grpo
  right: ev_alpaca_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_alpaca_opd
  left: ev_alpaca_opd
  right: ev_alpaca_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_alpaca_opd_grpo
  left: ev_alpaca_opd
  right: ev_alpaca_grpo
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: token-level on-policy distillation against a sequence-level judge reward, same data
  page: 7
- id: c_sepu_msa
  left: ev_sepu_msa
  right: ev_sepu_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_sepu_grpo
  left: ev_sepu_grpo
  right: ev_sepu_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_sepu_opd
  left: ev_sepu_opd
  right: ev_sepu_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_sepu_opd_grpo
  left: ev_sepu_opd
  right: ev_sepu_grpo
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: token-level on-policy distillation against a sequence-level judge reward, same data
  page: 7
- id: c_adu_msa
  left: ev_adu_msa
  right: ev_adu_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_adu_grpo
  left: ev_adu_grpo
  right: ev_adu_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_adu_opd
  left: ev_adu_opd
  right: ev_adu_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_adu_opd_grpo
  left: ev_adu_opd
  right: ev_adu_grpo
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: token-level on-policy distillation against a sequence-level judge reward, same data
  page: 7
- id: c_mmlupro_msa
  left: ev_mmlupro_msa
  right: ev_mmlupro_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_mmlupro_grpo
  left: ev_mmlupro_grpo
  right: ev_mmlupro_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_mmlupro_opd
  left: ev_mmlupro_opd
  right: ev_mmlupro_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_mmlupro_opd_grpo
  left: ev_mmlupro_opd
  right: ev_mmlupro_grpo
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: token-level on-policy distillation against a sequence-level judge reward, same data
  page: 7
- id: c_gpqa_msa
  left: ev_gpqa_msa
  right: ev_gpqa_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_gpqa_grpo
  left: ev_gpqa_grpo
  right: ev_gpqa_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_gpqa_opd
  left: ev_gpqa_opd
  right: ev_gpqa_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_gpqa_opd_grpo
  left: ev_gpqa_opd
  right: ev_gpqa_grpo
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: token-level on-policy distillation against a sequence-level judge reward, same data
  page: 7
- id: c_gsm8k_msa
  left: ev_gsm8k_msa
  right: ev_gsm8k_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_gsm8k_grpo
  left: ev_gsm8k_grpo
  right: ev_gsm8k_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_gsm8k_opd
  left: ev_gsm8k_opd
  right: ev_gsm8k_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_gsm8k_opd_grpo
  left: ev_gsm8k_opd
  right: ev_gsm8k_grpo
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: token-level on-policy distillation against a sequence-level judge reward, same data
  page: 7
- id: c_minerva_msa
  left: ev_minerva_msa
  right: ev_minerva_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_minerva_grpo
  left: ev_minerva_grpo
  right: ev_minerva_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_minerva_opd
  left: ev_minerva_opd
  right: ev_minerva_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_minerva_opd_grpo
  left: ev_minerva_opd
  right: ev_minerva_grpo
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: token-level on-policy distillation against a sequence-level judge reward, same data
  page: 7
- id: c_sepst_opd_msa
  left: ev_sepst_opd
  right: ev_sepst_msa
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_sepba_opd_msa
  left: ev_sepba_opd
  right: ev_sepba_msa
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_seppi_opd_msa
  left: ev_seppi_opd
  right: ev_seppi_msa
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_adst_opd_msa
  left: ev_adst_opd
  right: ev_adst_msa
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
claims:
- id: k1
  text: SecOPD reaches 9.0% ASR against PISmith, against 94.0% for Meta-SecAlign
  location: abstract
  anchor: achieves a 9.0% ASR against the SoTA PISmith adaptive
  jobs:
  - R1
  supported_by:
  - c_seppi_opd_msa
  - c_seppi_opd
  page: 1
- id: k2
  text: SecOPD's security generalises to unseen agentic tool calling (4.7% against 5.5% ASR)
  location: abstract
  anchor: in agentic tool calling, SecOPD achieves a 4.7% ASR compared
  jobs:
  - R1
  supported_by:
  - c_adst_opd_msa
  - c_adst_opd
  page: 1
coverage:
  R1: direct
  R2clean: direct
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 1
    R2clean: Table 2 (AgentDojo benign; SEP utility is a proxy)
    general_utility: Table 2
schema_gaps:
- where: defence msa_q
  note: Meta-SecAlign is re-trained by the authors on Qwen3.6-27B with the same 19K Cleaned-Alpaca data as SecOPD, not the released checkpoint (p. 6)

`overlays_arxiv2026` (2608.23873)

In [ ]:
%%writefile reports/records/overlays_arxiv2026.yaml
report:
  id: overlays_arxiv2026
  slug: '2608.23873'
  title: 'Semantic Overlays: Mitigating Prompt Injection with Annotations Beyond Tokens and Steering Vectors'
  first_author: Joshua Penman
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2608.23873'
  version_read: arXiv v4
  first_date: '2026-08-24'
  read: F
  role: proposes
  code_release: not checked
  extraction_scope: 'Section 3 and Appendices C, F and O: Table 1, all rows and every column except TensorTrust (whose attacker writes the user''s own input); the abstract claim (the GCG and adversarial-training results of
    Appendix K, the overlay experiments of Section 4 and Tables 2 to 4 not coded)'
defences:
- id: sand_q
  name: Sandwich
  origin: other_report
  base_checkpoint: Qwen3.5-9B
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  anchor: prompt-only defenses
- id: spot_q
  name: Spotlighting
  origin: other_report
  base_checkpoint: Qwen3.5-9B
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  anchor: prompt-only defenses
- id: ltok_q
  name: LoRA control, span marked by two new special tokens (StruQ-style)
  origin: this_report
  base_checkpoint: Qwen3.5-9B
  axis1:
  - delimiters_markers
  axis2:
  - sft
  - distillation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 17
  anchor: The special-token LoRA follows StruQ
- id: ltxt_q
  name: LoRA control, span wrapped in text delimiters
  origin: this_report
  base_checkpoint: Qwen3.5-9B
  axis1:
  - delimiters_markers
  axis2:
  - sft
  - distillation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 17
  anchor: The text-delimiter LoRA wraps the span in plain-text delimiters
- id: lnone_q
  name: LoRA control, no marker
  origin: this_report
  base_checkpoint: Qwen3.5-9B
  axis1:
  - no_signal
  axis2:
  - sft
  - distillation
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 17
  anchor: the third LoRA marks nothing
- id: dpo_q
  name: SecAlign objective (DPO) on the special-token LoRA
  origin: this_report
  base_checkpoint: Qwen3.5-9B
  axis1:
  - delimiters_markers
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 17
  anchor: trains the special-token LoRA with DPO
- id: plv_q
  name: learned per-layer vectors added on the marked span
  origin: this_report
  base_checkpoint: Qwen3.5-9B
  axis1:
  - learned_token_features
  - internal_intervention
  axis2:
  - sft
  - distillation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 17
  anchor: The per-layer vectors add one learned vector per layer
- id: loreft_q
  name: LoReFT on the marked span (rank-8 edit)
  origin: this_report
  base_checkpoint: Qwen3.5-9B
  axis1:
  - learned_token_features
  - internal_intervention
  axis2:
  - sft
  - distillation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  anchor: We also train LoReFT, a low-rank form of ReFT
- id: ov_q
  name: Semantic Overlays (do-not-execute overlay)
  origin: this_report
  base_checkpoint: Qwen3.5-9B
  axis1:
  - learned_token_features
  - internal_intervention
  axis2:
  - sft
  - distillation
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 1
  anchor: small trained adapters that edit a frozen model
  components:
  - a bottleneck adapter (dimension 128) at every layer but the first adds an edit to the residual stream at token positions the serving stack marks as untrusted; the frozen model runs unchanged elsewhere
  - targets are the frozen model's own completions (on clean items, and on attacked items with the planted instruction removed), plus copy items that quote the marked span
  layout: untrusted span marked out of band (no tokens added)
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  pages:
    components: 5
    layout_compat: 7
- id: spot_l
  name: Spotlighting
  origin: other_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  anchor: prompt-only defenses
- id: dt_l
  name: DefensiveTokens
  origin: other_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - chat_roles
  axis2:
  - soft_tokens_frozen
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  anchor: On Llama-3.1-8B we add DefensiveTokens
- id: msa_l
  name: Meta-SecAlign
  origin: other_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  anchor: On Llama-3.1-8B we add DefensiveTokens
- id: ov_l
  name: Semantic Overlays (do-not-execute overlay, earlier data mix)
  origin: this_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - learned_token_features
  - internal_intervention
  axis2:
  - sft
  - distillation
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 1
  anchor: small trained adapters that edit a frozen model
  components:
  - a bottleneck adapter (dimension 128) at every layer but the first adds an edit to the residual stream at token positions the serving stack marks as untrusted; the frozen model runs unchanged elsewhere
  - targets are the frozen model's own completions (on clean items, and on attacked items with the planted instruction removed), plus copy items that quote the marked span
  layout: untrusted span marked out of band (no tokens added)
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  pages:
    components: 5
    layout_compat: 7
evaluations:
- id: ev_q_none_sep
  defence_id: none:Qwen3.5-9B (frozen)
  benchmark: SEP, 9,160 items
  attacker_type: static
  attacker_access: black_box
  judge: corrected SEP grader
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
  sample_size: 9,160
  baseline_kind: off_the_shelf
- id: ev_q_none_util
  defence_id: none:Qwen3.5-9B (frozen)
  benchmark: SEP, 9,160 items
  attacker_type: none
  attacker_access: not_applicable
  judge: corrected SEP grader
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
  sample_size: 9,160
  baseline_kind: off_the_shelf
- id: ev_q_none_af
  defence_id: none:Qwen3.5-9B (frozen)
  benchmark: AlpacaFarm, StruQ's four static attacks
  attacker_type: static
  attacker_access: black_box
  judge: response begins with the injected marker
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
  baseline_kind: off_the_shelf
- id: ev_q_none_qipi
  defence_id: none:Qwen3.5-9B (frozen)
  benchmark: Quadrat-IPI, 2,000 documents, summarisation task
  attacker_type: static
  attacker_access: black_box
  judge: injection compliance
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
  baseline_kind: off_the_shelf
- id: ev_q_none_ad
  defence_id: none:Qwen3.5-9B (frozen)
  benchmark: AgentDojo, every tool output marked
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checkers
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
  baseline_kind: off_the_shelf
- id: ev_q_none_pia
  defence_id: none:Qwen3.5-9B (frozen)
  benchmark: PIArena HotpotQA and NQ
  attacker_type: static
  attacker_access: black_box
  judge: behaviour-based PIArena scoring
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
  baseline_kind: off_the_shelf
- id: ev_q_none_piau
  defence_id: none:Qwen3.5-9B (frozen)
  benchmark: PIArena HotpotQA and NQ
  attacker_type: static
  attacker_access: black_box
  judge: answer accuracy
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
  baseline_kind: off_the_shelf
- id: ev_q_none_sq
  defence_id: none:Qwen3.5-9B (frozen)
  benchmark: PIArena SQuAD v2
  attacker_type: static
  attacker_access: black_box
  judge: behaviour-based PIArena scoring
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
  baseline_kind: off_the_shelf
- id: ev_q_sand_sep
  defence_id: sand_q
  benchmark: SEP, 9,160 items
  attacker_type: static
  attacker_access: black_box
  judge: corrected SEP grader
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
  sample_size: 9,160
- id: ev_q_sand_util
  defence_id: sand_q
  benchmark: SEP, 9,160 items
  attacker_type: none
  attacker_access: not_applicable
  judge: corrected SEP grader
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
  sample_size: 9,160
- id: ev_q_sand_af
  defence_id: sand_q
  benchmark: AlpacaFarm, StruQ's four static attacks
  attacker_type: static
  attacker_access: black_box
  judge: response begins with the injected marker
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_q_sand_qipi
  defence_id: sand_q
  benchmark: Quadrat-IPI, 2,000 documents, summarisation task
  attacker_type: static
  attacker_access: black_box
  judge: injection compliance
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_q_sand_pia
  defence_id: sand_q
  benchmark: PIArena HotpotQA and NQ
  attacker_type: static
  attacker_access: black_box
  judge: behaviour-based PIArena scoring
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_q_sand_piau
  defence_id: sand_q
  benchmark: PIArena HotpotQA and NQ
  attacker_type: static
  attacker_access: black_box
  judge: answer accuracy
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_q_sand_sq
  defence_id: sand_q
  benchmark: PIArena SQuAD v2
  attacker_type: static
  attacker_access: black_box
  judge: behaviour-based PIArena scoring
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_q_spot_sep
  defence_id: spot_q
  benchmark: SEP, 9,160 items
  attacker_type: static
  attacker_access: black_box
  judge: corrected SEP grader
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
  sample_size: 9,160
- id: ev_q_spot_util
  defence_id: spot_q
  benchmark: SEP, 9,160 items
  attacker_type: none
  attacker_access: not_applicable
  judge: corrected SEP grader
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
  sample_size: 9,160
- id: ev_q_spot_af
  defence_id: spot_q
  benchmark: AlpacaFarm, StruQ's four static attacks
  attacker_type: static
  attacker_access: black_box
  judge: response begins with the injected marker
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_q_spot_qipi
  defence_id: spot_q
  benchmark: Quadrat-IPI, 2,000 documents, summarisation task
  attacker_type: static
  attacker_access: black_box
  judge: injection compliance
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_q_spot_pia
  defence_id: spot_q
  benchmark: PIArena HotpotQA and NQ
  attacker_type: static
  attacker_access: black_box
  judge: behaviour-based PIArena scoring
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_q_spot_piau
  defence_id: spot_q
  benchmark: PIArena HotpotQA and NQ
  attacker_type: static
  attacker_access: black_box
  judge: answer accuracy
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_q_spot_sq
  defence_id: spot_q
  benchmark: PIArena SQuAD v2
  attacker_type: static
  attacker_access: black_box
  judge: behaviour-based PIArena scoring
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_q_ltok_sep
  defence_id: ltok_q
  benchmark: SEP, 9,160 items
  attacker_type: static
  attacker_access: black_box
  judge: corrected SEP grader
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
  sample_size: 9,160
- id: ev_q_ltok_util
  defence_id: ltok_q
  benchmark: SEP, 9,160 items
  attacker_type: none
  attacker_access: not_applicable
  judge: corrected SEP grader
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
  sample_size: 9,160
- id: ev_q_ltok_af
  defence_id: ltok_q
  benchmark: AlpacaFarm, StruQ's four static attacks
  attacker_type: static
  attacker_access: black_box
  judge: response begins with the injected marker
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_q_ltok_qipi
  defence_id: ltok_q
  benchmark: Quadrat-IPI, 2,000 documents, summarisation task
  attacker_type: static
  attacker_access: black_box
  judge: injection compliance
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_q_ltok_pia
  defence_id: ltok_q
  benchmark: PIArena HotpotQA and NQ
  attacker_type: static
  attacker_access: black_box
  judge: behaviour-based PIArena scoring
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_q_ltok_piau
  defence_id: ltok_q
  benchmark: PIArena HotpotQA and NQ
  attacker_type: static
  attacker_access: black_box
  judge: answer accuracy
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_q_ltok_sq
  defence_id: ltok_q
  benchmark: PIArena SQuAD v2
  attacker_type: static
  attacker_access: black_box
  judge: behaviour-based PIArena scoring
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_q_ltxt_sep
  defence_id: ltxt_q
  benchmark: SEP, 9,160 items
  attacker_type: static
  attacker_access: black_box
  judge: corrected SEP grader
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
  sample_size: 9,160
- id: ev_q_ltxt_util
  defence_id: ltxt_q
  benchmark: SEP, 9,160 items
  attacker_type: none
  attacker_access: not_applicable
  judge: corrected SEP grader
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
  sample_size: 9,160
- id: ev_q_ltxt_af
  defence_id: ltxt_q
  benchmark: AlpacaFarm, StruQ's four static attacks
  attacker_type: static
  attacker_access: black_box
  judge: response begins with the injected marker
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_q_ltxt_qipi
  defence_id: ltxt_q
  benchmark: Quadrat-IPI, 2,000 documents, summarisation task
  attacker_type: static
  attacker_access: black_box
  judge: injection compliance
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_q_ltxt_pia
  defence_id: ltxt_q
  benchmark: PIArena HotpotQA and NQ
  attacker_type: static
  attacker_access: black_box
  judge: behaviour-based PIArena scoring
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_q_ltxt_piau
  defence_id: ltxt_q
  benchmark: PIArena HotpotQA and NQ
  attacker_type: static
  attacker_access: black_box
  judge: answer accuracy
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_q_ltxt_sq
  defence_id: ltxt_q
  benchmark: PIArena SQuAD v2
  attacker_type: static
  attacker_access: black_box
  judge: behaviour-based PIArena scoring
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_q_lnone_sep
  defence_id: lnone_q
  benchmark: SEP, 9,160 items
  attacker_type: static
  attacker_access: black_box
  judge: corrected SEP grader
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
  sample_size: 9,160
- id: ev_q_lnone_util
  defence_id: lnone_q
  benchmark: SEP, 9,160 items
  attacker_type: none
  attacker_access: not_applicable
  judge: corrected SEP grader
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
  sample_size: 9,160
- id: ev_q_lnone_af
  defence_id: lnone_q
  benchmark: AlpacaFarm, StruQ's four static attacks
  attacker_type: static
  attacker_access: black_box
  judge: response begins with the injected marker
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_q_lnone_qipi
  defence_id: lnone_q
  benchmark: Quadrat-IPI, 2,000 documents, summarisation task
  attacker_type: static
  attacker_access: black_box
  judge: injection compliance
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_q_lnone_pia
  defence_id: lnone_q
  benchmark: PIArena HotpotQA and NQ
  attacker_type: static
  attacker_access: black_box
  judge: behaviour-based PIArena scoring
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_q_lnone_piau
  defence_id: lnone_q
  benchmark: PIArena HotpotQA and NQ
  attacker_type: static
  attacker_access: black_box
  judge: answer accuracy
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_q_lnone_sq
  defence_id: lnone_q
  benchmark: PIArena SQuAD v2
  attacker_type: static
  attacker_access: black_box
  judge: behaviour-based PIArena scoring
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_q_dpo_sep
  defence_id: dpo_q
  benchmark: SEP, 9,160 items
  attacker_type: static
  attacker_access: black_box
  judge: corrected SEP grader
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
  sample_size: 9,160
- id: ev_q_dpo_util
  defence_id: dpo_q
  benchmark: SEP, 9,160 items
  attacker_type: none
  attacker_access: not_applicable
  judge: corrected SEP grader
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
  sample_size: 9,160
- id: ev_q_dpo_af
  defence_id: dpo_q
  benchmark: AlpacaFarm, StruQ's four static attacks
  attacker_type: static
  attacker_access: black_box
  judge: response begins with the injected marker
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_q_dpo_qipi
  defence_id: dpo_q
  benchmark: Quadrat-IPI, 2,000 documents, summarisation task
  attacker_type: static
  attacker_access: black_box
  judge: injection compliance
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_q_dpo_pia
  defence_id: dpo_q
  benchmark: PIArena HotpotQA and NQ
  attacker_type: static
  attacker_access: black_box
  judge: behaviour-based PIArena scoring
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_q_dpo_piau
  defence_id: dpo_q
  benchmark: PIArena HotpotQA and NQ
  attacker_type: static
  attacker_access: black_box
  judge: answer accuracy
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_q_dpo_sq
  defence_id: dpo_q
  benchmark: PIArena SQuAD v2
  attacker_type: static
  attacker_access: black_box
  judge: behaviour-based PIArena scoring
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_q_plv_sep
  defence_id: plv_q
  benchmark: SEP, 9,160 items
  attacker_type: static
  attacker_access: black_box
  judge: corrected SEP grader
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
  sample_size: 9,160
- id: ev_q_plv_util
  defence_id: plv_q
  benchmark: SEP, 9,160 items
  attacker_type: none
  attacker_access: not_applicable
  judge: corrected SEP grader
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
  sample_size: 9,160
- id: ev_q_plv_af
  defence_id: plv_q
  benchmark: AlpacaFarm, StruQ's four static attacks
  attacker_type: static
  attacker_access: black_box
  judge: response begins with the injected marker
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_q_plv_qipi
  defence_id: plv_q
  benchmark: Quadrat-IPI, 2,000 documents, summarisation task
  attacker_type: static
  attacker_access: black_box
  judge: injection compliance
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_q_plv_pia
  defence_id: plv_q
  benchmark: PIArena HotpotQA and NQ
  attacker_type: static
  attacker_access: black_box
  judge: behaviour-based PIArena scoring
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_q_plv_piau
  defence_id: plv_q
  benchmark: PIArena HotpotQA and NQ
  attacker_type: static
  attacker_access: black_box
  judge: answer accuracy
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_q_plv_sq
  defence_id: plv_q
  benchmark: PIArena SQuAD v2
  attacker_type: static
  attacker_access: black_box
  judge: behaviour-based PIArena scoring
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_q_loreft_sep
  defence_id: loreft_q
  benchmark: SEP, 9,160 items
  attacker_type: static
  attacker_access: black_box
  judge: corrected SEP grader
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
  sample_size: 9,160
- id: ev_q_loreft_util
  defence_id: loreft_q
  benchmark: SEP, 9,160 items
  attacker_type: none
  attacker_access: not_applicable
  judge: corrected SEP grader
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
  sample_size: 9,160
- id: ev_q_loreft_af
  defence_id: loreft_q
  benchmark: AlpacaFarm, StruQ's four static attacks
  attacker_type: static
  attacker_access: black_box
  judge: response begins with the injected marker
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_q_loreft_qipi
  defence_id: loreft_q
  benchmark: Quadrat-IPI, 2,000 documents, summarisation task
  attacker_type: static
  attacker_access: black_box
  judge: injection compliance
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_q_loreft_ad
  defence_id: loreft_q
  benchmark: AgentDojo, every tool output marked
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checkers
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_q_loreft_pia
  defence_id: loreft_q
  benchmark: PIArena HotpotQA and NQ
  attacker_type: static
  attacker_access: black_box
  judge: behaviour-based PIArena scoring
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_q_loreft_piau
  defence_id: loreft_q
  benchmark: PIArena HotpotQA and NQ
  attacker_type: static
  attacker_access: black_box
  judge: answer accuracy
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_q_loreft_sq
  defence_id: loreft_q
  benchmark: PIArena SQuAD v2
  attacker_type: static
  attacker_access: black_box
  judge: behaviour-based PIArena scoring
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_q_ov_sep
  defence_id: ov_q
  benchmark: SEP, 9,160 items
  attacker_type: static
  attacker_access: black_box
  judge: corrected SEP grader
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
  sample_size: 9,160
- id: ev_q_ov_util
  defence_id: ov_q
  benchmark: SEP, 9,160 items
  attacker_type: none
  attacker_access: not_applicable
  judge: corrected SEP grader
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
  sample_size: 9,160
- id: ev_q_ov_af
  defence_id: ov_q
  benchmark: AlpacaFarm, StruQ's four static attacks
  attacker_type: static
  attacker_access: black_box
  judge: response begins with the injected marker
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_q_ov_qipi
  defence_id: ov_q
  benchmark: Quadrat-IPI, 2,000 documents, summarisation task
  attacker_type: static
  attacker_access: black_box
  judge: injection compliance
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_q_ov_ad
  defence_id: ov_q
  benchmark: AgentDojo, every tool output marked
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checkers
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_q_ov_pia
  defence_id: ov_q
  benchmark: PIArena HotpotQA and NQ
  attacker_type: static
  attacker_access: black_box
  judge: behaviour-based PIArena scoring
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_q_ov_piau
  defence_id: ov_q
  benchmark: PIArena HotpotQA and NQ
  attacker_type: static
  attacker_access: black_box
  judge: answer accuracy
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_q_ov_sq
  defence_id: ov_q
  benchmark: PIArena SQuAD v2
  attacker_type: static
  attacker_access: black_box
  judge: behaviour-based PIArena scoring
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_l_none_sep
  defence_id: none:Llama-3.1-8B-Instruct (frozen)
  benchmark: SEP, 300 items
  attacker_type: static
  attacker_access: black_box
  judge: corrected SEP grader
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
  sample_size: '300'
  baseline_kind: off_the_shelf
- id: ev_l_none_util
  defence_id: none:Llama-3.1-8B-Instruct (frozen)
  benchmark: SEP, 300 items
  attacker_type: none
  attacker_access: not_applicable
  judge: corrected SEP grader
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
  sample_size: '300'
  baseline_kind: off_the_shelf
- id: ev_l_none_pia
  defence_id: none:Llama-3.1-8B-Instruct (frozen)
  benchmark: PIArena HotpotQA and NQ
  attacker_type: static
  attacker_access: black_box
  judge: behaviour-based PIArena scoring
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
  baseline_kind: off_the_shelf
- id: ev_l_none_piau
  defence_id: none:Llama-3.1-8B-Instruct (frozen)
  benchmark: PIArena HotpotQA and NQ
  attacker_type: static
  attacker_access: black_box
  judge: answer accuracy
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
  baseline_kind: off_the_shelf
- id: ev_l_none_sq
  defence_id: none:Llama-3.1-8B-Instruct (frozen)
  benchmark: PIArena SQuAD v2
  attacker_type: static
  attacker_access: black_box
  judge: behaviour-based PIArena scoring
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
  baseline_kind: off_the_shelf
- id: ev_l_spot_sep
  defence_id: spot_l
  benchmark: SEP, 300 items
  attacker_type: static
  attacker_access: black_box
  judge: corrected SEP grader
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
  sample_size: '300'
- id: ev_l_spot_util
  defence_id: spot_l
  benchmark: SEP, 300 items
  attacker_type: none
  attacker_access: not_applicable
  judge: corrected SEP grader
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
  sample_size: '300'
- id: ev_l_spot_pia
  defence_id: spot_l
  benchmark: PIArena HotpotQA and NQ
  attacker_type: static
  attacker_access: black_box
  judge: behaviour-based PIArena scoring
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_l_spot_piau
  defence_id: spot_l
  benchmark: PIArena HotpotQA and NQ
  attacker_type: static
  attacker_access: black_box
  judge: answer accuracy
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_l_spot_sq
  defence_id: spot_l
  benchmark: PIArena SQuAD v2
  attacker_type: static
  attacker_access: black_box
  judge: behaviour-based PIArena scoring
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_l_dt_sep
  defence_id: dt_l
  benchmark: SEP, 300 items
  attacker_type: static
  attacker_access: black_box
  judge: corrected SEP grader
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
  sample_size: '300'
- id: ev_l_dt_util
  defence_id: dt_l
  benchmark: SEP, 300 items
  attacker_type: none
  attacker_access: not_applicable
  judge: corrected SEP grader
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
  sample_size: '300'
- id: ev_l_dt_pia
  defence_id: dt_l
  benchmark: PIArena HotpotQA and NQ
  attacker_type: static
  attacker_access: black_box
  judge: behaviour-based PIArena scoring
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_l_dt_piau
  defence_id: dt_l
  benchmark: PIArena HotpotQA and NQ
  attacker_type: static
  attacker_access: black_box
  judge: answer accuracy
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_l_dt_sq
  defence_id: dt_l
  benchmark: PIArena SQuAD v2
  attacker_type: static
  attacker_access: black_box
  judge: behaviour-based PIArena scoring
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_l_msa_sep
  defence_id: msa_l
  benchmark: SEP, 300 items
  attacker_type: static
  attacker_access: black_box
  judge: corrected SEP grader
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
  sample_size: '300'
- id: ev_l_msa_util
  defence_id: msa_l
  benchmark: SEP, 300 items
  attacker_type: none
  attacker_access: not_applicable
  judge: corrected SEP grader
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
  sample_size: '300'
- id: ev_l_msa_pia
  defence_id: msa_l
  benchmark: PIArena HotpotQA and NQ
  attacker_type: static
  attacker_access: black_box
  judge: behaviour-based PIArena scoring
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_l_msa_piau
  defence_id: msa_l
  benchmark: PIArena HotpotQA and NQ
  attacker_type: static
  attacker_access: black_box
  judge: answer accuracy
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_l_msa_sq
  defence_id: msa_l
  benchmark: PIArena SQuAD v2
  attacker_type: static
  attacker_access: black_box
  judge: behaviour-based PIArena scoring
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_l_ov_sep
  defence_id: ov_l
  benchmark: SEP, 1,000 items
  attacker_type: static
  attacker_access: black_box
  judge: corrected SEP grader
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
  sample_size: 1,000
- id: ev_l_ov_util
  defence_id: ov_l
  benchmark: SEP, 1,000 items
  attacker_type: none
  attacker_access: not_applicable
  judge: corrected SEP grader
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
  sample_size: 1,000
- id: ev_l_ov_pia
  defence_id: ov_l
  benchmark: PIArena HotpotQA and NQ
  attacker_type: static
  attacker_access: black_box
  judge: behaviour-based PIArena scoring
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_l_ov_piau
  defence_id: ov_l
  benchmark: PIArena HotpotQA and NQ
  attacker_type: static
  attacker_access: black_box
  judge: answer accuracy
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
- id: ev_l_ov_sq
  defence_id: ov_l
  benchmark: PIArena SQuAD v2
  attacker_type: static
  attacker_access: black_box
  judge: behaviour-based PIArena scoring
  layout_same_as_defence: 'yes'
  measured_by: overlays_arxiv2026
  page: 6
  decoding: greedy decoding
results:
- id: r_q_none_sep
  evaluation_id: ev_q_none_sep
  quantity: other
  coverage: direct
  metric_as_printed: SEP separation (corrected grader)
  value: '24.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_none_util
  evaluation_id: ev_q_none_util
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP utility
  value: '92.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_none_af
  evaluation_id: ev_q_none_af
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm ASR
  value: '99.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_none_qipi
  evaluation_id: ev_q_none_qipi
  quantity: R1
  coverage: direct
  metric_as_printed: Quadrat-IPI injection compliance
  value: '6.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_none_ad
  evaluation_id: ev_q_none_ad
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '12.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_none_pia
  evaluation_id: ev_q_none_pia
  quantity: R1
  coverage: direct
  metric_as_printed: PIArena Hot/NQ ASR
  value: '53.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_none_piau
  evaluation_id: ev_q_none_piau
  quantity: R2attack
  coverage: direct
  metric_as_printed: PIArena utility (accuracy under attack)
  value: '68.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_none_sq
  evaluation_id: ev_q_none_sq
  quantity: R1
  coverage: direct
  metric_as_printed: PIArena SQuAD ASR
  value: '97.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_sand_sep
  evaluation_id: ev_q_sand_sep
  quantity: other
  coverage: direct
  metric_as_printed: SEP separation (corrected grader)
  value: '40.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_sand_util
  evaluation_id: ev_q_sand_util
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP utility
  value: '93.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_sand_af
  evaluation_id: ev_q_sand_af
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm ASR
  value: '97.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_sand_qipi
  evaluation_id: ev_q_sand_qipi
  quantity: R1
  coverage: direct
  metric_as_printed: Quadrat-IPI injection compliance
  value: '5.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_sand_pia
  evaluation_id: ev_q_sand_pia
  quantity: R1
  coverage: direct
  metric_as_printed: PIArena Hot/NQ ASR
  value: '42.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_sand_piau
  evaluation_id: ev_q_sand_piau
  quantity: R2attack
  coverage: direct
  metric_as_printed: PIArena utility (accuracy under attack)
  value: '69.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_sand_sq
  evaluation_id: ev_q_sand_sq
  quantity: R1
  coverage: direct
  metric_as_printed: PIArena SQuAD ASR
  value: '95.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_spot_sep
  evaluation_id: ev_q_spot_sep
  quantity: other
  coverage: direct
  metric_as_printed: SEP separation (corrected grader)
  value: '47.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_spot_util
  evaluation_id: ev_q_spot_util
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP utility
  value: '93.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_spot_af
  evaluation_id: ev_q_spot_af
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm ASR
  value: '84.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_spot_qipi
  evaluation_id: ev_q_spot_qipi
  quantity: R1
  coverage: direct
  metric_as_printed: Quadrat-IPI injection compliance
  value: '3.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_spot_pia
  evaluation_id: ev_q_spot_pia
  quantity: R1
  coverage: direct
  metric_as_printed: PIArena Hot/NQ ASR
  value: '32.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_spot_piau
  evaluation_id: ev_q_spot_piau
  quantity: R2attack
  coverage: direct
  metric_as_printed: PIArena utility (accuracy under attack)
  value: '71.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_spot_sq
  evaluation_id: ev_q_spot_sq
  quantity: R1
  coverage: direct
  metric_as_printed: PIArena SQuAD ASR
  value: '48.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_ltok_sep
  evaluation_id: ev_q_ltok_sep
  quantity: other
  coverage: direct
  metric_as_printed: SEP separation (corrected grader)
  value: '98.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_ltok_util
  evaluation_id: ev_q_ltok_util
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP utility
  value: '68.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_ltok_af
  evaluation_id: ev_q_ltok_af
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm ASR
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_ltok_qipi
  evaluation_id: ev_q_ltok_qipi
  quantity: R1
  coverage: direct
  metric_as_printed: Quadrat-IPI injection compliance
  value: '0.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_ltok_pia
  evaluation_id: ev_q_ltok_pia
  quantity: R1
  coverage: direct
  metric_as_printed: PIArena Hot/NQ ASR
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_ltok_piau
  evaluation_id: ev_q_ltok_piau
  quantity: R2attack
  coverage: direct
  metric_as_printed: PIArena utility (accuracy under attack)
  value: '73.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_ltok_sq
  evaluation_id: ev_q_ltok_sq
  quantity: R1
  coverage: direct
  metric_as_printed: PIArena SQuAD ASR
  value: '7.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_ltxt_sep
  evaluation_id: ev_q_ltxt_sep
  quantity: other
  coverage: direct
  metric_as_printed: SEP separation (corrected grader)
  value: '97.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_ltxt_util
  evaluation_id: ev_q_ltxt_util
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP utility
  value: '80.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_ltxt_af
  evaluation_id: ev_q_ltxt_af
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm ASR
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_ltxt_qipi
  evaluation_id: ev_q_ltxt_qipi
  quantity: R1
  coverage: direct
  metric_as_printed: Quadrat-IPI injection compliance
  value: '0.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_ltxt_pia
  evaluation_id: ev_q_ltxt_pia
  quantity: R1
  coverage: direct
  metric_as_printed: PIArena Hot/NQ ASR
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_ltxt_piau
  evaluation_id: ev_q_ltxt_piau
  quantity: R2attack
  coverage: direct
  metric_as_printed: PIArena utility (accuracy under attack)
  value: '74.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_ltxt_sq
  evaluation_id: ev_q_ltxt_sq
  quantity: R1
  coverage: direct
  metric_as_printed: PIArena SQuAD ASR
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_lnone_sep
  evaluation_id: ev_q_lnone_sep
  quantity: other
  coverage: direct
  metric_as_printed: SEP separation (corrected grader)
  value: '97.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_lnone_util
  evaluation_id: ev_q_lnone_util
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP utility
  value: '63.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_lnone_af
  evaluation_id: ev_q_lnone_af
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm ASR
  value: '14.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_lnone_qipi
  evaluation_id: ev_q_lnone_qipi
  quantity: R1
  coverage: direct
  metric_as_printed: Quadrat-IPI injection compliance
  value: '0.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_lnone_pia
  evaluation_id: ev_q_lnone_pia
  quantity: R1
  coverage: direct
  metric_as_printed: PIArena Hot/NQ ASR
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_lnone_piau
  evaluation_id: ev_q_lnone_piau
  quantity: R2attack
  coverage: direct
  metric_as_printed: PIArena utility (accuracy under attack)
  value: '72.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_lnone_sq
  evaluation_id: ev_q_lnone_sq
  quantity: R1
  coverage: direct
  metric_as_printed: PIArena SQuAD ASR
  value: '3.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_dpo_sep
  evaluation_id: ev_q_dpo_sep
  quantity: other
  coverage: direct
  metric_as_printed: SEP separation (corrected grader)
  value: '95.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_dpo_util
  evaluation_id: ev_q_dpo_util
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP utility
  value: '14.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_dpo_af
  evaluation_id: ev_q_dpo_af
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm ASR
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_dpo_qipi
  evaluation_id: ev_q_dpo_qipi
  quantity: R1
  coverage: direct
  metric_as_printed: Quadrat-IPI injection compliance
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_dpo_pia
  evaluation_id: ev_q_dpo_pia
  quantity: R1
  coverage: direct
  metric_as_printed: PIArena Hot/NQ ASR
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_dpo_piau
  evaluation_id: ev_q_dpo_piau
  quantity: R2attack
  coverage: direct
  metric_as_printed: PIArena utility (accuracy under attack)
  value: '63.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_dpo_sq
  evaluation_id: ev_q_dpo_sq
  quantity: R1
  coverage: direct
  metric_as_printed: PIArena SQuAD ASR
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_plv_sep
  evaluation_id: ev_q_plv_sep
  quantity: other
  coverage: direct
  metric_as_printed: SEP separation (corrected grader)
  value: '98.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_plv_util
  evaluation_id: ev_q_plv_util
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP utility
  value: '92.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_plv_af
  evaluation_id: ev_q_plv_af
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm ASR
  value: '2.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_plv_qipi
  evaluation_id: ev_q_plv_qipi
  quantity: R1
  coverage: direct
  metric_as_printed: Quadrat-IPI injection compliance
  value: '0.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_plv_pia
  evaluation_id: ev_q_plv_pia
  quantity: R1
  coverage: direct
  metric_as_printed: PIArena Hot/NQ ASR
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_plv_piau
  evaluation_id: ev_q_plv_piau
  quantity: R2attack
  coverage: direct
  metric_as_printed: PIArena utility (accuracy under attack)
  value: '75.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_plv_sq
  evaluation_id: ev_q_plv_sq
  quantity: R1
  coverage: direct
  metric_as_printed: PIArena SQuAD ASR
  value: '0.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_loreft_sep
  evaluation_id: ev_q_loreft_sep
  quantity: other
  coverage: direct
  metric_as_printed: SEP separation (corrected grader)
  value: '99.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_loreft_util
  evaluation_id: ev_q_loreft_util
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP utility
  value: '92.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_loreft_af
  evaluation_id: ev_q_loreft_af
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm ASR
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_loreft_qipi
  evaluation_id: ev_q_loreft_qipi
  quantity: R1
  coverage: direct
  metric_as_printed: Quadrat-IPI injection compliance
  value: '0.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_loreft_ad
  evaluation_id: ev_q_loreft_ad
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_loreft_pia
  evaluation_id: ev_q_loreft_pia
  quantity: R1
  coverage: direct
  metric_as_printed: PIArena Hot/NQ ASR
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_loreft_piau
  evaluation_id: ev_q_loreft_piau
  quantity: R2attack
  coverage: direct
  metric_as_printed: PIArena utility (accuracy under attack)
  value: '70.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_loreft_sq
  evaluation_id: ev_q_loreft_sq
  quantity: R1
  coverage: direct
  metric_as_printed: PIArena SQuAD ASR
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_ov_sep
  evaluation_id: ev_q_ov_sep
  quantity: other
  coverage: direct
  metric_as_printed: SEP separation (corrected grader)
  value: '98.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_ov_util
  evaluation_id: ev_q_ov_util
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP utility
  value: '93.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_ov_af
  evaluation_id: ev_q_ov_af
  quantity: R1
  coverage: direct
  metric_as_printed: AlpacaFarm ASR
  value: '0.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_ov_qipi
  evaluation_id: ev_q_ov_qipi
  quantity: R1
  coverage: direct
  metric_as_printed: Quadrat-IPI injection compliance
  value: '0.05'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_ov_ad
  evaluation_id: ev_q_ov_ad
  quantity: R1
  coverage: direct
  metric_as_printed: AgentDojo ASR
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_ov_pia
  evaluation_id: ev_q_ov_pia
  quantity: R1
  coverage: direct
  metric_as_printed: PIArena Hot/NQ ASR
  value: '0.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_ov_piau
  evaluation_id: ev_q_ov_piau
  quantity: R2attack
  coverage: direct
  metric_as_printed: PIArena utility (accuracy under attack)
  value: '74.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_q_ov_sq
  evaluation_id: ev_q_ov_sq
  quantity: R1
  coverage: direct
  metric_as_printed: PIArena SQuAD ASR
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l_none_sep
  evaluation_id: ev_l_none_sep
  quantity: other
  coverage: direct
  metric_as_printed: SEP separation (corrected grader)
  value: '29.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l_none_util
  evaluation_id: ev_l_none_util
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP utility
  value: '91.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l_none_pia
  evaluation_id: ev_l_none_pia
  quantity: R1
  coverage: direct
  metric_as_printed: PIArena Hot/NQ ASR
  value: '59.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l_none_piau
  evaluation_id: ev_l_none_piau
  quantity: R2attack
  coverage: direct
  metric_as_printed: PIArena utility (accuracy under attack)
  value: '68.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l_none_sq
  evaluation_id: ev_l_none_sq
  quantity: R1
  coverage: direct
  metric_as_printed: PIArena SQuAD ASR
  value: '95.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l_spot_sep
  evaluation_id: ev_l_spot_sep
  quantity: other
  coverage: direct
  metric_as_printed: SEP separation (corrected grader)
  value: '66.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l_spot_util
  evaluation_id: ev_l_spot_util
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP utility
  value: '91.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l_spot_pia
  evaluation_id: ev_l_spot_pia
  quantity: R1
  coverage: direct
  metric_as_printed: PIArena Hot/NQ ASR
  value: '30.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l_spot_piau
  evaluation_id: ev_l_spot_piau
  quantity: R2attack
  coverage: direct
  metric_as_printed: PIArena utility (accuracy under attack)
  value: '68.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l_spot_sq
  evaluation_id: ev_l_spot_sq
  quantity: R1
  coverage: direct
  metric_as_printed: PIArena SQuAD ASR
  value: '39.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l_dt_sep
  evaluation_id: ev_l_dt_sep
  quantity: other
  coverage: direct
  metric_as_printed: SEP separation (corrected grader)
  value: '80.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l_dt_util
  evaluation_id: ev_l_dt_util
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP utility
  value: '55.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l_dt_pia
  evaluation_id: ev_l_dt_pia
  quantity: R1
  coverage: direct
  metric_as_printed: PIArena Hot/NQ ASR
  value: '3.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l_dt_piau
  evaluation_id: ev_l_dt_piau
  quantity: R2attack
  coverage: direct
  metric_as_printed: PIArena utility (accuracy under attack)
  value: '71.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l_dt_sq
  evaluation_id: ev_l_dt_sq
  quantity: R1
  coverage: direct
  metric_as_printed: PIArena SQuAD ASR
  value: '17.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l_msa_sep
  evaluation_id: ev_l_msa_sep
  quantity: other
  coverage: direct
  metric_as_printed: SEP separation (corrected grader)
  value: '97.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l_msa_util
  evaluation_id: ev_l_msa_util
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP utility
  value: '78.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l_msa_pia
  evaluation_id: ev_l_msa_pia
  quantity: R1
  coverage: direct
  metric_as_printed: PIArena Hot/NQ ASR
  value: '1.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l_msa_piau
  evaluation_id: ev_l_msa_piau
  quantity: R2attack
  coverage: direct
  metric_as_printed: PIArena utility (accuracy under attack)
  value: '76.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l_msa_sq
  evaluation_id: ev_l_msa_sq
  quantity: R1
  coverage: direct
  metric_as_printed: PIArena SQuAD ASR
  value: '0.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l_ov_sep
  evaluation_id: ev_l_ov_sep
  quantity: other
  coverage: direct
  metric_as_printed: SEP separation (corrected grader)
  value: '99.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l_ov_util
  evaluation_id: ev_l_ov_util
  quantity: R2clean
  coverage: proxy
  metric_as_printed: SEP utility
  value: '89.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l_ov_pia
  evaluation_id: ev_l_ov_pia
  quantity: R1
  coverage: direct
  metric_as_printed: PIArena Hot/NQ ASR
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l_ov_piau
  evaluation_id: ev_l_ov_piau
  quantity: R2attack
  coverage: direct
  metric_as_printed: PIArena utility (accuracy under attack)
  value: '70.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_l_ov_sq
  evaluation_id: ev_l_ov_sq
  quantity: R1
  coverage: direct
  metric_as_printed: PIArena SQuAD ASR
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
comparisons:
- id: c_q_sand_sep
  left: ev_q_sand_sep
  right: ev_q_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_sand_util
  left: ev_q_sand_util
  right: ev_q_none_util
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_sand_af
  left: ev_q_sand_af
  right: ev_q_none_af
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_sand_qipi
  left: ev_q_sand_qipi
  right: ev_q_none_qipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_sand_pia
  left: ev_q_sand_pia
  right: ev_q_none_pia
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_sand_piau
  left: ev_q_sand_piau
  right: ev_q_none_piau
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_sand_sq
  left: ev_q_sand_sq
  right: ev_q_none_sq
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_spot_sep
  left: ev_q_spot_sep
  right: ev_q_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_spot_util
  left: ev_q_spot_util
  right: ev_q_none_util
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_spot_af
  left: ev_q_spot_af
  right: ev_q_none_af
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_spot_qipi
  left: ev_q_spot_qipi
  right: ev_q_none_qipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_spot_pia
  left: ev_q_spot_pia
  right: ev_q_none_pia
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_spot_piau
  left: ev_q_spot_piau
  right: ev_q_none_piau
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_spot_sq
  left: ev_q_spot_sq
  right: ev_q_none_sq
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_ltok_sep
  left: ev_q_ltok_sep
  right: ev_q_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_ltok_util
  left: ev_q_ltok_util
  right: ev_q_none_util
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_ltok_af
  left: ev_q_ltok_af
  right: ev_q_none_af
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_ltok_qipi
  left: ev_q_ltok_qipi
  right: ev_q_none_qipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_ltok_pia
  left: ev_q_ltok_pia
  right: ev_q_none_pia
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_ltok_piau
  left: ev_q_ltok_piau
  right: ev_q_none_piau
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_ltok_sq
  left: ev_q_ltok_sq
  right: ev_q_none_sq
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_ltxt_sep
  left: ev_q_ltxt_sep
  right: ev_q_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_ltxt_util
  left: ev_q_ltxt_util
  right: ev_q_none_util
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_ltxt_af
  left: ev_q_ltxt_af
  right: ev_q_none_af
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_ltxt_qipi
  left: ev_q_ltxt_qipi
  right: ev_q_none_qipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_ltxt_pia
  left: ev_q_ltxt_pia
  right: ev_q_none_pia
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_ltxt_piau
  left: ev_q_ltxt_piau
  right: ev_q_none_piau
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_ltxt_sq
  left: ev_q_ltxt_sq
  right: ev_q_none_sq
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_lnone_sep
  left: ev_q_lnone_sep
  right: ev_q_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_lnone_util
  left: ev_q_lnone_util
  right: ev_q_none_util
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_lnone_af
  left: ev_q_lnone_af
  right: ev_q_none_af
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_lnone_qipi
  left: ev_q_lnone_qipi
  right: ev_q_none_qipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_lnone_pia
  left: ev_q_lnone_pia
  right: ev_q_none_pia
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_lnone_piau
  left: ev_q_lnone_piau
  right: ev_q_none_piau
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_lnone_sq
  left: ev_q_lnone_sq
  right: ev_q_none_sq
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_dpo_sep
  left: ev_q_dpo_sep
  right: ev_q_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_dpo_util
  left: ev_q_dpo_util
  right: ev_q_none_util
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_dpo_af
  left: ev_q_dpo_af
  right: ev_q_none_af
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_dpo_qipi
  left: ev_q_dpo_qipi
  right: ev_q_none_qipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_dpo_pia
  left: ev_q_dpo_pia
  right: ev_q_none_pia
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_dpo_piau
  left: ev_q_dpo_piau
  right: ev_q_none_piau
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_dpo_sq
  left: ev_q_dpo_sq
  right: ev_q_none_sq
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_plv_sep
  left: ev_q_plv_sep
  right: ev_q_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_plv_util
  left: ev_q_plv_util
  right: ev_q_none_util
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_plv_af
  left: ev_q_plv_af
  right: ev_q_none_af
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_plv_qipi
  left: ev_q_plv_qipi
  right: ev_q_none_qipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_plv_pia
  left: ev_q_plv_pia
  right: ev_q_none_pia
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_plv_piau
  left: ev_q_plv_piau
  right: ev_q_none_piau
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_plv_sq
  left: ev_q_plv_sq
  right: ev_q_none_sq
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_loreft_sep
  left: ev_q_loreft_sep
  right: ev_q_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_loreft_util
  left: ev_q_loreft_util
  right: ev_q_none_util
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_loreft_af
  left: ev_q_loreft_af
  right: ev_q_none_af
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_loreft_qipi
  left: ev_q_loreft_qipi
  right: ev_q_none_qipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_loreft_ad
  left: ev_q_loreft_ad
  right: ev_q_none_ad
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_loreft_pia
  left: ev_q_loreft_pia
  right: ev_q_none_pia
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_loreft_piau
  left: ev_q_loreft_piau
  right: ev_q_none_piau
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_loreft_sq
  left: ev_q_loreft_sq
  right: ev_q_none_sq
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_ov_sep
  left: ev_q_ov_sep
  right: ev_q_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_ov_util
  left: ev_q_ov_util
  right: ev_q_none_util
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_ov_af
  left: ev_q_ov_af
  right: ev_q_none_af
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_ov_qipi
  left: ev_q_ov_qipi
  right: ev_q_none_qipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_ov_ad
  left: ev_q_ov_ad
  right: ev_q_none_ad
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_ov_pia
  left: ev_q_ov_pia
  right: ev_q_none_pia
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_ov_piau
  left: ev_q_ov_piau
  right: ev_q_none_piau
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_ov_sq
  left: ev_q_ov_sq
  right: ev_q_none_sq
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l_spot_sep
  left: ev_l_spot_sep
  right: ev_l_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l_spot_util
  left: ev_l_spot_util
  right: ev_l_none_util
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l_spot_pia
  left: ev_l_spot_pia
  right: ev_l_none_pia
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l_spot_piau
  left: ev_l_spot_piau
  right: ev_l_none_piau
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l_spot_sq
  left: ev_l_spot_sq
  right: ev_l_none_sq
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l_dt_sep
  left: ev_l_dt_sep
  right: ev_l_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l_dt_util
  left: ev_l_dt_util
  right: ev_l_none_util
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l_dt_pia
  left: ev_l_dt_pia
  right: ev_l_none_pia
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l_dt_piau
  left: ev_l_dt_piau
  right: ev_l_none_piau
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l_dt_sq
  left: ev_l_dt_sq
  right: ev_l_none_sq
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l_msa_sep
  left: ev_l_msa_sep
  right: ev_l_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l_msa_util
  left: ev_l_msa_util
  right: ev_l_none_util
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l_msa_pia
  left: ev_l_msa_pia
  right: ev_l_none_pia
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l_msa_piau
  left: ev_l_msa_piau
  right: ev_l_none_piau
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l_msa_sq
  left: ev_l_msa_sq
  right: ev_l_none_sq
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l_ov_sep
  left: ev_l_ov_sep
  right: ev_l_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: unmatched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l_ov_util
  left: ev_l_ov_util
  right: ev_l_none_util
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: unmatched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l_ov_pia
  left: ev_l_ov_pia
  right: ev_l_none_pia
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l_ov_piau
  left: ev_l_ov_piau
  right: ev_l_none_piau
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l_ov_sq
  left: ev_l_ov_sq
  right: ev_l_none_sq
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q_ov_ltok_sep
  left: ev_q_ov_sep
  right: ev_q_ltok_sep
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: out-of-band residual overlay against an in-band special-token LoRA, same data
  page: 6
- id: c_q_ov_loreft_sep
  left: ev_q_ov_sep
  right: ev_q_loreft_sep
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: out-of-band residual overlay against a LoReFT edit on the same span, same data
  page: 6
- id: c_q_ov_ltok_util
  left: ev_q_ov_util
  right: ev_q_ltok_util
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: out-of-band residual overlay against an in-band special-token LoRA, same data
  page: 6
- id: c_q_ov_loreft_util
  left: ev_q_ov_util
  right: ev_q_loreft_util
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: out-of-band residual overlay against a LoReFT edit on the same span, same data
  page: 6
- id: c_q_ov_ltok_af
  left: ev_q_ov_af
  right: ev_q_ltok_af
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: out-of-band residual overlay against an in-band special-token LoRA, same data
  page: 6
- id: c_q_ov_loreft_af
  left: ev_q_ov_af
  right: ev_q_loreft_af
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: out-of-band residual overlay against a LoReFT edit on the same span, same data
  page: 6
- id: c_q_ov_ltok_qipi
  left: ev_q_ov_qipi
  right: ev_q_ltok_qipi
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: out-of-band residual overlay against an in-band special-token LoRA, same data
  page: 6
- id: c_q_ov_loreft_qipi
  left: ev_q_ov_qipi
  right: ev_q_loreft_qipi
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: out-of-band residual overlay against a LoReFT edit on the same span, same data
  page: 6
- id: c_q_ov_loreft_ad
  left: ev_q_ov_ad
  right: ev_q_loreft_ad
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: out-of-band residual overlay against a LoReFT edit on the same span, same data
  page: 6
- id: c_q_ov_ltok_pia
  left: ev_q_ov_pia
  right: ev_q_ltok_pia
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: out-of-band residual overlay against an in-band special-token LoRA, same data
  page: 6
- id: c_q_ov_loreft_pia
  left: ev_q_ov_pia
  right: ev_q_loreft_pia
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: out-of-band residual overlay against a LoReFT edit on the same span, same data
  page: 6
- id: c_q_ov_ltok_piau
  left: ev_q_ov_piau
  right: ev_q_ltok_piau
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: out-of-band residual overlay against an in-band special-token LoRA, same data
  page: 6
- id: c_q_ov_loreft_piau
  left: ev_q_ov_piau
  right: ev_q_loreft_piau
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: out-of-band residual overlay against a LoReFT edit on the same span, same data
  page: 6
- id: c_q_ov_ltok_sq
  left: ev_q_ov_sq
  right: ev_q_ltok_sq
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: out-of-band residual overlay against an in-band special-token LoRA, same data
  page: 6
- id: c_q_ov_loreft_sq
  left: ev_q_ov_sq
  right: ev_q_loreft_sq
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: out-of-band residual overlay against a LoReFT edit on the same span, same data
  page: 6
claims:
- id: k1
  text: against attacks written without knowledge of the defence the overlay matches the best published defences on a frozen model
  location: abstract
  anchor: AlpacaFarm attack success falls from 99.0% to 0.5%
  jobs:
  - R1
  - R2
  supported_by:
  - c_q_ov_sep
  - c_q_ov_af
  - c_q_ov_ad
  - c_q_ov_util
  page: 1
coverage:
  R1: direct
  R2clean: proxy
  R2attack: direct
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 1 (AlpacaFarm, Quadrat-IPI, AgentDojo, PIArena)
    R2clean: Table 1 (SEP utility)
    R2attack: Table 1 (PIArena accuracy under attack)
schema_gaps:
- where: comparisons c_l_ov_sep, c_l_ov_util
  note: for Llama, SEP is scored on a fixed 300-item subset for every row except the overlay, which uses 1,000 items, and the Llama overlay was trained on an earlier data mix (p. 6)
- where: defences ov_*
  note: training includes copy items that quote the marked span; the paper does not say whether those spans carry instructions, so trains_carry_through is coded no
- where: evaluations ev_*
  note: TensorTrust hijacking is left out because its attacker writes the user's own input; the paper notes that SEP's items resemble the training data while the other benchmarks share no attack text with it (p. 5)

`coer_arxiv2026` (2609.07529)

In [ ]:
%%writefile reports/records/coer_arxiv2026.yaml
report:
  id: coer_arxiv2026
  slug: '2609.07529'
  title: 'CoER: Defending against Adaptive Indirect Prompt Injection via Adversarial Co-Evolution and Refinement'
  first_author: Boyang Zhang
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2609.07529'
  version_read: arXiv v3
  first_date: '2026-09-07'
  read: F
  role: proposes
  code_release: not checked
  extraction_scope: 'Section 5: Table 1(a) (same-backbone defenders, means over three evaluation seeds) and the Qwen3.5-9B rows of Table 2 (external benchmarks); the abstract claim (Table 1(b), cross-model references from
    other protocols, Table 1(c) attribution study and the appendix sensitivity tables not coded)'
defences:
- id: ppo_q
  name: PPO defender trained against fixed templates
  origin: this_report
  base_checkpoint: Qwen3.5-9B
  axis1:
  - no_signal
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  layout: agent loop over AgentDojo and AgentDyn suites; tool outputs carry the injection
  layout_compat: &id001
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: PPO trains only the defender against
- id: grpo_q
  name: GRPO defender trained against fixed templates
  origin: this_report
  base_checkpoint: Qwen3.5-9B
  axis1:
  - no_signal
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  layout: agent loop over AgentDojo and AgentDyn suites; tool outputs carry the injection
  layout_compat: *id001
  anchor: GRPO replaces PPO on the same data
- id: nopop_q
  name: co-evolution without historical opponents (NoPop)
  origin: this_report
  base_checkpoint: Qwen3.5-9B
  axis1:
  - no_signal
  axis2:
  - adversarial_rl
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  layout: agent loop over AgentDojo and AgentDyn suites; tool outputs carry the injection
  layout_compat: *id001
  anchor: NoPop co-trains both roles without historical
- id: barl_q
  name: BA-RL (bilateral co-evolution with historical populations)
  origin: this_report
  base_checkpoint: Qwen3.5-9B
  axis1:
  - no_signal
  axis2:
  - adversarial_rl
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  layout: agent loop over AgentDojo and AgentDyn suites; tool outputs carry the injection
  layout_compat: *id001
  anchor: BA-RL is the selected online defender
- id: coer_q
  name: CoER (BA-RL then population-guided Defender SFT)
  origin: this_report
  base_checkpoint: Qwen3.5-9B
  axis1:
  - no_signal
  axis2:
  - adversarial_rl
  - sft
  - distillation
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  layout: agent loop over AgentDojo and AgentDyn suites; tool outputs carry the injection
  layout_compat: *id001
  anchor: CoER adds population-guided
  components:
  - defender and attacker co-trained as a general-sum Markov game with PPO; the attacker may inject several times within an execution
  - historical populations of both roles sampled during training
  - Defender SFT on 5,760 verified safe trajectories from GLM-5.2 and DeepSeek V4 Pro under the retained attackers
  pages:
    components: 6
evaluations:
- id: ev_base_clean
  defence_id: none:Qwen3.5-9B
  benchmark: main workload over three AgentDyn and four AgentDojo suites, 157 clean cases
  attacker_type: none
  attacker_access: not_applicable
  judge: 'verifier: task success and attack success indicators'
  layout_same_as_defence: 'yes'
  measured_by: coer_arxiv2026
  page: 7
  decoding: not reported
  sample_size: '157'
  pages:
    benchmark: 6
  baseline_kind: off_the_shelf
- id: ev_base_fixed
  defence_id: none:Qwen3.5-9B
  benchmark: main workload over three AgentDyn and four AgentDojo suites, 168 fixed-template cases
  attacker_type: static
  attacker_access: black_box
  judge: 'verifier: task success and attack success indicators'
  layout_same_as_defence: 'yes'
  measured_by: coer_arxiv2026
  page: 7
  decoding: not reported
  sample_size: '168'
  pages:
    benchmark: 6
  baseline_kind: off_the_shelf
- id: ev_base_adapt
  defence_id: none:Qwen3.5-9B
  benchmark: main workload over three AgentDyn and four AgentDojo suites, 1,187 cases, common adaptive attacker (frozen weights, adapts injections within each execution)
  attacker_type: adaptive
  attacker_access: black_box
  judge: 'verifier: task success and attack success indicators'
  layout_same_as_defence: 'yes'
  measured_by: coer_arxiv2026
  page: 7
  attacker_budget: multiple injections within one execution
  decoding: not reported
  sample_size: 1,187
  pages:
    benchmark: 6
  baseline_kind: off_the_shelf
- id: ev_ppo_clean
  defence_id: ppo_q
  benchmark: main workload over three AgentDyn and four AgentDojo suites, 157 clean cases
  attacker_type: none
  attacker_access: not_applicable
  judge: 'verifier: task success and attack success indicators'
  layout_same_as_defence: 'yes'
  measured_by: coer_arxiv2026
  page: 7
  decoding: not reported
  sample_size: '157'
  pages:
    benchmark: 6
- id: ev_ppo_fixed
  defence_id: ppo_q
  benchmark: main workload over three AgentDyn and four AgentDojo suites, 168 fixed-template cases
  attacker_type: static
  attacker_access: black_box
  judge: 'verifier: task success and attack success indicators'
  layout_same_as_defence: 'yes'
  measured_by: coer_arxiv2026
  page: 7
  decoding: not reported
  sample_size: '168'
  pages:
    benchmark: 6
- id: ev_ppo_adapt
  defence_id: ppo_q
  benchmark: main workload over three AgentDyn and four AgentDojo suites, 1,187 cases, common adaptive attacker (frozen weights, adapts injections within each execution)
  attacker_type: adaptive
  attacker_access: black_box
  judge: 'verifier: task success and attack success indicators'
  layout_same_as_defence: 'yes'
  measured_by: coer_arxiv2026
  page: 7
  attacker_budget: multiple injections within one execution
  decoding: not reported
  sample_size: 1,187
  pages:
    benchmark: 6
- id: ev_grpo_clean
  defence_id: grpo_q
  benchmark: main workload over three AgentDyn and four AgentDojo suites, 157 clean cases
  attacker_type: none
  attacker_access: not_applicable
  judge: 'verifier: task success and attack success indicators'
  layout_same_as_defence: 'yes'
  measured_by: coer_arxiv2026
  page: 7
  decoding: not reported
  sample_size: '157'
  pages:
    benchmark: 6
- id: ev_grpo_fixed
  defence_id: grpo_q
  benchmark: main workload over three AgentDyn and four AgentDojo suites, 168 fixed-template cases
  attacker_type: static
  attacker_access: black_box
  judge: 'verifier: task success and attack success indicators'
  layout_same_as_defence: 'yes'
  measured_by: coer_arxiv2026
  page: 7
  decoding: not reported
  sample_size: '168'
  pages:
    benchmark: 6
- id: ev_grpo_adapt
  defence_id: grpo_q
  benchmark: main workload over three AgentDyn and four AgentDojo suites, 1,187 cases, common adaptive attacker (frozen weights, adapts injections within each execution)
  attacker_type: adaptive
  attacker_access: black_box
  judge: 'verifier: task success and attack success indicators'
  layout_same_as_defence: 'yes'
  measured_by: coer_arxiv2026
  page: 7
  attacker_budget: multiple injections within one execution
  decoding: not reported
  sample_size: 1,187
  pages:
    benchmark: 6
- id: ev_nopop_clean
  defence_id: nopop_q
  benchmark: main workload over three AgentDyn and four AgentDojo suites, 157 clean cases
  attacker_type: none
  attacker_access: not_applicable
  judge: 'verifier: task success and attack success indicators'
  layout_same_as_defence: 'yes'
  measured_by: coer_arxiv2026
  page: 7
  decoding: not reported
  sample_size: '157'
  pages:
    benchmark: 6
- id: ev_nopop_fixed
  defence_id: nopop_q
  benchmark: main workload over three AgentDyn and four AgentDojo suites, 168 fixed-template cases
  attacker_type: static
  attacker_access: black_box
  judge: 'verifier: task success and attack success indicators'
  layout_same_as_defence: 'yes'
  measured_by: coer_arxiv2026
  page: 7
  decoding: not reported
  sample_size: '168'
  pages:
    benchmark: 6
- id: ev_nopop_adapt
  defence_id: nopop_q
  benchmark: main workload over three AgentDyn and four AgentDojo suites, 1,187 cases, common adaptive attacker (frozen weights, adapts injections within each execution)
  attacker_type: adaptive
  attacker_access: black_box
  judge: 'verifier: task success and attack success indicators'
  layout_same_as_defence: 'yes'
  measured_by: coer_arxiv2026
  page: 7
  attacker_budget: multiple injections within one execution
  decoding: not reported
  sample_size: 1,187
  pages:
    benchmark: 6
- id: ev_barl_clean
  defence_id: barl_q
  benchmark: main workload over three AgentDyn and four AgentDojo suites, 157 clean cases
  attacker_type: none
  attacker_access: not_applicable
  judge: 'verifier: task success and attack success indicators'
  layout_same_as_defence: 'yes'
  measured_by: coer_arxiv2026
  page: 7
  decoding: not reported
  sample_size: '157'
  pages:
    benchmark: 6
- id: ev_barl_fixed
  defence_id: barl_q
  benchmark: main workload over three AgentDyn and four AgentDojo suites, 168 fixed-template cases
  attacker_type: static
  attacker_access: black_box
  judge: 'verifier: task success and attack success indicators'
  layout_same_as_defence: 'yes'
  measured_by: coer_arxiv2026
  page: 7
  decoding: not reported
  sample_size: '168'
  pages:
    benchmark: 6
- id: ev_barl_adapt
  defence_id: barl_q
  benchmark: main workload over three AgentDyn and four AgentDojo suites, 1,187 cases, common adaptive attacker (frozen weights, adapts injections within each execution)
  attacker_type: adaptive
  attacker_access: black_box
  judge: 'verifier: task success and attack success indicators'
  layout_same_as_defence: 'yes'
  measured_by: coer_arxiv2026
  page: 7
  attacker_budget: multiple injections within one execution
  decoding: not reported
  sample_size: 1,187
  pages:
    benchmark: 6
- id: ev_coer_clean
  defence_id: coer_q
  benchmark: main workload over three AgentDyn and four AgentDojo suites, 157 clean cases
  attacker_type: none
  attacker_access: not_applicable
  judge: 'verifier: task success and attack success indicators'
  layout_same_as_defence: 'yes'
  measured_by: coer_arxiv2026
  page: 7
  decoding: not reported
  sample_size: '157'
  pages:
    benchmark: 6
- id: ev_coer_fixed
  defence_id: coer_q
  benchmark: main workload over three AgentDyn and four AgentDojo suites, 168 fixed-template cases
  attacker_type: static
  attacker_access: black_box
  judge: 'verifier: task success and attack success indicators'
  layout_same_as_defence: 'yes'
  measured_by: coer_arxiv2026
  page: 7
  decoding: not reported
  sample_size: '168'
  pages:
    benchmark: 6
- id: ev_coer_adapt
  defence_id: coer_q
  benchmark: main workload over three AgentDyn and four AgentDojo suites, 1,187 cases, common adaptive attacker (frozen weights, adapts injections within each execution)
  attacker_type: adaptive
  attacker_access: black_box
  judge: 'verifier: task success and attack success indicators'
  layout_same_as_defence: 'yes'
  measured_by: coer_arxiv2026
  page: 7
  attacker_budget: multiple injections within one execution
  decoding: not reported
  sample_size: 1,187
  pages:
    benchmark: 6
- id: ev_xadj_base
  defence_id: none:Qwen3.5-9B
  benchmark: AgentDojo, official important_instructions protocol
  attacker_type: static
  attacker_access: black_box
  judge: benchmark checkers
  layout_same_as_defence: 'yes'
  measured_by: coer_arxiv2026
  page: 8
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_xady_base
  defence_id: none:Qwen3.5-9B
  benchmark: AgentDyn, official important_instructions protocol
  attacker_type: static
  attacker_access: black_box
  judge: benchmark checkers
  layout_same_as_defence: 'yes'
  measured_by: coer_arxiv2026
  page: 8
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_xalab_base
  defence_id: none:Qwen3.5-9B
  benchmark: AgentLAB Task-Injection, 949 pairs, GPT-5.4 adaptive attacker
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark checkers
  layout_same_as_defence: 'yes'
  measured_by: coer_arxiv2026
  page: 8
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_xiab_base
  defence_id: none:Qwen3.5-9B
  benchmark: InjecAgent, Base payload
  attacker_type: static
  attacker_access: black_box
  judge: benchmark checkers
  layout_same_as_defence: 'yes'
  measured_by: coer_arxiv2026
  page: 8
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_xiae_base
  defence_id: none:Qwen3.5-9B
  benchmark: InjecAgent, Enhanced payload
  attacker_type: static
  attacker_access: black_box
  judge: benchmark checkers
  layout_same_as_defence: 'yes'
  measured_by: coer_arxiv2026
  page: 8
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_xadj_barl
  defence_id: barl_q
  benchmark: AgentDojo, official important_instructions protocol
  attacker_type: static
  attacker_access: black_box
  judge: benchmark checkers
  layout_same_as_defence: 'yes'
  measured_by: coer_arxiv2026
  page: 8
  decoding: not reported
- id: ev_xady_barl
  defence_id: barl_q
  benchmark: AgentDyn, official important_instructions protocol
  attacker_type: static
  attacker_access: black_box
  judge: benchmark checkers
  layout_same_as_defence: 'yes'
  measured_by: coer_arxiv2026
  page: 8
  decoding: not reported
- id: ev_xalab_barl
  defence_id: barl_q
  benchmark: AgentLAB Task-Injection, 949 pairs, GPT-5.4 adaptive attacker
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark checkers
  layout_same_as_defence: 'yes'
  measured_by: coer_arxiv2026
  page: 8
  decoding: not reported
- id: ev_xiab_barl
  defence_id: barl_q
  benchmark: InjecAgent, Base payload
  attacker_type: static
  attacker_access: black_box
  judge: benchmark checkers
  layout_same_as_defence: 'yes'
  measured_by: coer_arxiv2026
  page: 8
  decoding: not reported
- id: ev_xiae_barl
  defence_id: barl_q
  benchmark: InjecAgent, Enhanced payload
  attacker_type: static
  attacker_access: black_box
  judge: benchmark checkers
  layout_same_as_defence: 'yes'
  measured_by: coer_arxiv2026
  page: 8
  decoding: not reported
- id: ev_xadj_coer
  defence_id: coer_q
  benchmark: AgentDojo, official important_instructions protocol
  attacker_type: static
  attacker_access: black_box
  judge: benchmark checkers
  layout_same_as_defence: 'yes'
  measured_by: coer_arxiv2026
  page: 8
  decoding: not reported
- id: ev_xady_coer
  defence_id: coer_q
  benchmark: AgentDyn, official important_instructions protocol
  attacker_type: static
  attacker_access: black_box
  judge: benchmark checkers
  layout_same_as_defence: 'yes'
  measured_by: coer_arxiv2026
  page: 8
  decoding: not reported
- id: ev_xalab_coer
  defence_id: coer_q
  benchmark: AgentLAB Task-Injection, 949 pairs, GPT-5.4 adaptive attacker
  attacker_type: adaptive
  attacker_access: black_box
  judge: benchmark checkers
  layout_same_as_defence: 'yes'
  measured_by: coer_arxiv2026
  page: 8
  decoding: not reported
- id: ev_xiab_coer
  defence_id: coer_q
  benchmark: InjecAgent, Base payload
  attacker_type: static
  attacker_access: black_box
  judge: benchmark checkers
  layout_same_as_defence: 'yes'
  measured_by: coer_arxiv2026
  page: 8
  decoding: not reported
- id: ev_xiae_coer
  defence_id: coer_q
  benchmark: InjecAgent, Enhanced payload
  attacker_type: static
  attacker_access: black_box
  judge: benchmark checkers
  layout_same_as_defence: 'yes'
  measured_by: coer_arxiv2026
  page: 8
  decoding: not reported
results:
- id: r_base_cu
  evaluation_id: ev_base_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Clean U
  value: '77.92'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1(a)
  spread: sample SD 1.60 over evaluation seeds 0, 1, 2
- id: r_base_fasr
  evaluation_id: ev_base_fixed
  quantity: R1
  coverage: direct
  metric_as_printed: Fixed ASR
  value: '13.10'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1(a)
  spread: sample SD 0.60 over evaluation seeds 0, 1, 2
- id: r_base_au
  evaluation_id: ev_base_adapt
  quantity: R2attack
  coverage: direct
  metric_as_printed: Adaptive U
  value: '61.08'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1(a)
  spread: sample SD 0.73 over evaluation seeds 0, 1, 2
- id: r_base_aasr
  evaluation_id: ev_base_adapt
  quantity: R1
  coverage: direct
  metric_as_printed: Adaptive ASR
  value: '41.31'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1(a)
  spread: sample SD 0.89 over evaluation seeds 0, 1, 2
- id: r_base_asafe
  evaluation_id: ev_base_adapt
  quantity: R3safe
  coverage: direct
  metric_as_printed: Adaptive Safe-U (task done and attack not executed)
  value: '39.60'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1(a)
  spread: sample SD 1.11 over evaluation seeds 0, 1, 2
- id: r_ppo_cu
  evaluation_id: ev_ppo_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Clean U
  value: '75.58'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1(a)
  spread: sample SD 1.33 over evaluation seeds 0, 1, 2
- id: r_ppo_fasr
  evaluation_id: ev_ppo_fixed
  quantity: R1
  coverage: direct
  metric_as_printed: Fixed ASR
  value: '4.96'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1(a)
  spread: sample SD 1.24 over evaluation seeds 0, 1, 2
- id: r_ppo_au
  evaluation_id: ev_ppo_adapt
  quantity: R2attack
  coverage: direct
  metric_as_printed: Adaptive U
  value: '63.21'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1(a)
  spread: sample SD 2.31 over evaluation seeds 0, 1, 2
- id: r_ppo_aasr
  evaluation_id: ev_ppo_adapt
  quantity: R1
  coverage: direct
  metric_as_printed: Adaptive ASR
  value: '31.03'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1(a)
  spread: sample SD 0.42 over evaluation seeds 0, 1, 2
- id: r_ppo_asafe
  evaluation_id: ev_ppo_adapt
  quantity: R3safe
  coverage: direct
  metric_as_printed: Adaptive Safe-U (task done and attack not executed)
  value: '46.70'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1(a)
  spread: sample SD 0.63 over evaluation seeds 0, 1, 2
- id: r_grpo_cu
  evaluation_id: ev_grpo_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Clean U
  value: '78.77'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1(a)
  spread: sample SD 1.33 over evaluation seeds 0, 1, 2
- id: r_grpo_fasr
  evaluation_id: ev_grpo_fixed
  quantity: R1
  coverage: direct
  metric_as_printed: Fixed ASR
  value: '6.15'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1(a)
  spread: sample SD 0.34 over evaluation seeds 0, 1, 2
- id: r_grpo_au
  evaluation_id: ev_grpo_adapt
  quantity: R2attack
  coverage: direct
  metric_as_printed: Adaptive U
  value: '56.25'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1(a)
  spread: sample SD 0.05 over evaluation seeds 0, 1, 2
- id: r_grpo_aasr
  evaluation_id: ev_grpo_adapt
  quantity: R1
  coverage: direct
  metric_as_printed: Adaptive ASR
  value: '39.82'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1(a)
  spread: sample SD 0.27 over evaluation seeds 0, 1, 2
- id: r_grpo_asafe
  evaluation_id: ev_grpo_adapt
  quantity: R3safe
  coverage: direct
  metric_as_printed: Adaptive Safe-U (task done and attack not executed)
  value: '36.90'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1(a)
  spread: sample SD 0.53 over evaluation seeds 0, 1, 2
- id: r_nopop_cu
  evaluation_id: ev_nopop_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Clean U
  value: '80.04'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1(a)
  spread: sample SD 3.51 over evaluation seeds 0, 1, 2
- id: r_nopop_fasr
  evaluation_id: ev_nopop_fixed
  quantity: R1
  coverage: direct
  metric_as_printed: Fixed ASR
  value: '4.56'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1(a)
  spread: sample SD 0.34 over evaluation seeds 0, 1, 2
- id: r_nopop_au
  evaluation_id: ev_nopop_adapt
  quantity: R2attack
  coverage: direct
  metric_as_printed: Adaptive U
  value: '68.49'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1(a)
  spread: sample SD 1.19 over evaluation seeds 0, 1, 2
- id: r_nopop_aasr
  evaluation_id: ev_nopop_adapt
  quantity: R1
  coverage: direct
  metric_as_printed: Adaptive ASR
  value: '28.14'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1(a)
  spread: sample SD 0.30 over evaluation seeds 0, 1, 2
- id: r_nopop_asafe
  evaluation_id: ev_nopop_adapt
  quantity: R3safe
  coverage: direct
  metric_as_printed: Adaptive Safe-U (task done and attack not executed)
  value: '52.93'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1(a)
  spread: sample SD 1.05 over evaluation seeds 0, 1, 2
- id: r_barl_cu
  evaluation_id: ev_barl_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Clean U
  value: '79.19'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1(a)
  spread: sample SD 0.74 over evaluation seeds 0, 1, 2
- id: r_barl_fasr
  evaluation_id: ev_barl_fixed
  quantity: R1
  coverage: direct
  metric_as_printed: Fixed ASR
  value: '4.37'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1(a)
  spread: sample SD 1.24 over evaluation seeds 0, 1, 2
- id: r_barl_au
  evaluation_id: ev_barl_adapt
  quantity: R2attack
  coverage: direct
  metric_as_printed: Adaptive U
  value: '69.08'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1(a)
  spread: sample SD 0.59 over evaluation seeds 0, 1, 2
- id: r_barl_aasr
  evaluation_id: ev_barl_adapt
  quantity: R1
  coverage: direct
  metric_as_printed: Adaptive ASR
  value: '25.78'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1(a)
  spread: sample SD 1.02 over evaluation seeds 0, 1, 2
- id: r_barl_asafe
  evaluation_id: ev_barl_adapt
  quantity: R3safe
  coverage: direct
  metric_as_printed: Adaptive Safe-U (task done and attack not executed)
  value: '54.84'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1(a)
  spread: sample SD 0.80 over evaluation seeds 0, 1, 2
- id: r_coer_cu
  evaluation_id: ev_coer_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Clean U
  value: '80.47'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1(a)
  spread: sample SD 1.47 over evaluation seeds 0, 1, 2
- id: r_coer_fasr
  evaluation_id: ev_coer_fixed
  quantity: R1
  coverage: direct
  metric_as_printed: Fixed ASR
  value: '0.40'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1(a)
  spread: sample SD 0.69 over evaluation seeds 0, 1, 2
- id: r_coer_au
  evaluation_id: ev_coer_adapt
  quantity: R2attack
  coverage: direct
  metric_as_printed: Adaptive U
  value: '76.47'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1(a)
  spread: sample SD 1.14 over evaluation seeds 0, 1, 2
- id: r_coer_aasr
  evaluation_id: ev_coer_adapt
  quantity: R1
  coverage: direct
  metric_as_printed: Adaptive ASR
  value: '0.22'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1(a)
  spread: sample SD 0.13 over evaluation seeds 0, 1, 2
- id: r_coer_asafe
  evaluation_id: ev_coer_adapt
  quantity: R3safe
  coverage: direct
  metric_as_printed: Adaptive Safe-U (task done and attack not executed)
  value: '76.24'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1(a)
  spread: sample SD 1.08 over evaluation seeds 0, 1, 2
- id: r_xadj_base_u
  evaluation_id: ev_xadj_base
  quantity: R2attack
  coverage: direct
  metric_as_printed: U
  value: '84.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_xadj_base_asr
  evaluation_id: ev_xadj_base
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '13.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_xady_base_u
  evaluation_id: ev_xady_base
  quantity: R2attack
  coverage: direct
  metric_as_printed: U
  value: '54.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_xady_base_asr
  evaluation_id: ev_xady_base
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '31.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_xalab_base_asr
  evaluation_id: ev_xalab_base
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '37.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_xiab_base_asr
  evaluation_id: ev_xiab_base
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Base)
  value: '7.49'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_xiae_base_asr
  evaluation_id: ev_xiae_base
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Enhanced)
  value: '22.69'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_xadj_barl_u
  evaluation_id: ev_xadj_barl
  quantity: R2attack
  coverage: direct
  metric_as_printed: U
  value: '86.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_xadj_barl_asr
  evaluation_id: ev_xadj_barl
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '5.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_xady_barl_u
  evaluation_id: ev_xady_barl
  quantity: R2attack
  coverage: direct
  metric_as_printed: U
  value: '57.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_xady_barl_asr
  evaluation_id: ev_xady_barl
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '17.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_xalab_barl_asr
  evaluation_id: ev_xalab_barl
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '15.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_xiab_barl_asr
  evaluation_id: ev_xiab_barl
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Base)
  value: '4.34'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_xiae_barl_asr
  evaluation_id: ev_xiae_barl
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Enhanced)
  value: '17.35'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_xadj_coer_u
  evaluation_id: ev_xadj_coer
  quantity: R2attack
  coverage: direct
  metric_as_printed: U
  value: '86.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_xadj_coer_asr
  evaluation_id: ev_xadj_coer
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '2.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_xady_coer_u
  evaluation_id: ev_xady_coer
  quantity: R2attack
  coverage: direct
  metric_as_printed: U
  value: '66.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_xady_coer_asr
  evaluation_id: ev_xady_coer
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '2.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_xalab_coer_asr
  evaluation_id: ev_xalab_coer
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '14.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_xiab_coer_asr
  evaluation_id: ev_xiab_coer
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Base)
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_xiae_coer_asr
  evaluation_id: ev_xiae_coer
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Enhanced)
  value: '1.97'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
comparisons:
- id: c_ppo_clean
  left: ev_ppo_clean
  right: ev_base_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_ppo_fixed
  left: ev_ppo_fixed
  right: ev_base_fixed
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_ppo_adapt
  left: ev_ppo_adapt
  right: ev_base_adapt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_grpo_clean
  left: ev_grpo_clean
  right: ev_base_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_grpo_fixed
  left: ev_grpo_fixed
  right: ev_base_fixed
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_grpo_adapt
  left: ev_grpo_adapt
  right: ev_base_adapt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_nopop_clean
  left: ev_nopop_clean
  right: ev_base_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_nopop_fixed
  left: ev_nopop_fixed
  right: ev_base_fixed
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_nopop_adapt
  left: ev_nopop_adapt
  right: ev_base_adapt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_barl_clean
  left: ev_barl_clean
  right: ev_base_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_barl_fixed
  left: ev_barl_fixed
  right: ev_base_fixed
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_barl_adapt
  left: ev_barl_adapt
  right: ev_base_adapt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_coer_clean
  left: ev_coer_clean
  right: ev_base_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_coer_fixed
  left: ev_coer_fixed
  right: ev_base_fixed
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_coer_adapt
  left: ev_coer_adapt
  right: ev_base_adapt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_coer_barl_clean
  left: ev_coer_clean
  right: ev_barl_clean
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: population-guided Defender SFT added on top of BA-RL
  page: 7
- id: c_coer_barl_fixed
  left: ev_coer_fixed
  right: ev_barl_fixed
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: population-guided Defender SFT added on top of BA-RL
  page: 7
- id: c_coer_barl_adapt
  left: ev_coer_adapt
  right: ev_barl_adapt
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: population-guided Defender SFT added on top of BA-RL
  page: 7
- id: c_xadj_barl
  left: ev_xadj_barl
  right: ev_xadj_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_xady_barl
  left: ev_xady_barl
  right: ev_xady_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_xalab_barl
  left: ev_xalab_barl
  right: ev_xalab_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_xiab_barl
  left: ev_xiab_barl
  right: ev_xiab_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_xiae_barl
  left: ev_xiae_barl
  right: ev_xiae_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_xadj_coer
  left: ev_xadj_coer
  right: ev_xadj_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_xady_coer
  left: ev_xady_coer
  right: ev_xady_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_xalab_coer
  left: ev_xalab_coer
  right: ev_xalab_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_xiab_coer
  left: ev_xiab_coer
  right: ev_xiab_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_xiae_coer
  left: ev_xiae_coer
  right: ev_xiae_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
claims:
- id: k1
  text: CoER cuts adaptive attack success from 41.3% to 0.2% and raises safe task completion from 39.6% to 76.2%
  location: abstract
  anchor: CoER reduces adaptive attack success from 41.3% to 0.2%
  jobs:
  - R1
  - R3
  supported_by:
  - c_coer_adapt
  - c_coer_barl_adapt
  - c_coer_fixed
  page: 1
coverage:
  R1: direct
  R2clean: direct
  R2attack: direct
  R3ns: not_reported
  R3cp: not_reported
  R3safe: direct
  where:
    R1: Tables 1(a) and 2
    R2clean: Table 1(a) Clean U
    R2attack: Table 1(a) adaptive U and Table 2 U
    R3safe: Table 1(a) Safe-U

`provtransformer_arxiv2026` (2609.21088)

In [ ]:
%%writefile reports/records/provtransformer_arxiv2026.yaml
report:
  id: provtransformer_arxiv2026
  slug: '2609.21088'
  title: 'Origin Is All You Need: Provenance-Aware Transformers for Structural Trust-Boundary Separation'
  first_author: Yuxuan Zhang
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2609.21088'
  version_read: arXiv v1
  first_date: '2026-09-17'
  read: F
  role: proposes
  code_release: not checked
  extraction_scope: 'Section 5: Table 3 rows for no defence, Instructional, Sandwich, Random Sequence, Delimiters, SecAlign, Baseline SFT and Provenance-Aware; Tables 4 to 6; the w/o Bias rows of Table 7; the abstract claim
    (Known-Answer, PromptSleuth, DataSentinel, PromptGuard, PromptLocate and Rennervate, which detect injections, the w/o Stage 1 rows, Table 8 and the jailbreak Table 9 not coded)'
defences:
- id: pat_smol
  name: Provenance-Aware Transformer
  origin: this_report
  base_checkpoint: SmolLM2-360M
  axis1:
  - learned_token_features
  - internal_intervention
  axis2:
  - sft
  - distillation
  isolation: ablation_present
  signal_dependence: measured
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 1
  components:
  - ring ID per token (0 system, 1 operator, 2 user and trusted data, 3 untrusted data) assigned by the application
  - origin embeddings, a learnable origin attention bias indexed by query and key rings, and a learnable origin scale
  - Stage 1 origin fine-tuning on general data; Stage 2 on 60,200 SEP-style examples whose reference answers were written by Claude
  anchor: Each in- put token is assigned a ring ID encoding its origin
  pages:
    components: 9
- id: sft_smol
  name: Baseline SFT (same data, no ring components)
  origin: this_report
  base_checkpoint: SmolLM2-360M
  axis1:
  - no_signal
  axis2:
  - sft
  - distillation
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 11
  anchor: same-data fine-tune without the ring components
- id: nob_smol
  name: Provenance-Aware Transformer with the origin attention bias zeroed at inference
  origin: this_report
  base_checkpoint: SmolLM2-360M
  axis1:
  - learned_token_features
  axis2:
  - sft
  - distillation
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 12
  anchor: zeros the origin attention bias at inference
- id: pat_l3
  name: Provenance-Aware Transformer
  origin: this_report
  base_checkpoint: LLaMA-3-8B-Instruct
  axis1:
  - learned_token_features
  - internal_intervention
  axis2:
  - sft
  - distillation
  isolation: ablation_present
  signal_dependence: measured
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 1
  components:
  - ring ID per token (0 system, 1 operator, 2 user and trusted data, 3 untrusted data) assigned by the application
  - origin embeddings, a learnable origin attention bias indexed by query and key rings, and a learnable origin scale
  - Stage 1 origin fine-tuning on general data; Stage 2 on 60,200 SEP-style examples whose reference answers were written by Claude
  anchor: Each in- put token is assigned a ring ID encoding its origin
  pages:
    components: 9
- id: sft_l3
  name: Baseline SFT (same data, no ring components)
  origin: this_report
  base_checkpoint: LLaMA-3-8B-Instruct
  axis1:
  - no_signal
  axis2:
  - sft
  - distillation
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 11
  anchor: same-data fine-tune without the ring components
- id: nob_l3
  name: Provenance-Aware Transformer with the origin attention bias zeroed at inference
  origin: this_report
  base_checkpoint: LLaMA-3-8B-Instruct
  axis1:
  - learned_token_features
  axis2:
  - sft
  - distillation
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 12
  anchor: zeros the origin attention bias at inference
- id: pat_q25
  name: Provenance-Aware Transformer
  origin: this_report
  base_checkpoint: Qwen2.5-7B
  axis1:
  - learned_token_features
  - internal_intervention
  axis2:
  - sft
  - distillation
  isolation: ablation_present
  signal_dependence: measured
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 1
  components:
  - ring ID per token (0 system, 1 operator, 2 user and trusted data, 3 untrusted data) assigned by the application
  - origin embeddings, a learnable origin attention bias indexed by query and key rings, and a learnable origin scale
  - Stage 1 origin fine-tuning on general data; Stage 2 on 60,200 SEP-style examples whose reference answers were written by Claude
  anchor: Each in- put token is assigned a ring ID encoding its origin
  pages:
    components: 9
- id: sft_q25
  name: Baseline SFT (same data, no ring components)
  origin: this_report
  base_checkpoint: Qwen2.5-7B
  axis1:
  - no_signal
  axis2:
  - sft
  - distillation
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 11
  anchor: same-data fine-tune without the ring components
- id: nob_q25
  name: Provenance-Aware Transformer with the origin attention bias zeroed at inference
  origin: this_report
  base_checkpoint: Qwen2.5-7B
  axis1:
  - learned_token_features
  axis2:
  - sft
  - distillation
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 12
  anchor: zeros the origin attention bias at inference
- id: pat_mi
  name: Provenance-Aware Transformer
  origin: this_report
  base_checkpoint: Mistral-7B
  axis1:
  - learned_token_features
  - internal_intervention
  axis2:
  - sft
  - distillation
  isolation: ablation_present
  signal_dependence: measured
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 1
  components:
  - ring ID per token (0 system, 1 operator, 2 user and trusted data, 3 untrusted data) assigned by the application
  - origin embeddings, a learnable origin attention bias indexed by query and key rings, and a learnable origin scale
  - Stage 1 origin fine-tuning on general data; Stage 2 on 60,200 SEP-style examples whose reference answers were written by Claude
  anchor: Each in- put token is assigned a ring ID encoding its origin
  pages:
    components: 9
- id: sft_mi
  name: Baseline SFT (same data, no ring components)
  origin: this_report
  base_checkpoint: Mistral-7B
  axis1:
  - no_signal
  axis2:
  - sft
  - distillation
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 11
  anchor: same-data fine-tune without the ring components
- id: nob_mi
  name: Provenance-Aware Transformer with the origin attention bias zeroed at inference
  origin: this_report
  base_checkpoint: Mistral-7B
  axis1:
  - learned_token_features
  axis2:
  - sft
  - distillation
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 12
  anchor: zeros the origin attention bias at inference
- id: instr_l3
  name: Instructional
  origin: other_report
  base_checkpoint: LLaMA-3-8B-Instruct
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 10
  anchor: Instructional [52]
- id: sand_l3
  name: Sandwich
  origin: other_report
  base_checkpoint: LLaMA-3-8B-Instruct
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 10
  anchor: Sandwich [26]
- id: rseq_l3
  name: Random Sequence
  origin: other_report
  base_checkpoint: LLaMA-3-8B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 10
  anchor: Random Sequence [1]
- id: delim_l3
  name: Delimiters
  origin: other_report
  base_checkpoint: LLaMA-3-8B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 10
  anchor: Delimiters [55]
- id: secalign_l3
  name: SecAlign
  origin: other_report
  base_checkpoint: LLaMA-3-8B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 10
  anchor: SecAlign [11]
evaluations:
- id: ev_smol_none_sep
  defence_id: none:SmolLM2-360M
  benchmark: SEP test (IID), 4,375 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: 4,375
  pages:
    benchmark: 9
  baseline_kind: off_the_shelf
- id: ev_smol_none_pia
  defence_id: none:SmolLM2-360M
  benchmark: PI-Attack (OOD classification), 8,256 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: 8,256
  pages:
    benchmark: 9
  baseline_kind: off_the_shelf
- id: ev_smol_none_dsen
  defence_id: none:SmolLM2-360M
  benchmark: DataSentinel set (OOD classification), 178 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: '178'
  pages:
    benchmark: 9
  baseline_kind: off_the_shelf
- id: ev_smol_none_math
  defence_id: none:SmolLM2-360M
  benchmark: Math-Tutor (OOD free-form math), 3,800 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: 3,800
  pages:
    benchmark: 9
  baseline_kind: off_the_shelf
- id: ev_smol_sft_sep
  defence_id: sft_smol
  benchmark: SEP test (IID), 4,375 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: 4,375
  pages:
    benchmark: 9
- id: ev_smol_sft_pia
  defence_id: sft_smol
  benchmark: PI-Attack (OOD classification), 8,256 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: 8,256
  pages:
    benchmark: 9
- id: ev_smol_sft_dsen
  defence_id: sft_smol
  benchmark: DataSentinel set (OOD classification), 178 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: '178'
  pages:
    benchmark: 9
- id: ev_smol_sft_math
  defence_id: sft_smol
  benchmark: Math-Tutor (OOD free-form math), 3,800 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: 3,800
  pages:
    benchmark: 9
- id: ev_smol_pat_sep
  defence_id: pat_smol
  benchmark: SEP test (IID), 4,375 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: 4,375
  pages:
    benchmark: 9
- id: ev_smol_pat_pia
  defence_id: pat_smol
  benchmark: PI-Attack (OOD classification), 8,256 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: 8,256
  pages:
    benchmark: 9
- id: ev_smol_pat_dsen
  defence_id: pat_smol
  benchmark: DataSentinel set (OOD classification), 178 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: '178'
  pages:
    benchmark: 9
- id: ev_smol_pat_math
  defence_id: pat_smol
  benchmark: Math-Tutor (OOD free-form math), 3,800 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: 3,800
  pages:
    benchmark: 9
- id: ev_l3_none_sep
  defence_id: none:LLaMA-3-8B-Instruct
  benchmark: SEP test (IID), 4,375 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: 4,375
  pages:
    benchmark: 9
  baseline_kind: off_the_shelf
- id: ev_l3_none_pia
  defence_id: none:LLaMA-3-8B-Instruct
  benchmark: PI-Attack (OOD classification), 8,256 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: 8,256
  pages:
    benchmark: 9
  baseline_kind: off_the_shelf
- id: ev_l3_none_dsen
  defence_id: none:LLaMA-3-8B-Instruct
  benchmark: DataSentinel set (OOD classification), 178 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: '178'
  pages:
    benchmark: 9
  baseline_kind: off_the_shelf
- id: ev_l3_none_math
  defence_id: none:LLaMA-3-8B-Instruct
  benchmark: Math-Tutor (OOD free-form math), 3,800 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: 3,800
  pages:
    benchmark: 9
  baseline_kind: off_the_shelf
- id: ev_l3_sft_sep
  defence_id: sft_l3
  benchmark: SEP test (IID), 4,375 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: 4,375
  pages:
    benchmark: 9
- id: ev_l3_sft_pia
  defence_id: sft_l3
  benchmark: PI-Attack (OOD classification), 8,256 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: 8,256
  pages:
    benchmark: 9
- id: ev_l3_sft_dsen
  defence_id: sft_l3
  benchmark: DataSentinel set (OOD classification), 178 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: '178'
  pages:
    benchmark: 9
- id: ev_l3_sft_math
  defence_id: sft_l3
  benchmark: Math-Tutor (OOD free-form math), 3,800 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: 3,800
  pages:
    benchmark: 9
- id: ev_l3_pat_sep
  defence_id: pat_l3
  benchmark: SEP test (IID), 4,375 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: 4,375
  pages:
    benchmark: 9
- id: ev_l3_pat_pia
  defence_id: pat_l3
  benchmark: PI-Attack (OOD classification), 8,256 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: 8,256
  pages:
    benchmark: 9
- id: ev_l3_pat_dsen
  defence_id: pat_l3
  benchmark: DataSentinel set (OOD classification), 178 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: '178'
  pages:
    benchmark: 9
- id: ev_l3_pat_math
  defence_id: pat_l3
  benchmark: Math-Tutor (OOD free-form math), 3,800 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: 3,800
  pages:
    benchmark: 9
- id: ev_q25_none_sep
  defence_id: none:Qwen2.5-7B
  benchmark: SEP test (IID), 4,375 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: 4,375
  pages:
    benchmark: 9
  baseline_kind: off_the_shelf
- id: ev_q25_none_pia
  defence_id: none:Qwen2.5-7B
  benchmark: PI-Attack (OOD classification), 8,256 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: 8,256
  pages:
    benchmark: 9
  baseline_kind: off_the_shelf
- id: ev_q25_none_dsen
  defence_id: none:Qwen2.5-7B
  benchmark: DataSentinel set (OOD classification), 178 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: '178'
  pages:
    benchmark: 9
  baseline_kind: off_the_shelf
- id: ev_q25_none_math
  defence_id: none:Qwen2.5-7B
  benchmark: Math-Tutor (OOD free-form math), 3,800 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: 3,800
  pages:
    benchmark: 9
  baseline_kind: off_the_shelf
- id: ev_q25_sft_sep
  defence_id: sft_q25
  benchmark: SEP test (IID), 4,375 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: 4,375
  pages:
    benchmark: 9
- id: ev_q25_sft_pia
  defence_id: sft_q25
  benchmark: PI-Attack (OOD classification), 8,256 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: 8,256
  pages:
    benchmark: 9
- id: ev_q25_sft_dsen
  defence_id: sft_q25
  benchmark: DataSentinel set (OOD classification), 178 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: '178'
  pages:
    benchmark: 9
- id: ev_q25_sft_math
  defence_id: sft_q25
  benchmark: Math-Tutor (OOD free-form math), 3,800 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: 3,800
  pages:
    benchmark: 9
- id: ev_q25_pat_sep
  defence_id: pat_q25
  benchmark: SEP test (IID), 4,375 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: 4,375
  pages:
    benchmark: 9
- id: ev_q25_pat_pia
  defence_id: pat_q25
  benchmark: PI-Attack (OOD classification), 8,256 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: 8,256
  pages:
    benchmark: 9
- id: ev_q25_pat_dsen
  defence_id: pat_q25
  benchmark: DataSentinel set (OOD classification), 178 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: '178'
  pages:
    benchmark: 9
- id: ev_q25_pat_math
  defence_id: pat_q25
  benchmark: Math-Tutor (OOD free-form math), 3,800 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: 3,800
  pages:
    benchmark: 9
- id: ev_mi_none_sep
  defence_id: none:Mistral-7B
  benchmark: SEP test (IID), 4,375 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: 4,375
  pages:
    benchmark: 9
  baseline_kind: off_the_shelf
- id: ev_mi_none_pia
  defence_id: none:Mistral-7B
  benchmark: PI-Attack (OOD classification), 8,256 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: 8,256
  pages:
    benchmark: 9
  baseline_kind: off_the_shelf
- id: ev_mi_none_dsen
  defence_id: none:Mistral-7B
  benchmark: DataSentinel set (OOD classification), 178 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: '178'
  pages:
    benchmark: 9
  baseline_kind: off_the_shelf
- id: ev_mi_none_math
  defence_id: none:Mistral-7B
  benchmark: Math-Tutor (OOD free-form math), 3,800 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: 3,800
  pages:
    benchmark: 9
  baseline_kind: off_the_shelf
- id: ev_mi_sft_sep
  defence_id: sft_mi
  benchmark: SEP test (IID), 4,375 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: 4,375
  pages:
    benchmark: 9
- id: ev_mi_sft_pia
  defence_id: sft_mi
  benchmark: PI-Attack (OOD classification), 8,256 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: 8,256
  pages:
    benchmark: 9
- id: ev_mi_sft_dsen
  defence_id: sft_mi
  benchmark: DataSentinel set (OOD classification), 178 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: '178'
  pages:
    benchmark: 9
- id: ev_mi_sft_math
  defence_id: sft_mi
  benchmark: Math-Tutor (OOD free-form math), 3,800 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: 3,800
  pages:
    benchmark: 9
- id: ev_mi_pat_sep
  defence_id: pat_mi
  benchmark: SEP test (IID), 4,375 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: 4,375
  pages:
    benchmark: 9
- id: ev_mi_pat_pia
  defence_id: pat_mi
  benchmark: PI-Attack (OOD classification), 8,256 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: 8,256
  pages:
    benchmark: 9
- id: ev_mi_pat_dsen
  defence_id: pat_mi
  benchmark: DataSentinel set (OOD classification), 178 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: '178'
  pages:
    benchmark: 9
- id: ev_mi_pat_math
  defence_id: pat_mi
  benchmark: Math-Tutor (OOD free-form math), 3,800 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: 3,800
  pages:
    benchmark: 9
- id: ev_t3_instr_l3_sep
  defence_id: instr_l3
  benchmark: SEP test (IID), 4,375 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 10
  decoding: not reported
  sample_size: 4,375
  pages:
    benchmark: 9
- id: ev_t3_instr_l3_pia
  defence_id: instr_l3
  benchmark: PI-Attack (OOD classification), 8,256 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 10
  decoding: not reported
  sample_size: 8,256
  pages:
    benchmark: 9
- id: ev_t3_instr_l3_dsen
  defence_id: instr_l3
  benchmark: DataSentinel set (OOD classification), 178 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 10
  decoding: not reported
  sample_size: '178'
  pages:
    benchmark: 9
- id: ev_t3_instr_l3_math
  defence_id: instr_l3
  benchmark: Math-Tutor (OOD free-form math), 3,800 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 10
  decoding: not reported
  sample_size: 3,800
  pages:
    benchmark: 9
- id: ev_t3_sand_l3_sep
  defence_id: sand_l3
  benchmark: SEP test (IID), 4,375 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 10
  decoding: not reported
  sample_size: 4,375
  pages:
    benchmark: 9
- id: ev_t3_sand_l3_pia
  defence_id: sand_l3
  benchmark: PI-Attack (OOD classification), 8,256 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 10
  decoding: not reported
  sample_size: 8,256
  pages:
    benchmark: 9
- id: ev_t3_sand_l3_dsen
  defence_id: sand_l3
  benchmark: DataSentinel set (OOD classification), 178 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 10
  decoding: not reported
  sample_size: '178'
  pages:
    benchmark: 9
- id: ev_t3_sand_l3_math
  defence_id: sand_l3
  benchmark: Math-Tutor (OOD free-form math), 3,800 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 10
  decoding: not reported
  sample_size: 3,800
  pages:
    benchmark: 9
- id: ev_t3_rseq_l3_sep
  defence_id: rseq_l3
  benchmark: SEP test (IID), 4,375 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 10
  decoding: not reported
  sample_size: 4,375
  pages:
    benchmark: 9
- id: ev_t3_rseq_l3_pia
  defence_id: rseq_l3
  benchmark: PI-Attack (OOD classification), 8,256 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 10
  decoding: not reported
  sample_size: 8,256
  pages:
    benchmark: 9
- id: ev_t3_rseq_l3_dsen
  defence_id: rseq_l3
  benchmark: DataSentinel set (OOD classification), 178 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 10
  decoding: not reported
  sample_size: '178'
  pages:
    benchmark: 9
- id: ev_t3_rseq_l3_math
  defence_id: rseq_l3
  benchmark: Math-Tutor (OOD free-form math), 3,800 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 10
  decoding: not reported
  sample_size: 3,800
  pages:
    benchmark: 9
- id: ev_t3_delim_l3_sep
  defence_id: delim_l3
  benchmark: SEP test (IID), 4,375 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 10
  decoding: not reported
  sample_size: 4,375
  pages:
    benchmark: 9
- id: ev_t3_delim_l3_pia
  defence_id: delim_l3
  benchmark: PI-Attack (OOD classification), 8,256 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 10
  decoding: not reported
  sample_size: 8,256
  pages:
    benchmark: 9
- id: ev_t3_delim_l3_dsen
  defence_id: delim_l3
  benchmark: DataSentinel set (OOD classification), 178 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 10
  decoding: not reported
  sample_size: '178'
  pages:
    benchmark: 9
- id: ev_t3_delim_l3_math
  defence_id: delim_l3
  benchmark: Math-Tutor (OOD free-form math), 3,800 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 10
  decoding: not reported
  sample_size: 3,800
  pages:
    benchmark: 9
- id: ev_t3_secalign_l3_sep
  defence_id: secalign_l3
  benchmark: SEP test (IID), 4,375 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 10
  decoding: not reported
  sample_size: 4,375
  pages:
    benchmark: 9
- id: ev_t3_secalign_l3_pia
  defence_id: secalign_l3
  benchmark: PI-Attack (OOD classification), 8,256 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 10
  decoding: not reported
  sample_size: 8,256
  pages:
    benchmark: 9
- id: ev_t3_secalign_l3_dsen
  defence_id: secalign_l3
  benchmark: DataSentinel set (OOD classification), 178 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 10
  decoding: not reported
  sample_size: '178'
  pages:
    benchmark: 9
- id: ev_t3_secalign_l3_math
  defence_id: secalign_l3
  benchmark: Math-Tutor (OOD free-form math), 3,800 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 10
  decoding: not reported
  sample_size: 3,800
  pages:
    benchmark: 9
- id: ev_smol_nob_sep
  defence_id: nob_smol
  benchmark: SEP test (IID), 4,375 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 12
  decoding: not reported
  sample_size: 4,375
  pages:
    benchmark: 9
- id: ev_smol_nob_pia
  defence_id: nob_smol
  benchmark: PI-Attack (OOD classification), 8,256 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 12
  decoding: not reported
  sample_size: 8,256
  pages:
    benchmark: 9
- id: ev_smol_nob_dsen
  defence_id: nob_smol
  benchmark: DataSentinel set (OOD classification), 178 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 12
  decoding: not reported
  sample_size: '178'
  pages:
    benchmark: 9
- id: ev_smol_nob_math
  defence_id: nob_smol
  benchmark: Math-Tutor (OOD free-form math), 3,800 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 12
  decoding: not reported
  sample_size: 3,800
  pages:
    benchmark: 9
- id: ev_l3_nob_sep
  defence_id: nob_l3
  benchmark: SEP test (IID), 4,375 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 12
  decoding: not reported
  sample_size: 4,375
  pages:
    benchmark: 9
- id: ev_l3_nob_pia
  defence_id: nob_l3
  benchmark: PI-Attack (OOD classification), 8,256 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 12
  decoding: not reported
  sample_size: 8,256
  pages:
    benchmark: 9
- id: ev_l3_nob_dsen
  defence_id: nob_l3
  benchmark: DataSentinel set (OOD classification), 178 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 12
  decoding: not reported
  sample_size: '178'
  pages:
    benchmark: 9
- id: ev_l3_nob_math
  defence_id: nob_l3
  benchmark: Math-Tutor (OOD free-form math), 3,800 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 12
  decoding: not reported
  sample_size: 3,800
  pages:
    benchmark: 9
- id: ev_q25_nob_sep
  defence_id: nob_q25
  benchmark: SEP test (IID), 4,375 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 12
  decoding: not reported
  sample_size: 4,375
  pages:
    benchmark: 9
- id: ev_q25_nob_pia
  defence_id: nob_q25
  benchmark: PI-Attack (OOD classification), 8,256 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 12
  decoding: not reported
  sample_size: 8,256
  pages:
    benchmark: 9
- id: ev_q25_nob_dsen
  defence_id: nob_q25
  benchmark: DataSentinel set (OOD classification), 178 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 12
  decoding: not reported
  sample_size: '178'
  pages:
    benchmark: 9
- id: ev_q25_nob_math
  defence_id: nob_q25
  benchmark: Math-Tutor (OOD free-form math), 3,800 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 12
  decoding: not reported
  sample_size: 3,800
  pages:
    benchmark: 9
- id: ev_mi_nob_sep
  defence_id: nob_mi
  benchmark: SEP test (IID), 4,375 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 12
  decoding: not reported
  sample_size: 4,375
  pages:
    benchmark: 9
- id: ev_mi_nob_pia
  defence_id: nob_mi
  benchmark: PI-Attack (OOD classification), 8,256 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 12
  decoding: not reported
  sample_size: 8,256
  pages:
    benchmark: 9
- id: ev_mi_nob_dsen
  defence_id: nob_mi
  benchmark: DataSentinel set (OOD classification), 178 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 12
  decoding: not reported
  sample_size: '178'
  pages:
    benchmark: 9
- id: ev_mi_nob_math
  defence_id: nob_mi
  benchmark: Math-Tutor (OOD free-form math), 3,800 examples
  attacker_type: static
  attacker_access: black_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 12
  decoding: not reported
  sample_size: 3,800
  pages:
    benchmark: 9
- id: ev_ae_smol_sft
  defence_id: sft_smol
  benchmark: AlpacaEval 2.0, 805 prompts, pairwise against the base pretrained model
  attacker_type: none
  attacker_access: not_applicable
  judge: Claude Sonnet 5 judge, length-controlled win rate
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: '805'
  pages:
    judge: 9
- id: ev_ae_smol_pat
  defence_id: pat_smol
  benchmark: AlpacaEval 2.0, 805 prompts, pairwise against the base pretrained model
  attacker_type: none
  attacker_access: not_applicable
  judge: Claude Sonnet 5 judge, length-controlled win rate
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: '805'
  pages:
    judge: 9
- id: ev_ae_l3_sft
  defence_id: sft_l3
  benchmark: AlpacaEval 2.0, 805 prompts, pairwise against the base pretrained model
  attacker_type: none
  attacker_access: not_applicable
  judge: Claude Sonnet 5 judge, length-controlled win rate
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: '805'
  pages:
    judge: 9
- id: ev_ae_l3_pat
  defence_id: pat_l3
  benchmark: AlpacaEval 2.0, 805 prompts, pairwise against the base pretrained model
  attacker_type: none
  attacker_access: not_applicable
  judge: Claude Sonnet 5 judge, length-controlled win rate
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: '805'
  pages:
    judge: 9
- id: ev_ae_q25_sft
  defence_id: sft_q25
  benchmark: AlpacaEval 2.0, 805 prompts, pairwise against the base pretrained model
  attacker_type: none
  attacker_access: not_applicable
  judge: Claude Sonnet 5 judge, length-controlled win rate
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: '805'
  pages:
    judge: 9
- id: ev_ae_q25_pat
  defence_id: pat_q25
  benchmark: AlpacaEval 2.0, 805 prompts, pairwise against the base pretrained model
  attacker_type: none
  attacker_access: not_applicable
  judge: Claude Sonnet 5 judge, length-controlled win rate
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: '805'
  pages:
    judge: 9
- id: ev_ae_mi_sft
  defence_id: sft_mi
  benchmark: AlpacaEval 2.0, 805 prompts, pairwise against the base pretrained model
  attacker_type: none
  attacker_access: not_applicable
  judge: Claude Sonnet 5 judge, length-controlled win rate
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: '805'
  pages:
    judge: 9
- id: ev_ae_mi_pat
  defence_id: pat_mi
  benchmark: AlpacaEval 2.0, 805 prompts, pairwise against the base pretrained model
  attacker_type: none
  attacker_access: not_applicable
  judge: Claude Sonnet 5 judge, length-controlled win rate
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  decoding: not reported
  sample_size: '805'
  pages:
    judge: 9
- id: ev_ad_none_gcg
  defence_id: none:LLaMA-3-8B-Instruct
  benchmark: PI-Attack and DataSentinel, universal GCG suffix or trigger optimised against this model
  attacker_type: adaptive
  attacker_access: white_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  attacker_budget: 2,000 optimisation steps, universal
  decoding: not reported
  pages:
    attacker_budget: 12
  baseline_kind: off_the_shelf
- id: ev_ad_none_neu
  defence_id: none:LLaMA-3-8B-Instruct
  benchmark: PI-Attack and DataSentinel, universal NeuExe suffix or trigger optimised against this model
  attacker_type: adaptive
  attacker_access: white_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  attacker_budget: 2,000 optimisation steps, universal
  decoding: not reported
  pages:
    attacker_budget: 12
  baseline_kind: off_the_shelf
- id: ev_ad_sft_gcg
  defence_id: sft_l3
  benchmark: PI-Attack and DataSentinel, universal GCG suffix or trigger optimised against this model
  attacker_type: adaptive
  attacker_access: white_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  attacker_budget: 2,000 optimisation steps, universal
  decoding: not reported
  pages:
    attacker_budget: 12
- id: ev_ad_sft_neu
  defence_id: sft_l3
  benchmark: PI-Attack and DataSentinel, universal NeuExe suffix or trigger optimised against this model
  attacker_type: adaptive
  attacker_access: white_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  attacker_budget: 2,000 optimisation steps, universal
  decoding: not reported
  pages:
    attacker_budget: 12
- id: ev_ad_pat_gcg
  defence_id: pat_l3
  benchmark: PI-Attack and DataSentinel, universal GCG suffix or trigger optimised against this model
  attacker_type: adaptive
  attacker_access: white_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  attacker_budget: 2,000 optimisation steps, universal
  decoding: not reported
  pages:
    attacker_budget: 12
- id: ev_ad_pat_neu
  defence_id: pat_l3
  benchmark: PI-Attack and DataSentinel, universal NeuExe suffix or trigger optimised against this model
  attacker_type: adaptive
  attacker_access: white_box
  judge: output matches the label implied by the injected instruction
  layout_same_as_defence: 'yes'
  measured_by: provtransformer_arxiv2026
  page: 11
  attacker_budget: 2,000 optimisation steps, universal
  decoding: not reported
  pages:
    attacker_budget: 12
results:
- id: r_smol_none_sep
  evaluation_id: ev_smol_none_sep
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '40.19'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 4
- id: r_smol_none_pia
  evaluation_id: ev_smol_none_pia
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '33.79'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 4
- id: r_smol_none_dsen
  evaluation_id: ev_smol_none_dsen
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '84.48'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 4
- id: r_smol_none_math
  evaluation_id: ev_smol_none_math
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '3.33'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 4
- id: r_smol_sft_sep
  evaluation_id: ev_smol_sft_sep
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '0.74'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 4
- id: r_smol_sft_pia
  evaluation_id: ev_smol_sft_pia
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '13.90'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 4
- id: r_smol_sft_dsen
  evaluation_id: ev_smol_sft_dsen
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '6.90'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 4
- id: r_smol_sft_math
  evaluation_id: ev_smol_sft_math
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '0.06'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 4
- id: r_smol_pat_sep
  evaluation_id: ev_smol_pat_sep
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '0.51'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 4
- id: r_smol_pat_pia
  evaluation_id: ev_smol_pat_pia
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '0.05'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 4
- id: r_smol_pat_dsen
  evaluation_id: ev_smol_pat_dsen
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 4
- id: r_smol_pat_math
  evaluation_id: ev_smol_pat_math
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 4
- id: r_l3_none_sep
  evaluation_id: ev_l3_none_sep
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '32.50'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 4
- id: r_l3_none_pia
  evaluation_id: ev_l3_none_pia
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '21.30'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 4
- id: r_l3_none_dsen
  evaluation_id: ev_l3_none_dsen
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '24.14'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 4
- id: r_l3_none_math
  evaluation_id: ev_l3_none_math
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '17.22'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 4
- id: r_l3_sft_sep
  evaluation_id: ev_l3_sft_sep
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '0.80'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 4
- id: r_l3_sft_pia
  evaluation_id: ev_l3_sft_pia
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '2.32'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 4
- id: r_l3_sft_dsen
  evaluation_id: ev_l3_sft_dsen
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '8.62'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 4
- id: r_l3_sft_math
  evaluation_id: ev_l3_sft_math
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '0.33'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 4
- id: r_l3_pat_sep
  evaluation_id: ev_l3_pat_sep
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '0.69'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 4
- id: r_l3_pat_pia
  evaluation_id: ev_l3_pat_pia
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '0.03'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 4
- id: r_l3_pat_dsen
  evaluation_id: ev_l3_pat_dsen
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 4
- id: r_l3_pat_math
  evaluation_id: ev_l3_pat_math
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 4
- id: r_q25_none_sep
  evaluation_id: ev_q25_none_sep
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '49.13'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 4
- id: r_q25_none_pia
  evaluation_id: ev_q25_none_pia
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '34.83'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 4
- id: r_q25_none_dsen
  evaluation_id: ev_q25_none_dsen
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '84.48'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 4
- id: r_q25_none_math
  evaluation_id: ev_q25_none_math
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '10.89'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 4
- id: r_q25_sft_sep
  evaluation_id: ev_q25_sft_sep
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '0.66'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 4
- id: r_q25_sft_pia
  evaluation_id: ev_q25_sft_pia
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '3.74'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 4
- id: r_q25_sft_dsen
  evaluation_id: ev_q25_sft_dsen
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '10.34'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 4
- id: r_q25_sft_math
  evaluation_id: ev_q25_sft_math
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '0.11'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 4
- id: r_q25_pat_sep
  evaluation_id: ev_q25_pat_sep
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '0.63'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 4
- id: r_q25_pat_pia
  evaluation_id: ev_q25_pat_pia
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 4
- id: r_q25_pat_dsen
  evaluation_id: ev_q25_pat_dsen
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 4
- id: r_q25_pat_math
  evaluation_id: ev_q25_pat_math
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 4
- id: r_mi_none_sep
  evaluation_id: ev_mi_none_sep
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '43.47'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 4
- id: r_mi_none_pia
  evaluation_id: ev_mi_none_pia
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '43.58'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 4
- id: r_mi_none_dsen
  evaluation_id: ev_mi_none_dsen
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '51.72'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 4
- id: r_mi_none_math
  evaluation_id: ev_mi_none_math
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '35.94'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 4
- id: r_mi_sft_sep
  evaluation_id: ev_mi_sft_sep
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '0.34'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 4
- id: r_mi_sft_pia
  evaluation_id: ev_mi_sft_pia
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '17.77'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 4
- id: r_mi_sft_dsen
  evaluation_id: ev_mi_sft_dsen
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '5.17'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 4
- id: r_mi_sft_math
  evaluation_id: ev_mi_sft_math
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '1.50'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 4
- id: r_mi_pat_sep
  evaluation_id: ev_mi_pat_sep
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '0.66'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 4
- id: r_mi_pat_pia
  evaluation_id: ev_mi_pat_pia
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '0.03'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 4
- id: r_mi_pat_dsen
  evaluation_id: ev_mi_pat_dsen
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 4
- id: r_mi_pat_math
  evaluation_id: ev_mi_pat_math
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 4
- id: r_t3_instr_l3_sep
  evaluation_id: ev_t3_instr_l3_sep
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '30.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 3
- id: r_t3_instr_l3_pia
  evaluation_id: ev_t3_instr_l3_pia
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '16.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 3
- id: r_t3_instr_l3_dsen
  evaluation_id: ev_t3_instr_l3_dsen
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '20.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 3
- id: r_t3_instr_l3_math
  evaluation_id: ev_t3_instr_l3_math
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '19.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 3
- id: r_t3_sand_l3_sep
  evaluation_id: ev_t3_sand_l3_sep
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '25.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 3
- id: r_t3_sand_l3_pia
  evaluation_id: ev_t3_sand_l3_pia
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '4.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 3
- id: r_t3_sand_l3_dsen
  evaluation_id: ev_t3_sand_l3_dsen
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '13.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 3
- id: r_t3_sand_l3_math
  evaluation_id: ev_t3_sand_l3_math
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '13.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 3
- id: r_t3_rseq_l3_sep
  evaluation_id: ev_t3_rseq_l3_sep
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '15.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 3
- id: r_t3_rseq_l3_pia
  evaluation_id: ev_t3_rseq_l3_pia
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '2.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 3
- id: r_t3_rseq_l3_dsen
  evaluation_id: ev_t3_rseq_l3_dsen
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '12.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 3
- id: r_t3_rseq_l3_math
  evaluation_id: ev_t3_rseq_l3_math
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '26.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 3
- id: r_t3_delim_l3_sep
  evaluation_id: ev_t3_delim_l3_sep
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '11.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 3
- id: r_t3_delim_l3_pia
  evaluation_id: ev_t3_delim_l3_pia
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '9.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 3
- id: r_t3_delim_l3_dsen
  evaluation_id: ev_t3_delim_l3_dsen
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '22.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 3
- id: r_t3_delim_l3_math
  evaluation_id: ev_t3_delim_l3_math
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '4.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 3
- id: r_t3_secalign_l3_sep
  evaluation_id: ev_t3_secalign_l3_sep
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '2.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 3
- id: r_t3_secalign_l3_pia
  evaluation_id: ev_t3_secalign_l3_pia
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 3
- id: r_t3_secalign_l3_dsen
  evaluation_id: ev_t3_secalign_l3_dsen
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '6.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 3
- id: r_t3_secalign_l3_math
  evaluation_id: ev_t3_secalign_l3_math
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 3
- id: r_smol_nob_sep
  evaluation_id: ev_smol_nob_sep
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '29.51'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 7 (w/o Bias)
- id: r_smol_nob_pia
  evaluation_id: ev_smol_nob_pia
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '45.77'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 7 (w/o Bias)
- id: r_smol_nob_dsen
  evaluation_id: ev_smol_nob_dsen
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '96.55'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 7 (w/o Bias)
- id: r_smol_nob_math
  evaluation_id: ev_smol_nob_math
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '11.56'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 7 (w/o Bias)
- id: r_l3_nob_sep
  evaluation_id: ev_l3_nob_sep
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '72.04'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 7 (w/o Bias)
- id: r_l3_nob_pia
  evaluation_id: ev_l3_nob_pia
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '77.80'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 7 (w/o Bias)
- id: r_l3_nob_dsen
  evaluation_id: ev_l3_nob_dsen
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '89.66'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 7 (w/o Bias)
- id: r_l3_nob_math
  evaluation_id: ev_l3_nob_math
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '67.56'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 7 (w/o Bias)
- id: r_q25_nob_sep
  evaluation_id: ev_q25_nob_sep
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '69.35'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 7 (w/o Bias)
- id: r_q25_nob_pia
  evaluation_id: ev_q25_nob_pia
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '70.62'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 7 (w/o Bias)
- id: r_q25_nob_dsen
  evaluation_id: ev_q25_nob_dsen
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '98.28'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 7 (w/o Bias)
- id: r_q25_nob_math
  evaluation_id: ev_q25_nob_math
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '39.67'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 7 (w/o Bias)
- id: r_mi_nob_sep
  evaluation_id: ev_mi_nob_sep
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '8.08'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 7 (w/o Bias)
- id: r_mi_nob_pia
  evaluation_id: ev_mi_nob_pia
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '12.22'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 7 (w/o Bias)
- id: r_mi_nob_dsen
  evaluation_id: ev_mi_nob_dsen
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '94.83'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 7 (w/o Bias)
- id: r_mi_nob_math
  evaluation_id: ev_mi_nob_math
  quantity: R1
  coverage: direct
  metric_as_printed: ASR%
  value: '32.89'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 7 (w/o Bias)
- id: r_ae_smol_sft
  evaluation_id: ev_ae_smol_sft
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaEval 2.0 LC win rate
  value: '57.1'
  unit: percent_of_reference
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 5
  reference: base pretrained model (50% = parity)
- id: r_ae_smol_pat
  evaluation_id: ev_ae_smol_pat
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaEval 2.0 LC win rate
  value: '49.5'
  unit: percent_of_reference
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 5
  reference: base pretrained model (50% = parity)
- id: r_ae_l3_sft
  evaluation_id: ev_ae_l3_sft
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaEval 2.0 LC win rate
  value: '58.1'
  unit: percent_of_reference
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 5
  reference: base pretrained model (50% = parity)
- id: r_ae_l3_pat
  evaluation_id: ev_ae_l3_pat
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaEval 2.0 LC win rate
  value: '48.8'
  unit: percent_of_reference
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 5
  reference: base pretrained model (50% = parity)
- id: r_ae_q25_sft
  evaluation_id: ev_ae_q25_sft
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaEval 2.0 LC win rate
  value: '60.7'
  unit: percent_of_reference
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 5
  reference: base pretrained model (50% = parity)
- id: r_ae_q25_pat
  evaluation_id: ev_ae_q25_pat
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaEval 2.0 LC win rate
  value: '60.9'
  unit: percent_of_reference
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 5
  reference: base pretrained model (50% = parity)
- id: r_ae_mi_sft
  evaluation_id: ev_ae_mi_sft
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaEval 2.0 LC win rate
  value: '28.0'
  unit: percent_of_reference
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 5
  reference: base pretrained model (50% = parity)
- id: r_ae_mi_pat
  evaluation_id: ev_ae_mi_pat
  quantity: general_utility
  coverage: direct
  metric_as_printed: AlpacaEval 2.0 LC win rate
  value: '42.7'
  unit: percent_of_reference
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 5
  reference: base pretrained model (50% = parity)
- id: r_ad_none_gcg
  evaluation_id: ev_ad_none_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: GCG ASR%
  value: '50.71'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 6
- id: r_ad_none_neu
  evaluation_id: ev_ad_none_neu
  quantity: R1
  coverage: direct
  metric_as_printed: NeuExe ASR%
  value: '48.44'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 6
- id: r_ad_sft_gcg
  evaluation_id: ev_ad_sft_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: GCG ASR%
  value: '3.19'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 6
- id: r_ad_sft_neu
  evaluation_id: ev_ad_sft_neu
  quantity: R1
  coverage: direct
  metric_as_printed: NeuExe ASR%
  value: '19.69'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 6
- id: r_ad_pat_gcg
  evaluation_id: ev_ad_pat_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: GCG ASR%
  value: '1.44'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 6
- id: r_ad_pat_neu
  evaluation_id: ev_ad_pat_neu
  quantity: R1
  coverage: direct
  metric_as_printed: NeuExe ASR%
  value: '1.56'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 6
comparisons:
- id: c_smol_pat_sep
  left: ev_smol_pat_sep
  right: ev_smol_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_smol_sft_sep
  left: ev_smol_sft_sep
  right: ev_smol_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_smol_pat_sft_sep
  left: ev_smol_pat_sep
  right: ev_smol_sft_sep
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: ring components (origin embeddings, attention bias, scale) against the same data without them
  page: 11
- id: c_smol_pat_pia
  left: ev_smol_pat_pia
  right: ev_smol_none_pia
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_smol_sft_pia
  left: ev_smol_sft_pia
  right: ev_smol_none_pia
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_smol_pat_sft_pia
  left: ev_smol_pat_pia
  right: ev_smol_sft_pia
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: ring components (origin embeddings, attention bias, scale) against the same data without them
  page: 11
- id: c_smol_pat_dsen
  left: ev_smol_pat_dsen
  right: ev_smol_none_dsen
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_smol_sft_dsen
  left: ev_smol_sft_dsen
  right: ev_smol_none_dsen
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_smol_pat_sft_dsen
  left: ev_smol_pat_dsen
  right: ev_smol_sft_dsen
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: ring components (origin embeddings, attention bias, scale) against the same data without them
  page: 11
- id: c_smol_pat_math
  left: ev_smol_pat_math
  right: ev_smol_none_math
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_smol_sft_math
  left: ev_smol_sft_math
  right: ev_smol_none_math
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_smol_pat_sft_math
  left: ev_smol_pat_math
  right: ev_smol_sft_math
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: ring components (origin embeddings, attention bias, scale) against the same data without them
  page: 11
- id: c_l3_pat_sep
  left: ev_l3_pat_sep
  right: ev_l3_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_l3_sft_sep
  left: ev_l3_sft_sep
  right: ev_l3_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_l3_pat_sft_sep
  left: ev_l3_pat_sep
  right: ev_l3_sft_sep
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: ring components (origin embeddings, attention bias, scale) against the same data without them
  page: 11
- id: c_l3_pat_pia
  left: ev_l3_pat_pia
  right: ev_l3_none_pia
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_l3_sft_pia
  left: ev_l3_sft_pia
  right: ev_l3_none_pia
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_l3_pat_sft_pia
  left: ev_l3_pat_pia
  right: ev_l3_sft_pia
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: ring components (origin embeddings, attention bias, scale) against the same data without them
  page: 11
- id: c_l3_pat_dsen
  left: ev_l3_pat_dsen
  right: ev_l3_none_dsen
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_l3_sft_dsen
  left: ev_l3_sft_dsen
  right: ev_l3_none_dsen
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_l3_pat_sft_dsen
  left: ev_l3_pat_dsen
  right: ev_l3_sft_dsen
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: ring components (origin embeddings, attention bias, scale) against the same data without them
  page: 11
- id: c_l3_pat_math
  left: ev_l3_pat_math
  right: ev_l3_none_math
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_l3_sft_math
  left: ev_l3_sft_math
  right: ev_l3_none_math
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_l3_pat_sft_math
  left: ev_l3_pat_math
  right: ev_l3_sft_math
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: ring components (origin embeddings, attention bias, scale) against the same data without them
  page: 11
- id: c_q25_pat_sep
  left: ev_q25_pat_sep
  right: ev_q25_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_q25_sft_sep
  left: ev_q25_sft_sep
  right: ev_q25_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_q25_pat_sft_sep
  left: ev_q25_pat_sep
  right: ev_q25_sft_sep
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: ring components (origin embeddings, attention bias, scale) against the same data without them
  page: 11
- id: c_q25_pat_pia
  left: ev_q25_pat_pia
  right: ev_q25_none_pia
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_q25_sft_pia
  left: ev_q25_sft_pia
  right: ev_q25_none_pia
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_q25_pat_sft_pia
  left: ev_q25_pat_pia
  right: ev_q25_sft_pia
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: ring components (origin embeddings, attention bias, scale) against the same data without them
  page: 11
- id: c_q25_pat_dsen
  left: ev_q25_pat_dsen
  right: ev_q25_none_dsen
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_q25_sft_dsen
  left: ev_q25_sft_dsen
  right: ev_q25_none_dsen
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_q25_pat_sft_dsen
  left: ev_q25_pat_dsen
  right: ev_q25_sft_dsen
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: ring components (origin embeddings, attention bias, scale) against the same data without them
  page: 11
- id: c_q25_pat_math
  left: ev_q25_pat_math
  right: ev_q25_none_math
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_q25_sft_math
  left: ev_q25_sft_math
  right: ev_q25_none_math
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_q25_pat_sft_math
  left: ev_q25_pat_math
  right: ev_q25_sft_math
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: ring components (origin embeddings, attention bias, scale) against the same data without them
  page: 11
- id: c_mi_pat_sep
  left: ev_mi_pat_sep
  right: ev_mi_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_mi_sft_sep
  left: ev_mi_sft_sep
  right: ev_mi_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_mi_pat_sft_sep
  left: ev_mi_pat_sep
  right: ev_mi_sft_sep
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: ring components (origin embeddings, attention bias, scale) against the same data without them
  page: 11
- id: c_mi_pat_pia
  left: ev_mi_pat_pia
  right: ev_mi_none_pia
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_mi_sft_pia
  left: ev_mi_sft_pia
  right: ev_mi_none_pia
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_mi_pat_sft_pia
  left: ev_mi_pat_pia
  right: ev_mi_sft_pia
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: ring components (origin embeddings, attention bias, scale) against the same data without them
  page: 11
- id: c_mi_pat_dsen
  left: ev_mi_pat_dsen
  right: ev_mi_none_dsen
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_mi_sft_dsen
  left: ev_mi_sft_dsen
  right: ev_mi_none_dsen
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_mi_pat_sft_dsen
  left: ev_mi_pat_dsen
  right: ev_mi_sft_dsen
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: ring components (origin embeddings, attention bias, scale) against the same data without them
  page: 11
- id: c_mi_pat_math
  left: ev_mi_pat_math
  right: ev_mi_none_math
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_mi_sft_math
  left: ev_mi_sft_math
  right: ev_mi_none_math
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_mi_pat_sft_math
  left: ev_mi_pat_math
  right: ev_mi_sft_math
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: ring components (origin embeddings, attention bias, scale) against the same data without them
  page: 11
- id: c_t3_instr_l3_sep
  left: ev_t3_instr_l3_sep
  right: ev_l3_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_t3_instr_l3_pia
  left: ev_t3_instr_l3_pia
  right: ev_l3_none_pia
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_t3_instr_l3_dsen
  left: ev_t3_instr_l3_dsen
  right: ev_l3_none_dsen
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_t3_instr_l3_math
  left: ev_t3_instr_l3_math
  right: ev_l3_none_math
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_t3_sand_l3_sep
  left: ev_t3_sand_l3_sep
  right: ev_l3_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_t3_sand_l3_pia
  left: ev_t3_sand_l3_pia
  right: ev_l3_none_pia
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_t3_sand_l3_dsen
  left: ev_t3_sand_l3_dsen
  right: ev_l3_none_dsen
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_t3_sand_l3_math
  left: ev_t3_sand_l3_math
  right: ev_l3_none_math
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_t3_rseq_l3_sep
  left: ev_t3_rseq_l3_sep
  right: ev_l3_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_t3_rseq_l3_pia
  left: ev_t3_rseq_l3_pia
  right: ev_l3_none_pia
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_t3_rseq_l3_dsen
  left: ev_t3_rseq_l3_dsen
  right: ev_l3_none_dsen
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_t3_rseq_l3_math
  left: ev_t3_rseq_l3_math
  right: ev_l3_none_math
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_t3_delim_l3_sep
  left: ev_t3_delim_l3_sep
  right: ev_l3_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_t3_delim_l3_pia
  left: ev_t3_delim_l3_pia
  right: ev_l3_none_pia
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_t3_delim_l3_dsen
  left: ev_t3_delim_l3_dsen
  right: ev_l3_none_dsen
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_t3_delim_l3_math
  left: ev_t3_delim_l3_math
  right: ev_l3_none_math
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_t3_secalign_l3_sep
  left: ev_t3_secalign_l3_sep
  right: ev_l3_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_t3_secalign_l3_pia
  left: ev_t3_secalign_l3_pia
  right: ev_l3_none_pia
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_t3_secalign_l3_dsen
  left: ev_t3_secalign_l3_dsen
  right: ev_l3_none_dsen
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_t3_secalign_l3_math
  left: ev_t3_secalign_l3_math
  right: ev_l3_none_math
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_smol_bias_sep
  left: ev_smol_pat_sep
  right: ev_smol_nob_sep
  purpose: M
  c1_base: matched
  c2_layout: manipulated
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: origin attention bias kept against zeroed at inference, same trained checkpoint
  page: 12
- id: c_smol_bias_pia
  left: ev_smol_pat_pia
  right: ev_smol_nob_pia
  purpose: M
  c1_base: matched
  c2_layout: manipulated
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: origin attention bias kept against zeroed at inference, same trained checkpoint
  page: 12
- id: c_smol_bias_dsen
  left: ev_smol_pat_dsen
  right: ev_smol_nob_dsen
  purpose: M
  c1_base: matched
  c2_layout: manipulated
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: origin attention bias kept against zeroed at inference, same trained checkpoint
  page: 12
- id: c_smol_bias_math
  left: ev_smol_pat_math
  right: ev_smol_nob_math
  purpose: M
  c1_base: matched
  c2_layout: manipulated
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: origin attention bias kept against zeroed at inference, same trained checkpoint
  page: 12
- id: c_l3_bias_sep
  left: ev_l3_pat_sep
  right: ev_l3_nob_sep
  purpose: M
  c1_base: matched
  c2_layout: manipulated
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: origin attention bias kept against zeroed at inference, same trained checkpoint
  page: 12
- id: c_l3_bias_pia
  left: ev_l3_pat_pia
  right: ev_l3_nob_pia
  purpose: M
  c1_base: matched
  c2_layout: manipulated
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: origin attention bias kept against zeroed at inference, same trained checkpoint
  page: 12
- id: c_l3_bias_dsen
  left: ev_l3_pat_dsen
  right: ev_l3_nob_dsen
  purpose: M
  c1_base: matched
  c2_layout: manipulated
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: origin attention bias kept against zeroed at inference, same trained checkpoint
  page: 12
- id: c_l3_bias_math
  left: ev_l3_pat_math
  right: ev_l3_nob_math
  purpose: M
  c1_base: matched
  c2_layout: manipulated
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: origin attention bias kept against zeroed at inference, same trained checkpoint
  page: 12
- id: c_q25_bias_sep
  left: ev_q25_pat_sep
  right: ev_q25_nob_sep
  purpose: M
  c1_base: matched
  c2_layout: manipulated
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: origin attention bias kept against zeroed at inference, same trained checkpoint
  page: 12
- id: c_q25_bias_pia
  left: ev_q25_pat_pia
  right: ev_q25_nob_pia
  purpose: M
  c1_base: matched
  c2_layout: manipulated
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: origin attention bias kept against zeroed at inference, same trained checkpoint
  page: 12
- id: c_q25_bias_dsen
  left: ev_q25_pat_dsen
  right: ev_q25_nob_dsen
  purpose: M
  c1_base: matched
  c2_layout: manipulated
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: origin attention bias kept against zeroed at inference, same trained checkpoint
  page: 12
- id: c_q25_bias_math
  left: ev_q25_pat_math
  right: ev_q25_nob_math
  purpose: M
  c1_base: matched
  c2_layout: manipulated
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: origin attention bias kept against zeroed at inference, same trained checkpoint
  page: 12
- id: c_mi_bias_sep
  left: ev_mi_pat_sep
  right: ev_mi_nob_sep
  purpose: M
  c1_base: matched
  c2_layout: manipulated
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: origin attention bias kept against zeroed at inference, same trained checkpoint
  page: 12
- id: c_mi_bias_pia
  left: ev_mi_pat_pia
  right: ev_mi_nob_pia
  purpose: M
  c1_base: matched
  c2_layout: manipulated
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: origin attention bias kept against zeroed at inference, same trained checkpoint
  page: 12
- id: c_mi_bias_dsen
  left: ev_mi_pat_dsen
  right: ev_mi_nob_dsen
  purpose: M
  c1_base: matched
  c2_layout: manipulated
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: origin attention bias kept against zeroed at inference, same trained checkpoint
  page: 12
- id: c_mi_bias_math
  left: ev_mi_pat_math
  right: ev_mi_nob_math
  purpose: M
  c1_base: matched
  c2_layout: manipulated
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: origin attention bias kept against zeroed at inference, same trained checkpoint
  page: 12
- id: c_ae_smol
  left: ev_ae_smol_pat
  right: ev_ae_smol_sft
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: ring components against the same data without them
  page: 11
- id: c_ae_l3
  left: ev_ae_l3_pat
  right: ev_ae_l3_sft
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: ring components against the same data without them
  page: 11
- id: c_ae_q25
  left: ev_ae_q25_pat
  right: ev_ae_q25_sft
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: ring components against the same data without them
  page: 11
- id: c_ae_mi
  left: ev_ae_mi_pat
  right: ev_ae_mi_sft
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: ring components against the same data without them
  page: 11
- id: c_ad_sft_gcg
  left: ev_ad_sft_gcg
  right: ev_ad_none_gcg
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_ad_sft_neu
  left: ev_ad_sft_neu
  right: ev_ad_none_neu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_ad_pat_gcg
  left: ev_ad_pat_gcg
  right: ev_ad_none_gcg
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_ad_pat_neu
  left: ev_ad_pat_neu
  right: ev_ad_none_neu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 11
claims:
- id: k1
  text: Provenance-Aware Transformers resist indirect prompt injection in and out of distribution while keeping utility close to the base model
  location: abstract
  anchor: maintain robust resis- tance to IPI both in-distribution and out-of-distribution
  jobs:
  - R1
  - general_utility
  supported_by:
  - c_smol_pat_sep
  - c_smol_pat_sft_sep
  - c_smol_pat_pia
  - c_smol_pat_sft_pia
  - c_smol_pat_dsen
  - c_smol_pat_sft_dsen
  - c_smol_pat_math
  - c_smol_pat_sft_math
  - c_l3_pat_sep
  - c_l3_pat_sft_sep
  - c_l3_pat_pia
  - c_l3_pat_sft_pia
  - c_l3_pat_dsen
  - c_l3_pat_sft_dsen
  - c_l3_pat_math
  - c_l3_pat_sft_math
  - c_q25_pat_sep
  - c_q25_pat_sft_sep
  - c_q25_pat_pia
  - c_q25_pat_sft_pia
  - c_q25_pat_dsen
  - c_q25_pat_sft_dsen
  - c_q25_pat_math
  - c_q25_pat_sft_math
  - c_mi_pat_sep
  - c_mi_pat_sft_sep
  - c_mi_pat_pia
  - c_mi_pat_sft_pia
  - c_mi_pat_dsen
  - c_mi_pat_sft_dsen
  - c_mi_pat_math
  - c_mi_pat_sft_math
  - c_ae_l3
  - c_ae_mi
  page: 1
  contradicted_by:
  - r_ae_mi_pat
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Tables 3, 4, 6 and 7
    general_utility: Table 5 (win rate against the base model)
schema_gaps:
- where: evaluations ev_*
  note: Stage 2 training data tags only the inserted probe as Ring 3 inside a Ring 2 context (Step 4, p. 9); the paper does not say how rings were assigned on the OOD test sets, so whether the injected span is given to the
    model at test time is not reported
- where: results r_ad_*_neu
  note: Table 6 prints NeuExe ASR 48.44 (no defence) and 1.56 (Provenance-Aware), while the text gives 47.81% and 0.87% (p. 12); the record codes the table
- where: results r_ae_*
  note: the win rate is measured against the undefended base model, so the undefended model has no value of its own and the only comparisons are Provenance-Aware against Baseline SFT

`codel_arxiv2026` (2609.34463)

In [ ]:
%%writefile reports/records/codel_arxiv2026.yaml
report:
  id: codel_arxiv2026
  slug: '2609.34463'
  title: 'CoDeL: Co-Evolutionary Defense against Indirect Prompt Injection in LLM-based Agents'
  first_author: Xiao Yang
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2609.34463'
  version_read: arXiv v1
  first_date: '2026-09-28'
  read: F
  role: proposes
  code_release: stated as available in a footnote; not checked
  extraction_scope: 'Section 5.2: Table 1 rows No defense, AgentAlign, DRIP, RETA, SecOPD, Meta-SecAlign and CoDeL on both models and all three benchmarks; the abstract claim (MELON and PromptArmor, which filter or rewrite
    outside the model, PIGuard and TSGuard, which are detectors, Figure 3, the generalisation study of Section 5.4 and the ablations not coded)'
defences:
- id: codel_q
  name: CoDeL
  origin: this_report
  base_checkpoint: Qwen2.5-7B-Instruct
  axis1:
  - no_signal
  axis2:
  - sft
  - adversarial_rl
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 1
  components:
  - defender updated each round with LoRA-based GDPO under a decoupled reward for safety, task progress and format
  - a co-evolving prober searches injection rounds, methods and payloads, rewarded for attack success and for late detection
  - SFT stage before GDPO
  layout: agent loop with a shared tool schema and system prompt; tool outputs carry the injection
  layout_compat: &id001
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: The defender is updated each round via LoRA-based GDPO
  pages:
    components: 5
- id: agentalign_q
  name: AgentAlign (re-trained by the authors on their splits)
  origin: other_report
  base_checkpoint: Qwen2.5-7B-Instruct
  axis1:
  - no_signal
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 16
  layout: agent loop with a shared tool schema and system prompt; tool outputs carry the injection
  layout_compat: *id001
  anchor: we retrain each method under our setup
- id: drip_q
  name: DRIP (re-trained by the authors on their splits)
  origin: other_report
  base_checkpoint: Qwen2.5-7B-Instruct
  axis1:
  - chat_roles
  - learned_token_features
  - internal_intervention
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 16
  layout: agent loop with a shared tool schema and system prompt; tool outputs carry the injection
  layout_compat: *id001
  anchor: we retrain each method under our setup
- id: reta_q
  name: RETA (re-trained by the authors on their splits)
  origin: other_report
  base_checkpoint: Qwen2.5-7B-Instruct
  axis1:
  - no_signal
  axis2:
  - sft
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 16
  layout: agent loop with a shared tool schema and system prompt; tool outputs carry the injection
  layout_compat: *id001
  anchor: we retrain each method under our setup
- id: secopd_q
  name: SecOPD (re-trained by the authors on their splits)
  origin: other_report
  base_checkpoint: Qwen2.5-7B-Instruct
  axis1:
  - chat_roles
  axis2:
  - distillation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 16
  layout: agent loop with a shared tool schema and system prompt; tool outputs carry the injection
  layout_compat: *id001
  anchor: we retrain each method under our setup
- id: msa_q
  name: Meta-SecAlign (re-trained by the authors on their splits)
  origin: other_report
  base_checkpoint: Qwen2.5-7B-Instruct
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 16
  layout: agent loop with a shared tool schema and system prompt; tool outputs carry the injection
  layout_compat: *id001
  anchor: we retrain each method under our setup
- id: codel_l
  name: CoDeL
  origin: this_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - no_signal
  axis2:
  - sft
  - adversarial_rl
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 1
  components:
  - defender updated each round with LoRA-based GDPO under a decoupled reward for safety, task progress and format
  - a co-evolving prober searches injection rounds, methods and payloads, rewarded for attack success and for late detection
  - SFT stage before GDPO
  layout: agent loop with a shared tool schema and system prompt; tool outputs carry the injection
  layout_compat: *id001
  anchor: The defender is updated each round via LoRA-based GDPO
  pages:
    components: 5
- id: agentalign_l
  name: AgentAlign (re-trained by the authors on their splits)
  origin: other_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - no_signal
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 16
  layout: agent loop with a shared tool schema and system prompt; tool outputs carry the injection
  layout_compat: *id001
  anchor: we retrain each method under our setup
- id: drip_l
  name: DRIP (re-trained by the authors on their splits)
  origin: other_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - chat_roles
  - learned_token_features
  - internal_intervention
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 16
  layout: agent loop with a shared tool schema and system prompt; tool outputs carry the injection
  layout_compat: *id001
  anchor: we retrain each method under our setup
- id: reta_l
  name: RETA (re-trained by the authors on their splits)
  origin: other_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - no_signal
  axis2:
  - sft
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 16
  layout: agent loop with a shared tool schema and system prompt; tool outputs carry the injection
  layout_compat: *id001
  anchor: we retrain each method under our setup
- id: secopd_l
  name: SecOPD (re-trained by the authors on their splits)
  origin: other_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - chat_roles
  axis2:
  - distillation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 16
  layout: agent loop with a shared tool schema and system prompt; tool outputs carry the injection
  layout_compat: *id001
  anchor: we retrain each method under our setup
- id: msa_l
  name: Meta-SecAlign (re-trained by the authors on their splits)
  origin: other_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 16
  layout: agent loop with a shared tool schema and system prompt; tool outputs carry the injection
  layout_compat: *id001
  anchor: we retrain each method under our setup
evaluations:
- id: ev_q_ad_none_clean
  defence_id: none:Qwen2.5-7B-Instruct
  benchmark: AgentDojo (step-level annotated release), no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_q_ad_none_att
  defence_id: none:Qwen2.5-7B-Instruct
  benchmark: AgentDojo (step-level annotated release), benchmark injections
  attacker_type: static
  attacker_access: black_box
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_q_ia_none_clean
  defence_id: none:Qwen2.5-7B-Instruct
  benchmark: InjecAgent (1,054 cases), no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_q_ia_none_att
  defence_id: none:Qwen2.5-7B-Instruct
  benchmark: InjecAgent (1,054 cases), benchmark injections
  attacker_type: static
  attacker_access: black_box
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_q_asb_none_clean
  defence_id: none:Qwen2.5-7B-Instruct
  benchmark: Agent Security Bench, indirect prompt injection branch (ASB-OPI), no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_q_asb_none_att
  defence_id: none:Qwen2.5-7B-Instruct
  benchmark: Agent Security Bench, indirect prompt injection branch (ASB-OPI), benchmark injections
  attacker_type: static
  attacker_access: black_box
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_l_ad_none_clean
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: AgentDojo (step-level annotated release), no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_l_ad_none_att
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: AgentDojo (step-level annotated release), benchmark injections
  attacker_type: static
  attacker_access: black_box
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_l_ia_none_clean
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: InjecAgent (1,054 cases), no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_l_ia_none_att
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: InjecAgent (1,054 cases), benchmark injections
  attacker_type: static
  attacker_access: black_box
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_l_asb_none_clean
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: Agent Security Bench, indirect prompt injection branch (ASB-OPI), no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_l_asb_none_att
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: Agent Security Bench, indirect prompt injection branch (ASB-OPI), benchmark injections
  attacker_type: static
  attacker_access: black_box
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_q_ad_agentalign_clean
  defence_id: agentalign_q
  benchmark: AgentDojo (step-level annotated release), no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_ad_agentalign_att
  defence_id: agentalign_q
  benchmark: AgentDojo (step-level annotated release), benchmark injections
  attacker_type: static
  attacker_access: black_box
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_ia_agentalign_clean
  defence_id: agentalign_q
  benchmark: InjecAgent (1,054 cases), no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_ia_agentalign_att
  defence_id: agentalign_q
  benchmark: InjecAgent (1,054 cases), benchmark injections
  attacker_type: static
  attacker_access: black_box
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_asb_agentalign_clean
  defence_id: agentalign_q
  benchmark: Agent Security Bench, indirect prompt injection branch (ASB-OPI), no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_asb_agentalign_att
  defence_id: agentalign_q
  benchmark: Agent Security Bench, indirect prompt injection branch (ASB-OPI), benchmark injections
  attacker_type: static
  attacker_access: black_box
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_ad_agentalign_clean
  defence_id: agentalign_l
  benchmark: AgentDojo (step-level annotated release), no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_ad_agentalign_att
  defence_id: agentalign_l
  benchmark: AgentDojo (step-level annotated release), benchmark injections
  attacker_type: static
  attacker_access: black_box
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_ia_agentalign_clean
  defence_id: agentalign_l
  benchmark: InjecAgent (1,054 cases), no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_ia_agentalign_att
  defence_id: agentalign_l
  benchmark: InjecAgent (1,054 cases), benchmark injections
  attacker_type: static
  attacker_access: black_box
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_asb_agentalign_clean
  defence_id: agentalign_l
  benchmark: Agent Security Bench, indirect prompt injection branch (ASB-OPI), no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_asb_agentalign_att
  defence_id: agentalign_l
  benchmark: Agent Security Bench, indirect prompt injection branch (ASB-OPI), benchmark injections
  attacker_type: static
  attacker_access: black_box
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_ad_drip_clean
  defence_id: drip_q
  benchmark: AgentDojo (step-level annotated release), no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_ad_drip_att
  defence_id: drip_q
  benchmark: AgentDojo (step-level annotated release), benchmark injections
  attacker_type: static
  attacker_access: black_box
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_ia_drip_clean
  defence_id: drip_q
  benchmark: InjecAgent (1,054 cases), no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_ia_drip_att
  defence_id: drip_q
  benchmark: InjecAgent (1,054 cases), benchmark injections
  attacker_type: static
  attacker_access: black_box
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_asb_drip_clean
  defence_id: drip_q
  benchmark: Agent Security Bench, indirect prompt injection branch (ASB-OPI), no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_asb_drip_att
  defence_id: drip_q
  benchmark: Agent Security Bench, indirect prompt injection branch (ASB-OPI), benchmark injections
  attacker_type: static
  attacker_access: black_box
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_ad_drip_clean
  defence_id: drip_l
  benchmark: AgentDojo (step-level annotated release), no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_ad_drip_att
  defence_id: drip_l
  benchmark: AgentDojo (step-level annotated release), benchmark injections
  attacker_type: static
  attacker_access: black_box
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_ia_drip_clean
  defence_id: drip_l
  benchmark: InjecAgent (1,054 cases), no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_ia_drip_att
  defence_id: drip_l
  benchmark: InjecAgent (1,054 cases), benchmark injections
  attacker_type: static
  attacker_access: black_box
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_asb_drip_clean
  defence_id: drip_l
  benchmark: Agent Security Bench, indirect prompt injection branch (ASB-OPI), no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_asb_drip_att
  defence_id: drip_l
  benchmark: Agent Security Bench, indirect prompt injection branch (ASB-OPI), benchmark injections
  attacker_type: static
  attacker_access: black_box
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_ad_reta_clean
  defence_id: reta_q
  benchmark: AgentDojo (step-level annotated release), no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_ad_reta_att
  defence_id: reta_q
  benchmark: AgentDojo (step-level annotated release), benchmark injections
  attacker_type: static
  attacker_access: black_box
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_ia_reta_clean
  defence_id: reta_q
  benchmark: InjecAgent (1,054 cases), no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_ia_reta_att
  defence_id: reta_q
  benchmark: InjecAgent (1,054 cases), benchmark injections
  attacker_type: static
  attacker_access: black_box
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_asb_reta_clean
  defence_id: reta_q
  benchmark: Agent Security Bench, indirect prompt injection branch (ASB-OPI), no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_asb_reta_att
  defence_id: reta_q
  benchmark: Agent Security Bench, indirect prompt injection branch (ASB-OPI), benchmark injections
  attacker_type: static
  attacker_access: black_box
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_ad_reta_clean
  defence_id: reta_l
  benchmark: AgentDojo (step-level annotated release), no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_ad_reta_att
  defence_id: reta_l
  benchmark: AgentDojo (step-level annotated release), benchmark injections
  attacker_type: static
  attacker_access: black_box
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_ia_reta_clean
  defence_id: reta_l
  benchmark: InjecAgent (1,054 cases), no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_ia_reta_att
  defence_id: reta_l
  benchmark: InjecAgent (1,054 cases), benchmark injections
  attacker_type: static
  attacker_access: black_box
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_asb_reta_clean
  defence_id: reta_l
  benchmark: Agent Security Bench, indirect prompt injection branch (ASB-OPI), no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_asb_reta_att
  defence_id: reta_l
  benchmark: Agent Security Bench, indirect prompt injection branch (ASB-OPI), benchmark injections
  attacker_type: static
  attacker_access: black_box
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_ad_secopd_clean
  defence_id: secopd_q
  benchmark: AgentDojo (step-level annotated release), no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_ad_secopd_att
  defence_id: secopd_q
  benchmark: AgentDojo (step-level annotated release), benchmark injections
  attacker_type: static
  attacker_access: black_box
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_ia_secopd_clean
  defence_id: secopd_q
  benchmark: InjecAgent (1,054 cases), no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_ia_secopd_att
  defence_id: secopd_q
  benchmark: InjecAgent (1,054 cases), benchmark injections
  attacker_type: static
  attacker_access: black_box
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_asb_secopd_clean
  defence_id: secopd_q
  benchmark: Agent Security Bench, indirect prompt injection branch (ASB-OPI), no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_asb_secopd_att
  defence_id: secopd_q
  benchmark: Agent Security Bench, indirect prompt injection branch (ASB-OPI), benchmark injections
  attacker_type: static
  attacker_access: black_box
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_ad_secopd_clean
  defence_id: secopd_l
  benchmark: AgentDojo (step-level annotated release), no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_ad_secopd_att
  defence_id: secopd_l
  benchmark: AgentDojo (step-level annotated release), benchmark injections
  attacker_type: static
  attacker_access: black_box
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_ia_secopd_clean
  defence_id: secopd_l
  benchmark: InjecAgent (1,054 cases), no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_ia_secopd_att
  defence_id: secopd_l
  benchmark: InjecAgent (1,054 cases), benchmark injections
  attacker_type: static
  attacker_access: black_box
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_asb_secopd_clean
  defence_id: secopd_l
  benchmark: Agent Security Bench, indirect prompt injection branch (ASB-OPI), no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_asb_secopd_att
  defence_id: secopd_l
  benchmark: Agent Security Bench, indirect prompt injection branch (ASB-OPI), benchmark injections
  attacker_type: static
  attacker_access: black_box
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_ad_msa_clean
  defence_id: msa_q
  benchmark: AgentDojo (step-level annotated release), no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_ad_msa_att
  defence_id: msa_q
  benchmark: AgentDojo (step-level annotated release), benchmark injections
  attacker_type: static
  attacker_access: black_box
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_ia_msa_clean
  defence_id: msa_q
  benchmark: InjecAgent (1,054 cases), no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_ia_msa_att
  defence_id: msa_q
  benchmark: InjecAgent (1,054 cases), benchmark injections
  attacker_type: static
  attacker_access: black_box
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_asb_msa_clean
  defence_id: msa_q
  benchmark: Agent Security Bench, indirect prompt injection branch (ASB-OPI), no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_asb_msa_att
  defence_id: msa_q
  benchmark: Agent Security Bench, indirect prompt injection branch (ASB-OPI), benchmark injections
  attacker_type: static
  attacker_access: black_box
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_ad_msa_clean
  defence_id: msa_l
  benchmark: AgentDojo (step-level annotated release), no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_ad_msa_att
  defence_id: msa_l
  benchmark: AgentDojo (step-level annotated release), benchmark injections
  attacker_type: static
  attacker_access: black_box
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_ia_msa_clean
  defence_id: msa_l
  benchmark: InjecAgent (1,054 cases), no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_ia_msa_att
  defence_id: msa_l
  benchmark: InjecAgent (1,054 cases), benchmark injections
  attacker_type: static
  attacker_access: black_box
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_asb_msa_clean
  defence_id: msa_l
  benchmark: Agent Security Bench, indirect prompt injection branch (ASB-OPI), no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_asb_msa_att
  defence_id: msa_l
  benchmark: Agent Security Bench, indirect prompt injection branch (ASB-OPI), benchmark injections
  attacker_type: static
  attacker_access: black_box
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_ad_codel_clean
  defence_id: codel_q
  benchmark: AgentDojo (step-level annotated release), no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_ad_codel_att
  defence_id: codel_q
  benchmark: AgentDojo (step-level annotated release), benchmark injections
  attacker_type: static
  attacker_access: black_box
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_ia_codel_clean
  defence_id: codel_q
  benchmark: InjecAgent (1,054 cases), no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_ia_codel_att
  defence_id: codel_q
  benchmark: InjecAgent (1,054 cases), benchmark injections
  attacker_type: static
  attacker_access: black_box
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_asb_codel_clean
  defence_id: codel_q
  benchmark: Agent Security Bench, indirect prompt injection branch (ASB-OPI), no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_q_asb_codel_att
  defence_id: codel_q
  benchmark: Agent Security Bench, indirect prompt injection branch (ASB-OPI), benchmark injections
  attacker_type: static
  attacker_access: black_box
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_ad_codel_clean
  defence_id: codel_l
  benchmark: AgentDojo (step-level annotated release), no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_ad_codel_att
  defence_id: codel_l
  benchmark: AgentDojo (step-level annotated release), benchmark injections
  attacker_type: static
  attacker_access: black_box
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_ia_codel_clean
  defence_id: codel_l
  benchmark: InjecAgent (1,054 cases), no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_ia_codel_att
  defence_id: codel_l
  benchmark: InjecAgent (1,054 cases), benchmark injections
  attacker_type: static
  attacker_access: black_box
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_asb_codel_clean
  defence_id: codel_l
  benchmark: Agent Security Bench, indirect prompt injection branch (ASB-OPI), no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_l_asb_codel_att
  defence_id: codel_l
  benchmark: Agent Security Bench, indirect prompt injection branch (ASB-OPI), benchmark injections
  attacker_type: static
  attacker_access: black_box
  judge: LLM judges for attack execution and task completion
  layout_same_as_defence: 'yes'
  measured_by: codel_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 7
results:
- id: r_q_ad_none_bu
  evaluation_id: ev_q_ad_none_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '0.632'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_ad_none_asr
  evaluation_id: ev_q_ad_none_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.364'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_ad_none_ua
  evaluation_id: ev_q_ad_none_att
  quantity: R3safe
  coverage: direct
  metric_as_printed: UA (task completed while the injection is not executed)
  value: '0.432'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_ia_none_bu
  evaluation_id: ev_q_ia_none_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '0.765'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_ia_none_asr
  evaluation_id: ev_q_ia_none_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.221'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_ia_none_ua
  evaluation_id: ev_q_ia_none_att
  quantity: R3safe
  coverage: direct
  metric_as_printed: UA (task completed while the injection is not executed)
  value: '0.544'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_asb_none_bu
  evaluation_id: ev_q_asb_none_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '0.857'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_asb_none_asr
  evaluation_id: ev_q_asb_none_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.377'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_asb_none_ua
  evaluation_id: ev_q_asb_none_att
  quantity: R3safe
  coverage: direct
  metric_as_printed: UA (task completed while the injection is not executed)
  value: '0.232'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_ad_none_bu
  evaluation_id: ev_l_ad_none_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '0.727'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_ad_none_asr
  evaluation_id: ev_l_ad_none_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.189'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_ad_none_ua
  evaluation_id: ev_l_ad_none_att
  quantity: R3safe
  coverage: direct
  metric_as_printed: UA (task completed while the injection is not executed)
  value: '0.679'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_ia_none_bu
  evaluation_id: ev_l_ia_none_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '0.647'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_ia_none_asr
  evaluation_id: ev_l_ia_none_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.299'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_ia_none_ua
  evaluation_id: ev_l_ia_none_att
  quantity: R3safe
  coverage: direct
  metric_as_printed: UA (task completed while the injection is not executed)
  value: '0.480'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_asb_none_bu
  evaluation_id: ev_l_asb_none_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '0.900'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_asb_none_asr
  evaluation_id: ev_l_asb_none_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.377'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_asb_none_ua
  evaluation_id: ev_l_asb_none_att
  quantity: R3safe
  coverage: direct
  metric_as_printed: UA (task completed while the injection is not executed)
  value: '0.522'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_ad_agentalign_bu
  evaluation_id: ev_q_ad_agentalign_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '0.361'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_ad_agentalign_asr
  evaluation_id: ev_q_ad_agentalign_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.116'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_ad_agentalign_ua
  evaluation_id: ev_q_ad_agentalign_att
  quantity: R3safe
  coverage: direct
  metric_as_printed: UA (task completed while the injection is not executed)
  value: '0.290'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_ia_agentalign_bu
  evaluation_id: ev_q_ia_agentalign_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '0.333'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_ia_agentalign_asr
  evaluation_id: ev_q_ia_agentalign_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.111'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_ia_agentalign_ua
  evaluation_id: ev_q_ia_agentalign_att
  quantity: R3safe
  coverage: direct
  metric_as_printed: UA (task completed while the injection is not executed)
  value: '0.333'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_asb_agentalign_bu
  evaluation_id: ev_q_asb_agentalign_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '0.429'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_asb_agentalign_asr
  evaluation_id: ev_q_asb_agentalign_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.029'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_asb_agentalign_ua
  evaluation_id: ev_q_asb_agentalign_att
  quantity: R3safe
  coverage: direct
  metric_as_printed: UA (task completed while the injection is not executed)
  value: '0.130'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_ad_agentalign_bu
  evaluation_id: ev_l_ad_agentalign_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '0.309'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_ad_agentalign_asr
  evaluation_id: ev_l_ad_agentalign_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.066'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_ad_agentalign_ua
  evaluation_id: ev_l_ad_agentalign_att
  quantity: R3safe
  coverage: direct
  metric_as_printed: UA (task completed while the injection is not executed)
  value: '0.179'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_ia_agentalign_bu
  evaluation_id: ev_l_ia_agentalign_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '0.361'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_ia_agentalign_asr
  evaluation_id: ev_l_ia_agentalign_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.139'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_ia_agentalign_ua
  evaluation_id: ev_l_ia_agentalign_att
  quantity: R3safe
  coverage: direct
  metric_as_printed: UA (task completed while the injection is not executed)
  value: '0.333'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_asb_agentalign_bu
  evaluation_id: ev_l_asb_agentalign_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '0.571'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_asb_agentalign_asr
  evaluation_id: ev_l_asb_agentalign_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.159'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_asb_agentalign_ua
  evaluation_id: ev_l_asb_agentalign_att
  quantity: R3safe
  coverage: direct
  metric_as_printed: UA (task completed while the injection is not executed)
  value: '0.145'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_ad_drip_bu
  evaluation_id: ev_q_ad_drip_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '0.636'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_ad_drip_asr
  evaluation_id: ev_q_ad_drip_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.151'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_ad_drip_ua
  evaluation_id: ev_q_ad_drip_att
  quantity: R3safe
  coverage: direct
  metric_as_printed: UA (task completed while the injection is not executed)
  value: '0.491'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_ia_drip_bu
  evaluation_id: ev_q_ia_drip_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '0.588'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_ia_drip_asr
  evaluation_id: ev_q_ia_drip_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.294'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_ia_drip_ua
  evaluation_id: ev_q_ia_drip_att
  quantity: R3safe
  coverage: direct
  metric_as_printed: UA (task completed while the injection is not executed)
  value: '0.304'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_asb_drip_bu
  evaluation_id: ev_q_asb_drip_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '0.700'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_asb_drip_asr
  evaluation_id: ev_q_asb_drip_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.232'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_asb_drip_ua
  evaluation_id: ev_q_asb_drip_att
  quantity: R3safe
  coverage: direct
  metric_as_printed: UA (task completed while the injection is not executed)
  value: '0.609'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_ad_drip_bu
  evaluation_id: ev_l_ad_drip_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '0.546'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_ad_drip_asr
  evaluation_id: ev_l_ad_drip_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.151'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_ad_drip_ua
  evaluation_id: ev_l_ad_drip_att
  quantity: R3safe
  coverage: direct
  metric_as_printed: UA (task completed while the injection is not executed)
  value: '0.264'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_ia_drip_bu
  evaluation_id: ev_l_ia_drip_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '0.588'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_ia_drip_asr
  evaluation_id: ev_l_ia_drip_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.078'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_ia_drip_ua
  evaluation_id: ev_l_ia_drip_att
  quantity: R3safe
  coverage: direct
  metric_as_printed: UA (task completed while the injection is not executed)
  value: '0.451'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_asb_drip_bu
  evaluation_id: ev_l_asb_drip_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '0.400'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_asb_drip_asr
  evaluation_id: ev_l_asb_drip_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.087'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_asb_drip_ua
  evaluation_id: ev_l_asb_drip_att
  quantity: R3safe
  coverage: direct
  metric_as_printed: UA (task completed while the injection is not executed)
  value: '0.290'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_ad_reta_bu
  evaluation_id: ev_q_ad_reta_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '0.618'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_ad_reta_asr
  evaluation_id: ev_q_ad_reta_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.119'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_ad_reta_ua
  evaluation_id: ev_q_ad_reta_att
  quantity: R3safe
  coverage: direct
  metric_as_printed: UA (task completed while the injection is not executed)
  value: '0.517'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_ia_reta_bu
  evaluation_id: ev_q_ia_reta_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '0.706'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_ia_reta_asr
  evaluation_id: ev_q_ia_reta_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.069'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_ia_reta_ua
  evaluation_id: ev_q_ia_reta_att
  quantity: R3safe
  coverage: direct
  metric_as_printed: UA (task completed while the injection is not executed)
  value: '0.578'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_asb_reta_bu
  evaluation_id: ev_q_asb_reta_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '0.647'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_asb_reta_asr
  evaluation_id: ev_q_asb_reta_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.073'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_asb_reta_ua
  evaluation_id: ev_q_asb_reta_att
  quantity: R3safe
  coverage: direct
  metric_as_printed: UA (task completed while the injection is not executed)
  value: '0.638'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_ad_reta_bu
  evaluation_id: ev_l_ad_reta_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '0.632'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_ad_reta_asr
  evaluation_id: ev_l_ad_reta_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.102'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_ad_reta_ua
  evaluation_id: ev_l_ad_reta_att
  quantity: R3safe
  coverage: direct
  metric_as_printed: UA (task completed while the injection is not executed)
  value: '0.542'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_ia_reta_bu
  evaluation_id: ev_l_ia_reta_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '0.735'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_ia_reta_asr
  evaluation_id: ev_l_ia_reta_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.059'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_ia_reta_ua
  evaluation_id: ev_l_ia_reta_att
  quantity: R3safe
  coverage: direct
  metric_as_printed: UA (task completed while the injection is not executed)
  value: '0.549'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_asb_reta_bu
  evaluation_id: ev_l_asb_reta_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '0.810'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_asb_reta_asr
  evaluation_id: ev_l_asb_reta_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.058'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_asb_reta_ua
  evaluation_id: ev_l_asb_reta_att
  quantity: R3safe
  coverage: direct
  metric_as_printed: UA (task completed while the injection is not executed)
  value: '0.739'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_ad_secopd_bu
  evaluation_id: ev_q_ad_secopd_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '0.273'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_ad_secopd_asr
  evaluation_id: ev_q_ad_secopd_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.046'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_ad_secopd_ua
  evaluation_id: ev_q_ad_secopd_att
  quantity: R3safe
  coverage: direct
  metric_as_printed: UA (task completed while the injection is not executed)
  value: '0.343'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_ia_secopd_bu
  evaluation_id: ev_q_ia_secopd_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '0.417'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_ia_secopd_asr
  evaluation_id: ev_q_ia_secopd_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.010'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_ia_secopd_ua
  evaluation_id: ev_q_ia_secopd_att
  quantity: R3safe
  coverage: direct
  metric_as_printed: UA (task completed while the injection is not executed)
  value: '0.446'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_asb_secopd_bu
  evaluation_id: ev_q_asb_secopd_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '0.286'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_asb_secopd_asr
  evaluation_id: ev_q_asb_secopd_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.319'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_asb_secopd_ua
  evaluation_id: ev_q_asb_secopd_att
  quantity: R3safe
  coverage: direct
  metric_as_printed: UA (task completed while the injection is not executed)
  value: '0.333'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_ad_secopd_bu
  evaluation_id: ev_l_ad_secopd_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '0.103'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_ad_secopd_asr
  evaluation_id: ev_l_ad_secopd_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.144'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_ad_secopd_ua
  evaluation_id: ev_l_ad_secopd_att
  quantity: R3safe
  coverage: direct
  metric_as_printed: UA (task completed while the injection is not executed)
  value: '0.228'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_ia_secopd_bu
  evaluation_id: ev_l_ia_secopd_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '0.529'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_ia_secopd_asr
  evaluation_id: ev_l_ia_secopd_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.203'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_ia_secopd_ua
  evaluation_id: ev_l_ia_secopd_att
  quantity: R3safe
  coverage: direct
  metric_as_printed: UA (task completed while the injection is not executed)
  value: '0.461'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_asb_secopd_bu
  evaluation_id: ev_l_asb_secopd_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '0.143'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_asb_secopd_asr
  evaluation_id: ev_l_asb_secopd_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_asb_secopd_ua
  evaluation_id: ev_l_asb_secopd_att
  quantity: R3safe
  coverage: direct
  metric_as_printed: UA (task completed while the injection is not executed)
  value: '0.101'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_ad_msa_bu
  evaluation_id: ev_q_ad_msa_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '0.456'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_ad_msa_asr
  evaluation_id: ev_q_ad_msa_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.068'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_ad_msa_ua
  evaluation_id: ev_q_ad_msa_att
  quantity: R3safe
  coverage: direct
  metric_as_printed: UA (task completed while the injection is not executed)
  value: '0.602'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_ia_msa_bu
  evaluation_id: ev_q_ia_msa_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '0.824'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_ia_msa_asr
  evaluation_id: ev_q_ia_msa_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.059'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_ia_msa_ua
  evaluation_id: ev_q_ia_msa_att
  quantity: R3safe
  coverage: direct
  metric_as_printed: UA (task completed while the injection is not executed)
  value: '0.678'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_asb_msa_bu
  evaluation_id: ev_q_asb_msa_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '0.714'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_asb_msa_asr
  evaluation_id: ev_q_asb_msa_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.073'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_asb_msa_ua
  evaluation_id: ev_q_asb_msa_att
  quantity: R3safe
  coverage: direct
  metric_as_printed: UA (task completed while the injection is not executed)
  value: '0.279'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_ad_msa_bu
  evaluation_id: ev_l_ad_msa_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '0.456'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_ad_msa_asr
  evaluation_id: ev_l_ad_msa_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.025'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_ad_msa_ua
  evaluation_id: ev_l_ad_msa_att
  quantity: R3safe
  coverage: direct
  metric_as_printed: UA (task completed while the injection is not executed)
  value: '0.610'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_ia_msa_bu
  evaluation_id: ev_l_ia_msa_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '0.765'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_ia_msa_asr
  evaluation_id: ev_l_ia_msa_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.059'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_ia_msa_ua
  evaluation_id: ev_l_ia_msa_att
  quantity: R3safe
  coverage: direct
  metric_as_printed: UA (task completed while the injection is not executed)
  value: '0.466'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_asb_msa_bu
  evaluation_id: ev_l_asb_msa_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '0.857'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_asb_msa_asr
  evaluation_id: ev_l_asb_msa_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.058'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_asb_msa_ua
  evaluation_id: ev_l_asb_msa_att
  quantity: R3safe
  coverage: direct
  metric_as_printed: UA (task completed while the injection is not executed)
  value: '0.812'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_ad_codel_bu
  evaluation_id: ev_q_ad_codel_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '0.779'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_ad_codel_asr
  evaluation_id: ev_q_ad_codel_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.042'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_ad_codel_ua
  evaluation_id: ev_q_ad_codel_att
  quantity: R3safe
  coverage: direct
  metric_as_printed: UA (task completed while the injection is not executed)
  value: '0.831'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_ia_codel_bu
  evaluation_id: ev_q_ia_codel_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '0.882'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_ia_codel_asr
  evaluation_id: ev_q_ia_codel_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.010'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_ia_codel_ua
  evaluation_id: ev_q_ia_codel_att
  quantity: R3safe
  coverage: direct
  metric_as_printed: UA (task completed while the injection is not executed)
  value: '0.843'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_asb_codel_bu
  evaluation_id: ev_q_asb_codel_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '0.857'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_asb_codel_asr
  evaluation_id: ev_q_asb_codel_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.043'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q_asb_codel_ua
  evaluation_id: ev_q_asb_codel_att
  quantity: R3safe
  coverage: direct
  metric_as_printed: UA (task completed while the injection is not executed)
  value: '0.913'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_ad_codel_bu
  evaluation_id: ev_l_ad_codel_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '0.819'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_ad_codel_asr
  evaluation_id: ev_l_ad_codel_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.051'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_ad_codel_ua
  evaluation_id: ev_l_ad_codel_att
  quantity: R3safe
  coverage: direct
  metric_as_printed: UA (task completed while the injection is not executed)
  value: '0.797'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_ia_codel_bu
  evaluation_id: ev_l_ia_codel_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '0.882'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_ia_codel_asr
  evaluation_id: ev_l_ia_codel_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.020'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_ia_codel_ua
  evaluation_id: ev_l_ia_codel_att
  quantity: R3safe
  coverage: direct
  metric_as_printed: UA (task completed while the injection is not executed)
  value: '0.809'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_asb_codel_bu
  evaluation_id: ev_l_asb_codel_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: BU
  value: '1.000'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_asb_codel_asr
  evaluation_id: ev_l_asb_codel_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l_asb_codel_ua
  evaluation_id: ev_l_asb_codel_att
  quantity: R3safe
  coverage: direct
  metric_as_printed: UA (task completed while the injection is not executed)
  value: '0.942'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
comparisons:
- id: c_q_ad_agentalign_clean
  left: ev_q_ad_agentalign_clean
  right: ev_q_ad_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_ad_agentalign_att
  left: ev_q_ad_agentalign_att
  right: ev_q_ad_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_ad_drip_clean
  left: ev_q_ad_drip_clean
  right: ev_q_ad_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_ad_drip_att
  left: ev_q_ad_drip_att
  right: ev_q_ad_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_ad_reta_clean
  left: ev_q_ad_reta_clean
  right: ev_q_ad_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_ad_reta_att
  left: ev_q_ad_reta_att
  right: ev_q_ad_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_ad_secopd_clean
  left: ev_q_ad_secopd_clean
  right: ev_q_ad_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_ad_secopd_att
  left: ev_q_ad_secopd_att
  right: ev_q_ad_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_ad_msa_clean
  left: ev_q_ad_msa_clean
  right: ev_q_ad_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_ad_msa_att
  left: ev_q_ad_msa_att
  right: ev_q_ad_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_ad_codel_clean
  left: ev_q_ad_codel_clean
  right: ev_q_ad_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_ad_codel_att
  left: ev_q_ad_codel_att
  right: ev_q_ad_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_ia_agentalign_clean
  left: ev_q_ia_agentalign_clean
  right: ev_q_ia_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_ia_agentalign_att
  left: ev_q_ia_agentalign_att
  right: ev_q_ia_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_ia_drip_clean
  left: ev_q_ia_drip_clean
  right: ev_q_ia_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_ia_drip_att
  left: ev_q_ia_drip_att
  right: ev_q_ia_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_ia_reta_clean
  left: ev_q_ia_reta_clean
  right: ev_q_ia_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_ia_reta_att
  left: ev_q_ia_reta_att
  right: ev_q_ia_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_ia_secopd_clean
  left: ev_q_ia_secopd_clean
  right: ev_q_ia_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_ia_secopd_att
  left: ev_q_ia_secopd_att
  right: ev_q_ia_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_ia_msa_clean
  left: ev_q_ia_msa_clean
  right: ev_q_ia_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_ia_msa_att
  left: ev_q_ia_msa_att
  right: ev_q_ia_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_ia_codel_clean
  left: ev_q_ia_codel_clean
  right: ev_q_ia_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_ia_codel_att
  left: ev_q_ia_codel_att
  right: ev_q_ia_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_asb_agentalign_clean
  left: ev_q_asb_agentalign_clean
  right: ev_q_asb_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_asb_agentalign_att
  left: ev_q_asb_agentalign_att
  right: ev_q_asb_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_asb_drip_clean
  left: ev_q_asb_drip_clean
  right: ev_q_asb_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_asb_drip_att
  left: ev_q_asb_drip_att
  right: ev_q_asb_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_asb_reta_clean
  left: ev_q_asb_reta_clean
  right: ev_q_asb_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_asb_reta_att
  left: ev_q_asb_reta_att
  right: ev_q_asb_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_asb_secopd_clean
  left: ev_q_asb_secopd_clean
  right: ev_q_asb_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_asb_secopd_att
  left: ev_q_asb_secopd_att
  right: ev_q_asb_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_asb_msa_clean
  left: ev_q_asb_msa_clean
  right: ev_q_asb_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_asb_msa_att
  left: ev_q_asb_msa_att
  right: ev_q_asb_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_asb_codel_clean
  left: ev_q_asb_codel_clean
  right: ev_q_asb_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q_asb_codel_att
  left: ev_q_asb_codel_att
  right: ev_q_asb_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_ad_agentalign_clean
  left: ev_l_ad_agentalign_clean
  right: ev_l_ad_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_ad_agentalign_att
  left: ev_l_ad_agentalign_att
  right: ev_l_ad_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_ad_drip_clean
  left: ev_l_ad_drip_clean
  right: ev_l_ad_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_ad_drip_att
  left: ev_l_ad_drip_att
  right: ev_l_ad_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_ad_reta_clean
  left: ev_l_ad_reta_clean
  right: ev_l_ad_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_ad_reta_att
  left: ev_l_ad_reta_att
  right: ev_l_ad_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_ad_secopd_clean
  left: ev_l_ad_secopd_clean
  right: ev_l_ad_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_ad_secopd_att
  left: ev_l_ad_secopd_att
  right: ev_l_ad_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_ad_msa_clean
  left: ev_l_ad_msa_clean
  right: ev_l_ad_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_ad_msa_att
  left: ev_l_ad_msa_att
  right: ev_l_ad_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_ad_codel_clean
  left: ev_l_ad_codel_clean
  right: ev_l_ad_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_ad_codel_att
  left: ev_l_ad_codel_att
  right: ev_l_ad_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_ia_agentalign_clean
  left: ev_l_ia_agentalign_clean
  right: ev_l_ia_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_ia_agentalign_att
  left: ev_l_ia_agentalign_att
  right: ev_l_ia_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_ia_drip_clean
  left: ev_l_ia_drip_clean
  right: ev_l_ia_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_ia_drip_att
  left: ev_l_ia_drip_att
  right: ev_l_ia_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_ia_reta_clean
  left: ev_l_ia_reta_clean
  right: ev_l_ia_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_ia_reta_att
  left: ev_l_ia_reta_att
  right: ev_l_ia_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_ia_secopd_clean
  left: ev_l_ia_secopd_clean
  right: ev_l_ia_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_ia_secopd_att
  left: ev_l_ia_secopd_att
  right: ev_l_ia_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_ia_msa_clean
  left: ev_l_ia_msa_clean
  right: ev_l_ia_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_ia_msa_att
  left: ev_l_ia_msa_att
  right: ev_l_ia_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_ia_codel_clean
  left: ev_l_ia_codel_clean
  right: ev_l_ia_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_ia_codel_att
  left: ev_l_ia_codel_att
  right: ev_l_ia_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_asb_agentalign_clean
  left: ev_l_asb_agentalign_clean
  right: ev_l_asb_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_asb_agentalign_att
  left: ev_l_asb_agentalign_att
  right: ev_l_asb_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_asb_drip_clean
  left: ev_l_asb_drip_clean
  right: ev_l_asb_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_asb_drip_att
  left: ev_l_asb_drip_att
  right: ev_l_asb_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_asb_reta_clean
  left: ev_l_asb_reta_clean
  right: ev_l_asb_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_asb_reta_att
  left: ev_l_asb_reta_att
  right: ev_l_asb_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_asb_secopd_clean
  left: ev_l_asb_secopd_clean
  right: ev_l_asb_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_asb_secopd_att
  left: ev_l_asb_secopd_att
  right: ev_l_asb_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_asb_msa_clean
  left: ev_l_asb_msa_clean
  right: ev_l_asb_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_asb_msa_att
  left: ev_l_asb_msa_att
  right: ev_l_asb_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_asb_codel_clean
  left: ev_l_asb_codel_clean
  right: ev_l_asb_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_asb_codel_att
  left: ev_l_asb_codel_att
  right: ev_l_asb_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
claims:
- id: k1
  text: CoDeL cuts ASR by 88.5% and clearly outperforms the other baselines
  location: abstract
  anchor: CODEL reduces attack success rate (ASR) by 88.5%
  jobs:
  - R1
  - R2
  - R3
  supported_by:
  - c_q_ad_codel_clean
  - c_q_ad_codel_att
  - c_q_ia_codel_clean
  - c_q_ia_codel_att
  - c_q_asb_codel_clean
  - c_q_asb_codel_att
  - c_l_ad_codel_clean
  - c_l_ad_codel_att
  - c_l_ia_codel_clean
  - c_l_ia_codel_att
  - c_l_asb_codel_clean
  - c_l_asb_codel_att
  page: 1
  contradicted_by:
  - r_l_ad_codel_asr
coverage:
  R1: direct
  R2clean: direct
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: direct
  where:
    R1: Table 1 (ASR)
    R2clean: Table 1 (BU)
    R3safe: Table 1 (UA, defined as completing the task while not executing the injection)
schema_gaps:
- where: results r_*_ua
  note: the paper's UA counts a task as done only when the injection is not executed (p. 7), so it is coded R3safe rather than R2attack

`pictionary_arxiv2026` (2609.36121)

In [ ]:
%%writefile reports/records/pictionary_arxiv2026.yaml
report:
  id: pictionary_arxiv2026
  slug: '2609.36121'
  title: 'Render Before Reading: Visual Rendering as a Prompt Injection Defense'
  first_author: Jie Zhang
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2609.36121'
  version_read: arXiv v1
  first_date: '2026-09-28'
  read: F
  role: proposes
  code_release: https://github.com/zj-jayzhang/Pictionary (linked from the abstract)
  extraction_scope: 'Sections 4 to 6: Table 1 (ten models, DirectInject and AgentDojo), Table 2 (union of adaptive attacks), Table 4 (SWE-bench and tau2-Bench), Table 5 (audio rendering) and the No defense and Ours rows of
    Table 3; the abstract claim (PromptLocate and DataFilter, which rewrite the data, and DataSentinel, a detector, are left out; Table 6 and the appendices not coded)'
defences:
- id: pic_opus
  name: Pictionary (untrusted content rendered as a typographic image)
  origin: this_report
  base_checkpoint: Claude Opus 4.7
  axis1:
  - encodings
  axis2:
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  components:
  - every tool return (or the user-supplied document) rendered as black text on a white image and sent in place of the text
  - user prompt, system prompt and tool schemas left as text
  layout: image attachment for documents; image content in tool messages
  layout_compat: &id001
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Pictionary converts untrusted text into a typographic image
- id: pic_haiku
  name: Pictionary (untrusted content rendered as a typographic image)
  origin: this_report
  base_checkpoint: Claude Haiku 4.5
  axis1:
  - encodings
  axis2:
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  components:
  - every tool return (or the user-supplied document) rendered as black text on a white image and sent in place of the text
  - user prompt, system prompt and tool schemas left as text
  layout: image attachment for documents; image content in tool messages
  layout_compat: *id001
  anchor: Pictionary converts untrusted text into a typographic image
- id: pic_g55
  name: Pictionary (untrusted content rendered as a typographic image)
  origin: this_report
  base_checkpoint: GPT-5.5
  axis1:
  - encodings
  axis2:
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  components:
  - every tool return (or the user-supplied document) rendered as black text on a white image and sent in place of the text
  - user prompt, system prompt and tool schemas left as text
  layout: image attachment for documents; image content in tool messages
  layout_compat: *id001
  anchor: Pictionary converts untrusted text into a typographic image
- id: pic_nano
  name: Pictionary (untrusted content rendered as a typographic image)
  origin: this_report
  base_checkpoint: GPT-5.4 nano
  axis1:
  - encodings
  axis2:
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  components:
  - every tool return (or the user-supplied document) rendered as black text on a white image and sent in place of the text
  - user prompt, system prompt and tool schemas left as text
  layout: image attachment for documents; image content in tool messages
  layout_compat: *id001
  anchor: Pictionary converts untrusted text into a typographic image
- id: pic_mini
  name: Pictionary (untrusted content rendered as a typographic image)
  origin: this_report
  base_checkpoint: GPT-5.4 mini
  axis1:
  - encodings
  axis2:
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  components:
  - every tool return (or the user-supplied document) rendered as black text on a white image and sent in place of the text
  - user prompt, system prompt and tool schemas left as text
  layout: image attachment for documents; image content in tool messages
  layout_compat: *id001
  anchor: Pictionary converts untrusted text into a typographic image
- id: pic_qwen
  name: Pictionary (untrusted content rendered as a typographic image)
  origin: this_report
  base_checkpoint: Qwen3.6-plus
  axis1:
  - encodings
  axis2:
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  components:
  - every tool return (or the user-supplied document) rendered as black text on a white image and sent in place of the text
  - user prompt, system prompt and tool schemas left as text
  layout: image attachment for documents; image content in tool messages
  layout_compat: *id001
  anchor: Pictionary converts untrusted text into a typographic image
- id: pic_kimi
  name: Pictionary (untrusted content rendered as a typographic image)
  origin: this_report
  base_checkpoint: Kimi-K2.6
  axis1:
  - encodings
  axis2:
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  components:
  - every tool return (or the user-supplied document) rendered as black text on a white image and sent in place of the text
  - user prompt, system prompt and tool schemas left as text
  layout: image attachment for documents; image content in tool messages
  layout_compat: *id001
  anchor: Pictionary converts untrusted text into a typographic image
- id: pic_grok
  name: Pictionary (untrusted content rendered as a typographic image)
  origin: this_report
  base_checkpoint: Grok 4.3
  axis1:
  - encodings
  axis2:
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  components:
  - every tool return (or the user-supplied document) rendered as black text on a white image and sent in place of the text
  - user prompt, system prompt and tool schemas left as text
  layout: image attachment for documents; image content in tool messages
  layout_compat: *id001
  anchor: Pictionary converts untrusted text into a typographic image
- id: pic_gpro
  name: Pictionary (untrusted content rendered as a typographic image)
  origin: this_report
  base_checkpoint: Gemini 3.1 Pro
  axis1:
  - encodings
  axis2:
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  components:
  - every tool return (or the user-supplied document) rendered as black text on a white image and sent in place of the text
  - user prompt, system prompt and tool schemas left as text
  layout: image attachment for documents; image content in tool messages
  layout_compat: *id001
  anchor: Pictionary converts untrusted text into a typographic image
- id: pic_glite
  name: Pictionary (untrusted content rendered as a typographic image)
  origin: this_report
  base_checkpoint: Gemini 3.1 Flash Lite
  axis1:
  - encodings
  axis2:
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  components:
  - every tool return (or the user-supplied document) rendered as black text on a white image and sent in place of the text
  - user prompt, system prompt and tool schemas left as text
  layout: image attachment for documents; image content in tool messages
  layout_compat: *id001
  anchor: Pictionary converts untrusted text into a typographic image
- id: pic_g54
  name: Pictionary (untrusted content rendered as a typographic image)
  origin: this_report
  base_checkpoint: GPT-5.4
  axis1:
  - encodings
  axis2:
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  components:
  - every tool return (or the user-supplied document) rendered as black text on a white image and sent in place of the text
  - user prompt, system prompt and tool schemas left as text
  layout: image attachment for documents; image content in tool messages
  layout_compat: *id001
  anchor: Pictionary converts untrusted text into a typographic image
- id: aud_gaud
  name: untrusted content synthesised as speech (audio rendering)
  origin: this_report
  base_checkpoint: GPT-Audio Mini
  axis1:
  - encodings
  axis2:
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 8
  components:
  - untrusted text synthesised with GPT-4o-mini-tts and supplied as audio, no transcript
  anchor: synthesized as speech and supplied as a waveform
- id: aud_qomni
  name: untrusted content synthesised as speech (audio rendering)
  origin: this_report
  base_checkpoint: Qwen3.5 Omni-Plus
  axis1:
  - encodings
  axis2:
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 8
  components:
  - untrusted text synthesised with GPT-4o-mini-tts and supplied as audio, no transcript
  anchor: synthesized as speech and supplied as a waveform
evaluations:
- id: ev_di_opus_txt
  defence_id: none:Claude Opus 4.7
  benchmark: DirectInject, 56 task-goal pairs, seven fixed templates
  attacker_type: static
  attacker_access: black_box
  judge: executable call to the target function; GPT-5.4 mini judges the answers
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 6
  decoding: not reported
  sample_size: '56'
  pages:
    benchmark: 5
    judge: 5
  baseline_kind: off_the_shelf
- id: ev_di_opus_img
  defence_id: pic_opus
  benchmark: DirectInject, 56 task-goal pairs, seven fixed templates
  attacker_type: static
  attacker_access: black_box
  judge: executable call to the target function; GPT-5.4 mini judges the answers
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 6
  decoding: not reported
  sample_size: '56'
  pages:
    benchmark: 5
    judge: 5
- id: ev_adj_opus_txt
  defence_id: none:Claude Opus 4.7
  benchmark: AgentDojo, seven fixed templates, strict attack criterion
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checkers (strict)
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 5
    judge: 5
  baseline_kind: off_the_shelf
- id: ev_adj_opus_img
  defence_id: pic_opus
  benchmark: AgentDojo, seven fixed templates, strict attack criterion
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checkers (strict)
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 5
    judge: 5
- id: ev_di_haiku_txt
  defence_id: none:Claude Haiku 4.5
  benchmark: DirectInject, 56 task-goal pairs, seven fixed templates
  attacker_type: static
  attacker_access: black_box
  judge: executable call to the target function; GPT-5.4 mini judges the answers
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 6
  decoding: not reported
  sample_size: '56'
  pages:
    benchmark: 5
    judge: 5
  baseline_kind: off_the_shelf
- id: ev_di_haiku_img
  defence_id: pic_haiku
  benchmark: DirectInject, 56 task-goal pairs, seven fixed templates
  attacker_type: static
  attacker_access: black_box
  judge: executable call to the target function; GPT-5.4 mini judges the answers
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 6
  decoding: not reported
  sample_size: '56'
  pages:
    benchmark: 5
    judge: 5
- id: ev_adj_haiku_txt
  defence_id: none:Claude Haiku 4.5
  benchmark: AgentDojo, seven fixed templates, strict attack criterion
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checkers (strict)
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 5
    judge: 5
  baseline_kind: off_the_shelf
- id: ev_adj_haiku_img
  defence_id: pic_haiku
  benchmark: AgentDojo, seven fixed templates, strict attack criterion
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checkers (strict)
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 5
    judge: 5
- id: ev_di_g55_txt
  defence_id: none:GPT-5.5
  benchmark: DirectInject, 56 task-goal pairs, seven fixed templates
  attacker_type: static
  attacker_access: black_box
  judge: executable call to the target function; GPT-5.4 mini judges the answers
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 6
  decoding: not reported
  sample_size: '56'
  pages:
    benchmark: 5
    judge: 5
  baseline_kind: off_the_shelf
- id: ev_di_g55_img
  defence_id: pic_g55
  benchmark: DirectInject, 56 task-goal pairs, seven fixed templates
  attacker_type: static
  attacker_access: black_box
  judge: executable call to the target function; GPT-5.4 mini judges the answers
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 6
  decoding: not reported
  sample_size: '56'
  pages:
    benchmark: 5
    judge: 5
- id: ev_adj_g55_txt
  defence_id: none:GPT-5.5
  benchmark: AgentDojo, seven fixed templates, strict attack criterion
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checkers (strict)
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 5
    judge: 5
  baseline_kind: off_the_shelf
- id: ev_adj_g55_img
  defence_id: pic_g55
  benchmark: AgentDojo, seven fixed templates, strict attack criterion
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checkers (strict)
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 5
    judge: 5
- id: ev_di_nano_txt
  defence_id: none:GPT-5.4 nano
  benchmark: DirectInject, 56 task-goal pairs, seven fixed templates
  attacker_type: static
  attacker_access: black_box
  judge: executable call to the target function; GPT-5.4 mini judges the answers
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 6
  decoding: not reported
  sample_size: '56'
  pages:
    benchmark: 5
    judge: 5
  baseline_kind: off_the_shelf
- id: ev_di_nano_img
  defence_id: pic_nano
  benchmark: DirectInject, 56 task-goal pairs, seven fixed templates
  attacker_type: static
  attacker_access: black_box
  judge: executable call to the target function; GPT-5.4 mini judges the answers
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 6
  decoding: not reported
  sample_size: '56'
  pages:
    benchmark: 5
    judge: 5
- id: ev_adj_nano_txt
  defence_id: none:GPT-5.4 nano
  benchmark: AgentDojo, seven fixed templates, strict attack criterion
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checkers (strict)
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 5
    judge: 5
  baseline_kind: off_the_shelf
- id: ev_adj_nano_img
  defence_id: pic_nano
  benchmark: AgentDojo, seven fixed templates, strict attack criterion
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checkers (strict)
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 5
    judge: 5
- id: ev_di_mini_txt
  defence_id: none:GPT-5.4 mini
  benchmark: DirectInject, 56 task-goal pairs, seven fixed templates
  attacker_type: static
  attacker_access: black_box
  judge: executable call to the target function; GPT-5.4 mini judges the answers
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 6
  decoding: not reported
  sample_size: '56'
  pages:
    benchmark: 5
    judge: 5
  baseline_kind: off_the_shelf
- id: ev_di_mini_img
  defence_id: pic_mini
  benchmark: DirectInject, 56 task-goal pairs, seven fixed templates
  attacker_type: static
  attacker_access: black_box
  judge: executable call to the target function; GPT-5.4 mini judges the answers
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 6
  decoding: not reported
  sample_size: '56'
  pages:
    benchmark: 5
    judge: 5
- id: ev_adj_mini_txt
  defence_id: none:GPT-5.4 mini
  benchmark: AgentDojo, seven fixed templates, strict attack criterion
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checkers (strict)
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 5
    judge: 5
  baseline_kind: off_the_shelf
- id: ev_adj_mini_img
  defence_id: pic_mini
  benchmark: AgentDojo, seven fixed templates, strict attack criterion
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checkers (strict)
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 5
    judge: 5
- id: ev_di_qwen_txt
  defence_id: none:Qwen3.6-plus
  benchmark: DirectInject, 56 task-goal pairs, seven fixed templates
  attacker_type: static
  attacker_access: black_box
  judge: executable call to the target function; GPT-5.4 mini judges the answers
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 6
  decoding: not reported
  sample_size: '56'
  pages:
    benchmark: 5
    judge: 5
  baseline_kind: off_the_shelf
- id: ev_di_qwen_img
  defence_id: pic_qwen
  benchmark: DirectInject, 56 task-goal pairs, seven fixed templates
  attacker_type: static
  attacker_access: black_box
  judge: executable call to the target function; GPT-5.4 mini judges the answers
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 6
  decoding: not reported
  sample_size: '56'
  pages:
    benchmark: 5
    judge: 5
- id: ev_adj_qwen_txt
  defence_id: none:Qwen3.6-plus
  benchmark: AgentDojo, seven fixed templates, strict attack criterion
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checkers (strict)
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 5
    judge: 5
  baseline_kind: off_the_shelf
- id: ev_adj_qwen_img
  defence_id: pic_qwen
  benchmark: AgentDojo, seven fixed templates, strict attack criterion
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checkers (strict)
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 5
    judge: 5
- id: ev_di_kimi_txt
  defence_id: none:Kimi-K2.6
  benchmark: DirectInject, 56 task-goal pairs, seven fixed templates
  attacker_type: static
  attacker_access: black_box
  judge: executable call to the target function; GPT-5.4 mini judges the answers
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 6
  decoding: not reported
  sample_size: '56'
  pages:
    benchmark: 5
    judge: 5
  baseline_kind: off_the_shelf
- id: ev_di_kimi_img
  defence_id: pic_kimi
  benchmark: DirectInject, 56 task-goal pairs, seven fixed templates
  attacker_type: static
  attacker_access: black_box
  judge: executable call to the target function; GPT-5.4 mini judges the answers
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 6
  decoding: not reported
  sample_size: '56'
  pages:
    benchmark: 5
    judge: 5
- id: ev_adj_kimi_txt
  defence_id: none:Kimi-K2.6
  benchmark: AgentDojo, seven fixed templates, strict attack criterion
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checkers (strict)
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 5
    judge: 5
  baseline_kind: off_the_shelf
- id: ev_adj_kimi_img
  defence_id: pic_kimi
  benchmark: AgentDojo, seven fixed templates, strict attack criterion
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checkers (strict)
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 5
    judge: 5
- id: ev_di_grok_txt
  defence_id: none:Grok 4.3
  benchmark: DirectInject, 56 task-goal pairs, seven fixed templates
  attacker_type: static
  attacker_access: black_box
  judge: executable call to the target function; GPT-5.4 mini judges the answers
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 6
  decoding: not reported
  sample_size: '56'
  pages:
    benchmark: 5
    judge: 5
  baseline_kind: off_the_shelf
- id: ev_di_grok_img
  defence_id: pic_grok
  benchmark: DirectInject, 56 task-goal pairs, seven fixed templates
  attacker_type: static
  attacker_access: black_box
  judge: executable call to the target function; GPT-5.4 mini judges the answers
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 6
  decoding: not reported
  sample_size: '56'
  pages:
    benchmark: 5
    judge: 5
- id: ev_adj_grok_txt
  defence_id: none:Grok 4.3
  benchmark: AgentDojo, seven fixed templates, strict attack criterion
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checkers (strict)
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 5
    judge: 5
  baseline_kind: off_the_shelf
- id: ev_adj_grok_img
  defence_id: pic_grok
  benchmark: AgentDojo, seven fixed templates, strict attack criterion
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checkers (strict)
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 5
    judge: 5
- id: ev_di_gpro_txt
  defence_id: none:Gemini 3.1 Pro
  benchmark: DirectInject, 56 task-goal pairs, seven fixed templates
  attacker_type: static
  attacker_access: black_box
  judge: executable call to the target function; GPT-5.4 mini judges the answers
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 6
  decoding: not reported
  sample_size: '56'
  pages:
    benchmark: 5
    judge: 5
  baseline_kind: off_the_shelf
- id: ev_di_gpro_img
  defence_id: pic_gpro
  benchmark: DirectInject, 56 task-goal pairs, seven fixed templates
  attacker_type: static
  attacker_access: black_box
  judge: executable call to the target function; GPT-5.4 mini judges the answers
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 6
  decoding: not reported
  sample_size: '56'
  pages:
    benchmark: 5
    judge: 5
- id: ev_adj_gpro_txt
  defence_id: none:Gemini 3.1 Pro
  benchmark: AgentDojo, seven fixed templates, strict attack criterion
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checkers (strict)
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 5
    judge: 5
  baseline_kind: off_the_shelf
- id: ev_adj_gpro_img
  defence_id: pic_gpro
  benchmark: AgentDojo, seven fixed templates, strict attack criterion
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checkers (strict)
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 5
    judge: 5
- id: ev_di_glite_txt
  defence_id: none:Gemini 3.1 Flash Lite
  benchmark: DirectInject, 56 task-goal pairs, seven fixed templates
  attacker_type: static
  attacker_access: black_box
  judge: executable call to the target function; GPT-5.4 mini judges the answers
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 6
  decoding: not reported
  sample_size: '56'
  pages:
    benchmark: 5
    judge: 5
  baseline_kind: off_the_shelf
- id: ev_di_glite_img
  defence_id: pic_glite
  benchmark: DirectInject, 56 task-goal pairs, seven fixed templates
  attacker_type: static
  attacker_access: black_box
  judge: executable call to the target function; GPT-5.4 mini judges the answers
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 6
  decoding: not reported
  sample_size: '56'
  pages:
    benchmark: 5
    judge: 5
- id: ev_adj_glite_txt
  defence_id: none:Gemini 3.1 Flash Lite
  benchmark: AgentDojo, seven fixed templates, strict attack criterion
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checkers (strict)
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 5
    judge: 5
  baseline_kind: off_the_shelf
- id: ev_adj_glite_img
  defence_id: pic_glite
  benchmark: AgentDojo, seven fixed templates, strict attack criterion
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checkers (strict)
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 6
  decoding: not reported
  pages:
    benchmark: 5
    judge: 5
- id: ev_union_opus_txt
  defence_id: none:Claude Opus 4.7
  benchmark: AgentDojo banking, union of RL suffix optimisation, agentic auto red teaming and human red teaming payloads replayed on both channels
  attacker_type: adaptive
  attacker_access: black_box
  judge: AgentDojo checkers (strict); any payload succeeds
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 7
  attacker_budget: pooled payloads from three methods, including 144 expert hours
  decoding: not reported
  pages:
    attacker_budget: 6
  baseline_kind: off_the_shelf
- id: ev_union_opus_img
  defence_id: pic_opus
  benchmark: AgentDojo banking, union of RL suffix optimisation, agentic auto red teaming and human red teaming payloads replayed on both channels
  attacker_type: adaptive
  attacker_access: black_box
  judge: AgentDojo checkers (strict); any payload succeeds
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 7
  attacker_budget: pooled payloads from three methods, including 144 expert hours
  decoding: not reported
  pages:
    attacker_budget: 6
- id: ev_union_haiku_txt
  defence_id: none:Claude Haiku 4.5
  benchmark: AgentDojo banking, union of RL suffix optimisation, agentic auto red teaming and human red teaming payloads replayed on both channels
  attacker_type: adaptive
  attacker_access: black_box
  judge: AgentDojo checkers (strict); any payload succeeds
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 7
  attacker_budget: pooled payloads from three methods, including 144 expert hours
  decoding: not reported
  pages:
    attacker_budget: 6
  baseline_kind: off_the_shelf
- id: ev_union_haiku_img
  defence_id: pic_haiku
  benchmark: AgentDojo banking, union of RL suffix optimisation, agentic auto red teaming and human red teaming payloads replayed on both channels
  attacker_type: adaptive
  attacker_access: black_box
  judge: AgentDojo checkers (strict); any payload succeeds
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 7
  attacker_budget: pooled payloads from three methods, including 144 expert hours
  decoding: not reported
  pages:
    attacker_budget: 6
- id: ev_union_g55_txt
  defence_id: none:GPT-5.5
  benchmark: AgentDojo banking, union of RL suffix optimisation, agentic auto red teaming and human red teaming payloads replayed on both channels
  attacker_type: adaptive
  attacker_access: black_box
  judge: AgentDojo checkers (strict); any payload succeeds
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 7
  attacker_budget: pooled payloads from three methods, including 144 expert hours
  decoding: not reported
  pages:
    attacker_budget: 6
  baseline_kind: off_the_shelf
- id: ev_union_g55_img
  defence_id: pic_g55
  benchmark: AgentDojo banking, union of RL suffix optimisation, agentic auto red teaming and human red teaming payloads replayed on both channels
  attacker_type: adaptive
  attacker_access: black_box
  judge: AgentDojo checkers (strict); any payload succeeds
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 7
  attacker_budget: pooled payloads from three methods, including 144 expert hours
  decoding: not reported
  pages:
    attacker_budget: 6
- id: ev_union_mini_txt
  defence_id: none:GPT-5.4 mini
  benchmark: AgentDojo banking, union of RL suffix optimisation, agentic auto red teaming and human red teaming payloads replayed on both channels
  attacker_type: adaptive
  attacker_access: black_box
  judge: AgentDojo checkers (strict); any payload succeeds
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 7
  attacker_budget: pooled payloads from three methods, including 144 expert hours
  decoding: not reported
  pages:
    attacker_budget: 6
  baseline_kind: off_the_shelf
- id: ev_union_mini_img
  defence_id: pic_mini
  benchmark: AgentDojo banking, union of RL suffix optimisation, agentic auto red teaming and human red teaming payloads replayed on both channels
  attacker_type: adaptive
  attacker_access: black_box
  judge: AgentDojo checkers (strict); any payload succeeds
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 7
  attacker_budget: pooled payloads from three methods, including 144 expert hours
  decoding: not reported
  pages:
    attacker_budget: 6
- id: ev_union_kimi_txt
  defence_id: none:Kimi-K2.6
  benchmark: AgentDojo banking, union of RL suffix optimisation, agentic auto red teaming and human red teaming payloads replayed on both channels
  attacker_type: adaptive
  attacker_access: black_box
  judge: AgentDojo checkers (strict); any payload succeeds
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 7
  attacker_budget: pooled payloads from three methods, including 144 expert hours
  decoding: not reported
  pages:
    attacker_budget: 6
  baseline_kind: off_the_shelf
- id: ev_union_kimi_img
  defence_id: pic_kimi
  benchmark: AgentDojo banking, union of RL suffix optimisation, agentic auto red teaming and human red teaming payloads replayed on both channels
  attacker_type: adaptive
  attacker_access: black_box
  judge: AgentDojo checkers (strict); any payload succeeds
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 7
  attacker_budget: pooled payloads from three methods, including 144 expert hours
  decoding: not reported
  pages:
    attacker_budget: 6
- id: ev_union_glite_txt
  defence_id: none:Gemini 3.1 Flash Lite
  benchmark: AgentDojo banking, union of RL suffix optimisation, agentic auto red teaming and human red teaming payloads replayed on both channels
  attacker_type: adaptive
  attacker_access: black_box
  judge: AgentDojo checkers (strict); any payload succeeds
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 7
  attacker_budget: pooled payloads from three methods, including 144 expert hours
  decoding: not reported
  pages:
    attacker_budget: 6
  baseline_kind: off_the_shelf
- id: ev_union_glite_img
  defence_id: pic_glite
  benchmark: AgentDojo banking, union of RL suffix optimisation, agentic auto red teaming and human red teaming payloads replayed on both channels
  attacker_type: adaptive
  attacker_access: black_box
  judge: AgentDojo checkers (strict); any payload succeeds
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 7
  attacker_budget: pooled payloads from three methods, including 144 expert hours
  decoding: not reported
  pages:
    attacker_budget: 6
- id: ev_art_mini_txt
  defence_id: none:GPT-5.4 mini
  benchmark: DirectInject, union of three ART-discovered templates
  attacker_type: static
  attacker_access: black_box
  judge: executable call to the target function; GPT-5.4 mini judges the answers
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 7
  decoding: not reported
  run: Table 3 comparison run
  baseline_kind: off_the_shelf
- id: ev_art_mini_img
  defence_id: pic_mini
  benchmark: DirectInject, union of three ART-discovered templates
  attacker_type: static
  attacker_access: black_box
  judge: executable call to the target function; GPT-5.4 mini judges the answers
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 7
  decoding: not reported
  run: Table 3 comparison run
- id: ev_art_g54_txt
  defence_id: none:GPT-5.4
  benchmark: DirectInject, union of three ART-discovered templates
  attacker_type: static
  attacker_access: black_box
  judge: executable call to the target function; GPT-5.4 mini judges the answers
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 7
  decoding: not reported
  run: Table 3 comparison run
  baseline_kind: off_the_shelf
- id: ev_art_g54_img
  defence_id: pic_g54
  benchmark: DirectInject, union of three ART-discovered templates
  attacker_type: static
  attacker_access: black_box
  judge: executable call to the target function; GPT-5.4 mini judges the answers
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 7
  decoding: not reported
  run: Table 3 comparison run
- id: ev_art_kimi_txt
  defence_id: none:Kimi-K2.6
  benchmark: DirectInject, union of three ART-discovered templates
  attacker_type: static
  attacker_access: black_box
  judge: executable call to the target function; GPT-5.4 mini judges the answers
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 7
  decoding: not reported
  run: Table 3 comparison run
  baseline_kind: off_the_shelf
- id: ev_art_kimi_img
  defence_id: pic_kimi
  benchmark: DirectInject, union of three ART-discovered templates
  attacker_type: static
  attacker_access: black_box
  judge: executable call to the target function; GPT-5.4 mini judges the answers
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 7
  decoding: not reported
  run: Table 3 comparison run
- id: ev_swe_kimi_txt
  defence_id: none:Kimi-K2.6
  benchmark: SWE-bench Verified, 100 instances
  attacker_type: none
  attacker_access: not_applicable
  judge: repository tests
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 8
  decoding: not reported
  sample_size: '100'
  baseline_kind: off_the_shelf
- id: ev_swe_kimi_img
  defence_id: pic_kimi
  benchmark: SWE-bench Verified, 100 instances
  attacker_type: none
  attacker_access: not_applicable
  judge: repository tests
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 8
  decoding: not reported
  sample_size: '100'
- id: ev_tau_kimi_txt
  defence_id: none:Kimi-K2.6
  benchmark: tau2-Bench telecom and airline, 164 tasks
  attacker_type: none
  attacker_access: not_applicable
  judge: tau2-Bench checker
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 8
  decoding: not reported
  sample_size: '164'
  baseline_kind: off_the_shelf
- id: ev_tau_kimi_img
  defence_id: pic_kimi
  benchmark: tau2-Bench telecom and airline, 164 tasks
  attacker_type: none
  attacker_access: not_applicable
  judge: tau2-Bench checker
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 8
  decoding: not reported
  sample_size: '164'
- id: ev_swe_mini_txt
  defence_id: none:GPT-5.4 mini
  benchmark: SWE-bench Verified, 100 instances
  attacker_type: none
  attacker_access: not_applicable
  judge: repository tests
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 8
  decoding: not reported
  sample_size: '100'
  baseline_kind: off_the_shelf
- id: ev_swe_mini_img
  defence_id: pic_mini
  benchmark: SWE-bench Verified, 100 instances
  attacker_type: none
  attacker_access: not_applicable
  judge: repository tests
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 8
  decoding: not reported
  sample_size: '100'
- id: ev_tau_mini_txt
  defence_id: none:GPT-5.4 mini
  benchmark: tau2-Bench telecom and airline, 164 tasks
  attacker_type: none
  attacker_access: not_applicable
  judge: tau2-Bench checker
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 8
  decoding: not reported
  sample_size: '164'
  baseline_kind: off_the_shelf
- id: ev_tau_mini_img
  defence_id: pic_mini
  benchmark: tau2-Bench telecom and airline, 164 tasks
  attacker_type: none
  attacker_access: not_applicable
  judge: tau2-Bench checker
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 8
  decoding: not reported
  sample_size: '164'
- id: ev_swe_haiku_txt
  defence_id: none:Claude Haiku 4.5
  benchmark: SWE-bench Verified, 100 instances
  attacker_type: none
  attacker_access: not_applicable
  judge: repository tests
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 8
  decoding: not reported
  sample_size: '100'
  baseline_kind: off_the_shelf
- id: ev_swe_haiku_img
  defence_id: pic_haiku
  benchmark: SWE-bench Verified, 100 instances
  attacker_type: none
  attacker_access: not_applicable
  judge: repository tests
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 8
  decoding: not reported
  sample_size: '100'
- id: ev_tau_haiku_txt
  defence_id: none:Claude Haiku 4.5
  benchmark: tau2-Bench telecom and airline, 164 tasks
  attacker_type: none
  attacker_access: not_applicable
  judge: tau2-Bench checker
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 8
  decoding: not reported
  sample_size: '164'
  baseline_kind: off_the_shelf
- id: ev_tau_haiku_img
  defence_id: pic_haiku
  benchmark: tau2-Bench telecom and airline, 164 tasks
  attacker_type: none
  attacker_access: not_applicable
  judge: tau2-Bench checker
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 8
  decoding: not reported
  sample_size: '164'
- id: ev_audclean_gaud_txt
  defence_id: none:GPT-Audio Mini
  benchmark: DirectInject, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-5.4 mini judges the answers
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 8
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_audatt_gaud_txt
  defence_id: none:GPT-Audio Mini
  benchmark: DirectInject, seven fixed templates
  attacker_type: static
  attacker_access: black_box
  judge: executable call to the target function; GPT-5.4 mini judges the answers
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 8
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_audclean_gaud_aud
  defence_id: aud_gaud
  benchmark: DirectInject, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-5.4 mini judges the answers
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 8
  decoding: not reported
- id: ev_audatt_gaud_aud
  defence_id: aud_gaud
  benchmark: DirectInject, seven fixed templates
  attacker_type: static
  attacker_access: black_box
  judge: executable call to the target function; GPT-5.4 mini judges the answers
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 8
  decoding: not reported
- id: ev_audclean_qomni_txt
  defence_id: none:Qwen3.5 Omni-Plus
  benchmark: DirectInject, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-5.4 mini judges the answers
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 8
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_audatt_qomni_txt
  defence_id: none:Qwen3.5 Omni-Plus
  benchmark: DirectInject, seven fixed templates
  attacker_type: static
  attacker_access: black_box
  judge: executable call to the target function; GPT-5.4 mini judges the answers
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 8
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_audclean_qomni_aud
  defence_id: aud_qomni
  benchmark: DirectInject, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-5.4 mini judges the answers
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 8
  decoding: not reported
- id: ev_audatt_qomni_aud
  defence_id: aud_qomni
  benchmark: DirectInject, seven fixed templates
  attacker_type: static
  attacker_access: black_box
  judge: executable call to the target function; GPT-5.4 mini judges the answers
  layout_same_as_defence: 'yes'
  measured_by: pictionary_arxiv2026
  page: 8
  decoding: not reported
results:
- id: r_di_opus_txt_asr
  evaluation_id: ev_di_opus_txt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%)
  value: '5.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_di_opus_txt_u
  evaluation_id: ev_di_opus_txt
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (%) under attack
  value: '99.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_di_opus_img_asr
  evaluation_id: ev_di_opus_img
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_di_opus_img_u
  evaluation_id: ev_di_opus_img
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (%) under attack
  value: '100'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_adj_opus_txt_asr
  evaluation_id: ev_adj_opus_txt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_adj_opus_txt_u
  evaluation_id: ev_adj_opus_txt
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (%) under attack
  value: '90.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_adj_opus_img_asr
  evaluation_id: ev_adj_opus_img
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_adj_opus_img_u
  evaluation_id: ev_adj_opus_img
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (%) under attack
  value: '90.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_di_haiku_txt_asr
  evaluation_id: ev_di_haiku_txt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%)
  value: '17.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_di_haiku_txt_u
  evaluation_id: ev_di_haiku_txt
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (%) under attack
  value: '99.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_di_haiku_img_asr
  evaluation_id: ev_di_haiku_img
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_di_haiku_img_u
  evaluation_id: ev_di_haiku_img
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (%) under attack
  value: '98.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_adj_haiku_txt_asr
  evaluation_id: ev_adj_haiku_txt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%)
  value: '1.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_adj_haiku_txt_u
  evaluation_id: ev_adj_haiku_txt
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (%) under attack
  value: '70.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_adj_haiku_img_asr
  evaluation_id: ev_adj_haiku_img
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%)
  value: '0.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_adj_haiku_img_u
  evaluation_id: ev_adj_haiku_img
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (%) under attack
  value: '62.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_di_g55_txt_asr
  evaluation_id: ev_di_g55_txt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_di_g55_txt_u
  evaluation_id: ev_di_g55_txt
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (%) under attack
  value: '100.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_di_g55_img_asr
  evaluation_id: ev_di_g55_img
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_di_g55_img_u
  evaluation_id: ev_di_g55_img
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (%) under attack
  value: '100.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_adj_g55_txt_asr
  evaluation_id: ev_adj_g55_txt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%)
  value: '0.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_adj_g55_txt_u
  evaluation_id: ev_adj_g55_txt
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (%) under attack
  value: '92.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_adj_g55_img_asr
  evaluation_id: ev_adj_g55_img
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_adj_g55_img_u
  evaluation_id: ev_adj_g55_img
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (%) under attack
  value: '92.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_di_nano_txt_asr
  evaluation_id: ev_di_nano_txt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%)
  value: '14.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_di_nano_txt_u
  evaluation_id: ev_di_nano_txt
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (%) under attack
  value: '100.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_di_nano_img_asr
  evaluation_id: ev_di_nano_img
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_di_nano_img_u
  evaluation_id: ev_di_nano_img
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (%) under attack
  value: '98.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_adj_nano_txt_asr
  evaluation_id: ev_adj_nano_txt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%)
  value: '14.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_adj_nano_txt_u
  evaluation_id: ev_adj_nano_txt
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (%) under attack
  value: '50.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_adj_nano_img_asr
  evaluation_id: ev_adj_nano_img
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_adj_nano_img_u
  evaluation_id: ev_adj_nano_img
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (%) under attack
  value: '43.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_di_mini_txt_asr
  evaluation_id: ev_di_mini_txt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%)
  value: '75.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_di_mini_txt_u
  evaluation_id: ev_di_mini_txt
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (%) under attack
  value: '99.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_di_mini_img_asr
  evaluation_id: ev_di_mini_img
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%)
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_di_mini_img_u
  evaluation_id: ev_di_mini_img
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (%) under attack
  value: '100.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_adj_mini_txt_asr
  evaluation_id: ev_adj_mini_txt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%)
  value: '29.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_adj_mini_txt_u
  evaluation_id: ev_adj_mini_txt
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (%) under attack
  value: '71.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_adj_mini_img_asr
  evaluation_id: ev_adj_mini_img
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%)
  value: '7.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_adj_mini_img_u
  evaluation_id: ev_adj_mini_img
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (%) under attack
  value: '64.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_di_qwen_txt_asr
  evaluation_id: ev_di_qwen_txt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%)
  value: '96.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_di_qwen_txt_u
  evaluation_id: ev_di_qwen_txt
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (%) under attack
  value: '98.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_di_qwen_img_asr
  evaluation_id: ev_di_qwen_img
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%)
  value: '58.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_di_qwen_img_u
  evaluation_id: ev_di_qwen_img
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (%) under attack
  value: '99.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_adj_qwen_txt_asr
  evaluation_id: ev_adj_qwen_txt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%)
  value: '60.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_adj_qwen_txt_u
  evaluation_id: ev_adj_qwen_txt
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (%) under attack
  value: '88.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_adj_qwen_img_asr
  evaluation_id: ev_adj_qwen_img
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%)
  value: '22.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_adj_qwen_img_u
  evaluation_id: ev_adj_qwen_img
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (%) under attack
  value: '90.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_di_kimi_txt_asr
  evaluation_id: ev_di_kimi_txt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%)
  value: '80.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_di_kimi_txt_u
  evaluation_id: ev_di_kimi_txt
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (%) under attack
  value: '76.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_di_kimi_img_asr
  evaluation_id: ev_di_kimi_img
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%)
  value: '7.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_di_kimi_img_u
  evaluation_id: ev_di_kimi_img
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (%) under attack
  value: '83.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_adj_kimi_txt_asr
  evaluation_id: ev_adj_kimi_txt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%)
  value: '50.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_adj_kimi_txt_u
  evaluation_id: ev_adj_kimi_txt
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (%) under attack
  value: '84.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_adj_kimi_img_asr
  evaluation_id: ev_adj_kimi_img
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%)
  value: '12.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_adj_kimi_img_u
  evaluation_id: ev_adj_kimi_img
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (%) under attack
  value: '84.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_di_grok_txt_asr
  evaluation_id: ev_di_grok_txt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%)
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_di_grok_txt_u
  evaluation_id: ev_di_grok_txt
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (%) under attack
  value: '97.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_di_grok_img_asr
  evaluation_id: ev_di_grok_img
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%)
  value: '64.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_di_grok_img_u
  evaluation_id: ev_di_grok_img
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (%) under attack
  value: '99.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_adj_grok_txt_asr
  evaluation_id: ev_adj_grok_txt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%)
  value: '32.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_adj_grok_txt_u
  evaluation_id: ev_adj_grok_txt
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (%) under attack
  value: '87.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_adj_grok_img_asr
  evaluation_id: ev_adj_grok_img
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%)
  value: '23.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_adj_grok_img_u
  evaluation_id: ev_adj_grok_img
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (%) under attack
  value: '72.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_di_gpro_txt_asr
  evaluation_id: ev_di_gpro_txt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%)
  value: '98.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_di_gpro_txt_u
  evaluation_id: ev_di_gpro_txt
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (%) under attack
  value: '87.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_di_gpro_img_asr
  evaluation_id: ev_di_gpro_img
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%)
  value: '55.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_di_gpro_img_u
  evaluation_id: ev_di_gpro_img
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (%) under attack
  value: '96.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_adj_gpro_txt_asr
  evaluation_id: ev_adj_gpro_txt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%)
  value: '83.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_adj_gpro_txt_u
  evaluation_id: ev_adj_gpro_txt
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (%) under attack
  value: '77.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_adj_gpro_img_asr
  evaluation_id: ev_adj_gpro_img
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%)
  value: '41.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_adj_gpro_img_u
  evaluation_id: ev_adj_gpro_img
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (%) under attack
  value: '87.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_di_glite_txt_asr
  evaluation_id: ev_di_glite_txt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%)
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_di_glite_txt_u
  evaluation_id: ev_di_glite_txt
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (%) under attack
  value: '100.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_di_glite_img_asr
  evaluation_id: ev_di_glite_img
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%)
  value: '46.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_di_glite_img_u
  evaluation_id: ev_di_glite_img
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (%) under attack
  value: '100.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_adj_glite_txt_asr
  evaluation_id: ev_adj_glite_txt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%)
  value: '90.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_adj_glite_txt_u
  evaluation_id: ev_adj_glite_txt
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (%) under attack
  value: '63.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_adj_glite_img_asr
  evaluation_id: ev_adj_glite_img
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%)
  value: '49.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_adj_glite_img_u
  evaluation_id: ev_adj_glite_img
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (%) under attack
  value: '52.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_union_opus_txt
  evaluation_id: ev_union_opus_txt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%) union
  value: '14.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_union_opus_img
  evaluation_id: ev_union_opus_img
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%) union
  value: '11.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_union_haiku_txt
  evaluation_id: ev_union_haiku_txt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%) union
  value: '97.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_union_haiku_img
  evaluation_id: ev_union_haiku_img
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%) union
  value: '9.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_union_g55_txt
  evaluation_id: ev_union_g55_txt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%) union
  value: '19.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_union_g55_img
  evaluation_id: ev_union_g55_img
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%) union
  value: '16.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_union_mini_txt
  evaluation_id: ev_union_mini_txt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%) union
  value: '83.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_union_mini_img
  evaluation_id: ev_union_mini_img
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%) union
  value: '31.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_union_kimi_txt
  evaluation_id: ev_union_kimi_txt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%) union
  value: '77.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_union_kimi_img
  evaluation_id: ev_union_kimi_img
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%) union
  value: '40.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_union_glite_txt
  evaluation_id: ev_union_glite_txt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%) union
  value: '100.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_union_glite_img
  evaluation_id: ev_union_glite_img
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%) union
  value: '97.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
- id: r_art_mini_txt_asr
  evaluation_id: ev_art_mini_txt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '82.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_art_mini_txt_u
  evaluation_id: ev_art_mini_txt
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '98'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_art_mini_img_asr
  evaluation_id: ev_art_mini_img
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '3.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_art_mini_img_u
  evaluation_id: ev_art_mini_img
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '100'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_art_g54_txt_asr
  evaluation_id: ev_art_g54_txt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '87.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_art_g54_txt_u
  evaluation_id: ev_art_g54_txt
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '99'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_art_g54_img_asr
  evaluation_id: ev_art_g54_img
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '3.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_art_g54_img_u
  evaluation_id: ev_art_g54_img
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '100'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_art_kimi_txt_asr
  evaluation_id: ev_art_kimi_txt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '71.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_art_kimi_txt_u
  evaluation_id: ev_art_kimi_txt
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '96'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_art_kimi_img_asr
  evaluation_id: ev_art_kimi_img
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '3.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_art_kimi_img_u
  evaluation_id: ev_art_kimi_img
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '99'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_swe_kimi_txt
  evaluation_id: ev_swe_kimi_txt
  quantity: R2clean
  coverage: direct
  metric_as_printed: Task success (%)
  value: '65.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_swe_kimi_img
  evaluation_id: ev_swe_kimi_img
  quantity: R2clean
  coverage: direct
  metric_as_printed: Task success (%)
  value: '65.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_tau_kimi_txt
  evaluation_id: ev_tau_kimi_txt
  quantity: R2clean
  coverage: direct
  metric_as_printed: Task success (%)
  value: '87.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_tau_kimi_img
  evaluation_id: ev_tau_kimi_img
  quantity: R2clean
  coverage: direct
  metric_as_printed: Task success (%)
  value: '85.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_swe_mini_txt
  evaluation_id: ev_swe_mini_txt
  quantity: R2clean
  coverage: direct
  metric_as_printed: Task success (%)
  value: '66.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_swe_mini_img
  evaluation_id: ev_swe_mini_img
  quantity: R2clean
  coverage: direct
  metric_as_printed: Task success (%)
  value: '56.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_tau_mini_txt
  evaluation_id: ev_tau_mini_txt
  quantity: R2clean
  coverage: direct
  metric_as_printed: Task success (%)
  value: '44.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_tau_mini_img
  evaluation_id: ev_tau_mini_img
  quantity: R2clean
  coverage: direct
  metric_as_printed: Task success (%)
  value: '43.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_swe_haiku_txt
  evaluation_id: ev_swe_haiku_txt
  quantity: R2clean
  coverage: direct
  metric_as_printed: Task success (%)
  value: '66.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_swe_haiku_img
  evaluation_id: ev_swe_haiku_img
  quantity: R2clean
  coverage: direct
  metric_as_printed: Task success (%)
  value: '57.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_tau_haiku_txt
  evaluation_id: ev_tau_haiku_txt
  quantity: R2clean
  coverage: direct
  metric_as_printed: Task success (%)
  value: '50.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_tau_haiku_img
  evaluation_id: ev_tau_haiku_img
  quantity: R2clean
  coverage: direct
  metric_as_printed: Task success (%)
  value: '48.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 4
- id: r_audclean_gaud_txt
  evaluation_id: ev_audclean_gaud_txt
  quantity: R2clean
  coverage: direct
  metric_as_printed: Utility, no attack
  value: '100.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
  spread: ± 0.0
- id: r_audatt_gaud_txt_asr
  evaluation_id: ev_audatt_gaud_txt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '27.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_audatt_gaud_txt_u
  evaluation_id: ev_audatt_gaud_txt
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility under attack
  value: '100.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_audclean_gaud_aud
  evaluation_id: ev_audclean_gaud_aud
  quantity: R2clean
  coverage: direct
  metric_as_printed: Utility, no attack
  value: '95.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
  spread: ± 22.4
- id: r_audatt_gaud_aud_asr
  evaluation_id: ev_audatt_gaud_aud
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '1.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_audatt_gaud_aud_u
  evaluation_id: ev_audatt_gaud_aud
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility under attack
  value: '91.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_audclean_qomni_txt
  evaluation_id: ev_audclean_qomni_txt
  quantity: R2clean
  coverage: direct
  metric_as_printed: Utility, no attack
  value: '100.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
  spread: ± 0.0
- id: r_audatt_qomni_txt_asr
  evaluation_id: ev_audatt_qomni_txt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '50.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_audatt_qomni_txt_u
  evaluation_id: ev_audatt_qomni_txt
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility under attack
  value: '82.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_audclean_qomni_aud
  evaluation_id: ev_audclean_qomni_aud
  quantity: R2clean
  coverage: direct
  metric_as_printed: Utility, no attack
  value: '100.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
  spread: ± 0.0
- id: r_audatt_qomni_aud_asr
  evaluation_id: ev_audatt_qomni_aud
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '15.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
- id: r_audatt_qomni_aud_u
  evaluation_id: ev_audatt_qomni_aud
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility under attack
  value: '93.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 5
comparisons:
- id: c_di_opus
  left: ev_di_opus_img
  right: ev_di_opus_txt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_adj_opus
  left: ev_adj_opus_img
  right: ev_adj_opus_txt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_di_haiku
  left: ev_di_haiku_img
  right: ev_di_haiku_txt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_adj_haiku
  left: ev_adj_haiku_img
  right: ev_adj_haiku_txt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_di_g55
  left: ev_di_g55_img
  right: ev_di_g55_txt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_adj_g55
  left: ev_adj_g55_img
  right: ev_adj_g55_txt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_di_nano
  left: ev_di_nano_img
  right: ev_di_nano_txt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_adj_nano
  left: ev_adj_nano_img
  right: ev_adj_nano_txt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_di_mini
  left: ev_di_mini_img
  right: ev_di_mini_txt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_adj_mini
  left: ev_adj_mini_img
  right: ev_adj_mini_txt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_di_qwen
  left: ev_di_qwen_img
  right: ev_di_qwen_txt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_adj_qwen
  left: ev_adj_qwen_img
  right: ev_adj_qwen_txt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_di_kimi
  left: ev_di_kimi_img
  right: ev_di_kimi_txt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_adj_kimi
  left: ev_adj_kimi_img
  right: ev_adj_kimi_txt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_di_grok
  left: ev_di_grok_img
  right: ev_di_grok_txt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_adj_grok
  left: ev_adj_grok_img
  right: ev_adj_grok_txt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_di_gpro
  left: ev_di_gpro_img
  right: ev_di_gpro_txt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_adj_gpro
  left: ev_adj_gpro_img
  right: ev_adj_gpro_txt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_di_glite
  left: ev_di_glite_img
  right: ev_di_glite_txt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_adj_glite
  left: ev_adj_glite_img
  right: ev_adj_glite_txt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_union_opus
  left: ev_union_opus_img
  right: ev_union_opus_txt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_union_haiku
  left: ev_union_haiku_img
  right: ev_union_haiku_txt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_union_g55
  left: ev_union_g55_img
  right: ev_union_g55_txt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_union_mini
  left: ev_union_mini_img
  right: ev_union_mini_txt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_union_kimi
  left: ev_union_kimi_img
  right: ev_union_kimi_txt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_union_glite
  left: ev_union_glite_img
  right: ev_union_glite_txt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_art_mini
  left: ev_art_mini_img
  right: ev_art_mini_txt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_art_g54
  left: ev_art_g54_img
  right: ev_art_g54_txt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_art_kimi
  left: ev_art_kimi_img
  right: ev_art_kimi_txt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_swe_kimi
  left: ev_swe_kimi_img
  right: ev_swe_kimi_txt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_tau_kimi
  left: ev_tau_kimi_img
  right: ev_tau_kimi_txt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_swe_mini
  left: ev_swe_mini_img
  right: ev_swe_mini_txt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_tau_mini
  left: ev_tau_mini_img
  right: ev_tau_mini_txt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_swe_haiku
  left: ev_swe_haiku_img
  right: ev_swe_haiku_txt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_tau_haiku
  left: ev_tau_haiku_img
  right: ev_tau_haiku_txt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_audclean_gaud
  left: ev_audclean_gaud_aud
  right: ev_audclean_gaud_txt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_audatt_gaud
  left: ev_audatt_gaud_aud
  right: ev_audatt_gaud_txt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_audclean_qomni
  left: ev_audclean_qomni_aud
  right: ev_audclean_qomni_txt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_audatt_qomni
  left: ev_audatt_qomni_aud
  right: ev_audatt_qomni_txt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
claims:
- id: k1
  text: Pictionary consistently reduces attack success, even against the strongest adaptive attacks and human red teamers, while largely keeping utility
  location: abstract
  anchor: consistently reduces attack success rates even against the strongest adaptive attacks
  jobs:
  - R1
  - R2
  supported_by:
  - c_di_opus
  - c_adj_opus
  - c_di_haiku
  - c_adj_haiku
  - c_di_g55
  - c_adj_g55
  - c_di_nano
  - c_adj_nano
  - c_di_mini
  - c_adj_mini
  - c_di_qwen
  - c_adj_qwen
  - c_di_kimi
  - c_adj_kimi
  - c_di_grok
  - c_adj_grok
  - c_di_gpro
  - c_adj_gpro
  - c_di_glite
  - c_adj_glite
  - c_union_opus
  - c_union_haiku
  - c_union_g55
  - c_union_mini
  - c_union_kimi
  - c_union_glite
  - c_swe_kimi
  - c_tau_kimi
  - c_swe_mini
  - c_tau_mini
  - c_swe_haiku
  - c_tau_haiku
  page: 1
  contradicted_by:
  - r_union_glite_img
  - r_swe_mini_img
  - r_adj_grok_img_u
coverage:
  R1: direct
  R2clean: direct
  R2attack: direct
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Tables 1, 2, 3 and 5
    R2clean: Tables 4 and 5
    R2attack: Tables 1, 3 and 5
schema_gaps:
- where: results r_audatt_qomni_txt_*
  note: 'Table 5 marks the Qwen3.5 Omni-Plus text values with a dagger: one attack template was rejected by the API before reaching the model (p. 8)'

`srft_arxiv2026` (2610.04269)

In [ ]:
%%writefile reports/records/srft_arxiv2026.yaml
report:
  id: srft_arxiv2026
  slug: '2610.04269'
  title: 'Self-Reflection Fine-Tuning: Enhancing Agent Security against Prompt Injection Attacks from Failure Experience'
  first_author: Zixuan Wang
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2610.04269'
  version_read: arXiv v1
  first_date: '2026-10-03'
  read: F
  role: proposes
  code_release: https://github.com/Eden-Wang1710/srft-repo (linked from the abstract)
  extraction_scope: 'Section 5 and Appendix B: the All rows of Table 1 (AgentDojo, Llama-3.1-8B), Table 2 (Qwen3-4B and Qwen3-8B) and the Final column of Table 7 (RL-Hammer on InjecAgent); the abstract claim (per-suite rows
    of Table 1, Figure 2, Table 6, the ablations of Table 8 and AgentDyn Table 9 not coded)'
defences:
- id: sr_l
  name: SR-Agent (Self-Reflection Fine-Tuning)
  origin: this_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - no_signal
  axis2:
  - sft
  - distillation
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  components:
  - 3,698 TOUCAN trajectories with injected tool returns; the agent's own sampled actions are contrasted with expert actions
  - Claude Sonnet 4.6 writes the injections and the structured self-reflection (goal summary, injection identification, action analysis)
  - LoRA rank 64 on attention projections, 3 epochs
  layout: AgentDojo agent loop; tool returns carry the injection
  layout_compat: &id001
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Our main model, SR-Agent, fine-tunes Llama-3.1-8B-Instruct
- id: sr_q8
  name: SR-Agent (Self-Reflection Fine-Tuning)
  origin: this_report
  base_checkpoint: Qwen3-8B
  axis1:
  - no_signal
  axis2:
  - sft
  - distillation
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  components:
  - 3,698 TOUCAN trajectories with injected tool returns; the agent's own sampled actions are contrasted with expert actions
  - Claude Sonnet 4.6 writes the injections and the structured self-reflection (goal summary, injection identification, action analysis)
  - LoRA rank 64 on attention projections, 3 epochs
  layout: AgentDojo agent loop; tool returns carry the injection
  layout_compat: *id001
  anchor: Our main model, SR-Agent, fine-tunes Llama-3.1-8B-Instruct
- id: sr_q4
  name: SR-Agent (Self-Reflection Fine-Tuning)
  origin: this_report
  base_checkpoint: Qwen3-4B
  axis1:
  - no_signal
  axis2:
  - sft
  - distillation
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  components:
  - 3,698 TOUCAN trajectories with injected tool returns; the agent's own sampled actions are contrasted with expert actions
  - Claude Sonnet 4.6 writes the injections and the structured self-reflection (goal summary, injection identification, action analysis)
  - LoRA rank 64 on attention projections, 3 epochs
  layout: AgentDojo agent loop; tool returns carry the injection
  layout_compat: *id001
  anchor: Our main model, SR-Agent, fine-tunes Llama-3.1-8B-Instruct
- id: msa8b
  name: Meta-SecAlign-8B (re-measured)
  origin: other_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  layout: AgentDojo agent loop; tool returns carry the injection
  layout_compat: *id001
  anchor: Meta-SecAlign- 8B, a strong prompt-injection defense built on the same base model
evaluations:
- id: ev_l_none_clean
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: AgentDojo, four suites, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo checkers
  layout_same_as_defence: 'yes'
  measured_by: srft_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 6
  baseline_kind: off_the_shelf
- id: ev_l_none_att
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: AgentDojo, four suites, 949 cases, important instructions attack
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checkers
  layout_same_as_defence: 'yes'
  measured_by: srft_arxiv2026
  page: 7
  decoding: not reported
  sample_size: '949'
  pages:
    benchmark: 6
  baseline_kind: off_the_shelf
- id: ev_l_msa_clean
  defence_id: msa8b
  benchmark: AgentDojo, four suites, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo checkers
  layout_same_as_defence: 'yes'
  measured_by: srft_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 6
- id: ev_l_msa_att
  defence_id: msa8b
  benchmark: AgentDojo, four suites, 949 cases, important instructions attack
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checkers
  layout_same_as_defence: 'yes'
  measured_by: srft_arxiv2026
  page: 7
  decoding: not reported
  sample_size: '949'
  pages:
    benchmark: 6
- id: ev_l_sr_clean
  defence_id: sr_l
  benchmark: AgentDojo, four suites, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo checkers
  layout_same_as_defence: 'yes'
  measured_by: srft_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 6
- id: ev_l_sr_att
  defence_id: sr_l
  benchmark: AgentDojo, four suites, 949 cases, important instructions attack
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checkers
  layout_same_as_defence: 'yes'
  measured_by: srft_arxiv2026
  page: 7
  decoding: not reported
  sample_size: '949'
  pages:
    benchmark: 6
- id: ev_q4_none_clean
  defence_id: none:Qwen3-4B
  benchmark: AgentDojo, four suites, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo checkers
  layout_same_as_defence: 'yes'
  measured_by: srft_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 6
  baseline_kind: off_the_shelf
- id: ev_q4_none_att
  defence_id: none:Qwen3-4B
  benchmark: AgentDojo, four suites, 949 cases, important instructions attack
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checkers
  layout_same_as_defence: 'yes'
  measured_by: srft_arxiv2026
  page: 8
  decoding: not reported
  sample_size: '949'
  pages:
    benchmark: 6
  baseline_kind: off_the_shelf
- id: ev_q4_sr_clean
  defence_id: sr_q4
  benchmark: AgentDojo, four suites, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo checkers
  layout_same_as_defence: 'yes'
  measured_by: srft_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 6
- id: ev_q4_sr_att
  defence_id: sr_q4
  benchmark: AgentDojo, four suites, 949 cases, important instructions attack
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checkers
  layout_same_as_defence: 'yes'
  measured_by: srft_arxiv2026
  page: 8
  decoding: not reported
  sample_size: '949'
  pages:
    benchmark: 6
- id: ev_q8_none_clean
  defence_id: none:Qwen3-8B
  benchmark: AgentDojo, four suites, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo checkers
  layout_same_as_defence: 'yes'
  measured_by: srft_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 6
  baseline_kind: off_the_shelf
- id: ev_q8_none_att
  defence_id: none:Qwen3-8B
  benchmark: AgentDojo, four suites, 949 cases, important instructions attack
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checkers
  layout_same_as_defence: 'yes'
  measured_by: srft_arxiv2026
  page: 8
  decoding: not reported
  sample_size: '949'
  pages:
    benchmark: 6
  baseline_kind: off_the_shelf
- id: ev_q8_sr_clean
  defence_id: sr_q8
  benchmark: AgentDojo, four suites, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo checkers
  layout_same_as_defence: 'yes'
  measured_by: srft_arxiv2026
  page: 8
  decoding: not reported
  pages:
    benchmark: 6
- id: ev_q8_sr_att
  defence_id: sr_q8
  benchmark: AgentDojo, four suites, 949 cases, important instructions attack
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo checkers
  layout_same_as_defence: 'yes'
  measured_by: srft_arxiv2026
  page: 8
  decoding: not reported
  sample_size: '949'
  pages:
    benchmark: 6
- id: ev_rlh_l_none
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: InjecAgent, 100 test cases, RL-Hammer attacker trained against this target (mean of two attacker trainings)
  attacker_type: rl
  attacker_access: black_box
  judge: InjecAgent attack success
  layout_same_as_defence: 'yes'
  measured_by: srft_arxiv2026
  page: 19
  attacker_budget: 20 attacker training epochs, final epoch reported
  decoding: not reported
  sample_size: '100'
  baseline_kind: off_the_shelf
- id: ev_rlh_l_msa
  defence_id: msa8b
  benchmark: InjecAgent, 100 test cases, RL-Hammer attacker trained against this target (mean of two attacker trainings)
  attacker_type: rl
  attacker_access: black_box
  judge: InjecAgent attack success
  layout_same_as_defence: 'yes'
  measured_by: srft_arxiv2026
  page: 19
  attacker_budget: 20 attacker training epochs, final epoch reported
  decoding: not reported
  sample_size: '100'
- id: ev_rlh_l_sr
  defence_id: sr_l
  benchmark: InjecAgent, 100 test cases, RL-Hammer attacker trained against this target (mean of two attacker trainings)
  attacker_type: rl
  attacker_access: black_box
  judge: InjecAgent attack success
  layout_same_as_defence: 'yes'
  measured_by: srft_arxiv2026
  page: 19
  attacker_budget: 20 attacker training epochs, final epoch reported
  decoding: not reported
  sample_size: '100'
- id: ev_rlh_q8_none
  defence_id: none:Qwen3-8B
  benchmark: InjecAgent, 100 test cases, RL-Hammer attacker trained against this target (mean of two attacker trainings)
  attacker_type: rl
  attacker_access: black_box
  judge: InjecAgent attack success
  layout_same_as_defence: 'yes'
  measured_by: srft_arxiv2026
  page: 19
  attacker_budget: 20 attacker training epochs, final epoch reported
  decoding: not reported
  sample_size: '100'
  baseline_kind: off_the_shelf
- id: ev_rlh_q8_sr
  defence_id: sr_q8
  benchmark: InjecAgent, 100 test cases, RL-Hammer attacker trained against this target (mean of two attacker trainings)
  attacker_type: rl
  attacker_access: black_box
  judge: InjecAgent attack success
  layout_same_as_defence: 'yes'
  measured_by: srft_arxiv2026
  page: 19
  attacker_budget: 20 attacker training epochs, final epoch reported
  decoding: not reported
  sample_size: '100'
results:
- id: r_l_none_bu
  evaluation_id: ev_l_none_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Benign Utility
  value: '27.84'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1, All row
- id: r_l_none_ua
  evaluation_id: ev_l_none_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility under Attack
  value: '23.08'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1, All row
- id: r_l_none_asr
  evaluation_id: ev_l_none_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '7.59'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1, All row
- id: r_l_msa_bu
  evaluation_id: ev_l_msa_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Benign Utility
  value: '23.71'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1, All row
- id: r_l_msa_ua
  evaluation_id: ev_l_msa_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility under Attack
  value: '19.07'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1, All row
- id: r_l_msa_asr
  evaluation_id: ev_l_msa_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.95'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1, All row
- id: r_l_sr_bu
  evaluation_id: ev_l_sr_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Benign Utility
  value: '37.11'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1, All row
- id: r_l_sr_ua
  evaluation_id: ev_l_sr_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility under Attack
  value: '29.08'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1, All row
- id: r_l_sr_asr
  evaluation_id: ev_l_sr_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '1.26'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1, All row
- id: r_q4_none_bu
  evaluation_id: ev_q4_none_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Benign Utility
  value: '57.73'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q4_none_ua
  evaluation_id: ev_q4_none_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility under Attack
  value: '50.58'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q4_none_asr
  evaluation_id: ev_q4_none_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '10.85'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q4_sr_bu
  evaluation_id: ev_q4_sr_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Benign Utility
  value: '50.52'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q4_sr_ua
  evaluation_id: ev_q4_sr_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility under Attack
  value: '49.21'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q4_sr_asr
  evaluation_id: ev_q4_sr_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.84'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q8_none_bu
  evaluation_id: ev_q8_none_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Benign Utility
  value: '60.82'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q8_none_ua
  evaluation_id: ev_q8_none_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility under Attack
  value: '50.47'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q8_none_asr
  evaluation_id: ev_q8_none_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '16.97'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q8_sr_bu
  evaluation_id: ev_q8_sr_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Benign Utility
  value: '51.55'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q8_sr_ua
  evaluation_id: ev_q8_sr_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility under Attack
  value: '46.68'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_q8_sr_asr
  evaluation_id: ev_q8_sr_att
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '1.05'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_rlh_l_none
  evaluation_id: ev_rlh_l_none
  quantity: R1
  coverage: direct
  metric_as_printed: Final ASR (RL-Hammer)
  value: '98.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 19
  location: Table 7
  spread: mean over two attacker trainings
- id: r_rlh_l_msa
  evaluation_id: ev_rlh_l_msa
  quantity: R1
  coverage: direct
  metric_as_printed: Final ASR (RL-Hammer)
  value: '69.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 19
  location: Table 7
  spread: mean over two attacker trainings
- id: r_rlh_l_sr
  evaluation_id: ev_rlh_l_sr
  quantity: R1
  coverage: direct
  metric_as_printed: Final ASR (RL-Hammer)
  value: '17.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 19
  location: Table 7
  spread: mean over two attacker trainings
- id: r_rlh_q8_none
  evaluation_id: ev_rlh_q8_none
  quantity: R1
  coverage: direct
  metric_as_printed: Final ASR (RL-Hammer)
  value: '59.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 19
  location: Table 7
  spread: mean over two attacker trainings
- id: r_rlh_q8_sr
  evaluation_id: ev_rlh_q8_sr
  quantity: R1
  coverage: direct
  metric_as_printed: Final ASR (RL-Hammer)
  value: '17.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 19
  location: Table 7
  spread: mean over two attacker trainings
comparisons:
- id: c_l_msa_clean
  left: ev_l_msa_clean
  right: ev_l_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l_msa_att
  left: ev_l_msa_att
  right: ev_l_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l_sr_clean
  left: ev_l_sr_clean
  right: ev_l_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l_sr_att
  left: ev_l_sr_att
  right: ev_l_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q4_sr_clean
  left: ev_q4_sr_clean
  right: ev_q4_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q4_sr_att
  left: ev_q4_sr_att
  right: ev_q4_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q8_sr_clean
  left: ev_q8_sr_clean
  right: ev_q8_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q8_sr_att
  left: ev_q8_sr_att
  right: ev_q8_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l_sr_msa_clean
  left: ev_l_sr_clean
  right: ev_l_msa_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l_sr_msa_att
  left: ev_l_sr_att
  right: ev_l_msa_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_rlh_l_msa
  left: ev_rlh_l_msa
  right: ev_rlh_l_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 19
- id: c_rlh_l_sr
  left: ev_rlh_l_sr
  right: ev_rlh_l_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 19
- id: c_rlh_q8_sr
  left: ev_rlh_q8_sr
  right: ev_rlh_q8_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 19
- id: c_rlh_l_sr_msa
  left: ev_rlh_l_sr
  right: ev_rlh_l_msa
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 19
claims:
- id: k1
  text: SRFT substantially reduces attack success while preserving task performance and generalises under adaptive attacks
  location: abstract
  anchor: SRFT substantially reduces attack success rates while preserving task per- formance
  jobs:
  - R1
  - R2
  supported_by:
  - c_l_sr_att
  - c_l_sr_clean
  - c_q8_sr_att
  - c_q4_sr_att
  - c_rlh_l_sr
  - c_rlh_q8_sr
  page: 1
  contradicted_by:
  - r_q4_sr_bu
  - r_q8_sr_bu
coverage:
  R1: direct
  R2clean: direct
  R2attack: direct
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Tables 1, 2 and 7
    R2clean: Tables 1 and 2
    R2attack: Tables 1 and 2

## Stage 2. The PDFs read are the PDFs registered, and every report is an included study

In [ ]:
import csv, logging, datetime as dt
from collections import Counter
import schema, search, corpus, paths
for m in (schema, search, corpus, paths):
    importlib.reload(m)
logging.getLogger("pypdf").setLevel(logging.ERROR)
from pypdf import PdfReader

BATCH = "04e"
cfg_exclusions = 0
cfg_next = "batch 04f"
IDS = ['astra_arxiv2025', 'autodojo_arxiv2026', 'reta_arxiv2026', 'ihdec_arxiv2026', 'copa_arxiv2026', 'secopd_emnlp2026', 'overlays_arxiv2026', 'coer_arxiv2026', 'provtransformer_arxiv2026', 'codel_arxiv2026', 'pictionary_arxiv2026', 'srft_arxiv2026']
man = {m["slug"]: m for m in search.read_csv(paths.REPORTS / "defences.csv")}
problems, texts = [], {}
for rid in IDS:
    rec = schema.load_record(paths.RECORDS / f"{rid}.yaml")
    slug = rec["report"]["slug"]
    pdf = paths.DATA_DEFENCES / f"{slug}.pdf"
    m = man.get(slug, {})
    if not pdf.exists():
        problems.append(f"{rid}: {pdf.name} not in data/defences"); continue
    sha = corpus.sha256(pdf)
    rd = PdfReader(str(pdf))
    texts[slug] = ([p.extract_text() or "" for p in rd.pages], [p.extract_text(extraction_mode="layout") or "" for p in rd.pages])
    ok_sha = sha == m.get("sha256")
    ok_inc = m.get("decision") == "include"
    print(f"{rid:<28} {slug:<12} pages {len(rd.pages):>3}  sha256 {'matches' if ok_sha else 'DIFFERS'}  decision {m.get('decision')}")
    if not ok_sha:
        problems.append(f"{rid}: PDF differs from the one registered in reports/defences.csv")
    if not ok_inc:
        problems.append(f"{rid}: not an included study")

## Stage 3. Validate against the frozen schema and the PDFs; grade the comparisons; extra checks

In [ ]:
import re
import pypdfium2 as pdfium
_docs = {}
def pdfium_page(slug, page):
    if slug not in _docs:
        _docs[slug] = pdfium.PdfDocument(str(paths.DATA_DEFENCES / f"{slug}.pdf"))
    return _docs[slug][page - 1].get_textpage().get_text_range()
S1 = schema.load_schema(paths.SCHEMA / "record_schema.yaml")
check_rows, summary, residual, third_rows = [], [], [], []
for rid in IDS:
    rec = schema.load_record(paths.RECORDS / f"{rid}.yaml")
    slug = rec["report"]["slug"]
    plain, lay = texts.get(slug, (None, None))
    probs, notes = schema.validate_record(rec, S1, plain, lay)
    kept, third = [], []                       # a value pypdf glues to its neighbour is looked for in a third extraction (pypdfium2)
    for p in probs:
        m = re.match(r"results:(\S+): value '(.+)' not found on page (\d+)$", p)
        if m and schema.value_on_page(m.group(2), pdfium_page(slug, int(m.group(3)))):
            third.append(f"{m.group(1)} ({m.group(2)!r}, page {m.group(3)})")
        else:
            kept.append(p)
    probs = kept
    for kind in ("defences", "evaluations", "results", "comparisons", "claims"):          # ids unique within each kind
        dup = [i for i, n in Counter(r["id"] for r in rec.get(kind) or []).items() if n > 1]
        probs += [f"{kind}: id {i} used more than once" for i in dup]
    fig_kept, third_chart = 0, []
    for r in rec.get("results") or []:                                                     # chart values kept as printed
        if r.get("value_source") == "figure_only" and r.get("value") not in (None, ""):
            fig_kept += 1
            pg = int(r["page"]) - 1
            if plain and not (schema.value_on_page(r["value"], plain[pg]) or schema.value_on_page(r["value"], lay[pg])):
                if schema.value_on_page(r["value"], pdfium_page(slug, int(r["page"]))):
                    third_chart.append(f"{r['id']} ({r['value']!r}, page {r['page']}, chart label)")
                else:
                    probs.append(f"results:{r['id']}: chart value {r['value']!r} not found on page {r['page']}")
    third_rows += [{"report": rid, "status": "value found in the third extraction", "detail": t} for t in third + third_chart]
    grades = Counter(schema.grade(c) for c in rec.get("comparisons") or [])
    n_val = sum(1 for r in rec.get("results") or [] if r.get("value_source") != "figure_only")
    n_anc = sum(1 for k in ("defences", "claims") for r in rec.get(k) or [] if r.get("anchor"))
    summary.append({"batch": BATCH, "report": rid, "slug": slug, "role": rec["report"]["role"],
                    "defences": len(rec.get("defences") or []), "evaluations": len(rec.get("evaluations") or []),
                    "results": len(rec.get("results") or []), "values_checked": n_val, "chart_values_checked": fig_kept,
                    "anchors_checked": n_anc, "comparisons": len(rec.get("comparisons") or []),
                    "grade_A": grades["A"], "grade_B": grades["B"], "grade_C": grades["C"], "grade_U": grades["U"],
                    "claims": len(rec.get("claims") or []),
                    **{f"cov_{q}": rec["coverage"][q] for q in ("R1", "R2clean", "R2attack", "R3ns", "R3cp", "R3safe")},
                    "values_third_extraction": len(third), "chart_values_third_extraction": len(third_chart), "residual_notes": len(notes), "problems": len(probs)})
    check_rows += [{"report": rid, "status": "problem", "detail": p} for p in probs]
    residual += [{"report": rid, "where": n.split(": ", 1)[0], "note": n.split(": ", 1)[1]} for n in notes]
    print(f"{rid:<28} problems {len(probs)}; values {n_val} ({len(third)} via the third extraction); chart values {fig_kept} ({len(third_chart)} via the third extraction); anchors {n_anc}; grades {dict(grades)}")
    for p in probs:
        print("   ", p)
    problems += [f"{rid}: {p}" for p in probs]
check_rows += third_rows + [{"report": g["report"], "status": "residual note", "detail": f"{g['where']}: {g['note']}"} for g in residual]
search.write_csv(paths.REPORTS / f"extraction_{BATCH}_check.csv", check_rows or [{"report": "all", "status": "ok", "detail": "no problems"}],
                 ["report", "status", "detail"])
search.write_csv(paths.REPORTS / f"extraction_{BATCH}_summary.csv", summary, list(summary[0].keys()))
for g in residual:
    print(f"  residual note, {g['report']}: {g['note'][:150]}")

## Stage 4. Batch register (one row per coded report, kept across batches)

In [ ]:
reg_path = paths.REPORTS / "extraction_batches.csv"
reg = [r for r in search.read_csv(reg_path) if r["batch"] != BATCH] if reg_path.exists() else []
today = dt.date.today().isoformat()
for s in summary:
    reg.append({"batch": BATCH, "report": s["report"], "slug": s["slug"], "coded": today,
                "status": "checked" if s["problems"] == 0 else "problems"})
reg.sort(key=lambda r: (r["batch"], r["report"]))
search.write_csv(reg_path, reg, ["batch", "report", "slug", "coded", "status"])
included = sum(1 for m in search.read_csv(paths.REPORTS / "defences.csv") if m.get("decision") == "include")   # read again: Stage 2b may change it
pilot = len(list(paths.RECORDS.glob("*.yaml"))) - len({r["report"] for r in reg})
print(f"records in reports/records: {len(list(paths.RECORDS.glob('*.yaml')))} (pilot {pilot}, batches {len({r['report'] for r in reg})}); included studies: {included}")

## Stage 5. Decision record and tests

In [ ]:
tot = {k: sum(s[k] for s in summary) for k in ("defences", "evaluations", "results", "values_checked", "values_third_extraction", "chart_values_checked",
                                              "chart_values_third_extraction", "anchors_checked", "comparisons", "grade_A", "grade_B", "grade_C", "grade_U", "claims")}
status = "BATCH CHECKED" if not problems else "BATCH NOT CHECKED: PROBLEMS BELOW"
lines = [f"decision_{BATCH}: {status}", f"this step: {dt.date.today().isoformat()}",
         "reports: " + ", ".join(IDS),
         f"records: {tot['defences']} defences, {tot['evaluations']} evaluations, {tot['results']} results, "
         f"{tot['comparisons']} comparisons, {tot['claims']} claims",
         f"checked against the PDFs: {tot['values_checked']} printed values ({tot['values_third_extraction']} of them found only in the third "
         f"extraction), {tot['chart_values_checked']} chart values ({tot['chart_values_third_extraction']} of them found only in the third extraction) "
         f"and {tot['anchors_checked']} anchors, each found on its page",
         f"comparison grades: A {tot['grade_A']}, B {tot['grade_B']}, C {tot['grade_C']}, U {tot['grade_U']}",
         f"residual notes: {len(residual)}; full-text exclusions found while coding: {cfg_exclusions}",
         f"next: {cfg_next}"] + [f"- problem: {p}" for p in problems]
(paths.REPORTS / f"decision_{BATCH}.txt").write_text("\n".join(lines) + "\n")
print("\n".join(lines))
tests = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print(tests.stdout.strip().splitlines()[-1] if tests.stdout.strip() else tests.stderr.strip()[-400:])

## Stage 6. Commit

In [ ]:
import os, subprocess
os.chdir(os.environ.get("SOURCEMAP_REPO_ROOT", "/content/drive/MyDrive/SourceMap_Research/sourcemap-research"))
r = subprocess.run(["python", "tools/commit_cell.py",
                    "04e extraction batch e: twelve included studies coded from their PDFs, values and anchors checked, comparisons graded"],
                   capture_output=True, text=True)
print(r.stdout)
print(r.stderr)